# CS781 Phase 1 - Legal LM from scratch (fast, resumable, multi-run pipeline)

**Run order:** 1 install -> 2 write script -> 3 HF login -> 4 self-test (once) -> 5 config -> 6 `all` (or the stages one by one) -> 7 compare runs -> 8 copy `submission.zip`.

* Everything is cached under `WORK` (tokenizer, tokenized shards, checkpoints). **Re-running a cell never redoes finished work**; an interrupted `train` resumes from `ckpt_last.pt`.
* A new `RUN` name = a new experiment (new architecture / mixture / context ...). Tokenizer + data are shared between runs when their settings match.
* Every train/sweep appends a row to `experiments.csv` - that is your experiment table for the report/viva.
* **Speed:** the `turbo` preset uses a ~55M-parameter model (inside the 50M-300M rule), all visible GPUs (DataParallel, e.g. 2xT4),
  `torch.compile` only if it benchmarks faster, and a faster data-prep path. Expect roughly 2-3x less wall-clock than the original defaults (my estimate; the log prints the measured speed).
* Kaggle: turn **Internet ON** (HF dataset + BERTScore's roberta-large) and pick a **GPU** accelerator.

In [1]:
!pip -q install tokenizers bert-score ijson huggingface_hub
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.2/150.2 kB 5.2 MB/s eta 0:00:00
torch 2.11.0+cu128 | CUDA: True Tesla T4


## 2. Write the pipeline script (this cell only writes a file)

In [ ]:
%%writefile cs781_phase1.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
CS781 Phase 1 - Legal language model from scratch  ->  128-word continuations  ->  submission.zip

Stages (each one is cached / resumable, so the whole thing can be re-run any number of times):

  selftest   unit tests for all 4 architectures + tiny end-to-end run on synthetic data (CPU, ~1-3 min)
  prepare    train tokenizer, tokenize corpus into uint16 .bin shards, build validation sets   (cached)
  train      train from scratch, auto-fits batch size + step budget to the GPU, resumes automatically
  sweep      decoding sweep on held-out prompts, scored with BERTScore / ROUGE-L / BLEU-4
  predict    generate 128 words for every test prompt -> prediction.json -> submission.zip
  all        prepare -> train -> sweep -> predict

Examples
  python cs781_phase1.py selftest
  python cs781_phase1.py all --run llama_v1 --cfg cfg.json
  python cs781_phase1.py train --run llama_v1                   # resumes if interrupted
  python cs781_phase1.py all --run retnet_v1 --cfg '{"arch": "retnet"}'

Directory layout under work_dir (nothing is ever overwritten by a different run):
  tok/<tok_sig>/tokenizer.json         shared by all runs with the same tokenizer settings
  data/<data_sig>/*.bin, meta.json     tokenized shards, shared by all runs with the same data settings
  runs/<run>/                          config.json, ckpt_best.pt, ckpt_last.pt, log.jsonl, sweep.json,
                                       best_decode.json, prediction.json, submission.zip
  experiments.csv                      one row per train / sweep (your experiment table for the report)
"""
from __future__ import annotations

import argparse
import contextlib
import csv
import gc
import hashlib
import json
import math
import os
import random
import shutil
import sys
import time
import warnings
import zipfile
import zlib
from dataclasses import dataclass, field, fields, asdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore", message=".*all input tensors were scalars.*")  # DataParallel loss gather

# ============================================================================================
# 0. Constants, presets, config
# ============================================================================================
EOS = "<|endoftext|>"
ON_KAGGLE = os.path.exists("/kaggle/working")
DEFAULT_WORK = "/kaggle/working/cs781" if ON_KAGGLE else os.path.abspath("./cs781_work")

SOURCE_FILES = {
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
}

# Four architectures that differ in design (not just in width / depth), all 50M-300M params at V=32k.
ARCH_PRESETS = {
    # 1) classic GPT-2: learned absolute positions, LayerNorm, GELU MLP, full multi-head attention (~110M)
    "gpt2": dict(n_embd=768, n_layer=12, n_head=12),
    # 2) modern decoder: RoPE, RMSNorm, SwiGLU, grouped-query attention (~107M)
    "llama": dict(n_embd=768, n_layer=12, n_head=12, n_kv_head=4, ffn_hidden=2304),
    # 3) RetNet-style retention (decaying linear attention, recurrent O(1) decoding) instead of softmax attention (~117M)
    "retnet": dict(n_embd=768, n_layer=12, n_head=12, ffn_hidden=2048),
    # 4) sparse Mixture-of-Experts decoder: 8 SwiGLU experts, top-2 routing (~124M total params, ~45M active)
    "moe": dict(n_embd=512, n_layer=8, n_head=8, n_kv_head=4, ffn_hidden=1024,
                n_experts=8, top_k=2, aux_coef=0.01),
}

# "small" is ~2x cheaper per token (~55-72M params): the better choice when training time is limited.
SIZE_PRESETS = {
    "base": ARCH_PRESETS,
    "small": {
        "gpt2": dict(n_embd=512, n_layer=12, n_head=8),
        "llama": dict(n_embd=512, n_layer=12, n_head=8, n_kv_head=4, ffn_hidden=1536),
        "retnet": dict(n_embd=512, n_layer=12, n_head=8, ffn_hidden=1536),
        "moe": dict(n_embd=384, n_layer=8, n_head=6, n_kv_head=3, ffn_hidden=768,
                    n_experts=8, top_k=2, aux_coef=0.01),
    },
}

DEFAULT_GRID = [
    {"name": "greedy", "temperature": 0.0, "rep": 1.0},
    {"name": "greedy_rp1.05", "temperature": 0.0, "rep": 1.05},
    {"name": "greedy_rp1.1", "temperature": 0.0, "rep": 1.1},
    {"name": "greedy_rp1.2", "temperature": 0.0, "rep": 1.2},
    {"name": "greedy_nr3", "temperature": 0.0, "nr": 3},
    {"name": "greedy_rp1.05_nr4", "temperature": 0.0, "rep": 1.05, "nr": 4},
    {"name": "greedy_rp1.1_nr4", "temperature": 0.0, "rep": 1.1, "nr": 4},
    {"name": "t0.3_p0.9_rp1.1", "temperature": 0.3, "top_p": 0.9, "rep": 1.1},
    {"name": "t0.5_p0.9_rp1.1", "temperature": 0.5, "top_p": 0.9, "rep": 1.1},
]
FALLBACK_DECODE = {"name": "greedy_rp1.1", "temperature": 0.0, "rep": 1.1}


@dataclass
class Cfg:
    # ---- identity / paths
    run: str = "run1"
    work_dir: str = DEFAULT_WORK
    hf_repo: str = "Exploration-Lab/CS-781-Capstone"
    local_data_dir: str = ""      # if set, read <dir>/<file basename> instead of downloading from HF
    test_file: str = "Phase_1_Pretraining/Test/test.json"
    test_path: str = ""           # local test.json override
    # ---- data  (token caps per source; mix = sampling weights during training)
    sources: dict = field(default_factory=lambda: {
        "supreme_court": 300e6, "high_courts": 200e6, "tribunals": 40e6, "acts": 10e6})
    mix: dict = field(default_factory=lambda: {
        "supreme_court": 0.45, "high_courts": 0.40, "tribunals": 0.10, "acts": 0.05})
    normalize_ws: bool = True     # collapse whitespace (model then emits plain running text)
    min_words: int = 30
    val_frac: float = 0.01        # fraction of documents held out (hash-based => deterministic)
    val_tokens_per_source: int = 1_000_000
    eval_docs_per_source: int = 60   # held-out docs turned into (512-word prompt -> 128-word reference)
    eval_prompt_words: int = 512
    eval_ref_words: int = 128
    split_seed: int = 1234
    json_loader: str = "auto"     # auto | json | stream (ijson for files > 2.5 GB)
    prep_coverage: float = 0.4    # scan only the first ~40% of each huge file (uniformly subsampled) -> much faster prepare
    # ---- tokenizer
    tokenizer: str = "bpe"        # "bpe" (trained here) | "gpt2"
    vocab_size: int = 32000
    tok_sample_chars: int = 30_000_000   # per source (stops early once reached)
    # ---- model / training
    arch: str = "llama"
    size: str = "base"            # base (~110M) | small (~55M, ~2x faster per token)
    ctx: int = 1024
    model: dict = field(default_factory=dict)   # overrides on top of ARCH_PRESETS[arch]
    dropout: float = 0.0
    time_budget_hours: float = 8.0   # used when train_tokens == 0 (steps are fitted to measured speed)
    train_tokens: int = 0
    batch_tokens: int = 65536
    max_micro_bs: int = 16
    lr: float = 6e-4
    min_lr_ratio: float = 0.1
    warmup_steps: int = 300
    weight_decay: float = 0.1
    grad_clip: float = 1.0
    seed: int = 1337
    eval_every: int = 500
    eval_blocks: int = 128
    log_every: int = 50
    ckpt_every_min: float = 20.0
    max_wall_hours: float = 0.0   # stop this invocation after N hours (checkpoint saved; rerun to resume)
    compile: str = "auto"         # auto = use torch.compile only if it is measurably faster (gpt2/llama, 1 GPU)
    multi_gpu: bool = True        # DataParallel over all visible GPUs (e.g. Kaggle 2xT4)
    # ---- generation / evaluation
    target_words: int = 128
    max_new_tokens: int = 256
    gen_batch: int = 32
    sweep_n: int = 96
    sweep_grid: list = field(default_factory=lambda: [dict(d) for d in DEFAULT_GRID])
    decode: dict = field(default_factory=dict)   # if non-empty: use this decoding config, skip choosing from sweep
    select_metric: str = "bertscore"
    use_bertscore: bool = True


PRESETS = {
    "full": {},
    "fast": {"time_budget_hours": 1.5,
             "sources": {"supreme_court": 80e6, "high_courts": 50e6, "tribunals": 15e6, "acts": 5e6}},
    # fast + strong: ~55M model, both GPUs, fast prepare, ~150M-300M tokens
    "turbo": {"size": "small", "time_budget_hours": 2.5,
              "sources": {"supreme_court": 110e6, "high_courts": 80e6, "tribunals": 20e6, "acts": 6e6},
              "prep_coverage": 0.4, "tok_sample_chars": 25_000_000,
              "batch_tokens": 32768, "lr": 8e-4, "warmup_steps": 200, "min_lr_ratio": 0.05,
              "eval_every": 400, "eval_blocks": 64, "sweep_n": 64, "gen_batch": 64},
    "smoke": {"time_budget_hours": 0, "train_tokens": 200_000, "ctx": 128, "vocab_size": 600,
              "model": {"n_embd": 64, "n_layer": 2, "n_head": 4, "n_kv_head": 2, "ffn_hidden": 128,
                        "n_experts": 4, "top_k": 2},
              "batch_tokens": 2048, "max_micro_bs": 4, "eval_every": 25, "eval_blocks": 8,
              "warmup_steps": 10, "log_every": 10,
              "sources": {"acts": 3e6}, "mix": {"acts": 1.0}, "tok_sample_chars": 2_000_000,
              "min_words": 10, "eval_prompt_words": 60, "eval_ref_words": 20, "target_words": 20,
              "max_new_tokens": 48, "sweep_n": 8, "gen_batch": 4, "eval_docs_per_source": 8,
              "use_bertscore": False, "val_tokens_per_source": 50_000, "val_frac": 0.05,
              "select_metric": "rougeL"},
}

# keys that may be changed when re-invoking a stage on an existing run (everything else is frozen in config.json)
RESUME_KEYS = {"work_dir", "local_data_dir", "max_wall_hours", "ckpt_every_min", "log_every", "eval_every"}
GEN_KEYS = RESUME_KEYS | {"test_file", "test_path", "target_words", "max_new_tokens", "gen_batch", "sweep_n",
                          "sweep_grid", "decode", "select_metric", "use_bertscore", "hf_repo"}


def apply_overrides(cfg: Cfg, ov: dict) -> Cfg:
    names = {f.name for f in fields(Cfg)}
    for k, v in ov.items():
        if k not in names:
            raise SystemExit(f"Unknown config key: {k!r}. Valid keys: {sorted(names)}")
        setattr(cfg, k, v)
    return cfg


# ============================================================================================
# 1. Small utilities
# ============================================================================================
def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


def norm_ws(t: str) -> str:
    return " ".join(t.split())


def short_hash(obj) -> str:
    return hashlib.md5(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:8]


def jdump(obj, path):
    path = str(path)
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    os.replace(tmp, path)


def jload(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def pad64(n: int) -> int:
    return ((n + 63) // 64) * 64


def run_dir_of(cfg: Cfg) -> Path:
    return Path(cfg.work_dir) / "runs" / cfg.run


def get_hf_token():
    t = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if t:
        return t
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        pass
    return None


def resolve_file(cfg: Cfg, rel: str) -> str:
    if cfg.local_data_dir:
        p = os.path.join(cfg.local_data_dir, os.path.basename(rel))
        if not os.path.exists(p):
            raise FileNotFoundError(p)
        return p
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=cfg.hf_repo, filename=rel, repo_type="dataset", token=get_hf_token())


def _rec_text(r) -> str:
    if isinstance(r, str):
        return r
    if isinstance(r, dict):
        t = r.get("text")
        if isinstance(t, str):
            return t
    return ""


def iter_texts(path: str, loader: str = "auto"):
    """Yield the text of every document. Handles list-of-records and dict-of-records JSON."""
    size = os.path.getsize(path)
    stream = loader == "stream" or (loader == "auto" and size > 2.5e9)
    if stream:
        try:
            import ijson
        except ImportError:
            log("ijson not installed -> falling back to json.load (needs RAM ~ 2x file size)")
            stream = False
    if stream:
        with open(path, "rb") as f:
            head = f.read(1024).lstrip()[:1]
            f.seek(0)
            it = ijson.items(f, "item") if head == b"[" else (v for _, v in ijson.kvitems(f, ""))
            for rec in it:
                yield _rec_text(rec)
    else:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, dict):
            data = list(data.values())
        for i in range(len(data)):
            r = data[i]
            data[i] = None          # free memory progressively
            yield _rec_text(r)


# ============================================================================================
# 2. Tokenizer + data preparation (cached, signature-keyed)
# ============================================================================================
def tok_dir_of(cfg: Cfg) -> Path:
    sig = short_hash({"kind": cfg.tokenizer, "vocab": cfg.vocab_size, "norm": cfg.normalize_ws,
                      "src": sorted(cfg.sources), "chars": cfg.tok_sample_chars, "local": bool(cfg.local_data_dir)})
    return Path(cfg.work_dir) / "tok" / sig


def build_tokenizer(cfg: Cfg):
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
    d = tok_dir_of(cfg)
    f = d / "tokenizer.json"
    if f.exists():
        return Tokenizer.from_file(str(f)), f
    d.mkdir(parents=True, exist_ok=True)
    if cfg.tokenizer == "gpt2":
        tok = Tokenizer.from_pretrained("openai-community/gpt2")
    else:
        tok = Tokenizer(models.BPE())
        tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
        tok.decoder = decoders.ByteLevel()
        trainer = trainers.BpeTrainer(vocab_size=int(cfg.vocab_size), special_tokens=[EOS], min_frequency=2,
                                      initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False)

        def gen():
            for name in cfg.sources:
                path = resolve_file(cfg, SOURCE_FILES[name])
                frac = min(1.0, 3.0 * cfg.tok_sample_chars / max(1, os.path.getsize(path)))
                got = 0
                log(f"[tokenizer] sampling {name} (fraction {frac:.3f}, up to {cfg.tok_sample_chars:,} chars)")
                for i, t in enumerate(iter_texts(path, cfg.json_loader)):
                    if frac < 1.0 and (zlib.crc32(f"tok{i}".encode()) % 100000) / 100000.0 >= frac:
                        continue
                    if cfg.normalize_ws:
                        t = norm_ws(t)
                    if t:
                        got += len(t)
                        yield t
                        if got >= cfg.tok_sample_chars:
                            break

        log("[tokenizer] training BPE ...")
        tok.train_from_iterator(gen(), trainer=trainer)
    tok.save(str(f))
    log(f"[tokenizer] saved {f}  vocab={tok.get_vocab_size()}")
    return tok, f


class _SrcWriter:
    def __init__(self, tok, eos, ftr, fva, cap, val_cap):
        self.tok, self.eos, self.ftr, self.fva = tok, eos, ftr, fva
        self.cap, self.val_cap = cap, val_cap
        self.n_tr = self.n_va = self.val_words = 0
        self.buf = []

    def add(self, text, is_val):
        self.buf.append((text, is_val))
        if len(self.buf) >= 512:
            self.flush()

    def flush(self):
        if not self.buf:
            return
        encs = self.tok.encode_batch([t for t, _ in self.buf])
        for (t, is_val), e in zip(self.buf, encs):
            arr = np.asarray(e.ids + [self.eos], dtype=np.uint16)
            if is_val:
                if self.n_va < self.val_cap:
                    arr.tofile(self.fva)
                    self.n_va += len(arr)
                    self.val_words += len(t.split())
            elif self.n_tr < self.cap:
                arr.tofile(self.ftr)
                self.n_tr += len(arr)
        self.buf = []

    def full(self):
        return self.n_tr >= self.cap and self.n_va >= self.val_cap


def prepare_source(cfg: Cfg, name: str, cap: int, tok, eos: int, ddir: Path):
    path = resolve_file(cfg, SOURCE_FILES[name])
    est_tokens = os.path.getsize(path) / 4.2
    keep = min(1.0, 1.1 * cap / max(1.0, cfg.prep_coverage * est_tokens))   # uniform subsample => no bias towards file start
    log(f"[prepare] {name}: cap={cap:,} tokens, keep-prob={keep:.3f}")
    tr_tmp, va_tmp = ddir / f"{name}.train.bin.tmp", ddir / f"{name}.val.bin.tmp"
    P, R = cfg.eval_prompt_words, cfg.eval_ref_words
    eval_items, docs = [], 0
    with open(tr_tmp, "wb") as ftr, open(va_tmp, "wb") as fva:
        w = _SrcWriter(tok, eos, ftr, fva, cap, cfg.val_tokens_per_source)
        t0 = time.time()
        for i, text in enumerate(iter_texts(path, cfg.json_loader)):
            if cfg.normalize_ws:
                text = norm_ws(text)
            words = text.split()
            if len(words) < cfg.min_words:
                continue
            h = zlib.crc32(f"{cfg.split_seed}:{name}:{i}".encode()) % 100000
            is_val = h < cfg.val_frac * 100000
            if not is_val:
                if (zlib.crc32(f"keep:{name}:{i}".encode()) % 100000) >= keep * 100000:
                    continue
            else:
                if len(words) >= P + R and len(eval_items) < cfg.eval_docs_per_source:
                    k = len(eval_items)
                    span = len(words) - (P + R)
                    start = 0 if (k % 2 == 0 or span == 0) else zlib.crc32(f"s:{name}:{i}".encode()) % (span + 1)
                    eval_items.append({"prompt": " ".join(words[start:start + P]),
                                       "reference": " ".join(words[start + P:start + P + R]),
                                       "source": name, "start": start})
            docs += 1
            w.add(text, is_val)
            if docs % 5000 == 0:
                log(f"[prepare] {name}: docs={docs:,} train_tok={w.n_tr:,} val_tok={w.n_va:,} ({time.time()-t0:.0f}s)")
            if not w.buf and w.full() and len(eval_items) >= cfg.eval_docs_per_source:
                break
        w.flush()
    os.replace(tr_tmp, ddir / f"{name}.train.bin")
    os.replace(va_tmp, ddir / f"{name}.val.bin")
    jdump(eval_items, ddir / f"{name}.eval.json")
    info = {"train_tokens": w.n_tr, "val_tokens": w.n_va, "val_words": w.val_words,
            "docs": docs, "eval_items": len(eval_items)}
    jdump(info, ddir / f"{name}.done.json")
    log(f"[prepare] {name}: {info}")
    return info


def prepare(cfg: Cfg) -> Path:
    tok, tok_file = build_tokenizer(cfg)
    eos = tok.token_to_id(EOS)
    sig = short_hash({"tok": str(tok_file), "caps": {k: int(v) for k, v in cfg.sources.items()},
                      "norm": cfg.normalize_ws, "minw": cfg.min_words, "vf": cfg.val_frac,
                      "vt": cfg.val_tokens_per_source, "ed": cfg.eval_docs_per_source,
                      "pw": cfg.eval_prompt_words, "rw": cfg.eval_ref_words, "ss": cfg.split_seed, "cov": cfg.prep_coverage})
    ddir = Path(cfg.work_dir) / "data" / sig
    ddir.mkdir(parents=True, exist_ok=True)
    if (ddir / "meta.json").exists():
        log(f"[prepare] cached data found: {ddir}")
        return ddir
    infos, evals = {}, []
    for name, cap in cfg.sources.items():
        if name not in SOURCE_FILES:
            raise SystemExit(f"Unknown source {name!r}; valid: {list(SOURCE_FILES)}")
        done = ddir / f"{name}.done.json"
        if done.exists():
            infos[name] = jload(done)
            log(f"[prepare] {name}: cached {infos[name]}")
        else:
            infos[name] = prepare_source(cfg, name, int(cap), tok, eos, ddir)
        evals += jload(ddir / f"{name}.eval.json")
    with open(ddir / "val.bin", "wb") as out:
        for name in infos:
            with open(ddir / f"{name}.val.bin", "rb") as f:
                shutil.copyfileobj(f, out)
    random.Random(cfg.split_seed).shuffle(evals)
    ev = {f"v{i}": e for i, e in enumerate(evals)}
    jdump(ev, ddir / "eval.json")
    jdump({k: {"text": v["reference"]} for k, v in ev.items()}, ddir / "eval_truth.json")   # for evaluate_phase1.py
    meta = {"sources": infos, "vocab": tok.get_vocab_size(), "eos": eos, "tok_file": str(tok_file),
            "val_tokens": sum(i["val_tokens"] for i in infos.values()),
            "val_words": sum(i["val_words"] for i in infos.values()),
            "n_eval_prompts": len(ev)}
    jdump(meta, ddir / "meta.json")
    tpw = meta["val_tokens"] / max(1, meta["val_words"])
    log(f"[prepare] done. tokens/word on validation = {tpw:.3f}; eval prompts = {len(ev)}")
    return ddir


# ============================================================================================
# 3. Models: gpt2 | llama | retnet | moe  (all with KV / recurrent cache and left-padding support)
# ============================================================================================
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.w = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        xf = x.float()
        y = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return y.to(x.dtype) * self.w.to(x.dtype)


def rope_tables(dim, maxlen, base=10000.0):
    inv = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
    f = torch.outer(torch.arange(maxlen).float(), inv)
    return f.cos(), f.sin()


def apply_rope(x, cos, sin):           # x: (B,H,T,hd)  cos/sin: (B,1,T,hd/2)
    x1, x2 = x.float().chunk(2, dim=-1)
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1).to(x.dtype)


class Attn(nn.Module):
    def __init__(self, mc):
        super().__init__()
        d, gpt2 = mc["n_embd"], mc["arch"] == "gpt2"
        self.h = mc["n_head"]
        self.kvh = self.h if gpt2 else mc["n_kv_head"]
        self.hd = d // self.h
        self.rope = not gpt2
        self.p = mc.get("dropout", 0.0)
        self.q = nn.Linear(d, self.h * self.hd, bias=gpt2)
        self.k = nn.Linear(d, self.kvh * self.hd, bias=gpt2)
        self.v = nn.Linear(d, self.kvh * self.hd, bias=gpt2)
        self.o = nn.Linear(d, d, bias=gpt2)

    def forward(self, x, rope, amask, valid, cache):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.kvh, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.kvh, self.hd).transpose(1, 2)
        if self.rope:
            q, k = apply_rope(q, *rope), apply_rope(k, *rope)
        if cache is not None:
            if "k" in cache:
                k = torch.cat([cache["k"], k], dim=2)
                v = torch.cat([cache["v"], v], dim=2)
            cache["k"], cache["v"] = k, v
        if self.kvh != self.h:
            r = self.h // self.kvh
            k, v = k.repeat_interleave(r, dim=1), v.repeat_interleave(r, dim=1)
        if amask is not None:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=amask)
        else:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                               dropout_p=self.p if self.training else 0.0)
        return self.o(y.transpose(1, 2).reshape(B, T, -1))


class Retention(nn.Module):
    """RetNet-style multi-scale retention. Parallel form for training / prefill, recurrent form for decoding."""

    def __init__(self, mc):
        super().__init__()
        d = mc["n_embd"]
        self.h = mc["n_head"]
        self.hd = d // self.h
        self.q, self.k, self.v, self.g, self.o = (nn.Linear(d, d, bias=False) for _ in range(5))
        self.qn, self.kn, self.on = RMSNorm(self.hd), RMSNorm(self.hd), RMSNorm(self.hd)
        self.register_buffer("gamma", 1.0 - 2.0 ** (-5.0 - torch.arange(self.h).float()), persistent=False)
        self._D = {}

    def decay_matrix(self, T, device, dtype):
        key = (T, str(device), dtype)
        if key not in self._D:
            i = torch.arange(T, device=device)
            diff = i[:, None] - i[None, :]
            D = (self.gamma.view(-1, 1, 1) ** diff.clamp(min=0).float()) * (diff >= 0)
            self._D = {key: D.to(dtype)}          # keep only the most recent size
        return self._D[key]

    def forward(self, x, rope, amask, valid, cache):
        B, T, d = x.shape
        H, hd = self.h, self.hd
        q = self.qn(self.q(x).view(B, T, H, hd).transpose(1, 2)) * (hd ** -0.5)
        k = self.kn(self.k(x).view(B, T, H, hd).transpose(1, 2))
        v = self.v(x).view(B, T, H, hd).transpose(1, 2)
        g = self.g(x)
        q, k = apply_rope(q, *rope), apply_rope(k, *rope)
        if valid is not None:                      # left-padding: padded tokens contribute nothing
            vm = valid[:, None, :, None].to(k.dtype)
            k, v = k * vm, v * vm
        dev_type = "cuda" if x.is_cuda else "cpu"
        if cache is not None and "S" in cache:     # recurrent decoding step
            assert T == 1, "recurrent decoding expects one token at a time"
            with torch.autocast(device_type=dev_type, enabled=False):
                S = self.gamma.view(1, -1, 1, 1) * cache["S"] + k.float().transpose(-1, -2) @ v.float()
                o = q.float() @ S
            cache["S"] = S
        else:                                      # parallel form
            D = self.decay_matrix(T, x.device, q.dtype)
            o = ((q @ k.transpose(-1, -2)) * D) @ v
            if cache is not None:                  # state after the prefill, for later recurrent steps
                with torch.autocast(device_type=dev_type, enabled=False):
                    dec = self.gamma.view(-1, 1, 1) ** (T - 1 - torch.arange(T, device=x.device).float()).view(1, -1, 1)
                    cache["S"] = (k.float() * dec[None]).transpose(-1, -2) @ v.float()
        o = self.on(o).transpose(1, 2).reshape(B, T, d)
        return self.o(o * F.silu(g))


class MLP(nn.Module):               # GPT-2 feed-forward
    def __init__(self, mc):
        super().__init__()
        d = mc["n_embd"]
        self.fc = nn.Linear(d, 4 * d)
        self.w2 = nn.Linear(4 * d, d)

    def forward(self, x):
        return self.w2(F.gelu(self.fc(x)))


class SwiGLU(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.w1 = nn.Linear(d, h, bias=False)
        self.w3 = nn.Linear(d, h, bias=False)
        self.w2 = nn.Linear(h, d, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class MoE(nn.Module):
    """Top-k routed mixture of SwiGLU experts with a Switch-style load-balancing loss."""

    def __init__(self, mc):
        super().__init__()
        d, E = mc["n_embd"], mc["n_experts"]
        self.E, self.k = E, mc["top_k"]
        self.router = nn.Linear(d, E, bias=False)
        self.experts = nn.ModuleList([SwiGLU(d, mc["ffn_hidden"]) for _ in range(E)])
        self.aux = torch.zeros(())

    def forward(self, x):
        B, T, D = x.shape
        xf = x.reshape(-1, D)
        probs = self.router(xf).float().softmax(-1)
        w, idx = probs.topk(self.k, dim=-1)
        w = w / w.sum(-1, keepdim=True)
        out = torch.zeros_like(xf)
        for e, ex in enumerate(self.experts):
            tok_idx, slot = (idx == e).nonzero(as_tuple=True)
            if tok_idx.numel() == 0:
                continue
            y = ex(xf[tok_idx]) * w[tok_idx, slot].unsqueeze(-1).to(out.dtype)
            out.index_add_(0, tok_idx, y.to(out.dtype))
        frac = F.one_hot(idx, self.E).float().sum(1).mean(0) / self.k
        self.aux = self.E * (frac * probs.mean(0)).sum()
        return out.view(B, T, D)


class Block(nn.Module):
    def __init__(self, mc):
        super().__init__()
        arch, d = mc["arch"], mc["n_embd"]
        Norm = nn.LayerNorm if arch == "gpt2" else RMSNorm
        self.n1, self.n2 = Norm(d), Norm(d)
        self.mix = Retention(mc) if arch == "retnet" else Attn(mc)
        if arch == "gpt2":
            self.ffn = MLP(mc)
        elif arch == "moe":
            self.ffn = MoE(mc)
        else:
            self.ffn = SwiGLU(d, mc["ffn_hidden"])
        self.drop = nn.Dropout(mc.get("dropout", 0.0))

    def forward(self, x, rope, amask, valid, cache):
        x = x + self.drop(self.mix(self.n1(x), rope, amask, valid, cache))
        return x + self.drop(self.ffn(self.n2(x)))


class LM(nn.Module):
    def __init__(self, mc):
        super().__init__()
        self.mc = mc
        d, V, arch, L = mc["n_embd"], mc["vocab"], mc["arch"], mc["n_layer"]
        self.wte = nn.Embedding(V, d)
        self.wpe = nn.Embedding(mc["ctx"], d) if arch == "gpt2" else None
        self.drop = nn.Dropout(mc.get("dropout", 0.0))
        self.blocks = nn.ModuleList([Block(mc) for _ in range(L)])
        self.norm_f = nn.LayerNorm(d) if arch == "gpt2" else RMSNorm(d)
        self.lm_head = nn.Linear(d, V, bias=False)
        self.lm_head.weight = self.wte.weight           # tied embeddings
        if arch == "gpt2":
            self.rope_cos = self.rope_sin = None
        else:
            cos, sin = rope_tables(d // mc["n_head"], mc["ctx"])
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init)
        for n, p in self.named_parameters():
            if n.endswith("o.weight") or n.endswith("w2.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * L))

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def new_cache(self):
        return {"past": 0, "layers": [{} for _ in self.blocks]}

    def forward(self, idx, targets=None, pos=None, kmask=None, cache=None, last_only=False):
        B, T = idx.shape
        dev = idx.device
        past = cache["past"] if cache is not None else 0
        if pos is None:
            pos = torch.arange(past, past + T, device=dev).unsqueeze(0).expand(B, -1)
        x = self.wte(idx)
        if self.wpe is not None:
            x = x + self.wpe(pos)
        x = self.drop(x)
        rope = None
        if self.rope_cos is not None:
            rope = (self.rope_cos[pos].unsqueeze(1), self.rope_sin[pos].unsqueeze(1))
        amask = valid = None
        if kmask is not None:
            S = past + T
            qi = torch.arange(past, past + T, device=dev).unsqueeze(1)
            kj = torch.arange(S, device=dev).unsqueeze(0)
            amask = ((kj <= qi)[None, None] & kmask[:, None, None, :]) | (kj == qi)[None, None]
            valid = kmask[:, -T:]
        for i, blk in enumerate(self.blocks):
            x = blk(x, rope, amask, valid, cache["layers"][i] if cache is not None else None)
        if cache is not None:
            cache["past"] = past + T
        x = self.norm_f(x)
        if last_only:
            x = x[:, -1:]
        logits = self.lm_head(x)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        if self.training and self.mc["arch"] == "moe":
            aux = sum(b.ffn.aux for b in self.blocks) / len(self.blocks)
            loss = loss + self.mc.get("aux_coef", 0.01) * aux
        return None, loss


def count_params(m) -> int:
    return sum(p.numel() for p in m.parameters())


def model_config(cfg: Cfg, vocab: int) -> dict:
    if cfg.arch not in ARCH_PRESETS:
        raise SystemExit(f"Unknown arch {cfg.arch!r}; valid: {list(ARCH_PRESETS)}")
    if cfg.size not in SIZE_PRESETS:
        raise SystemExit(f"Unknown size {cfg.size!r}; valid: {list(SIZE_PRESETS)}")
    mc = dict(SIZE_PRESETS[cfg.size][cfg.arch])
    mc.update(cfg.model)
    mc.update(arch=cfg.arch, ctx=int(cfg.ctx), vocab=int(vocab), dropout=float(cfg.dropout))
    mc.setdefault("n_kv_head", mc["n_head"])
    assert mc["n_embd"] % mc["n_head"] == 0 and mc["n_head"] % mc["n_kv_head"] == 0
    assert (mc["n_embd"] // mc["n_head"]) % 2 == 0, "head dim must be even (RoPE)"
    return mc


# ============================================================================================
# 4. Training data loader (deterministic + resumable) and training loop
# ============================================================================================
class MixLoader:
    """Weighted mixture of uint16 token shards. Per-sample source assignment and per-source block order are
    pure functions of (seed, sample index) => identical batches after a resume. Blocks are drawn without
    replacement within an epoch of each source."""

    def __init__(self, bins, weights, T, seed, total_samples):
        self.names = list(bins)
        self.T, self.seed = T, seed
        self.mm = [np.memmap(bins[n], dtype=np.uint16, mode="r") for n in self.names]
        self.nblk = [(len(m) - 1) // T for m in self.mm]
        w = np.array([weights[n] for n in self.names], dtype=np.float64)
        w = w / w.sum()
        self.src = np.random.default_rng(seed).choice(len(self.names), size=total_samples, p=w).astype(np.int16)
        self.cnt = np.zeros(total_samples, dtype=np.int64)
        for k in range(len(self.names)):
            m = np.nonzero(self.src == k)[0]
            self.cnt[m] = np.arange(len(m))
        self._perm = {}
        for k, n in enumerate(self.names):
            used = int((self.src == k).sum())
            if used > 2 * self.nblk[k]:
                log(f"[data] WARNING: source {n} will be seen ~{used / self.nblk[k]:.1f} times; prepare more tokens or lower its weight")

    def _block(self, k, c):
        n = self.nblk[k]
        ep = c // n
        if (k, ep) not in self._perm:
            self._perm = {kk: v for kk, v in self._perm.items() if kk[0] != k}
            self._perm[(k, ep)] = np.random.default_rng([self.seed, k, ep]).permutation(n)
        return int(self._perm[(k, ep)][c % n])

    def batch(self, g0, n):
        arr = np.empty((n, self.T + 1), dtype=np.int64)
        for j in range(n):
            g = g0 + j
            k = int(self.src[g])
            b = self._block(k, int(self.cnt[g]))
            arr[j] = self.mm[k][b * self.T: b * self.T + self.T + 1]
        t = torch.from_numpy(arr)
        return t[:, :-1].contiguous(), t[:, 1:].contiguous()


def lr_at(step, total, cfg: Cfg):
    w = min(cfg.warmup_steps, max(1, total // 10))
    if step < w:
        return cfg.lr * (step + 1) / w
    p = (step - w) / max(1, total - w)
    return cfg.lr * (cfg.min_lr_ratio + (1 - cfg.min_lr_ratio) * 0.5 * (1 + math.cos(math.pi * p)))


def make_opt(model, cfg: Cfg, dev):
    decay = [p for p in model.parameters() if p.requires_grad and p.dim() >= 2]
    other = [p for p in model.parameters() if p.requires_grad and p.dim() < 2]
    groups = [{"params": decay, "weight_decay": cfg.weight_decay}, {"params": other, "weight_decay": 0.0}]
    kw = dict(lr=cfg.lr, betas=(0.9, 0.95), eps=1e-8)
    if dev == "cuda":
        try:
            return torch.optim.AdamW(groups, fused=True, **kw)
        except Exception:
            pass
    return torch.optim.AdamW(groups, **kw)


def make_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=enabled)


@torch.no_grad()
def evaluate(model, val_mm, T, n_blocks, mb, dev, amp):
    model.eval()
    nb = (len(val_mm) - 1) // T
    idxs = np.linspace(0, nb - 1, min(n_blocks, nb)).astype(int)
    tot, cnt = 0.0, 0
    for s in range(0, len(idxs), mb):
        ch = idxs[s:s + mb]
        arr = np.stack([val_mm[i * T: i * T + T + 1] for i in ch]).astype(np.int64)
        x = torch.from_numpy(arr[:, :-1]).to(dev)
        y = torch.from_numpy(arr[:, 1:]).to(dev)
        with amp():
            _, loss = model(x, y)
        tot += float(loss) * len(ch)
        cnt += len(ch)
    model.train()
    return tot / cnt


def save_ckpt(path, model, mc, state, opt=None, scaler=None):
    obj = {"model": model.state_dict(), "mc": mc, "state": dict(state)}
    if opt is not None:
        obj["opt"] = opt.state_dict()
        obj["scaler"] = scaler.state_dict()
    tmp = str(path) + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, str(path))


def amp_setup(dev):
    if dev != "cuda":
        return None, contextlib.nullcontext
    dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    return dtype, (lambda: torch.autocast("cuda", dtype=dtype))


def train(cfg: Cfg) -> bool:
    """Returns True if training finished, False if it stopped early (wall limit / Ctrl-C) and can be resumed."""
    ddir = prepare(cfg)
    meta = jload(ddir / "meta.json")
    rd = run_dir_of(cfg)
    rd.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(cfg.seed)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    if dev == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    amp_dtype, amp = amp_setup(dev)
    V = pad64(meta["vocab"])
    mc = model_config(cfg, V)
    T = mc["ctx"]
    model = LM(mc).to(dev)
    n_params = count_params(model)
    log(f"[train] run={cfg.run} arch={cfg.arch} params={n_params/1e6:.1f}M ctx={T} device={dev} amp={amp_dtype}")
    if not (50e6 <= n_params <= 300e6) and "smoke" not in cfg.run and not cfg.run.startswith("selftest"):
        log("[train] WARNING: parameter count is outside the 50M-300M competition range!")

    jdump(asdict(cfg), rd / "config.json")
    shutil.copy(meta["tok_file"], rd / "tokenizer.json")
    shutil.copy(ddir / "eval.json", rd / "eval.json")
    shutil.copy(ddir / "eval_truth.json", rd / "eval_truth.json")
    jdump(meta, rd / "meta.json")

    weights = {n: float(cfg.mix.get(n, 0.0)) for n, i in meta["sources"].items() if i["train_tokens"] > T + 1}
    weights = {n: w for n, w in weights.items() if w > 0} or \
              {n: 1.0 for n, i in meta["sources"].items() if i["train_tokens"] > T + 1}
    if not weights:
        raise SystemExit("No training tokens available. Check sources / token caps.")
    bins = {n: ddir / f"{n}.train.bin" for n in weights}
    val_mm = np.memmap(ddir / "val.bin", dtype=np.uint16, mode="r")
    if len(val_mm) < T + 2:
        raise SystemExit("Validation shard too small for this context length.")
    tpw = meta["val_tokens"] / max(1, meta["val_words"])

    opt = make_opt(model, cfg, dev)
    scaler = make_scaler(dev == "cuda" and amp_dtype == torch.float16)
    state = {"step": 0, "best_val": float("inf"), "elapsed": 0.0, "total_steps": 0, "micro_bs": 0, "accum": 0}
    last_path, best_path = rd / "ckpt_last.pt", rd / "ckpt_best.pt"
    if last_path.exists():
        ck = torch.load(last_path, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        scaler.load_state_dict(ck["scaler"])
        state = ck["state"]
        log(f"[train] RESUMED from step {state['step']}/{state['total_steps']}")

    # ---- fit parallelism / micro-batch / step budget to this machine (fresh runs only)
    n_gpu = torch.cuda.device_count() if dev == "cuda" else 0
    if state["total_steps"] == 0:
        def bench(m, mb):
            if dev == "cuda":
                torch.cuda.reset_peak_memory_stats()
            x = torch.randint(0, V, (mb, T + 1), device=dev)
            m.train()
            t0 = time.time()
            for i in range(5):
                if i == 2:
                    if dev == "cuda":
                        torch.cuda.synchronize()
                    t0 = time.time()
                with amp():
                    _, loss = m(x[:, :-1], x[:, 1:])
                if loss.dim() > 0:                      # DataParallel returns one loss per GPU
                    loss = loss.mean()
                scaler.scale(loss).backward()
                model.zero_grad(set_to_none=True)
            if dev == "cuda":
                torch.cuda.synchronize()
            return 3 * mb * T / (time.time() - t0)

        def make_cands(dp):
            if dev == "cpu":
                return [min(cfg.max_micro_bs, 4)]
            unit, cap_ = (n_gpu, cfg.max_micro_bs * n_gpu) if dp else (1, cfg.max_micro_bs)
            return [m for m in (64, 32, 16, 8, 4, 2, 1) if m % unit == 0 and m <= cap_] or [unit]

        def cleanup():
            model.zero_grad(set_to_none=True)
            gc.collect()
            if dev == "cuda":
                torch.cuda.empty_cache()

        def tune(bm, cands):
            for c in cands:
                oom = False
                try:
                    return c, bench(bm, c)
                except RuntimeError as e:
                    if "out of memory" not in str(e).lower():
                        raise
                    oom = True
                if oom:
                    cleanup()
            return None, None

        use_dp = bool(cfg.multi_gpu) and n_gpu > 1
        bm = nn.DataParallel(model) if use_dp else model
        try:
            mb, tps = tune(bm, make_cands(use_dp))
        except Exception as e:
            if not use_dp:
                raise
            log(f"[train] DataParallel failed ({type(e).__name__}: {str(e)[:120]}); falling back to a single GPU")
            use_dp, bm = False, model
            cleanup()
            mb, tps = tune(bm, make_cands(False))
        if mb is None:
            raise SystemExit("Even the smallest micro-batch does not fit in GPU memory; reduce ctx or model size.")
        unit = n_gpu if use_dp else 1
        if dev == "cuda" and mb > unit:
            if torch.cuda.max_memory_allocated() > 0.9 * torch.cuda.get_device_properties(0).total_memory:
                half = mb // 2
                if half >= unit and half % unit == 0:
                    mb = half
        compiled = False
        if (not use_dp and dev == "cuda" and str(cfg.compile).lower() in ("auto", "true", "on", "1")
                and cfg.arch in ("gpt2", "llama")):
            try:
                tps_c = bench(torch.compile(model), mb)
                log(f"[train] torch.compile {tps_c:,.0f} tok/s vs eager {tps:,.0f} tok/s")
                if tps_c > 1.08 * tps:
                    compiled, tps = True, tps_c
            except Exception as e:
                log(f"[train] torch.compile unavailable ({type(e).__name__}); using eager mode")
            cleanup()
        accum = max(1, round(cfg.batch_tokens / (mb * T)))
        tps_step = accum * mb * T
        if cfg.train_tokens > 0:
            total = max(1, math.ceil(cfg.train_tokens / tps_step))
        elif cfg.time_budget_hours > 0:
            total = max(50, int(cfg.time_budget_hours * 3600 * tps * 0.92 / tps_step))
        else:
            raise SystemExit("Set time_budget_hours > 0 or train_tokens > 0")
        state.update(total_steps=total, micro_bs=mb, accum=accum, use_dp=use_dp, compile=compiled)
        log(f"[train] measured {tps:,.0f} tok/s | gpus={n_gpu} dp={use_dp} compile={compiled} | micro_bs={mb} "
            f"accum={accum} ({tps_step:,} tok/step) | total_steps={total:,} "
            f"(~{total * tps_step / 1e6:,.0f}M tokens, ~{total * tps_step / tps / 3600:.2f} h)")
        cleanup()
        jdump(state, rd / "plan.json")

    mb, accum, total = state["micro_bs"], state["accum"], state["total_steps"]
    loader = MixLoader(bins, weights, T, cfg.seed, total * accum * mb)
    log(f"[train] mixture (normalised): { {n: round(w / sum(weights.values()), 3) for n, w in weights.items()} }")
    use_dp = bool(state.get("use_dp", False)) and n_gpu > 1
    if use_dp:
        cmodel = nn.DataParallel(model)
    elif state.get("compile") and dev == "cuda":
        cmodel = torch.compile(model)
    else:
        cmodel = model
    log(f"[train] parallel={'DataParallel x%d' % n_gpu if use_dp else 'single'} compile={bool(state.get('compile'))}")

    model.train()
    t_session, last_ck, t_log = time.time(), time.time(), time.time()
    loss_acc, n_acc, step = torch.zeros((), device=dev), 0, state["step"]
    elapsed0 = state["elapsed"]
    logf = open(rd / "log.jsonl", "a")
    finished = True
    try:
        while step < total:
            lr = lr_at(step, total, cfg)
            for gp in opt.param_groups:
                gp["lr"] = lr
            for micro in range(accum):
                x, y = loader.batch((step * accum + micro) * mb, mb)
                x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
                with amp():
                    _, loss = cmodel(x, y)
                if loss.dim() > 0:
                    loss = loss.mean()
                scaler.scale(loss / accum).backward()
                loss_acc += loss.detach()
                n_acc += 1
            scaler.unscale_(opt)
            gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            step += 1
            state["step"] = step
            state["elapsed"] = elapsed0 + time.time() - t_session

            if step % cfg.log_every == 0 or step == total:
                l = float(loss_acc) / n_acc
                if not math.isfinite(l):
                    raise RuntimeError("Loss is NaN/inf. Lower lr, or use a GPU with bf16 support.")
                dt = time.time() - t_log
                tok_s = cfg.log_every * accum * mb * T / max(dt, 1e-9) if step % cfg.log_every == 0 else 0
                eta = (total - step) * (dt / cfg.log_every) / 3600 if tok_s else 0
                log(f"step {step}/{total} loss {l:.4f} lr {lr:.2e} gnorm {float(gn):.2f} {tok_s:,.0f} tok/s eta {eta:.2f}h")
                logf.write(json.dumps({"type": "train", "step": step, "loss": l, "lr": lr, "tok_s": tok_s}) + "\n")
                logf.flush()
                loss_acc.zero_()
                n_acc, t_log = 0, time.time()

            if step % cfg.eval_every == 0 or step == total:
                vl = evaluate(model, val_mm, T, cfg.eval_blocks, mb, dev, amp)
                rec = {"type": "eval", "step": step, "val_loss": vl, "val_ppl": math.exp(vl),
                       "val_ppl_word": math.exp(vl * tpw)}
                log(f"  EVAL step {step}: val_loss {vl:.4f} ppl {rec['val_ppl']:.2f} (per-word ppl {rec['val_ppl_word']:.1f})")
                logf.write(json.dumps(rec) + "\n")
                logf.flush()
                if vl < state["best_val"]:
                    state["best_val"] = vl
                    save_ckpt(best_path, model, mc, state)
                t_log = time.time()

            if time.time() - last_ck > cfg.ckpt_every_min * 60:
                save_ckpt(last_path, model, mc, state, opt, scaler)
                last_ck = time.time()
            if cfg.max_wall_hours > 0 and time.time() - t_session > cfg.max_wall_hours * 3600 and step < total:
                log(f"[train] wall limit reached at step {step}; checkpointing. Re-run the same command to resume.")
                finished = False
                break
    except KeyboardInterrupt:
        log("[train] interrupted; checkpointing. Re-run the same command to resume.")
        finished = False
    finally:
        logf.close()
    # reached only on a clean finish / wall limit / Ctrl-C (a crash such as NaN loss propagates and does NOT
    # overwrite the last good checkpoint)
    save_ckpt(last_path, model, mc, state, opt, scaler)
    if not finished:
        return False

    ev = [json.loads(l) for l in open(rd / "log.jsonl") if '"eval"' in l]
    last_ev = ev[-1] if ev else {}
    jdump({"params": n_params, "steps": step, "best_val_loss": state["best_val"],
           "last": last_ev, "train_hours": state["elapsed"] / 3600}, rd / "train_summary.json")
    log_experiment(cfg, {"stage": "train", "params_M": round(n_params / 1e6, 1), "steps": step,
                         "val_loss": round(state["best_val"], 4), "val_ppl": round(math.exp(state["best_val"]), 2),
                         "val_ppl_word": round(math.exp(state["best_val"] * tpw), 1)})
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        tr = [json.loads(l) for l in open(rd / "log.jsonl") if '"train"' in l]
        plt.figure(figsize=(9, 4.5))
        plt.plot([r["step"] for r in tr], [r["loss"] for r in tr], label="train loss")
        plt.plot([r["step"] for r in ev], [r["val_loss"] for r in ev], label="val loss", marker="o")
        plt.xlabel("step"); plt.ylabel("loss"); plt.grid(True); plt.legend(); plt.tight_layout()
        plt.savefig(rd / "loss.png", dpi=150)
        plt.close()
    except Exception as e:
        log(f"[train] (plot skipped: {e})")
    log(f"[train] finished. best val loss {state['best_val']:.4f}")
    return True


# ============================================================================================
# 5. Generation (batched, left-padded, KV/recurrent cache, exact word-count output)
# ============================================================================================
def load_for_inference(cfg: Cfg, which="best"):
    from tokenizers import Tokenizer
    rd = run_dir_of(cfg)
    p = rd / f"ckpt_{which}.pt"
    if not p.exists():
        p = rd / "ckpt_last.pt"
    if not p.exists():
        raise SystemExit(f"No checkpoint in {rd}. Run the train stage first.")
    ck = torch.load(p, map_location="cpu", weights_only=False)
    model = LM(ck["mc"])
    model.load_state_dict(ck["model"])
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(dev).eval()
    tok = Tokenizer.from_file(str(rd / "tokenizer.json"))
    eos = tok.token_to_id(EOS)
    amp_dtype, _ = amp_setup(dev)
    log(f"[infer] loaded {p.name} (step {ck['state']['step']}) on {dev}")
    return model, tok, eos, dev, amp_dtype


@torch.no_grad()
def _gen_batch(model, tok, ids_list, dec, cfg: Cfg, dev, amp_dtype, eos):
    B = len(ids_list)
    L = max(len(x) for x in ids_list)
    inp = torch.zeros((B, L), dtype=torch.long)
    km = torch.zeros((B, L), dtype=torch.bool)
    for b, x in enumerate(ids_list):
        inp[b, L - len(x):] = torch.tensor(x, dtype=torch.long)
        km[b, L - len(x):] = True
    inp, km = inp.to(dev), km.to(dev)
    pos = (km.long().cumsum(1) - 1).clamp(min=0)
    last_pos = pos[:, -1:]
    cache = model.new_cache()
    temp, top_p = float(dec.get("temperature", 0.0)), float(dec.get("top_p", 1.0))
    top_k, rep, nr = int(dec.get("top_k", 0)), float(dec.get("rep", 1.0)), int(dec.get("nr", 0))
    gcpu = torch.Generator(device=dev).manual_seed(cfg.seed)
    gen = torch.zeros((B, 0), dtype=torch.long, device=dev)
    seqs = [[] for _ in range(B)]
    maps = [dict() for _ in range(B)]
    ctxm = torch.autocast("cuda", dtype=amp_dtype) if dev == "cuda" else contextlib.nullcontext()
    target = cfg.target_words
    with ctxm:
        logits, _ = model(inp, pos=pos, kmask=km, cache=cache, last_only=True)
        for t in range(cfg.max_new_tokens):
            lg = logits[:, -1].float().clone()
            lg[:, eos] = -float("inf")                 # never stop early: we need a full 128 words
            if rep != 1.0 and gen.shape[1] > 0:
                sc = lg.gather(1, gen)
                lg.scatter_(1, gen, torch.where(sc < 0, sc * rep, sc / rep))
            if nr >= 2:
                rows, cols = [], []
                for b in range(B):
                    s = seqs[b]
                    if len(s) >= nr - 1:
                        for tk in maps[b].get(tuple(s[-(nr - 1):]), ()):
                            rows.append(b)
                            cols.append(tk)
                if rows:
                    lg[torch.tensor(rows, device=dev), torch.tensor(cols, device=dev)] = -float("inf")
            if temp <= 0:
                nxt = lg.argmax(-1)
            else:
                lg = lg / temp
                if top_k > 0:
                    kth = lg.topk(min(top_k, lg.shape[-1]), dim=-1).values[:, -1:]
                    lg = lg.masked_fill(lg < kth, -float("inf"))
                if top_p < 1.0:
                    sl, si = lg.sort(descending=True, dim=-1)
                    pr = sl.softmax(-1)
                    rm = (pr.cumsum(-1) - pr) > top_p
                    sl = sl.masked_fill(rm, -float("inf"))
                    lg = torch.full_like(lg, -float("inf")).scatter(1, si, sl)
                nxt = torch.multinomial(lg.softmax(-1), 1, generator=gcpu)[:, 0]
            nl = nxt.tolist()
            if nr >= 2:
                for b in range(B):
                    s = seqs[b]
                    if len(s) >= nr - 1:
                        maps[b].setdefault(tuple(s[-(nr - 1):]), []).append(nl[b])
            for b in range(B):
                seqs[b].append(nl[b])
            gen = torch.cat([gen, nxt[:, None]], dim=1)
            if (t + 1) % 16 == 0:
                texts = tok.decode_batch(gen.tolist(), skip_special_tokens=True)
                if all(len(x.split()) >= target + 1 for x in texts):
                    break
            km = torch.cat([km, torch.ones((B, 1), dtype=torch.bool, device=dev)], dim=1)
            last_pos = last_pos + 1
            logits, _ = model(nxt[:, None], pos=last_pos, kmask=km, cache=cache, last_only=True)
    texts = tok.decode_batch(gen.tolist(), skip_special_tokens=True)
    return [" ".join(x.split()[:target]) for x in texts]


def generate_texts(model, tok, prompts, dec, cfg: Cfg, dev, amp_dtype, eos, progress=True):
    budget = model.mc["ctx"] - cfg.max_new_tokens
    enc = []
    for p in prompts:
        ids = tok.encode(norm_ws(p) if cfg.normalize_ws else p).ids
        ids = ids[-budget:] if len(ids) > budget else ids
        enc.append(ids if ids else [eos])
    order = sorted(range(len(enc)), key=lambda i: len(enc[i]))
    out = [None] * len(enc)
    bs, i, t0 = cfg.gen_batch, 0, time.time()
    while i < len(order):
        idxs = order[i:i + bs]
        oom = False
        try:
            texts = _gen_batch(model, tok, [enc[j] for j in idxs], dec, cfg, dev, amp_dtype, eos)
        except RuntimeError as e:
            if "out of memory" in str(e).lower() and bs > 1:
                oom = True
            else:
                raise
        if oom:
            gc.collect()
            torch.cuda.empty_cache()
            bs = max(1, bs // 2)
            log(f"[gen] OOM -> batch size {bs}")
            continue
        for j, t in zip(idxs, texts):
            out[j] = t
        i += len(idxs)
        if progress:
            log(f"[gen] {i}/{len(order)} ({time.time() - t0:.0f}s)")
    return out


# ============================================================================================
# 6. Metrics (same formulas as evaluate_phase1.py) + sweep + predict
# ============================================================================================
def lcs_length(a, b):
    if not a or not b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    masks = {}
    for j, t in enumerate(a):
        masks[t] = masks.get(t, 0) | (1 << j)
    state = 0
    for t in b:
        m = masks.get(t, 0)
        x = m | state
        y = (state << 1) | 1
        state = x & ~(x - y)
    return bin(state).count("1")


def rouge_l_f1(p, r):
    pt, rt = p.split(), r.split()
    if not pt or not rt:
        return 0.0
    l = lcs_length(pt, rt)
    if not l:
        return 0.0
    pp, rr = l / len(pt), l / len(rt)
    return 2 * pp * rr / (pp + rr)


def _ngrams(t, n):
    from collections import Counter
    return Counter(tuple(t[i:i + n]) for i in range(len(t) - n + 1)) if len(t) >= n else Counter()


def bleu4(preds, refs):
    order, clipped, totals, pl, rl = 4, [0] * 4, [0] * 4, 0, 0
    for p, r in zip(preds, refs):
        pl += len(p)
        rl += len(r)
        for n in range(1, order + 1):
            pn, rn = _ngrams(p, n), _ngrams(r, n)
            totals[n - 1] += sum(pn.values())
            for g, c in pn.items():
                clipped[n - 1] += min(c, rn.get(g, 0))
    prec = [clipped[i] / totals[i] if totals[i] else 0.0 for i in range(order)]
    if any(x <= 0 for x in prec) or pl == 0:
        return 0.0
    bp = 1.0 if pl > rl else math.exp(1.0 - rl / pl)
    return bp * math.exp(sum(math.log(x) for x in prec) / order)


_BERT = None


def bert_f1(preds, refs, dev):
    global _BERT
    if _BERT is None:
        from bert_score import BERTScorer
        _BERT = BERTScorer(lang="en", device=dev, batch_size=16)    # roberta-large, layer 17, no rescaling = official
    _, _, f = _BERT.score(preds, refs)
    return float(f.mean().item())


def score_preds(preds, refs, cfg: Cfg, dev):
    out = {"rougeL": sum(rouge_l_f1(p, r) for p, r in zip(preds, refs)) / len(refs),
           "bleu4": bleu4([p.split() for p in preds], [r.split() for r in refs])}
    if cfg.use_bertscore:
        try:
            out["bertscore"] = bert_f1(preds, refs, dev)
        except Exception as e:
            log(f"[score] BERTScore unavailable ({type(e).__name__}: {e}); selecting by ROUGE-L instead")
            cfg.use_bertscore = False
    return out


def log_experiment(cfg: Cfg, row: dict):
    fields_ = ["time", "run", "stage", "arch", "params_M", "steps", "val_loss", "val_ppl", "val_ppl_word",
               "decode", "bertscore", "rougeL", "bleu4"]
    p = Path(cfg.work_dir) / "experiments.csv"
    new = not p.exists()
    with open(p, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields_, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerow({"time": time.strftime("%Y-%m-%d %H:%M:%S"), "run": cfg.run, "arch": cfg.arch, **row})


def sweep(cfg: Cfg, which="best") -> dict:
    rd = run_dir_of(cfg)
    model, tok, eos, dev, amp_dtype = load_for_inference(cfg, which)
    ev = jload(rd / "eval.json")
    keys = list(ev)[:cfg.sweep_n]
    prompts, refs = [ev[k]["prompt"] for k in keys], [ev[k]["reference"] for k in keys]
    log(f"[sweep] {len(keys)} held-out prompts, {len(cfg.sweep_grid)} decoding configs")
    results = []
    for dec in cfg.sweep_grid:
        t0 = time.time()
        preds = generate_texts(model, tok, prompts, dec, cfg, dev, amp_dtype, eos, progress=False)
        m = score_preds(preds, refs, cfg, dev)
        r = {"decode": dec, **m, "avg_words": float(np.mean([len(p.split()) for p in preds])),
             "sec": round(time.time() - t0, 1)}
        results.append(r)
        log(f"[sweep] {dec['name']:<22} " + "  ".join(f"{k}={m[k]:.4f}" for k in m) + f"  ({r['sec']}s)")
        log_experiment(cfg, {"stage": "sweep", "decode": dec["name"], **{k: round(v, 5) for k, v in m.items()}})
    key = cfg.select_metric if cfg.select_metric in results[0] else "rougeL"
    best = max(results, key=lambda r: r[key])
    jdump(results, rd / "sweep.json")
    jdump(best["decode"], rd / "best_decode.json")
    # a few qualitative samples for the report
    jdump([{"id": keys[i], "prompt_tail": " ".join(prompts[i].split()[-40:]), "reference": refs[i],
            "generated": generate_texts(model, tok, [prompts[i]], best["decode"], cfg, dev, amp_dtype, eos, False)[0]}
           for i in range(min(5, len(keys)))], rd / "samples.json")
    log(f"[sweep] BEST by {key}: {best['decode']['name']}  ({key}={best[key]:.4f})")
    return best


def predict(cfg: Cfg, which="best") -> str:
    rd = run_dir_of(cfg)
    model, tok, eos, dev, amp_dtype = load_for_inference(cfg, which)
    if cfg.decode:
        dec = cfg.decode
    elif (rd / "best_decode.json").exists():
        dec = jload(rd / "best_decode.json")
    else:
        dec = FALLBACK_DECODE
        log("[predict] no sweep results found; using fallback decoding config")
    log(f"[predict] decoding config: {dec}")
    tp = cfg.test_path or resolve_file(cfg, cfg.test_file)
    data = jload(tp)
    items = list(data.items()) if isinstance(data, dict) else [(str(i + 1), r) for i, r in enumerate(data)]
    ids = [str(k) for k, _ in items]
    prompts = [_rec_text(r) for _, r in items]
    log(f"[predict] {len(prompts)} test prompts from {tp}")
    preds = generate_texts(model, tok, prompts, dec, cfg, dev, amp_dtype, eos)
    out = {}
    for i, p in zip(ids, preds):
        out[i] = {"text": p if p.strip() else "the"}
    bad = [i for i in out if len(out[i]["text"].split()) != cfg.target_words]
    if bad:
        log(f"[predict] WARNING: {len(bad)} outputs do not have exactly {cfg.target_words} words (e.g. ids {bad[:5]})")
    pj = rd / "prediction.json"
    jdump(out, pj)
    zp = rd / "submission.zip"
    with zipfile.ZipFile(zp, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")        # must sit at the root of the zip
    hist = Path(cfg.work_dir) / "submissions"
    hist.mkdir(parents=True, exist_ok=True)
    shutil.copy(zp, hist / f"{cfg.run}_{time.strftime('%m%d_%H%M%S')}.zip")
    log(f"[predict] wrote {pj}  and  {zp}   ({len(out)} predictions)")
    return str(zp)


# ============================================================================================
# 7. Self-test
# ============================================================================================
def selftest(e2e=True) -> bool:
    import tempfile
    torch.manual_seed(0)
    ok = True
    log("=== unit tests: cached / left-padded decoding must equal full forward pass ===")
    for arch in ARCH_PRESETS:
        mc = {**ARCH_PRESETS[arch], "n_embd": 64, "n_layer": 2, "n_head": 4, "n_kv_head": 2, "ffn_hidden": 96,
              "n_experts": 4, "top_k": 2, "arch": arch, "ctx": 64, "vocab": 128, "dropout": 0.0}
        m = LM(mc).eval()
        lens, ext_n = (11, 7), 4
        seqs = [torch.randint(1, 128, (n + ext_n,)) for n in lens]
        with torch.no_grad():
            ref = [m(s[None])[0][0] for s in seqs]                   # (n+ext, V) full forward, no padding
            L = max(lens)
            inp = torch.zeros((2, L), dtype=torch.long)
            km = torch.zeros((2, L), dtype=torch.bool)
            for b, n in enumerate(lens):
                inp[b, L - n:] = seqs[b][:n]
                km[b, L - n:] = True
            pos = (km.long().cumsum(1) - 1).clamp(min=0)
            lp = pos[:, -1:]
            cache = m.new_cache()
            lg, _ = m(inp, pos=pos, kmask=km, cache=cache, last_only=True)
            diffs = [(lg[b, -1] - ref[b][lens[b] - 1]).abs().max().item() for b in range(2)]
            for j in range(ext_n):
                tk = torch.stack([seqs[b][lens[b] + j] for b in range(2)])[:, None]
                km = torch.cat([km, torch.ones((2, 1), dtype=torch.bool)], 1)
                lp = lp + 1
                lg, _ = m(tk, pos=lp, kmask=km, cache=cache, last_only=True)
                diffs += [(lg[b, -1] - ref[b][lens[b] + j]).abs().max().item() for b in range(2)]
        good = max(diffs) < 2e-3
        ok &= good
        log(f"  {arch:<7} max |cached - full| = {max(diffs):.2e}  {'PASS' if good else 'FAIL'}")
        # can it learn? (overfit one random batch)
        m.train()
        o = torch.optim.AdamW(m.parameters(), lr=3e-3)
        x = torch.randint(1, 128, (4, 33))
        l0 = None
        for _ in range(100):
            _, loss = m(x[:, :-1], x[:, 1:])
            l0 = l0 or loss.item()
            o.zero_grad()
            loss.backward()
            o.step()
        good = loss.item() < 0.8 * l0
        ok &= good
        log(f"  {arch:<7} overfit test: loss {l0:.2f} -> {loss.item():.2f}  {'PASS' if good else 'FAIL'}")

    log("=== parameter counts at real size (V=32064) ===")
    for size, presets in SIZE_PRESETS.items():
        for arch in presets:
            try:
                mc = {**presets[arch], "arch": arch, "ctx": 1024, "vocab": 32064, "dropout": 0.0}
                mc.setdefault("n_kv_head", mc["n_head"])
                with torch.device("meta"):
                    n = count_params(LM(mc))
                good = 50e6 <= n <= 300e6
                ok &= good
                log(f"  {size:<5} {arch:<7} {n / 1e6:6.1f}M  {'PASS' if good else 'FAIL (outside 50M-300M)'}")
            except Exception as e:
                log(f"  {size:<5} {arch:<7} (skipped: {type(e).__name__}: {e})")

    if e2e:
        log("=== end-to-end smoke run on synthetic data (prepare -> train -> sweep -> predict) ===")
        tmp = tempfile.mkdtemp(prefix="cs781_selftest_")
        rng = random.Random(0)
        words = ["".join(rng.choice("abcdefghilmnoprstu") for _ in range(rng.randint(2, 8))) for _ in range(300)]

        def doc(n):
            return " ".join(rng.choice(words) for _ in range(n))

        dd = os.path.join(tmp, "data")
        os.makedirs(dd)
        for rel in SOURCE_FILES.values():
            json.dump([{"text": doc(rng.randint(150, 400))} for _ in range(300)],
                      open(os.path.join(dd, os.path.basename(rel)), "w"))
        json.dump({str(i): {"text": doc(60)} for i in range(1, 7)}, open(os.path.join(dd, "test.json"), "w"))
        for arch in ARCH_PRESETS:
            cfg = apply_overrides(Cfg(), PRESETS["smoke"])
            apply_overrides(cfg, {"work_dir": os.path.join(tmp, "work"), "local_data_dir": dd,
                                  "run": f"selftest_{arch}", "arch": arch, "train_tokens": 60_000,
                                  "ctx": 192, "max_new_tokens": 96,
                                  "compile": "off",
                                  "sources": {"acts": 3e6, "tribunals": 3e6}, "mix": {"acts": 0.5, "tribunals": 0.5},
                                  "test_path": os.path.join(dd, "test.json"),
                                  "sweep_grid": [DEFAULT_GRID[1], DEFAULT_GRID[4], DEFAULT_GRID[6]]})
            try:
                prepare(cfg)
                assert train(cfg), "train did not finish"
                sweep(cfg)
                zp = predict(cfg)
                with zipfile.ZipFile(zp) as z:
                    assert z.namelist() == ["prediction.json"], z.namelist()
                    pr = json.loads(z.read("prediction.json"))
                assert set(pr) == {str(i) for i in range(1, 7)}
                # an untrained 27-step model can emit few whitespace-separated words, so only check the contract:
                # non-empty, never longer than the target (exact length is expected from a trained model)
                assert all(1 <= len(v["text"].split()) <= cfg.target_words for v in pr.values()), "bad output length"
                log(f"  e2e {arch:<7} PASS")
            except Exception as e:
                import traceback
                traceback.print_exc()
                log(f"  e2e {arch:<7} FAIL: {type(e).__name__}: {e}")
                ok = False
        shutil.rmtree(tmp, ignore_errors=True)
    log("SELFTEST " + ("PASSED" if ok else "FAILED"))
    return ok


# ============================================================================================
# 8. CLI
# ============================================================================================
def build_cfg(args) -> Cfg:
    cfg = Cfg()
    if args.preset:
        apply_overrides(cfg, PRESETS[args.preset])
    ov = {}
    if args.cfg:
        ov = json.loads(args.cfg) if args.cfg.strip().startswith("{") else jload(args.cfg)
    if args.run:
        ov["run"] = args.run
    apply_overrides(cfg, ov)
    saved = run_dir_of(cfg) / "config.json"
    if saved.exists() and not args.fresh and args.stage in ("train", "sweep", "predict", "all"):
        frozen = Cfg()
        apply_overrides(frozen, {k: v for k, v in jload(saved).items() if k in {f.name for f in fields(Cfg)}})
        allowed = RESUME_KEYS if args.stage == "train" else GEN_KEYS
        ignored = []
        for k, v in ov.items():
            if k in allowed:
                setattr(frozen, k, v)
            elif getattr(frozen, k, None) != v:
                ignored.append(k)
        if ignored:
            log(f"[config] run '{cfg.run}' already exists -> using its saved config; ignored: {ignored}. "
                f"(use a new --run name or --fresh to change them)")
        cfg = frozen
    return cfg


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("stage", choices=["selftest", "prepare", "train", "sweep", "predict", "all"])
    ap.add_argument("--run", default=None, help="run name (a new name = a fresh experiment)")
    ap.add_argument("--cfg", default="", help="JSON string or path to a JSON file with config overrides")
    ap.add_argument("--preset", default="", choices=["", "full", "fast", "turbo", "smoke"])
    ap.add_argument("--fresh", action="store_true", help="discard checkpoints/logs of this run and start over")
    ap.add_argument("--ckpt", default="best", choices=["best", "last"], help="checkpoint used by sweep/predict")
    ap.add_argument("--no-e2e", action="store_true", help="selftest: skip the synthetic end-to-end part")
    args = ap.parse_args(argv)

    if args.stage == "selftest":
        return 0 if selftest(e2e=not args.no_e2e) else 1
    cfg = build_cfg(args)
    Path(cfg.work_dir).mkdir(parents=True, exist_ok=True)
    if args.fresh:
        rd = run_dir_of(cfg)
        if rd.exists():
            shutil.rmtree(rd)
            log(f"[fresh] removed {rd}")
    log(f"work_dir={cfg.work_dir} run={cfg.run} arch={cfg.arch}")
    if args.stage == "prepare":
        prepare(cfg)
    elif args.stage == "train":
        return 0 if train(cfg) else 2
    elif args.stage == "sweep":
        sweep(cfg, args.ckpt)
    elif args.stage == "predict":
        predict(cfg, args.ckpt)
    elif args.stage == "all":
        if not train(cfg):
            log("Training not finished -> stop here. Re-run the same command to resume.")
            return 2
        sweep(cfg, args.ckpt)
        predict(cfg, args.ckpt)
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
# 3. Hugging Face login (Kaggle secret named "HF", or Colab secret "HF_TOKEN", or set HF_TOKEN yourself)
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF")
except Exception:
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass
print("HF token found:", bool(os.environ.get("HF_TOKEN")))

## 4. Self-test (run once, ~1-3 min on CPU/GPU). Must end with `SELFTEST PASSED`.

In [ ]:
!python cs781_phase1.py selftest

## 5. Configuration
Only change what you want to experiment with - every key is optional (see `Cfg` in the script).
`arch` is one of `gpt2`, `llama`, `retnet`, `moe`. The step budget is **fitted automatically** to your GPU speed from `time_budget_hours`
(or set `"train_tokens": 250_000_000` to fix it explicitly).

In [ ]:
import json
RUN = "turbo_llama_v1"     # change the name for every new experiment
PRESET = "turbo"           # turbo (~55M, fast) | full (~110M, slow) | fast | smoke
CFG = {
    "arch": "llama",                 # gpt2 | llama | retnet | moe  (llama/gpt2 are fastest)
    "time_budget_hours": 2.5,        # training time for THIS run; steps are fitted to the measured speed
    "max_wall_hours": 0,             # >0: stop this invocation after N hours (checkpoint kept; rerun to resume)
    # anything else is optional, e.g.: "size": "base", "ctx": 1024, "lr": 8e-4, "mix": {...}, "sources": {...}
    # "decode": {"name": "greedy_rp1.1", "temperature": 0.0, "rep": 1.1},   # skip the sweep choice
}
json.dump(CFG, open("cfg.json", "w"), indent=1)
print(RUN, PRESET, CFG)

## 6a. Everything in one go (prepare -> train -> sweep -> predict). Safe to re-run: it resumes.

In [ ]:
!python cs781_phase1.py all --run {RUN} --preset {PRESET} --cfg cfg.json

### 6b. ...or stage by stage
`prepare` downloads + tokenizes (slow once, then cached). `train` resumes automatically. `sweep` picks the best decoding setup on held-out prompts
(BERTScore F1 = the competition metric). `predict` writes `prediction.json` and `submission.zip` (with `prediction.json` at the zip root).

In [ ]:
!python cs781_phase1.py prepare --run {RUN} --preset {PRESET} --cfg cfg.json

In [ ]:
!python cs781_phase1.py train --run {RUN} --preset {PRESET} --cfg cfg.json

In [ ]:
!python cs781_phase1.py sweep --run {RUN}

In [ ]:
!python cs781_phase1.py predict --run {RUN}

### Optional (for the report): compare all four architectures with an identical short budget
Uses the same prepared data, so only the first run pays for preparation. Then re-run section 6a with the winner and a longer `time_budget_hours`.

In [ ]:
for arch in ["gpt2", "llama", "retnet", "moe"]:
    fn = f"cfg_{arch}.json"
    json.dump({**CFG, "arch": arch, "time_budget_hours": 0.4}, open(fn, "w"))
    !python cs781_phase1.py train --run explore_{arch} --preset {PRESET} --cfg {fn}
    !python cs781_phase1.py sweep --run explore_{arch}

## 7. Compare experiments (val loss / per-word perplexity / local BERTScore, ROUGE-L, BLEU-4)
Per-word perplexity is the fair number when comparing different tokenizers.

In [ ]:
import pandas as pd, glob
WORK = "/kaggle/working/cs781" if os.path.exists("/kaggle/working") else "./cs781_work"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
df = pd.read_csv(f"{WORK}/experiments.csv")
display(df[df.stage == "train"].drop(columns=["decode","bertscore","rougeL","bleu4"]))
display(df[df.stage == "sweep"].sort_values("bertscore", ascending=False).head(15) if "bertscore" in df else df[df.stage=="sweep"])

## 8. Copy the submission to the notebook output folder and look at a few samples

In [ ]:
import shutil, json
RD = f"{WORK}/runs/{RUN}"
shutil.copy(f"{RD}/submission.zip", "submission.zip")
print(open(f"{RD}/best_decode.json").read())
for s in json.load(open(f"{RD}/samples.json"))[:2]:
    print("\n--- prompt tail:", s["prompt_tail"], "\n--- reference:", s["reference"], "\n--- generated:", s["generated"])

# CS781 Phase 2 - Legal NER (FINAL)

Append at the end of your Phase-1 notebook (same session) or run standalone with your Phase-1 output attached.

**Order:** write scripts -> self-test -> setup -> (train missing backbones) -> `check` -> E1..E4 -> report -> (extra runs) -> ensemble -> collect submissions.

| Run | Backbone | Technique |
|---|---|---|
| `final_full_llama` | llama | full fine-tuning (teacher for E4) |
| `final_lorma_gpt2` | gpt2 | **LoRMA+** (compulsory) |
| `final_lora_moe` | moe | LoRA |
| `final_kd_retnet` | retnet | **knowledge distillation** (compulsory) |
| `final_ensemble` | best subset of all `final_*` runs | extra submission |

What is new in this final version: line-break + document-position features (zero-initialised, so training starts exactly at your Phase-1 model), layer-wise LR decay, early stopping,
decoding calibrated on validation (decoder, background threshold, per-class thresholds for macro-F1) and an automatic ensemble. Realistic expectations: the score is bounded by the quality of your
Phase-1 backbones; full fine-tuning and the ensemble will be your best submissions, adapter methods usually trail them.

In [ ]:
import os, base64
if not os.path.exists("cs781_phase1.py"):
    open("cs781_phase1.py", "w", encoding="utf-8").write(base64.b64decode("...").decode("utf-8"))
    print("wrote cs781_phase1.py")
else:
    print("cs781_phase1.py present")

In [ ]:
%%writefile cs781_phase2_ner.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
CS781 Phase 2 - Legal NER on top of YOUR Phase-1 language models  [FINAL VERSION]
(four adaptation strategies: full fine-tuning | LoRA | LoRMA (multiplicative low-rank) | knowledge distillation)

Needs cs781_phase1.py in the same folder (it provides the model classes) and a Phase-1 run folder that
contains ckpt_best.pt + tokenizer.json (e.g. <work_dir>/runs/turbo_llama_v1).

Pipeline
  text --(collapse whitespace, keep char map)--> Phase-1 BPE tokens (+char offsets) --> sliding windows
       --> Phase-1 backbone (bidirectional attention) + token-classification head (BIO, 25 classes)
       --> overlap-averaged probabilities --> greedy / constrained-Viterbi BIO decoding
       --> token spans --> ORIGINAL character offsets --> prediction.json / submission.zip

Methods (cfg "method")
  full     fine-tune every weight of the Phase-1 backbone + head
  partial  freeze embeddings and the first N blocks
  lora     additive low-rank adapters:           W = W0 + (alpha/r) B A                       (Hu et al.)
  lorma    multiplicative low-rank adapters:     W = I(BA) W0   (LoRMA+ : (alpha/r BA + I) W0 ;  LoRMAπ : Iπ(BA) W0)   (Bihany et al. 2025)
  kd       knowledge distillation: a smaller student (layer-pruned Phase-1 model) learns from a fine-tuned teacher

What makes this version strong (all switchable in the config)
  * layout features: line-break structure + position-in-document buckets, injected through ZERO-initialised embeddings (the model
    starts exactly as the Phase-1 model, then learns to use them) - court preambles are line-structured, we collapse whitespace otherwise
  * layer-wise learning-rate decay (full fine-tuning), early stopping, class-balanced loss
  * decoding calibrated on validation: decoder, background-class threshold and (macro-F1!) per-class thresholds
  * ensemble stage that greedily picks the best subset of your finished runs

Stages
  recalibrate  re-tune the entity/background decision threshold + decoder of a FINISHED run on validation and re-predict (no retraining)
  ensemble     average the token probabilities of several finished runs (same tokenizer) and decode once
  selftest   offsets + adapter identity-at-init + tiny end-to-end run of every method on synthetic data (~3-6 min)
  check      REAL data diagnostics: gold format, end-offset convention, token-boundary alignment, oracle strict F1
  train      fine-tune, validate (strict macro-F1), error analysis, predict test, write submission.zip
  predict    re-predict the test set from a saved run (best.pt)

Examples
  python cs781_phase2_ner.py check  --run ner_full_llama --cfg '{"p1_dir": "/kaggle/working/cs781/runs/turbo_llama_v1"}'
  python cs781_phase2_ner.py train  --run ner_full_llama --cfg cfg_ner.json
"""
from __future__ import annotations

import argparse
import contextlib
import csv
import gc
import json
import math
import os
import random
import re
import shutil
import sys
import time
import zipfile
import zlib
from dataclasses import dataclass, field, fields, asdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import cs781_phase1 as P1

log = P1.log
LABELS_DEFAULT = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]


# ============================================================================================
# 0. Config
# ============================================================================================
@dataclass
class NCfg:
    run: str = "ner_full_v1"
    work_dir: str = P1.DEFAULT_WORK
    p1_dir: str = ""                  # Phase-1 run dir (contains ckpt_best.pt, tokenizer.json)
    p1_ckpt: str = "ckpt_best.pt"
    hf_repo: str = "Exploration-Lab/CS-781-Capstone"
    local_data_dir: str = ""          # folder holding fold1-text.json, fold1-gold.json, ..., labels.txt, fold3-text.json
    train_prefix: str = "Phase_2_LNER/Train"
    test_file: str = "Phase_2_LNER/Test/fold3-text.json"
    # ---- data / context windows
    max_len: int = 512                # tokens per window (<= Phase-1 context)
    stride: int = 384                 # window step (overlap = max_len - stride)
    val_frac: float = 0.10
    split_seed: int = 1234
    neg_keep: float = 0.7             # fraction of entity-free windows kept per epoch (low values make the model over-predict entities)
    max_train_docs: int = 0           # debug only
    # ---- model
    method: str = "full"              # full | partial | lora | lorma | kd
    freeze_layers: int = 0            # partial: freeze embeddings + first N blocks
    # ---- parameter-efficient adapters (lora / lorma)
    peft_r: int = 16                  # rank
    peft_alpha: float = 32.0          # scale = alpha / r  (LoRA and LoRMA+; LoRMAπ uses scale 1 as in the paper)
    peft_dropout: float = 0.05        # LoRA only
    peft_lr: float = 5e-4             # learning rate of the adapter weights
    peft_targets: str = "all"         # attn (q,k,v,o + retention gate) | all (+ dense feed-forward linears)
    peft_experts: bool = False        # also adapt the MoE experts (off: only attention is adapted in MoE models)
    lorma_variant: str = "plus"       # plus (LoRMA+, additive rank inflation) | pi (LoRMAπ, permutation inflation)
    # ---- knowledge distillation
    teacher_run: str = ""             # name of a finished NER run (same labels + same tokenizer) used as teacher
    student_layers: int = 0           # student keeps this many evenly spaced Phase-1 blocks (0 = half)
    kd_alpha: float = 0.5             # loss = alpha * CE(hard labels) + (1 - alpha) * T^2 * KL(teacher || student)
    kd_temp: float = 2.0
    bidir: str = "auto"               # auto = bidirectional attention for gpt2/llama/moe, causal for retnet
    head: str = "auto"                # linear | bigru | auto (bigru when the backbone stays causal)
    head_dropout: float = 0.1
    head_hidden: int = 256            # BiGRU hidden size per direction (head = bigru)
    head_layers: int = 1              # BiGRU layers
    class_weight: str = "sqrt"        # none | sqrt  (up-weights rare entity classes)
    use_feats: bool = True            # line-break + document-position feature embeddings (zero-initialised)
    class_bias: bool = True           # tune a per-class decision threshold on validation (macro-F1 treats classes equally)
    # ---- optimisation
    epochs: int = 10
    patience: int = 4                 # early stopping: epochs without validation improvement
    batch_size: int = 16
    lr: float = 3e-4                  # backbone
    llrd: float = 0.85                # layer-wise lr decay (full / partial / kd); 1.0 = off
    head_lr: float = 1e-3
    weight_decay: float = 0.01
    warmup_frac: float = 0.06
    grad_clip: float = 1.0
    seed: int = 1337
    eval_batch: int = 32
    decode: str = "auto"              # greedy | viterbi | auto (best on validation)
    predict_after: bool = True
    ens_runs: list = field(default_factory=list)   # ensemble stage: names of finished runs whose probabilities are averaged
    ens_auto: bool = False            # ensemble stage: pick the best subset of ALL finished runs automatically
    ens_prefix: str = ""              # auto ensemble: only consider runs whose name starts with this
    ens_max: int = 6


def apply_overrides(cfg: NCfg, ov: dict) -> NCfg:
    names = {f.name for f in fields(NCfg)}
    for k, v in ov.items():
        if k not in names:
            raise SystemExit(f"Unknown config key {k!r}. Valid: {sorted(names)}")
        setattr(cfg, k, v)
    return cfg


NER_FIELDS = ["time", "run", "p1_arch", "method", "variant", "head", "bidir", "params_total_M", "params_trainable_M",
              "val_macro_f1", "val_macro_p", "val_macro_r", "decode", "train_s", "peak_gpu_mb", "infer_tok_s", "teacher_run"]


def run_dir(cfg: NCfg) -> Path:
    return Path(cfg.work_dir) / "ner_runs" / cfg.run


# ============================================================================================
# 1. Raw data loading (tolerant to the exact JSON layout) + end-offset convention
# ============================================================================================
def _doc_text(v) -> str:
    if isinstance(v, str):
        return v
    if isinstance(v, dict):
        for k in ("text", "document", "content"):
            if isinstance(v.get(k), str):
                return v[k]
    return ""


def _first(d: dict, keys, default=None):
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return default


def parse_gold_items(items):
    """-> list of (start, end, label, text_or_None)  (end exactly as stored in the file)"""
    if isinstance(items, dict):
        items = _first(items, ("entities", "annotations", "labels", "spans"), [])
    out = []
    for it in items or []:
        if isinstance(it, (list, tuple)):
            if len(it) < 3:
                continue
            s, e, l = it[0], it[1], it[2]
            t = it[3] if len(it) > 3 and isinstance(it[3], str) else None
        elif isinstance(it, dict):
            s = _first(it, ("start", "start_char", "start_offset", "begin", "start_pos"))
            e = _first(it, ("end", "end_char", "end_offset", "stop", "end_pos"))
            l = _first(it, ("label", "type", "entity", "tag", "entity_type", "category"))
            t = _first(it, ("text", "entity_text", "span", "value", "mention"))
            if s is None or e is None or l is None:
                continue
        else:
            continue
        out.append((int(s), int(e), str(l), t if isinstance(t, str) else None))
    return out


def _load_map(path):
    raw = P1.jload(path)
    if isinstance(raw, list):                       # [{"id":..,"text":..}, ...]
        raw = {str(_first(r, ("id", "doc_id", "document_id"), i)): r for i, r in enumerate(raw)}
    return {str(k): v for k, v in raw.items()}


def load_raw(cfg: NCfg):
    res = lambda rel: P1.resolve_file(cfg, rel)
    lp = res(f"{cfg.train_prefix}/labels.txt")
    labels = [x.strip() for x in open(lp, encoding="utf-8") if x.strip()] or LABELS_DEFAULT
    texts, gold = {}, {}
    for k in (1, 2):
        t = _load_map(res(f"{cfg.train_prefix}/fold{k}-text.json"))
        g = _load_map(res(f"{cfg.train_prefix}/fold{k}-gold.json"))
        for did, v in t.items():
            texts[did] = _doc_text(v)
            gold[did] = parse_gold_items(g.get(did, []))
    unknown = sorted({l for gs in gold.values() for _, _, l, _ in gs} - set(labels))
    if unknown:
        raise SystemExit(f"Gold labels not in labels.txt: {unknown}  (labels.txt: {labels})")
    return labels, texts, gold


def load_test(cfg: NCfg):
    t = _load_map(P1.resolve_file(cfg, cfg.test_file))
    return {k: _doc_text(v) for k, v in t.items()}


def detect_end_shift(texts, gold):
    """0 if gold end is exclusive (text[s:e] == entity text), 1 if inclusive."""
    ex = inc = n = 0
    for did, gs in gold.items():
        tx = texts[did]
        for s, e, _, t in gs:
            if t is None:
                continue
            n += 1
            ex += tx[s:e] == t
            inc += tx[s:e + 1] == t
    if n == 0:
        return 0, {"n_with_text": 0}
    shift = 1 if (inc > ex and ex < 0.5 * n) else 0
    return shift, {"n_with_text": n, "exclusive_match": ex / n, "inclusive_match": inc / n}


# ============================================================================================
# 2. Normalisation with a character map, tokenisation, BIO labels, windows
# ============================================================================================
_WORD = re.compile(r"\S+")


def normalize_with_feats(text: str):
    """Collapse whitespace runs (the Phase-1 model never saw newlines / double spaces) and keep
      n2o[i] : index of the original character of normalised character i
      nlb[i] : newlines (0, 1, 2+) in the whitespace BEFORE the word that starts at normalised char i (2 for the first word)
      nla[i] : newlines (0, 1, 2+) in the whitespace AFTER the word that ends at normalised char i (2 for the last word)
    The layout of a court preamble (party names on their own lines, 'Versus', judge names ...) is thus not lost."""
    words = list(_WORD.finditer(text))
    if not words:
        z = np.zeros(0, dtype=np.uint8)
        return "", np.zeros(0, dtype=np.int64), z, z
    nlw = np.empty(len(words) + 1, dtype=np.uint8)
    nlw[0], nlw[len(words)] = 2, 2
    for i in range(1, len(words)):
        nlw[i] = min(2, text.count("\n", words[i - 1].end(), words[i].start()))
    total = sum(m.end() - m.start() for m in words) + len(words) - 1
    nlb, nla = np.zeros(total, dtype=np.uint8), np.zeros(total, dtype=np.uint8)
    n2o, chars, pos = np.empty(total, dtype=np.int64), [], 0
    for i, m in enumerate(words):
        if i:
            chars.append(" ")
            n2o[pos] = m.start() - 1                    # separator -> a whitespace char before the word
            pos += 1
        w = m.group()
        n = len(w)
        chars.append(w)
        n2o[pos:pos + n] = np.arange(m.start(), m.end())
        nlb[pos] = nlw[i]
        nla[pos + n - 1] = nlw[i + 1]
        pos += n
    return "".join(chars), n2o, nlb, nla


def normalize_with_map(text: str):
    """(normalised text, char map) - see normalize_with_feats."""
    n, m, _, _ = normalize_with_feats(text)
    return n, m


POS_EDGES = np.array([64, 128, 256, 512, 1024, 2048, 4096, 8192])     # document-position buckets (in tokens), 9 buckets


class Doc:
    pass


def build_labels(n_tok, ts, te, ents, label2i, empty):
    lab = np.zeros(n_tok, dtype=np.int64)
    lab[empty] = -100
    mono = bool(np.all(np.diff(te) >= 0)) and bool(np.all(np.diff(ts) >= 0))
    unaligned = 0
    for cs, ce, l in sorted(ents, key=lambda x: -(x[1] - x[0])):         # long first, nested short ones overwrite
        if mono:
            i0 = int(np.searchsorted(te, cs, side="right"))
            i1 = int(np.searchsorted(ts, ce, side="left"))
            idx = [i for i in range(i0, i1) if not empty[i]]
        else:
            idx = np.nonzero((te > cs) & (ts < ce) & ~empty)[0].tolist()
        if not idx:
            unaligned += 1
            continue
        b = 1 + 2 * label2i[l]
        lab[idx[0]] = b
        lab[idx[1:]] = b + 1
    return lab, unaligned


def make_docs(tok, texts: dict, gold: dict | None, label2i: dict, end_shift: int, stats: dict):
    ids_order = list(texts)
    norms, maps, nlbs, nlas = [], [], [], []
    for d in ids_order:
        n, m, nb_, na_ = normalize_with_feats(texts[d])
        norms.append(n)
        maps.append(m)
        nlbs.append(nb_)
        nlas.append(na_)
    encs = tok.encode_batch([n if n else " " for n in norms])
    docs = []
    for did, norm, n2o, nlb, nla, enc in zip(ids_order, norms, maps, nlbs, nlas, encs):
        d = Doc()
        d.id, d.text, d.norm, d.n2o = did, texts[did], norm, n2o
        if not norm:
            enc_ids, off = [], np.zeros((0, 2), dtype=np.int64)
        else:
            enc_ids, off = enc.ids, np.asarray(enc.offsets, dtype=np.int64).reshape(-1, 2)
        d.ids = np.asarray(enc_ids, dtype=np.int64)
        ts, te = off[:, 0].copy(), off[:, 1].copy()
        if len(d.ids):
            nb = np.frombuffer(norm.encode("utf-32-le"), dtype=np.uint32)
            lead = (ts < te) & (nb[np.minimum(ts, len(nb) - 1)] == 32)    # byte-level tokens carry their leading space
            ts[lead] += 1
        d.ts, d.te = ts, te
        empty = te <= ts
        d.keep = np.nonzero(~empty)[0]
        d.feat = np.zeros(len(d.ids), dtype=np.int64)
        if len(d.ids):                                     # layout feature id = newlines-before * 3 + newlines-after
            d.feat = (nlb[np.minimum(ts, len(nlb) - 1)].astype(np.int64) * 3
                      + nla[np.clip(te - 1, 0, len(nla) - 1)].astype(np.int64))
            d.feat[empty] = 0
        d.gold, d.lab = None, None
        if gold is not None:
            ents_orig, ents_norm = [], []
            o2n = np.full(len(d.text), -1, dtype=np.int64)
            if len(n2o):
                nbn = np.frombuffer(norm.encode("utf-32-le"), dtype=np.uint32)
                nonsep = np.nonzero(nbn != 32)[0]
                o2n[n2o[nonsep]] = nonsep
            for s, e, l, _ in gold.get(did, []):
                e = e + end_shift
                stats["n_gold"] = stats.get("n_gold", 0) + 1
                if s < 0 or e > len(d.text) or e <= s:
                    stats["n_bad_offsets"] = stats.get("n_bad_offsets", 0) + 1
                    continue
                ents_orig.append((s, e, l))
                a, b = s, e
                while a < b and o2n[a] < 0:
                    a += 1
                while b > a and o2n[b - 1] < 0:
                    b -= 1
                if a != s or b != e:
                    stats["n_ws_edge"] = stats.get("n_ws_edge", 0) + 1
                if b <= a:
                    stats["n_empty_after_trim"] = stats.get("n_empty_after_trim", 0) + 1
                    continue
                ents_norm.append((int(o2n[a]), int(o2n[b - 1]) + 1, l))
            srt = sorted(ents_norm)
            stats["n_overlap_gold"] = stats.get("n_overlap_gold", 0) + sum(
                1 for x, y in zip(srt, srt[1:]) if y[0] < x[1])
            d.gold = set(ents_orig)
            d.lab, un = build_labels(len(d.ids), ts, te, ents_norm, label2i, empty)
            stats["n_unaligned"] = stats.get("n_unaligned", 0) + un
        docs.append(d)
    return docs


def make_windows(n, W, S):
    if n <= W:
        return [(0, n)] if n else []
    starts = list(range(0, n - W + 1, S))
    if starts[-1] + W < n:
        starts.append(n - W)
    return [(s, s + W) for s in starts]


# ============================================================================================
# 3. Decoding (greedy / constrained Viterbi) -> ORIGINAL character offsets
# ============================================================================================
def build_trans(L):
    C = 1 + 2 * L
    allowed = np.zeros((C, C), dtype=bool)
    for j in range(C):
        if j == 0 or (j - 1) % 2 == 0:
            allowed[:, j] = True                                  # O / B-x can follow anything
        else:
            lab = (j - 1) // 2
            allowed[1 + 2 * lab, j] = allowed[2 + 2 * lab, j] = True   # I-x only after B-x / I-x
    start = np.array([0.0 if (j == 0 or (j - 1) % 2 == 0) else -1e9 for j in range(C)])
    return np.where(allowed, 0.0, -1e9), start


def viterbi(logp, trans, start):
    T, C = logp.shape
    score = logp[0] + start
    back = np.zeros((T, C), dtype=np.int16)
    ar = np.arange(C)
    for t in range(1, T):
        cand = score[:, None] + trans
        idx = cand.argmax(0)
        back[t] = idx
        score = cand[idx, ar] + logp[t]
    path = np.empty(T, dtype=np.int64)
    path[-1] = int(score.argmax())
    for t in range(T - 1, 0, -1):
        path[t - 1] = back[t, path[t]]
    return path


def path_to_spans(path, ts, te, n2o):
    spans, cur = [], None

    def close():
        if cur is not None:
            spans.append((int(n2o[cur[1]]), int(n2o[cur[2] - 1]) + 1, cur[0]))

    for i, p in enumerate(path):
        if p == 0:
            close()
            cur = None
            continue
        lab, is_b = (int(p) - 1) // 2, (int(p) - 1) % 2 == 0
        if cur is not None and not is_b and cur[0] == lab:
            cur[2] = int(te[i])
            continue
        close()
        cur = [lab, int(ts[i]), int(te[i])]
    close()
    return spans


def decode_doc(doc, probs, mode, trans, start, o_bias=0.0, cbias=None):
    """o_bias is added to the log-probability of the background class 'O': > 0 -> fewer / more confident entities
    (higher precision), < 0 -> more entities (higher recall). It is tuned on validation data."""
    k = doc.keep
    if len(k) == 0:
        return []
    lp = np.log(np.clip(probs[k], 1e-9, 1.0))
    lp[:, 0] += o_bias
    if cbias is not None:
        lp += cbias[None, :]
    path = viterbi(lp, trans, start) if mode == "viterbi" else lp.argmax(1)
    return path_to_spans(path, doc.ts[k], doc.te[k], doc.n2o)


# ============================================================================================
# 4. Strict metrics (same formulas as evaluate_phase2.py) + error analysis
# ============================================================================================
def strict_scores(gold: dict, pred: dict, labels):
    stats = {l: [0, 0, 0] for l in labels}
    for did, g in gold.items():
        p = pred.get(did, set())
        for l in labels:
            gs = {x for x in g if x[2] == l}
            ps = {x for x in p if x[2] == l}
            stats[l][0] += len(gs & ps)
            stats[l][1] += len(ps - gs)
            stats[l][2] += len(gs - ps)
    per, ps_, rs_, fs_ = {}, [], [], []
    for l, (tp, fp, fn) in stats.items():
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
        per[l] = {"P": pr, "R": rc, "F1": f, "tp": tp, "fp": fp, "fn": fn}
        ps_.append(pr), rs_.append(rc), fs_.append(f)
    n = len(labels)
    return {"per_label": per, "macro_p": sum(ps_) / n, "macro_r": sum(rs_) / n, "macro_f1": sum(fs_) / n}


def present_f1(sc):
    """mean F1 over the labels that occur in the gold data (strict_scores' macro_f1 counts absent labels as 0)"""
    v = [x["F1"] for x in sc["per_label"].values() if x["tp"] + x["fn"] > 0]
    return sum(v) / max(1, len(v))


def analyze_errors(gold: dict, pred: dict):
    c = {"exact": 0, "boundary_error": 0, "label_confusion": 0, "overlap_other_label": 0, "spurious": 0, "missed": 0}
    for did, g in gold.items():
        p = pred.get(did, set())
        for x in p:
            if x in g:
                c["exact"] += 1
                continue
            ov = [y for y in g if y[0] < x[1] and x[0] < y[1]]
            if any(y[2] == x[2] for y in ov):
                c["boundary_error"] += 1
            elif any((y[0], y[1]) == (x[0], x[1]) for y in ov):
                c["label_confusion"] += 1
            elif ov:
                c["overlap_other_label"] += 1
            else:
                c["spurious"] += 1
        for y in g:
            if not any(x[0] < y[1] and y[0] < x[1] for x in p):
                c["missed"] += 1
    return c


def to_sets(spans_by_doc: dict, labels):
    return {d: {(s, e, labels[l]) for s, e, l in sp} for d, sp in spans_by_doc.items()}


# ============================================================================================
# 5. Model: Phase-1 backbone + token-classification head
# ============================================================================================
def load_p1(cfg: NCfg):
    from tokenizers import Tokenizer
    d = Path(cfg.p1_dir)
    ck_path = d / cfg.p1_ckpt
    if not ck_path.exists():
        raise SystemExit(f"Phase-1 checkpoint not found: {ck_path}  (set p1_dir to your Phase-1 run folder)")
    ck = torch.load(ck_path, map_location="cpu", weights_only=False)
    lm = P1.LM(ck["mc"])
    lm.load_state_dict(ck["model"])
    tok = Tokenizer.from_file(str(d / "tokenizer.json"))
    log(f"[p1] loaded {ck_path} arch={ck['mc']['arch']} params={P1.count_params(lm) / 1e6:.1f}M")
    return tok, lm, ck["mc"]


class NERNet(nn.Module):
    def __init__(self, lm, n_classes, bidir, head, head_dropout, head_hidden=256, head_layers=1, use_feats=False):
        super().__init__()
        self.lm, self.bidir, self.head_type, self.use_feats = lm, bidir, head, use_feats
        d = lm.mc["n_embd"]
        if use_feats:                                   # zero-init: the network starts exactly as the Phase-1 model
            self.f_emb = nn.Embedding(9, d)
            self.p_emb = nn.Embedding(len(POS_EDGES) + 1, d)
            nn.init.zeros_(self.f_emb.weight)
            nn.init.zeros_(self.p_emb.weight)
        out_d = d
        if head == "bigru":
            self.rnn = nn.GRU(d, head_hidden, num_layers=head_layers, batch_first=True, bidirectional=True,
                              dropout=0.1 if head_layers > 1 else 0.0)
            out_d = 2 * head_hidden
        self.drop = nn.Dropout(head_dropout)
        self.cls = nn.Linear(out_d, n_classes)
        nn.init.normal_(self.cls.weight, 0.0, 0.02)
        nn.init.zeros_(self.cls.bias)

    def forward(self, ids, km, feat=None, apos=None):
        lm = self.lm
        B, T = ids.shape
        dev = ids.device
        pos = torch.arange(T, device=dev).unsqueeze(0)
        x = lm.wte(ids)
        if lm.wpe is not None:
            x = x + lm.wpe(pos)
        if self.use_feats and feat is not None:
            x = x + self.f_emb(feat) + self.p_emb(apos)
        rope = None
        if lm.rope_cos is not None:
            rope = (lm.rope_cos[pos].unsqueeze(1), lm.rope_sin[pos].unsqueeze(1))
        if self.bidir:
            amask = km[:, None, None, :]                                       # every token sees all real tokens
        else:
            qi, kj = torch.arange(T, device=dev).unsqueeze(1), torch.arange(T, device=dev).unsqueeze(0)
            amask = ((kj <= qi)[None, None] & km[:, None, None, :]) | (kj == qi)[None, None]
        for blk in lm.blocks:
            x = blk(x, rope, amask, km, None)
        h = lm.norm_f(x)
        if self.head_type == "bigru":
            lengths = km.sum(1).cpu()
            pk = nn.utils.rnn.pack_padded_sequence(h, lengths, batch_first=True, enforce_sorted=False)
            h, _ = self.rnn(pk)
            h, _ = nn.utils.rnn.pad_packed_sequence(h, batch_first=True, total_length=T)
        return self.cls(self.drop(h))


def build_net(cfg: NCfg, lm, n_classes):
    arch = lm.mc["arch"]
    bidir = (arch != "retnet") if cfg.bidir == "auto" else (cfg.bidir in ("true", "yes", "1", True))
    if arch == "retnet":
        bidir = False                               # retention is inherently causal
    head = cfg.head if cfg.head != "auto" else ("linear" if bidir else "bigru")
    log(f"[model] arch={arch} bidirectional_attention={bidir} head={head}")
    return NERNet(lm, n_classes, bidir, head, cfg.head_dropout, cfg.head_hidden, cfg.head_layers, cfg.use_feats)


# --------------------------------------------------------------------------------------------
# Parameter-efficient adapters
# --------------------------------------------------------------------------------------------
class LoRALinear(nn.Module):
    """Additive low-rank adapter (LoRA):  y = W0 x + b + (alpha/r) * B A x ;  B = 0 at init so the model starts unchanged."""

    def __init__(self, base: nn.Linear, r: int, alpha: float, dropout: float):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)


_PI_IDX = {}


def _pi_index(d, device):
    """idx[i, j] = (j - i) mod d : gathering row i of a matrix with idx rotates that row to the right by i."""
    key = (d, str(device))
    if key not in _PI_IDX:
        i = torch.arange(d, device=device)
        _PI_IDX[key] = (i.unsqueeze(0) - i.unsqueeze(1)) % d
    return _PI_IDX[key]


class LoRMALinear(nn.Module):
    """Low-Rank Multiplicative Adaptation (LoRMA, Bihany et al. 2025): the frozen weight W0 (d x k) is multiplied
    from the left by a low-rank-parametrised d x d matrix, h = (I(BA) W0) x, with A in R^{r x d}, B in R^{d x r}.
      plus : I+(BA)  = (alpha/r) BA + I_d          init B = 0, A random      -> identity at start (LoRMA+)
      pi   : Ipi(BA) = rows of BA rotated by 0,1,2.. init B[:,0] = 1, A[0,0] = 1, rest of A = 0 -> identity at start (LoRMAπ)
    Operations are re-ordered as in the paper: W0 x is computed first, then the low-rank map acts on that d-dim vector,
    so the cost is comparable to LoRA. A bias (if any) is added after the multiplicative update."""

    def __init__(self, base: nn.Linear, r: int, alpha: float, variant: str):
        super().__init__()
        if variant not in ("plus", "pi"):
            raise SystemExit(f"lorma_variant must be 'plus' or 'pi', got {variant!r}")
        self.base, self.variant = base, variant
        d = base.out_features
        self.scale = alpha / r if variant == "plus" else 1.0
        self.A = nn.Parameter(torch.zeros(r, d))
        self.B = nn.Parameter(torch.zeros(d, r))
        if variant == "plus":
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        else:
            nn.init.normal_(self.B, 0.0, 0.02)
            with torch.no_grad():
                self.B[:, 0] = 1.0
                self.A[0, 0] = 1.0

    def forward(self, x):
        y = F.linear(x, self.base.weight)                       # W0 x
        if self.variant == "plus":
            y = y + self.scale * F.linear(F.linear(y, self.A), self.B)       # (I + s B A) W0 x
        else:
            m = (self.B @ self.A).gather(1, _pi_index(self.A.shape[1], self.A.device))   # Ipi(BA)  (d x d)
            y = F.linear(y, m)                                  # Ipi(BA) W0 x
        if self.base.bias is not None:
            y = y + self.base.bias
        return y


PEFT_ALL = {"q", "k", "v", "o", "g", "fc", "w1", "w2", "w3"}
PEFT_ATTN = {"q", "k", "v", "o", "g"}


def inject_peft(lm, cfg: NCfg) -> int:
    """Freeze the whole Phase-1 backbone and wrap its linear layers with LoRA / LoRMA adapters. Returns #layers wrapped."""
    for p in lm.parameters():
        p.requires_grad = False
    names = PEFT_ATTN if cfg.peft_targets == "attn" else PEFT_ALL
    wrap = (lambda base: LoRALinear(base, cfg.peft_r, cfg.peft_alpha, cfg.peft_dropout)) if cfg.method == "lora" else \
           (lambda base: LoRMALinear(base, cfg.peft_r, cfg.peft_alpha, cfg.lorma_variant))
    n = 0
    for blk in lm.blocks:
        for part in (blk.mix, blk.ffn):
            for name, child in list(part.named_children()):
                if isinstance(child, nn.Linear) and name in names:
                    setattr(part, name, wrap(child))
                    n += 1
                elif name == "experts" and cfg.peft_experts:
                    for ex in child:
                        for en, ec in list(ex.named_children()):
                            if isinstance(ec, nn.Linear) and en in names:
                                setattr(ex, en, wrap(ec))
                                n += 1
    return n


def apply_method(net: NERNet, cfg: NCfg):
    """Prepare the network for the chosen adaptation method (must run before the optimizer is built)."""
    m = cfg.method
    if m in ("full", "kd"):
        return
    if m == "partial":
        for p in net.lm.wte.parameters():
            p.requires_grad = False
        if net.lm.wpe is not None:
            for p in net.lm.wpe.parameters():
                p.requires_grad = False
        for blk in net.lm.blocks[:cfg.freeze_layers]:
            for p in blk.parameters():
                p.requires_grad = False
        return
    if m in ("lora", "lorma"):
        n = inject_peft(net.lm, cfg)
        log(f"[peft] {m}{'-' + cfg.lorma_variant if m == 'lorma' else ''}: wrapped {n} linear layers "
            f"(r={cfg.peft_r}, targets={cfg.peft_targets})")
        return
    raise SystemExit(f"unknown method {cfg.method!r} (use full | partial | lora | lorma | kd)")


# --------------------------------------------------------------------------------------------
# Knowledge distillation
# --------------------------------------------------------------------------------------------
def remap_student_keys(sd: dict, keep: list) -> dict:
    """Keep only the Phase-1 blocks listed in `keep` and renumber them 0..len(keep)-1."""
    out = {}
    for k, v in sd.items():
        if k.startswith("blocks."):
            _, i, rest = k.split(".", 2)
            if int(i) in keep:
                out[f"blocks.{keep.index(int(i))}.{rest}"] = v
        else:
            out[k] = v
    return out


def make_student(lm, mc: dict, cfg: NCfg):
    """Student = Phase-1 model with fewer (evenly spaced) blocks, initialised from the Phase-1 weights."""
    L = mc["n_layer"]
    n = cfg.student_layers or max(1, L // 2)
    if n >= L:
        raise SystemExit(f"student_layers ({n}) must be smaller than the Phase-1 depth ({L})")
    keep = [int(x) for x in np.linspace(0, L - 1, n).round()]
    mc2 = dict(mc)
    mc2["n_layer"] = n
    st = P1.LM(mc2)
    st.load_state_dict(remap_student_keys(lm.state_dict(), keep))
    log(f"[kd] student keeps Phase-1 blocks {keep} of {L}: {P1.count_params(lm) / 1e6:.1f}M -> {P1.count_params(st) / 1e6:.1f}M params")
    return st, mc2


def kd_loss(s_logits, t_logits, lab, T: float):
    """T^2 * KL(softmax(teacher/T) || softmax(student/T)) averaged over labelled (non-padding) tokens."""
    m = (lab != -100).float()
    sl = F.log_softmax(s_logits.float() / T, dim=-1)
    tp = F.softmax(t_logits.float() / T, dim=-1)
    kl = (tp * (torch.log(tp.clamp_min(1e-9)) - sl)).sum(-1)
    return (kl * m).sum() / m.sum().clamp_min(1.0) * (T * T)


def load_teacher(cfg: NCfg, dev, labels, tok_path):
    """Load a finished NER run as the (frozen) teacher; it must use the same labels and the same tokenizer."""
    tr = Path(cfg.work_dir) / "ner_runs" / cfg.teacher_run
    if not (tr / "best.pt").exists():
        raise SystemExit(f"teacher run not found: {tr}/best.pt (train it first)")
    ck = torch.load(tr / "best.pt", map_location="cpu", weights_only=False)
    if ck["labels"] != labels:
        raise SystemExit("teacher and student use different label sets")
    if (tr / "tokenizer.json").read_bytes() != Path(tok_path).read_bytes():
        raise SystemExit("teacher and student use different tokenizers (distillation needs identical token ids)")
    saved = NCfg()
    apply_overrides(saved, {k: v for k, v in ck["cfg"].items() if k in {f.name for f in fields(NCfg)}})
    net = build_net(saved, P1.LM(ck["mc"]), 1 + 2 * len(labels))
    apply_method(net, saved)
    net.load_state_dict(ck["net"])
    net.to(dev).eval()
    for p in net.parameters():
        p.requires_grad = False
    n = sum(p.numel() for p in net.parameters())
    log(f"[kd] teacher {cfg.teacher_run}: arch={ck['mc']['arch']} method={saved.method} params={n / 1e6:.1f}M")
    return net, n, ck["mc"]


# ============================================================================================
# 6. Batching, inference over windows, training
# ============================================================================================
def collate(docs, items, dev):
    B = len(items)
    L = max(e - s for _, s, e in items)
    ids = np.zeros((B, L), dtype=np.int64)
    lab = np.full((B, L), -100, dtype=np.int64)
    km = np.zeros((B, L), dtype=bool)
    feat = np.zeros((B, L), dtype=np.int64)
    apos = np.zeros((B, L), dtype=np.int64)
    for j, (di, s, e) in enumerate(items):
        d, n = docs[di], e - s
        ids[j, :n] = d.ids[s:e]
        km[j, :n] = True
        feat[j, :n] = d.feat[s:e]
        apos[j, :n] = np.searchsorted(POS_EDGES, np.arange(s, e), side="right")   # position of each token in its DOCUMENT
        if d.lab is not None:
            lab[j, :n] = d.lab[s:e]
    t = lambda a: torch.from_numpy(a).to(dev)
    return t(ids), t(km), t(lab), t(feat), t(apos)


@torch.no_grad()
def doc_probs(net, docs, cfg, dev, amp, C):
    net.eval()
    items = [(i, s, e) for i, d in enumerate(docs) for s, e in make_windows(len(d.ids), cfg.max_len, cfg.stride)]
    order = sorted(range(len(items)), key=lambda k: items[k][2] - items[k][1])
    probs = [np.zeros((len(d.ids), C), dtype=np.float32) for d in docs]
    wsum = [np.zeros(len(d.ids), dtype=np.float32) for d in docs]
    for b in range(0, len(order), cfg.eval_batch):
        chunk = [items[k] for k in order[b:b + cfg.eval_batch]]
        ids, km, _, ft, ap = collate(docs, chunk, dev)
        with amp():
            logits = net(ids, km, ft, ap)
        pr = logits.float().softmax(-1).cpu().numpy()
        for j, (di, s, e) in enumerate(chunk):
            L = e - s
            w = np.minimum(np.arange(1, L + 1), np.arange(L, 0, -1)).astype(np.float32)   # centre tokens count more
            probs[di][s:e] += pr[j, :L] * w[:, None]
            wsum[di][s:e] += w
    return [p / np.maximum(w, 1e-6)[:, None] for p, w in zip(probs, wsum)]


def predict_docs(net, docs, cfg, dev, amp, C, mode, trans, start, o_bias=0.0, cbias=None, group=64):
    out = {}
    for g in range(0, len(docs), group):
        sub = docs[g:g + group]
        for d, p in zip(sub, doc_probs(net, sub, cfg, dev, amp, C)):
            out[d.id] = decode_doc(d, p, mode, trans, start, o_bias, cbias)
    return out


def tune_class_bias(va, probs, labels, trans, start, o_bias, gold, min_support=5, grid=(-1.0, -0.5, 0.5, 1.0), passes=2):
    """Coordinate ascent on one additive log-prob bias per entity class (applied to its B and I states), greedy decoding.
    Macro-F1 weighs every class equally, so rare classes profit from their own threshold. To limit over-fitting to a small
    validation set, only classes with >= min_support gold spans are tuned and a change must gain >= 0.005 macro-F1."""
    C = 1 + 2 * len(labels)
    support = {l: sum(1 for g in gold.values() for x in g if x[2] == l) for l in labels}

    def score(cb):
        pr = {d.id: decode_doc(d, p, "greedy", trans, start, o_bias, cb) for d, p in zip(va, probs)}
        return strict_scores(gold, to_sets(pr, labels), labels)["macro_f1"]

    cb = np.zeros(C, dtype=np.float32)
    cur = score(cb)
    for _ in range(passes):
        improved = False
        for li, l in enumerate(labels):
            if support[l] < min_support:
                continue
            best_b, best_s = 0.0, cur
            for b in grid:
                trial = cb.copy()
                trial[1 + 2 * li] += b
                trial[2 + 2 * li] += b
                sc = score(trial)
                if sc > best_s + 0.005:
                    best_s, best_b = sc, b
            if best_b != 0.0:
                cb[1 + 2 * li] += best_b
                cb[2 + 2 * li] += best_b
                cur, improved = best_s, True
        if not improved:
            break
    return cb, cur


O_GRID = [-1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0, 3.0]
O_GRID_VIT = [0.0, 1.0, 2.0, 3.0]


def tune_decoding(va, probs, cfg, labels, trans, start):
    """Pick (decoder, O-bias) with the best strict macro-F1 on the validation documents."""
    gold = eval_gold_sets(va)
    best, table = None, []
    for mode, grid in (("greedy", O_GRID), ("viterbi", O_GRID_VIT)):
        if cfg.decode in ("greedy", "viterbi") and cfg.decode != mode:
            continue
        for b in grid:
            pr = {d.id: decode_doc(d, p, mode, trans, start, b) for d, p in zip(va, probs)}
            sets = to_sets(pr, labels)
            sc = strict_scores(gold, sets, labels)
            table.append({"decode": mode, "o_bias": b, "macro_f1": sc["macro_f1"], "macro_p": sc["macro_p"],
                          "macro_r": sc["macro_r"]})
            log(f"[val] decode={mode:<8} o_bias={b:+.1f}  macro-F1 {sc['macro_f1']:.4f} (P {sc['macro_p']:.4f} R {sc['macro_r']:.4f})")
            if best is None or sc["macro_f1"] > best["sc"]["macro_f1"] + 1e-9:
                best = {"sc": sc, "sets": sets, "mode": mode, "o_bias": b}
    best.setdefault("cbias", None)
    if cfg.class_bias:
        g_best = max((t for t in table if t["decode"] == "greedy"), key=lambda t: t["macro_f1"], default=None)
        if g_best is not None:
            cb, f_cb = tune_class_bias(va, probs, labels, trans, start, g_best["o_bias"], gold)
            log(f"[val] per-class thresholds: macro-F1 {f_cb:.4f} (best global setting {best['sc']['macro_f1']:.4f}); "
                f"bias per class {[round(float(cb[1 + 2 * i]), 1) for i in range(len(labels))]}")
            if f_cb > best["sc"]["macro_f1"] + 0.005:
                pr = {d.id: decode_doc(d, p, "greedy", trans, start, g_best["o_bias"], cb) for d, p in zip(va, probs)}
                sets = to_sets(pr, labels)
                best = {"sc": strict_scores(gold, sets, labels), "sets": sets, "mode": "greedy",
                        "o_bias": g_best["o_bias"], "cbias": cb}
    return best, table


def error_examples(va, gold, pred, n=40):
    """A random sample of concrete mistakes (type, predicted text, gold text) to see what the model gets wrong."""
    byid = {d.id: d for d in va}
    rows = []
    for did, g in gold.items():
        text, p = byid[did].text, pred.get(did, set())
        for x in sorted(p):
            if x in g:
                continue
            ov = [y for y in g if y[0] < x[1] and x[0] < y[1]]
            same = [y for y in ov if y[2] == x[2]]
            if same:
                kind, y = "boundary_error", same[0]
            elif ov:
                kind, y = ("label_confusion" if (ov[0][0], ov[0][1]) == (x[0], x[1]) else "overlap_other_label"), ov[0]
            else:
                kind, y = "spurious", None
            rows.append({"doc": did, "type": kind, "pred": [x[0], x[1], x[2], text[x[0]:x[1]][:80]],
                         "gold": [y[0], y[1], y[2], text[y[0]:y[1]][:80]] if y else None})
        for y in g:
            if not any(x[0] < y[1] and y[0] < x[1] for x in p):
                rows.append({"doc": did, "type": "missed", "pred": None,
                             "gold": [y[0], y[1], y[2], text[y[0]:y[1]][:80]]})
    random.Random(0).shuffle(rows)
    return rows[:n]


def lr_at(step, total, base, warm):
    w = max(1, int(total * warm))
    return base * (step + 1) / w if step < w else base * max(0.0, (total - step) / max(1, total - w))


def class_weights(docs, C, mode):
    if mode == "none":
        return None
    lab = np.concatenate([d.lab[d.lab >= 0] for d in docs if d.lab is not None])
    cnt = np.bincount(lab, minlength=C).astype(np.float64)
    w = np.clip(np.sqrt(cnt.max() / (cnt + 1.0)), 1.0, 6.0)
    return torch.tensor(w, dtype=torch.float32)


def eval_gold_sets(docs):
    return {d.id: d.gold for d in docs}


# ============================================================================================
# 7. Experiment driver
# ============================================================================================
def prepare_data(cfg: NCfg, tok, labels):
    labels_, texts, gold = load_raw(cfg)
    labels = labels_
    label2i = {l: i for i, l in enumerate(labels)}
    end_shift, info = detect_end_shift(texts, gold)
    log(f"[data] gold end-offset convention: {'INCLUSIVE (+1 applied)' if end_shift else 'exclusive'}  {info}")
    ids = sorted(texts)
    if cfg.max_train_docs:
        ids = ids[:cfg.max_train_docs]
    is_val = lambda d: (zlib.crc32(f"{cfg.split_seed}:{d}".encode()) % 1000) < cfg.val_frac * 1000
    va_ids = [d for d in ids if is_val(d)] or ids[:1]
    tr_ids = [d for d in ids if d not in set(va_ids)] or ids
    stats = {}
    t0 = time.time()
    tr = make_docs(tok, {d: texts[d] for d in tr_ids}, gold, label2i, end_shift, stats)
    va = make_docs(tok, {d: texts[d] for d in va_ids}, gold, label2i, end_shift, stats)
    log(f"[data] {len(tr)} train docs, {len(va)} val docs, "
        f"{sum(len(d.ids) for d in tr):,} train tokens ({time.time() - t0:.0f}s)  stats={stats}")
    return labels, label2i, end_shift, tr, va, stats


def oracle(docs, labels, trans=None):
    """gold -> token labels -> decode -> ORIGINAL offsets. Measures what the tokenisation / alignment can ever reach."""
    pred = {}
    for d in docs:
        k = d.keep
        pred[d.id] = path_to_spans(d.lab[k], d.ts[k], d.te[k], d.n2o) if len(k) else []
    return strict_scores(eval_gold_sets(docs), to_sets(pred, labels), labels)


def diagnose(cfg: NCfg):
    tok, _, mc = load_p1(cfg)
    labels, label2i, end_shift, tr, va, stats = prepare_data(cfg, tok, None)
    docs = tr + va
    lens = np.array([len(d.ids) for d in docs])
    nwin = sum(len(make_windows(n, cfg.max_len, cfg.stride)) for n in lens)
    log(f"[check] tokens/doc: mean {lens.mean():.0f}  median {np.median(lens):.0f}  max {lens.max()}  "
        f"| windows of {cfg.max_len}: {nwin}")
    if mc["ctx"] < cfg.max_len:
        log(f"[check] WARNING: max_len {cfg.max_len} > Phase-1 context {mc['ctx']}")
    o = oracle(docs, labels)
    log(f"[check] ORACLE strict macro-F1 (upper bound of tokenisation + offset mapping): {o['macro_f1']:.4f} "
        f"(over labels present in the data: {present_f1(o):.4f})")
    cnt = {l: o["per_label"][l]["tp"] + o["per_label"][l]["fn"] for l in labels}
    for l in labels:
        pl = o["per_label"][l]
        log(f"   {l:<10} gold={cnt[l]:>6}  oracle-recoverable={pl['R']:.4f}")
    if present_f1(o) < 0.97:
        log("[check] WARNING: oracle < 0.97 -> many gold boundaries fall inside BPE tokens or have whitespace edges; "
            "look at n_ws_edge / n_unaligned in the stats above.")
    return o


def run_train(cfg: NCfg):
    rd = run_dir(cfg)
    rd.mkdir(parents=True, exist_ok=True)
    jd = asdict(cfg)
    P1.jdump(jd, rd / "config.json")
    random.seed(cfg.seed), np.random.seed(cfg.seed), torch.manual_seed(cfg.seed)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    amp_dtype, amp = P1.amp_setup(dev)
    tok, lm, mc = load_p1(cfg)
    shutil.copy(Path(cfg.p1_dir) / "tokenizer.json", rd / "tokenizer.json")
    if cfg.method == "kd":
        if not cfg.teacher_run:
            raise SystemExit('method "kd" needs "teacher_run": the name of a finished NER run')
        lm, mc = make_student(lm, mc, cfg)
    labels, label2i, end_shift, tr, va, stats = prepare_data(cfg, tok, None)
    L = len(labels)
    C = 1 + 2 * L
    assert cfg.max_len <= mc["ctx"], "max_len exceeds the Phase-1 context length"
    teacher, teacher_params = None, 0
    if cfg.method == "kd":
        teacher, teacher_params, t_mc = load_teacher(cfg, dev, labels, rd / "tokenizer.json")
        assert cfg.max_len <= t_mc["ctx"], "max_len exceeds the teacher's context length"
    log(f"[oracle] strict macro-F1 upper bound on val docs: {oracle(va, labels)['macro_f1']:.4f}")

    net = build_net(cfg, lm, C)
    apply_method(net, cfg)
    net.to(dev)
    n_total = sum(p.numel() for p in net.parameters())
    n_train = sum(p.numel() for p in net.parameters() if p.requires_grad)
    log(f"[model] total params {n_total / 1e6:.1f}M | trainable {n_train / 1e6:.2f}M | method={cfg.method}")

    peft = cfg.method in ("lora", "lorma")
    body_lr, body_wd = (cfg.peft_lr, 0.0) if peft else (cfg.lr, cfg.weight_decay)   # no weight decay on adapters (it would pull LoRMAπ away from identity)
    n_blocks = mc["n_layer"]

    def lr_scale(name):                       # layer-wise lr decay: lower layers (and embeddings) move less
        if peft or cfg.llrd >= 1.0 or not name.startswith("lm."):
            return 1.0
        m_ = re.match(r"lm\.blocks\.(\d+)\.", name)
        if m_:
            return cfg.llrd ** (n_blocks - 1 - int(m_.group(1)))
        return cfg.llrd ** n_blocks if name.startswith(("lm.wte", "lm.wpe")) else 1.0

    buckets = {}
    for n_, p_ in net.named_parameters():
        if not p_.requires_grad:
            continue
        if n_.startswith("lm."):
            base, wd = body_lr * lr_scale(n_), (body_wd if p_.dim() >= 2 else 0.0)
        else:                                  # NER head, BiGRU, feature embeddings
            base, wd = cfg.head_lr, (cfg.weight_decay if p_.dim() >= 2 else 0.0)
        buckets.setdefault((round(base, 12), wd), []).append(p_)
    groups = [{"params": ps, "lr": b_, "base": b_, "weight_decay": wd} for (b_, wd), ps in buckets.items()]
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.98), eps=1e-8)
    scaler = P1.make_scaler(dev == "cuda" and amp_dtype == torch.float16)
    cw = class_weights(tr, C, cfg.class_weight)
    cw = cw.to(dev) if cw is not None else None

    all_items = [(i, s, e) for i, d in enumerate(tr) for s, e in make_windows(len(d.ids), cfg.max_len, cfg.stride)]
    has_ent = [bool((tr[i].lab[s:e] > 0).any()) for i, s, e in all_items]
    pos_items = [it for it, h in zip(all_items, has_ent) if h]
    neg_items = [it for it, h in zip(all_items, has_ent) if not h]
    per_epoch = len(pos_items) + int(cfg.neg_keep * len(neg_items))
    total_steps = max(1, cfg.epochs * math.ceil(per_epoch / cfg.batch_size))
    log(f"[train] windows: {len(pos_items)} with entities, {len(neg_items)} without (keep {cfg.neg_keep}) "
        f"-> ~{per_epoch}/epoch, {total_steps} steps")
    trans, start = build_trans(L)
    rng = random.Random(cfg.seed)
    best, step, t_start, bad = {"f1": -1.0}, 0, time.time(), 0
    if dev == "cuda":
        torch.cuda.reset_peak_memory_stats()
    logf = open(rd / "log.jsonl", "a")
    for ep in range(cfg.epochs):
        net.train()
        items = pos_items + [it for it in neg_items if rng.random() < cfg.neg_keep]
        rng.shuffle(items)
        run_loss, n_b, t_ep = 0.0, 0, time.time()
        for b in range(0, len(items), cfg.batch_size):
            ids, km, lab, ft, ap = collate(tr, items[b:b + cfg.batch_size], dev)
            if int((lab != -100).sum()) == 0:
                continue
            for g in opt.param_groups:
                g["lr"] = lr_at(step, total_steps, g["base"], cfg.warmup_frac)
            with amp():
                logits = net(ids, km, ft, ap)
                if teacher is not None:
                    with torch.no_grad():
                        t_logits = teacher(ids, km, ft, ap)
            loss = F.cross_entropy(logits.float().view(-1, C), lab.view(-1), weight=cw, ignore_index=-100)
            if teacher is not None:
                loss = cfg.kd_alpha * loss + (1.0 - cfg.kd_alpha) * kd_loss(logits, t_logits, lab, cfg.kd_temp)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in net.parameters() if p.requires_grad], cfg.grad_clip)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            step += 1
            run_loss += float(loss)
            n_b += 1
            if step % 20 == 0:
                log(f"  ep {ep + 1} step {step}/{total_steps} loss {run_loss / n_b:.4f}")
        pred = predict_docs(net, va, cfg, dev, amp, C, "greedy", trans, start)
        sc = strict_scores(eval_gold_sets(va), to_sets(pred, labels), labels)
        log(f"[epoch {ep + 1}/{cfg.epochs}] loss {run_loss / max(1, n_b):.4f} | val strict macro-F1 {sc['macro_f1']:.4f} "
            f"(P {sc['macro_p']:.4f} R {sc['macro_r']:.4f}) | {time.time() - t_ep:.0f}s")
        logf.write(json.dumps({"epoch": ep + 1, "loss": run_loss / max(1, n_b), "val_f1": sc["macro_f1"]}) + "\n")
        logf.flush()
        if sc["macro_f1"] > best["f1"]:
            best = {"f1": sc["macro_f1"], "epoch": ep + 1}
            torch.save({"net": net.state_dict(), "mc": mc, "labels": labels, "end_shift": end_shift,
                        "cfg": jd, "decode": "greedy"}, rd / "best.pt")
            bad = 0
        else:
            bad += 1
            if bad >= cfg.patience:
                log(f"[train] early stopping: no validation improvement for {bad} epochs (best epoch {best['epoch']})")
                break
    logf.close()
    train_s = time.time() - t_start
    peak = torch.cuda.max_memory_allocated() / 2 ** 20 if dev == "cuda" else 0.0

    # ---- reload best epoch, choose decoder, full analysis on validation
    ck = torch.load(rd / "best.pt", map_location="cpu", weights_only=False)
    net.load_state_dict(ck["net"])
    if dev == "cuda":
        torch.cuda.synchronize()
    t_inf = time.time()
    probs = doc_probs(net, va, cfg, dev, amp, C)
    if dev == "cuda":
        torch.cuda.synchronize()
    infer_tok_s = sum(len(d.ids) for d in va) / max(1e-9, time.time() - t_inf)
    best_dec, table = tune_decoding(va, probs, cfg, labels, trans, start)
    sc, ps, mode, o_bias = best_dec["sc"], best_dec["sets"], best_dec["mode"], best_dec["o_bias"]
    err = analyze_errors(eval_gold_sets(va), ps)
    log(f"[val] CHOSEN decoder: {mode} with O-bias {o_bias:+.1f} | macro-F1 {sc['macro_f1']:.4f} "
        f"P {sc['macro_p']:.4f} R {sc['macro_r']:.4f}")
    for l in labels:
        pl = sc["per_label"][l]
        log(f"   {l:<10} P={pl['P']:.3f} R={pl['R']:.3f} F1={pl['F1']:.3f}  (tp {pl['tp']} fp {pl['fp']} fn {pl['fn']})")
    log(f"[val] error breakdown (predictions): {err}")
    ex = error_examples(va, eval_gold_sets(va), ps)
    P1.jdump(ex, rd / "error_examples.json")
    for e in ex[:8]:
        log(f"   [{e['type']}] pred={e['pred']}  gold={e['gold']}")
    ck["decode"], ck["o_bias"] = mode, o_bias
    ck["cbias"] = None if best_dec["cbias"] is None else [float(x) for x in best_dec["cbias"]]
    torch.save(ck, rd / "best.pt")
    variant = cfg.lorma_variant if cfg.method == "lorma" else ""
    P1.jdump({"best_epoch": best["epoch"], "decode": mode, "o_bias": o_bias, "cbias": ck["cbias"], "decode_table": table, "scores": sc, "errors": err, "stats": stats,
              "params_total": n_total, "params_trainable": n_train, "train_seconds": train_s,
              "peak_gpu_mb": peak, "p1_arch": mc["arch"], "method": cfg.method, "variant": variant,
              "infer_tokens_per_s": infer_tok_s, "teacher_run": cfg.teacher_run,
              "teacher_params": teacher_params}, rd / "val_report.json")
    row = {"time": time.strftime("%Y-%m-%d %H:%M:%S"), "run": cfg.run, "p1_arch": mc["arch"], "method": cfg.method,
           "variant": variant, "head": net.head_type, "bidir": net.bidir, "params_total_M": round(n_total / 1e6, 2),
           "params_trainable_M": round(n_train / 1e6, 3), "val_macro_f1": round(sc["macro_f1"], 4),
           "val_macro_p": round(sc["macro_p"], 4), "val_macro_r": round(sc["macro_r"], 4), "decode": f"{mode}/o{o_bias:+.1f}",
           "train_s": round(train_s), "peak_gpu_mb": round(peak), "infer_tok_s": round(infer_tok_s),
           "teacher_run": cfg.teacher_run}
    p = Path(cfg.work_dir) / "ner_experiments.csv"
    if p.exists() and open(p).readline().strip().split(",") != NER_FIELDS:
        p.rename(p.with_suffix(".old.csv"))                 # header of an older version: keep it, start a new table
    new = not p.exists()
    with open(p, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=NER_FIELDS, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerow(row)
    if cfg.predict_after:
        run_predict(cfg)


def load_run(cfg: NCfg, run_name: str, dev):
    """Rebuild a finished NER run (network, saved config, tokenizer, labels) from its best.pt."""
    from tokenizers import Tokenizer
    rd = Path(cfg.work_dir) / "ner_runs" / run_name
    ck = torch.load(rd / "best.pt", map_location="cpu", weights_only=False)
    saved = NCfg()
    apply_overrides(saved, {k: v for k, v in ck["cfg"].items() if k in {f.name for f in fields(NCfg)}})
    saved.work_dir = cfg.work_dir
    if "use_feats" not in ck["cfg"]:
        saved.use_feats = False                  # run trained with an older version of this script (no feature embeddings)
    if cfg.local_data_dir:
        saved.local_data_dir = cfg.local_data_dir
    C = 1 + 2 * len(ck["labels"])
    net = build_net(saved, P1.LM(ck["mc"]), C)
    apply_method(net, saved)                     # re-create the LoRA / LoRMA adapters before loading their weights
    net.load_state_dict(ck["net"])
    net.to(dev).eval()
    return {"rd": rd, "ck": ck, "saved": saved, "net": net, "labels": ck["labels"], "C": C,
            "tok": Tokenizer.from_file(str(rd / "tokenizer.json"))}


def write_submission(cfg: NCfg, rd: Path, spans: dict, texts: dict, labels, end_shift: int):
    """spans: doc id -> [(start, end_exclusive, label_idx)] in ORIGINAL characters -> prediction.json + submission.zip"""
    out = {}
    for did in texts:
        seen, lst = set(), []
        for s_, e_, l_ in sorted(spans.get(did, [])):
            t = (s_, e_ - end_shift, labels[l_])
            if t not in seen and t[1] > t[0] - (1 if end_shift else 0):
                seen.add(t)
                lst.append([t[0], t[1], t[2]])
        out[did] = lst
    problems = validate_prediction(out, texts, labels, end_shift)
    if problems:
        log(f"[predict] WARNING: {problems[:5]}")
    rd.mkdir(parents=True, exist_ok=True)
    pj = rd / "prediction.json"
    P1.jdump(out, pj)
    zp = rd / "submission.zip"
    with zipfile.ZipFile(zp, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    hist = Path(cfg.work_dir) / "ner_submissions"
    hist.mkdir(parents=True, exist_ok=True)
    shutil.copy(zp, hist / f"{cfg.run}_{time.strftime('%m%d_%H%M%S')}.zip")
    n_ent = sum(len(v) for v in out.values())
    log(f"[predict] wrote {pj} ({n_ent} entities, {n_ent / max(1, len(out)):.1f} per doc) and {zp}")
    return str(zp)


def run_predict(cfg: NCfg):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    _, amp = P1.amp_setup(dev)
    r = load_run(cfg, cfg.run, dev)
    ck, labels = r["ck"], r["labels"]
    mode, o_bias, end_shift = ck["decode"], float(ck.get("o_bias", 0.0)), ck["end_shift"]
    cbias = np.array(ck["cbias"], dtype=np.float32) if ck.get("cbias") else None
    texts = load_test(cfg)
    log(f"[predict] {len(texts)} test documents, decoder={mode}, O-bias={o_bias:+.1f}")
    docs = make_docs(r["tok"], texts, None, {l: i for i, l in enumerate(labels)}, end_shift, {})
    trans, start = build_trans(len(labels))
    spans = predict_docs(r["net"], docs, r["saved"], dev, amp, r["C"], mode, trans, start, o_bias, cbias)
    return write_submission(cfg, r["rd"], spans, texts, labels, end_shift)


def _log_csv_row(cfg: NCfg, row: dict):
    p = Path(cfg.work_dir) / "ner_experiments.csv"
    if p.exists() and open(p).readline().strip().split(",") != NER_FIELDS:
        p.rename(p.with_suffix(".old.csv"))
    new = not p.exists()
    with open(p, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=NER_FIELDS, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerow({"time": time.strftime("%Y-%m-%d %H:%M:%S"), **row})


def run_recalibrate(cfg: NCfg):
    """No retraining: re-tune decoder + O-bias of a finished run on its validation documents, then re-predict."""
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    _, amp = P1.amp_setup(dev)
    r = load_run(cfg, cfg.run, dev)
    saved, ck, labels = r["saved"], r["ck"], r["labels"]
    saved.decode = cfg.decode
    _, _, end_shift, tr, va, stats = prepare_data(saved, r["tok"], None)
    trans, start = build_trans(len(labels))
    probs = doc_probs(r["net"], va, saved, dev, amp, r["C"])
    best, table = tune_decoding(va, probs, saved, labels, trans, start)
    sc, mode, o_bias = best["sc"], best["mode"], best["o_bias"]
    log(f"[recalibrate] {cfg.run}: CHOSEN {mode} O-bias {o_bias:+.1f} -> val macro-F1 {sc['macro_f1']:.4f} "
        f"(P {sc['macro_p']:.4f} R {sc['macro_r']:.4f})")
    for l in labels:
        pl = sc["per_label"][l]
        log(f"   {l:<10} P={pl['P']:.3f} R={pl['R']:.3f} F1={pl['F1']:.3f}  (tp {pl['tp']} fp {pl['fp']} fn {pl['fn']})")
    err = analyze_errors(eval_gold_sets(va), best["sets"])
    log(f"[recalibrate] error breakdown: {err}")
    P1.jdump(error_examples(va, eval_gold_sets(va), best["sets"]), r["rd"] / "error_examples.json")
    ck["decode"], ck["o_bias"] = mode, o_bias
    ck["cbias"] = None if best["cbias"] is None else [float(x) for x in best["cbias"]]
    torch.save(ck, r["rd"] / "best.pt")
    rep_path = r["rd"] / "val_report.json"
    rep = P1.jload(rep_path) if rep_path.exists() else {}
    rep.update({"decode": mode, "o_bias": o_bias, "cbias": ck["cbias"], "decode_table": table, "scores": sc, "errors": err})
    P1.jdump(rep, rep_path)
    _log_csv_row(cfg, {"run": cfg.run, "p1_arch": ck["mc"]["arch"], "method": saved.method, "variant": "recalibrated",
                       "val_macro_f1": round(sc["macro_f1"], 4), "val_macro_p": round(sc["macro_p"], 4),
                       "val_macro_r": round(sc["macro_r"], 4), "decode": f"{mode}/o{o_bias:+.1f}"})
    del r["net"]
    return run_predict(cfg)


def _ens_candidates(cfg: NCfg):
    out = []
    for f in sorted((Path(cfg.work_dir) / "ner_runs").glob("*/best.pt")):
        n = f.parent.name
        if cfg.ens_prefix:
            if not n.startswith(cfg.ens_prefix):
                continue
        elif n.startswith(("selftest", "check")):
            continue
        out.append(n)
    return out


def run_ensemble(cfg: NCfg):
    """Average the per-token class probabilities of several finished runs (they share the Phase-1 tokenizer and the
    validation split), tune the decoder on validation, write one submission.
      "ens_runs": [...]   explicit members            "ens_auto": true   greedy forward selection among ALL finished runs"""
    names = list(cfg.ens_runs) or (_ens_candidates(cfg) if cfg.ens_auto else [])
    if len(names) < 2:
        raise SystemExit('need >= 2 finished runs: set "ens_runs": [...] or "ens_auto": true')
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    _, amp = P1.amp_setup(dev)
    ref, vp, va = {}, {}, None
    for n in names:                                   # validation probabilities of every candidate (one network in memory at a time)
        r = load_run(cfg, n, dev)
        if not ref:
            ref = {"labels": r["labels"], "tok": r["tok"].to_str(), "C": r["C"], "end_shift": r["ck"]["end_shift"],
                   "split": (r["saved"].val_frac, r["saved"].split_seed)}
            _, _, _, tr, va, _ = prepare_data(r["saved"], r["tok"], None)
        if (r["labels"] != ref["labels"] or r["tok"].to_str() != ref["tok"]
                or (r["saved"].val_frac, r["saved"].split_seed) != ref["split"]):
            log(f"[ensemble] skipping {n}: different labels / tokenizer / validation split")
        else:
            vp[n] = doc_probs(r["net"], va, r["saved"], dev, amp, ref["C"])
        del r
        gc.collect()
        if dev == "cuda":
            torch.cuda.empty_cache()
    labels, C, end_shift = ref["labels"], ref["C"], ref["end_shift"]
    trans, start = build_trans(len(labels))
    gold = eval_gold_sets(va)

    def mean(keys):
        return [sum(vp[k][i] for k in keys) / len(keys) for i in range(len(va))]

    def quick(plist):                                 # tuned greedy macro-F1 (cheap)
        top = 0.0
        for b_ in (-0.5, 0.0, 0.5, 1.0):
            pr = {d.id: decode_doc(d, p, "greedy", trans, start, b_) for d, p in zip(va, plist)}
            top = max(top, strict_scores(gold, to_sets(pr, labels), labels)["macro_f1"])
        return top

    single = {k: quick(vp[k]) for k in vp}
    for k, v in sorted(single.items(), key=lambda kv: -kv[1]):
        log(f"[ensemble] single model {k:<28} val macro-F1 {v:.4f}")
    if cfg.ens_runs or not cfg.ens_auto:
        chosen = list(vp)
    else:
        order = sorted(single, key=lambda k: -single[k])
        chosen, cur = [order[0]], single[order[0]]
        for k in order[1:]:
            if len(chosen) >= cfg.ens_max:
                break
            f = quick(mean(chosen + [k]))
            log(f"[ensemble] try adding {k:<28} -> {f:.4f} (current {cur:.4f})")
            if f > cur + 0.002:
                chosen.append(k)
                cur = f
    log(f"[ensemble] members: {chosen}")
    best, table = tune_decoding(va, mean(chosen), cfg, labels, trans, start)
    sc, mode, o_bias, cbias = best["sc"], best["mode"], best["o_bias"], best["cbias"]
    log(f"[ensemble] val macro-F1 {sc['macro_f1']:.4f} (P {sc['macro_p']:.4f} R {sc['macro_r']:.4f}) with {mode}, "
        f"O-bias {o_bias:+.1f}{', per-class thresholds' if cbias is not None else ''}")
    runs = [load_run(cfg, n, dev) for n in chosen]    # now the test set, with only the chosen networks
    texts = load_test(cfg)
    docs = make_docs(runs[0]["tok"], texts, None, {l: i for i, l in enumerate(labels)}, end_shift, {})
    spans = {}
    for g in range(0, len(docs), 16):
        sub = docs[g:g + 16]
        probs = [doc_probs(r["net"], sub, r["saved"], dev, amp, C) for r in runs]
        for i, d in enumerate(sub):
            spans[d.id] = decode_doc(d, sum(p[i] for p in probs) / len(runs), mode, trans, start, o_bias, cbias)
    rd = run_dir(cfg)
    rd.mkdir(parents=True, exist_ok=True)
    P1.jdump({"members": chosen, "single_model_f1": single, "decode": mode, "o_bias": o_bias,
              "cbias": None if cbias is None else [float(x) for x in cbias], "scores": sc, "decode_table": table},
             rd / "val_report.json")
    _log_csv_row(cfg, {"run": cfg.run, "p1_arch": "ensemble", "method": "ensemble", "val_macro_f1": round(sc["macro_f1"], 4),
                       "val_macro_p": round(sc["macro_p"], 4), "val_macro_r": round(sc["macro_r"], 4),
                       "decode": f"{mode}/o{o_bias:+.1f}", "teacher_run": "+".join(chosen)})
    return write_submission(cfg, rd, spans, texts, labels, end_shift)


def validate_prediction(pred, texts, labels, end_shift=0):
    """Same checks as evaluate_phase2.py (+ bounds against the document text)."""
    bad, allowed = [], set(labels)
    if set(pred) != set(texts):
        bad.append("id mismatch")
    for did, lst in pred.items():
        seen = set()
        for j, s in enumerate(lst):
            if not (isinstance(s, list) and len(s) in (3, 4)):
                bad.append(f"{did}[{j}] bad shape")
                continue
            a, b, l = s[:3]
            if not (isinstance(a, int) and isinstance(b, int)) or a < 0 or b <= a:
                bad.append(f"{did}[{j}] bad offsets {a},{b}")
            if b > len(texts[did]):
                bad.append(f"{did}[{j}] beyond text")
            if l not in allowed:
                bad.append(f"{did}[{j}] bad label {l}")
            if (a, b, l) in seen:
                bad.append(f"{did}[{j}] duplicate")
            seen.add((a, b, l))
    return bad


# ============================================================================================
# 8. Self-test (synthetic data, tiny models, all 4 Phase-1 architectures)
# ============================================================================================
def _synthetic(tmp, rng):
    first = ["Ram", "Shyam", "Anita", "Suresh", "Meena", "Kiran", "Vijay", "Latha", "Arun", "Geeta"]
    last = ["Kumar", "Sharma", "Reddy", "Singh", "Iyer", "Das", "Patel", "Nair"]
    courts = ["SUPREME COURT OF INDIA", "High Court of Kerala", "High Court of Delhi"]
    months = ["January", "March", "June", "October"]
    filler = "the learned counsel submitted that the impugned order was passed without hearing and hence liable to be set aside".split()

    def doc():
        text, ents = "", []

        def add(s, label=None):
            nonlocal text
            if label:
                ents.append([len(text), len(text) + len(s), label, s])
            text += s
        add("IN THE ")
        add(rng.choice(courts), "COURT")
        add("\n\n  CIVIL APPEAL NO. ")
        add(f"{rng.randint(100, 999)} OF {rng.randint(1990, 2020)}", "CASENO")
        add("\n\n")
        add(f"{rng.choice(first)} {rng.choice(last)}", "APP")
        add("   ...Appellant\n  Versus \n")
        add(f"State of {rng.choice(last)}", "RESP")
        add("  ...Respondent\n\nJudgment by ")
        add(f"{rng.choice(first)[0]}. {rng.choice(last).upper()}", "JUDGE")
        add(", J. on ")
        add(f"{rng.randint(1, 28)} {rng.choice(months)} {rng.randint(1990, 2020)}", "DATE")
        add(".\n\n")
        for _ in range(rng.randint(3, 12)):
            add(" ".join(rng.choice(filler) for _ in range(rng.randint(8, 20))) + ".\n")
        if rng.random() < 0.5:
            add("Counsel ")
            add(f"{rng.choice(first)} {rng.choice(last)}", "A.COUNSEL")
            add(" appeared for the appellant.\n")
        return text, ents

    data = {}
    for k in (1, 2):
        texts, gold = {}, {}
        for i in range(20):
            did = f"{k}{i:03d}"
            texts[did], gold[did] = doc()
        data[k] = (texts, gold)
    test = {f"t{i}": doc()[0] for i in range(5)}
    os.makedirs(tmp, exist_ok=True)
    for k, (t, g) in data.items():
        json.dump(t, open(os.path.join(tmp, f"fold{k}-text.json"), "w"))
        json.dump(g, open(os.path.join(tmp, f"fold{k}-gold.json"), "w"))
    json.dump(test, open(os.path.join(tmp, "fold3-text.json"), "w"))
    open(os.path.join(tmp, "labels.txt"), "w").write("\n".join(LABELS_DEFAULT) + "\n")
    return data, test


def _tiny_mc(arch, V):
    return {**P1.ARCH_PRESETS[arch], "n_embd": 64, "n_layer": 2, "n_head": 4, "n_kv_head": 2, "ffn_hidden": 96,
            "n_experts": 4, "top_k": 2, "arch": arch, "ctx": 128, "vocab": V, "dropout": 0.0}


def selftest() -> bool:
    import tempfile
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
    ok = True
    rng = random.Random(0)
    tmp = tempfile.mkdtemp(prefix="cs781_ner_selftest_")
    dd = os.path.join(tmp, "data")
    data, test = _synthetic(dd, rng)
    log("=== 1. text normalisation / offset map ===")
    for s in ["a  b\n\nc", "  lead trail  ", "x\u00a0y\tz", ""]:
        n, m = normalize_with_map(s)
        good = all(s[m[i]] == n[i] or (n[i] == " " and s[m[i]].isspace()) for i in range(len(n)))
        ok &= good
        log(f"  {s!r:<22} -> {n!r:<16} {'PASS' if good else 'FAIL'}")

    log("=== 2. tokeniser + oracle (gold -> BIO -> char offsets must reproduce gold exactly) ===")
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    corpus = [normalize_with_map(t)[0] for k in data for t in data[k][0].values()]
    tok.train_from_iterator(corpus, trainers.BpeTrainer(
        vocab_size=500, special_tokens=["<|endoftext|>"], initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        show_progress=False))
    texts = {**data[1][0], **data[2][0]}
    gold = {**data[1][1], **data[2][1]}
    label2i = {l: i for i, l in enumerate(LABELS_DEFAULT)}
    shift, info = detect_end_shift(texts, gold)
    stats = {}
    docs = make_docs(tok, texts, {d: parse_gold_items(g) for d, g in gold.items()}, label2i, shift, stats)
    o = oracle(docs, LABELS_DEFAULT)
    good = shift == 0 and present_f1(o) > 0.999
    ok &= good
    log(f"  end_shift={shift} {info}\n  oracle strict F1 (labels present) = {present_f1(o):.4f} stats={stats}  "
        f"{'PASS' if good else 'FAIL'}")

    log("=== 2b. layout features (line breaks) and position buckets ===")
    n_, m_, nlb_, nla_ = normalize_with_feats("AAA BBB\n\nCCC DDD\nEEE")
    good = (n_ == "AAA BBB CCC DDD EEE" and [int(nlb_[i]) for i in (0, 4, 8, 12, 16)] == [2, 0, 2, 0, 1]
            and [int(nla_[i]) for i in (2, 6, 10, 14, 18)] == [0, 2, 0, 1, 2])
    good2 = any(int(d.feat.max()) > 0 for d in docs) and int(np.searchsorted(POS_EDGES, 63, side="right")) == 0 \
        and int(np.searchsorted(POS_EDGES, 64, side="right")) == 1 and int(np.searchsorted(POS_EDGES, 10 ** 6, side="right")) == 8
    ok &= good and good2
    log(f"  newline features {'PASS' if good else 'FAIL'} | per-token features / position buckets {'PASS' if good2 else 'FAIL'}")

    log("=== 3. viterbi / windows ===")
    trans, start = build_trans(2)
    lp = np.log(np.array([[.1, .1, .6, .1, .1], [.6, .1, .1, .1, .1], [.1, .1, .1, .1, .6]]))
    path = viterbi(lp, trans, start)
    good = path[0] not in (2, 4) and path[2] not in (2, 4) or path[1] in (1, 2, 3, 4)   # I-x never starts a span
    good = bool(path[0] not in (2, 4))
    ok &= good
    wins = make_windows(1000, 512, 384)
    good2 = wins[0][0] == 0 and wins[-1][1] == 1000 and all(b - a <= 512 for a, b in wins)
    ok &= good2
    log(f"  viterbi start constraint {'PASS' if good else 'FAIL'} | windows {wins} {'PASS' if good2 else 'FAIL'}")

    V = P1.pad64(tok.get_vocab_size())
    log("=== 3b. adapters are the IDENTITY at initialisation; only adapters + head are trainable; KD helpers ===")
    for arch in P1.ARCH_PRESETS:
        for meth, var in (("lora", "plus"), ("lorma", "plus"), ("lorma", "pi")):
            tag = f"{arch}/{meth}" + (f"-{var}" if meth == "lorma" else "")
            try:
                torch.manual_seed(0)
                c = apply_overrides(NCfg(), {"method": meth, "lorma_variant": var, "peft_r": 4, "peft_alpha": 8.0,
                                             "peft_dropout": 0.0})
                net = build_net(c, P1.LM(_tiny_mc(arch, V)), 25).eval()
                ids = torch.randint(1, V, (2, 16))
                km = torch.ones((2, 16), dtype=torch.bool)
                with torch.no_grad():
                    ref = net(ids, km)
                apply_method(net, c)
                net.eval()
                with torch.no_grad():
                    out = net(ids, km)
                diff = (ref - out).abs().max().item()
                n_tr = sum(p.numel() for p in net.parameters() if p.requires_grad)
                n_all = sum(p.numel() for p in net.parameters())
                net.train()
                net(ids, km).float().pow(2).mean().backward()
                grads = any(p.grad is not None and float(p.grad.abs().sum()) > 0 for p in net.parameters() if p.requires_grad)
                frozen_ok = all(p.grad is None for p in net.parameters() if not p.requires_grad)
                good = diff < 1e-4 and n_tr < n_all and grads and frozen_ok
                ok &= good
                log(f"  {tag:<18} |out_adapted - out_base| = {diff:.1e}  trainable {n_tr:,}/{n_all:,}  "
                    f"{'PASS' if good else 'FAIL'}")
            except Exception as e:
                import traceback
                traceback.print_exc()
                log(f"  {tag:<18} FAIL: {type(e).__name__}: {e}")
                ok = False
    sd = {"wte.weight": 0, "blocks.0.a": 1, "blocks.1.a": 2, "blocks.2.a": 3, "norm_f.weight": 4}
    good = remap_student_keys(sd, [0, 2]) == {"wte.weight": 0, "blocks.0.a": 1, "blocks.1.a": 3, "norm_f.weight": 4}
    tl = torch.randn(2, 5, 25)
    lb = torch.randint(0, 25, (2, 5))
    lb[0, 0] = -100
    good2 = abs(float(kd_loss(tl, tl, lb, 2.0))) < 1e-5 and float(kd_loss(tl, torch.randn(2, 5, 25), lb, 2.0)) > 0
    ok &= good and good2
    log(f"  layer remap {'PASS' if good else 'FAIL'} | kd_loss (0 for identical logits, > 0 otherwise) {'PASS' if good2 else 'FAIL'}")

    log("=== 4. end-to-end fine-tuning on synthetic data, all 4 Phase-1 architectures ===")
    p1dirs = {}
    for arch in P1.ARCH_PRESETS:
        try:
            mc = _tiny_mc(arch, V)
            p1 = os.path.join(tmp, f"p1_{arch}")
            os.makedirs(p1, exist_ok=True)
            p1dirs[arch] = p1
            torch.save({"model": P1.LM(mc).state_dict(), "mc": mc, "state": {"step": 0}}, os.path.join(p1, "ckpt_best.pt"))
            tok.save(os.path.join(p1, "tokenizer.json"))
            cfg = apply_overrides(NCfg(), {"run": f"selftest_{arch}", "work_dir": os.path.join(tmp, "work"), "p1_dir": p1,
                                           "local_data_dir": dd, "max_len": 96, "stride": 64, "epochs": 2,
                                           "batch_size": 8, "eval_batch": 8, "lr": 1e-3, "val_frac": 0.2})
            run_train(cfg)
            with zipfile.ZipFile(run_dir(cfg) / "submission.zip") as z:
                assert z.namelist() == ["prediction.json"]
                pr = json.loads(z.read("prediction.json"))
            assert set(pr) == set(test), "ids"
            assert not validate_prediction(pr, test, LABELS_DEFAULT), "format"
            log(f"  e2e {arch:<7} PASS")
        except Exception as e:
            import traceback
            traceback.print_exc()
            log(f"  e2e {arch:<7} FAIL: {type(e).__name__}: {e}")
            ok = False
    log("=== 5. every adaptation method end-to-end (LoRA, LoRMA+, LoRMAπ, distillation) ===")
    base_ov = {"work_dir": os.path.join(tmp, "work"), "local_data_dir": dd, "max_len": 96, "stride": 64, "epochs": 2,
               "batch_size": 8, "eval_batch": 8, "lr": 1e-3, "peft_lr": 3e-3, "val_frac": 0.2, "peft_r": 4,
               "peft_alpha": 8.0}
    cases = [("lora", "llama", {"method": "lora"}),
             ("lorma_plus", "gpt2", {"method": "lorma", "lorma_variant": "plus"}),
             ("lorma_pi", "retnet", {"method": "lorma", "lorma_variant": "pi"}),
             ("kd", "gpt2", {"method": "kd", "teacher_run": "selftest_llama", "student_layers": 1})]
    for tag, arch, extra in cases:
        try:
            cfg = apply_overrides(NCfg(), {**base_ov, **extra, "run": f"selftest_{tag}", "p1_dir": p1dirs[arch]})
            run_train(cfg)
            with zipfile.ZipFile(run_dir(cfg) / "submission.zip") as z:
                assert z.namelist() == ["prediction.json"]
                pr = json.loads(z.read("prediction.json"))
            assert set(pr) == set(test) and not validate_prediction(pr, test, LABELS_DEFAULT), "format"
            rep_ = P1.jload(run_dir(cfg) / "val_report.json")
            if extra["method"] in ("lora", "lorma"):
                assert rep_["params_trainable"] < rep_["params_total"], "adapters must be a small subset"
            if extra["method"] == "kd":
                assert rep_["params_total"] < rep_["teacher_params"], "student must be smaller than the teacher"
            log(f"  e2e {tag:<11} PASS  (trainable {rep_['params_trainable']:,} / total {rep_['params_total']:,})")
        except Exception as e:
            import traceback
            traceback.print_exc()
            log(f"  e2e {tag:<11} FAIL: {type(e).__name__}: {e}")
            ok = False
    log("=== 6. recalibrate (no retraining) + ensemble ===")
    try:
        cfg = apply_overrides(NCfg(), {"work_dir": os.path.join(tmp, "work"), "local_data_dir": dd, "run": "selftest_llama"})
        zp = run_recalibrate(cfg)
        with zipfile.ZipFile(zp) as z:
            assert z.namelist() == ["prediction.json"]
            assert not validate_prediction(json.loads(z.read("prediction.json")), test, LABELS_DEFAULT)
        cfg = apply_overrides(NCfg(), {"work_dir": os.path.join(tmp, "work"), "local_data_dir": dd, "run": "selftest_ens",
                                       "ens_runs": ["selftest_llama", "selftest_gpt2"]})
        zp = run_ensemble(cfg)
        with zipfile.ZipFile(zp) as z:
            pr = json.loads(z.read("prediction.json"))
        assert set(pr) == set(test) and not validate_prediction(pr, test, LABELS_DEFAULT)
        cfg = apply_overrides(NCfg(), {"work_dir": os.path.join(tmp, "work"), "local_data_dir": dd, "run": "selftest_ens_auto",
                                       "ens_auto": True, "ens_prefix": "selftest_"})
        zp = run_ensemble(cfg)
        with zipfile.ZipFile(zp) as z:
            pr = json.loads(z.read("prediction.json"))
        assert set(pr) == set(test) and not validate_prediction(pr, test, LABELS_DEFAULT)
        log("  recalibrate + explicit ensemble + automatic ensemble PASS")
    except Exception as e:
        import traceback
        traceback.print_exc()
        log(f"  recalibrate / ensemble FAIL: {type(e).__name__}: {e}")
        ok = False
    shutil.rmtree(tmp, ignore_errors=True)
    log("SELFTEST " + ("PASSED" if ok else "FAILED"))
    return ok


# ============================================================================================
# 9. CLI
# ============================================================================================
def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("stage", choices=["selftest", "check", "train", "predict", "recalibrate", "ensemble"])
    ap.add_argument("--run", default=None)
    ap.add_argument("--cfg", default="", help="JSON string or JSON file with config overrides")
    ap.add_argument("--p1_dir", default=None, help="Phase-1 run folder (ckpt_best.pt + tokenizer.json)")
    args = ap.parse_args(argv)
    if args.stage == "selftest":
        return 0 if selftest() else 1
    cfg = NCfg()
    ov = {}
    if args.cfg:
        ov = json.loads(args.cfg) if args.cfg.strip().startswith("{") else P1.jload(args.cfg)
    if args.run:
        ov["run"] = args.run
    if args.p1_dir:
        ov["p1_dir"] = args.p1_dir
    apply_overrides(cfg, ov)
    Path(cfg.work_dir).mkdir(parents=True, exist_ok=True)
    if args.stage == "check":
        diagnose(cfg)
    elif args.stage == "train":
        run_train(cfg)
    elif args.stage == "predict":
        run_predict(cfg)
    elif args.stage == "recalibrate":
        run_recalibrate(cfg)
    elif args.stage == "ensemble":
        run_ensemble(cfg)
    return 0


if __name__ == "__main__":
    sys.exit(main())

## Self-test (once). Must end with `SELFTEST PASSED`. It checks the offset logic, that LoRA/LoRMA adapters start as the identity, and runs every method end to end on tiny synthetic data.

In [ ]:
!python cs781_phase2_ner.py selftest

## Setup: locate the four Phase-1 backbones

In [ ]:
import os, json, glob, subprocess
WORK = "/kaggle/working/cs781" if os.path.exists("/kaggle/working") else "./cs781_work"
P1_DIRS = {a: f"{WORK}/runs/explore_{a}" for a in ["llama", "gpt2", "retnet", "moe"]}
print("runs folder:", sorted(os.listdir(f"{WORK}/runs")) if os.path.exists(f"{WORK}/runs") else "DOES NOT EXIST")
P1_CKPT, missing = {}, []
for a, d in P1_DIRS.items():
    if not os.path.exists(f"{d}/tokenizer.json"): missing.append(a)
    elif os.path.exists(f"{d}/ckpt_best.pt"): P1_CKPT[a] = "ckpt_best.pt"
    elif os.path.exists(f"{d}/ckpt_last.pt"): P1_CKPT[a] = "ckpt_last.pt"
    else: missing.append(a)
for a, d in P1_DIRS.items():
    print(f"{a:<7}", f"OK ({P1_CKPT[a]})" if a in P1_CKPT else "MISSING", d)
if missing:
    print("\nCandidates on this machine:")
    for f in glob.glob("/kaggle/**/ckpt_best.pt", recursive=True) + glob.glob("./**/ckpt_best.pt", recursive=True): print("  ", os.path.dirname(f))
    print("-> fix P1_DIRS, or train the missing backbone(s) with the next cell")

BASE = {"max_len": 512, "stride": 384}

def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print(line, end="")
    p.wait(); return p.returncode

def run_exp(run, arch, extra):
    if os.path.exists(f"{WORK}/ner_runs/{run}/submission.zip"):
        print("already done:", run, "(delete that folder to re-run)"); return
    if arch not in P1_CKPT:
        print(f"Phase-1 backbone '{arch}' is missing -> train it first, then re-run this setup cell."); return
    cfg = {"p1_dir": P1_DIRS[arch], "p1_ckpt": P1_CKPT[arch], **BASE, **extra}
    fn = f"cfg_ner_{run}.json"; json.dump(cfg, open(fn, "w"), indent=1)
    print("=" * 100, "\n", run, "|", arch, "|", extra, "\n", "=" * 100)
    print("exit code", sh(f"python cs781_phase2_ner.py train --run {run} --cfg {fn}"))

def report(run, n_examples=8):
    p = f"{WORK}/ner_runs/{run}/val_report.json"
    if not os.path.exists(p): print(f"\n{run}: no val_report.json (not finished)"); return
    r = json.load(open(p)); sc = r["scores"]
    print(f"\n=== {run} | {r.get('p1_arch', 'ensemble')} / {r.get('method', 'ensemble')} | best epoch {r.get('best_epoch')} | decode {r.get('decode')} o_bias {r.get('o_bias', 0)}")
    print(f"validation strict macro-F1 {sc['macro_f1']:.3f}   P {sc['macro_p']:.3f}   R {sc['macro_r']:.3f}")
    for l, v in sc["per_label"].items():
        print(f"   {l:<10} P={v['P']:.2f} R={v['R']:.2f} F1={v['F1']:.2f}   tp={v['tp']} fp={v['fp']} fn={v['fn']}")
    print("   errors:", r.get("errors"))
    ep = f"{WORK}/ner_runs/{run}/error_examples.json"
    if os.path.exists(ep):
        for e in json.load(open(ep))[:n_examples]: print("   ", e["type"], "| pred:", e["pred"], "| gold:", e["gold"])

## (Only if a backbone is MISSING) train it now - about 0.75 h each on a T4
Uses the same Phase-1 script and the already prepared data (so no new preparation). A partially finished run resumes from its last checkpoint.
After it finishes, **re-run the setup cell above** so the paths are refreshed.

In [ ]:
for a in missing:
    fn = f"cfg_p1_{a}.json"; json.dump({"arch": a, "time_budget_hours": 0.75}, open(fn, "w"))
    print("training Phase-1 backbone:", a); sh(f"python cs781_phase1.py train --run explore_{a} --preset turbo --cfg {fn}")
if not missing: print("nothing to train - all four backbones exist")

## Check the real data (1-2 min) - do this before the long runs
Prints the gold format, whether end offsets are exclusive/inclusive, how many gold spans fall inside BPE tokens, and the **oracle strict F1**
(the ceiling of tokenisation + offset mapping). If it is clearly below ~0.97, send the printed stats before continuing.

In [ ]:
sh("python cs781_phase2_ner.py check --run check_llama --cfg " + json.dumps(json.dumps({"p1_dir": P1_DIRS["llama"], "p1_ckpt": P1_CKPT.get("llama", "ckpt_best.pt"), **BASE})))

## E1 - full fine-tuning (llama): all weights, bidirectional attention, layout features, layer-wise LR decay, early stopping

In [ ]:
run_exp("final_full_llama", "llama", {"method": "full", "epochs": 12, "lr": 3e-4})
report("final_full_llama")

## E2 - LoRMA+ (gpt2): multiplicative low-rank adaptation
`h = (alpha/r * B A + I) W0 x` (additive rank inflation, B initialised to 0 so training starts exactly at the Phase-1 model). Backbone frozen; only the adapters and the NER head train.
Use `"lorma_variant": "pi"` for the permutation-inflated variant LoRMAπ.

In [ ]:
ADAPT = {"bidir": "false", "head": "bigru", "head_hidden": 384, "head_layers": 2, "peft_r": 32, "peft_alpha": 64, "epochs": 16, "peft_lr": 1e-3}
run_exp("final_lorma_gpt2", "gpt2", {"method": "lorma", "lorma_variant": "plus", **ADAPT})
report("final_lorma_gpt2")

## E3 - LoRA (moe): additive low-rank adaptation of the attention projections (experts stay frozen)

In [ ]:
run_exp("final_lora_moe", "moe", {"method": "lora", **ADAPT})
report("final_lora_moe")

## E4 - knowledge distillation (retnet student, E1 llama teacher)
The student is the Phase-1 retnet **pruned to 6 evenly spaced blocks of 12** (initialised from the Phase-1 weights) and trained with
`0.5 * CE(hard labels) + 0.5 * T^2 * KL(teacher || student)` (T = 2). Needs E1 to be finished (same labels and tokenizer).

In [ ]:
run_exp("final_kd_retnet", "retnet", {"method": "kd", "teacher_run": "final_full_llama", "student_layers": 6, "epochs": 12, "lr": 3e-4})
report("final_kd_retnet")

## Results, efficiency comparison and the four files to submit
Picks, for each Phase-1 architecture, the run with the best validation strict macro-F1, and copies the best four (one per architecture) to `phase2_submissions/`.

In [ ]:
import pandas as pd, shutil
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
df = pd.read_csv(f"{WORK}/ner_experiments.csv")
df = df[~df.run.str.startswith(("selftest", "check"))]
show = ["run", "p1_arch", "method", "variant", "params_total_M", "params_trainable_M", "val_macro_f1", "val_macro_p", "val_macro_r",
        "decode", "train_s", "peak_gpu_mb", "infer_tok_s"]
display(df[show].sort_values("val_macro_f1", ascending=False))
single = df[df.p1_arch != "ensemble"].sort_values("val_macro_f1", ascending=False)
best4 = single.groupby("p1_arch").head(1).head(4)
os.makedirs("phase2_submissions", exist_ok=True)
for _, r in best4.iterrows():
    src = f"{WORK}/ner_runs/{r.run}/submission.zip"
    if os.path.exists(src): shutil.copy(src, f"phase2_submissions/{r.run}.zip"); print("submit:", r.run, "|", r.p1_arch, r.method, "| val F1", r.val_macro_f1)
ens = f"{WORK}/ner_runs/final_ensemble/submission.zip"
if os.path.exists(ens): shutil.copy(ens, "phase2_submissions/final_ensemble.zip"); print("extra: final_ensemble.zip")

# CS781 Phase 3 - Rhetorical Role Prediction (CL + IT)

**Append these cells at the end of your existing notebook** (same session, so `/kaggle/working/cs781/runs/explore_*` exist), or run standalone with your Phase-1 output attached.

One model per run handles **both domains** (the official scorer pools CL + IT); every submission is a single `prediction.json` with one label per sentence for all CL and IT test documents.

| # | Run | Phase-1 backbone | Technique (from the task's list) |
|---|---|---|---|
| R1 | `rr_lorma_gpt2` | gpt2 | **LoRMA+** (compulsory) |
| R2 | `rr_rslora_llama` | llama | **rsLoRA** |
| R3 | `rr_ia3_moe` | moe | **IA3** |
| R4 | `rr_kd_retnet` | retnet | **knowledge distillation** (compulsory): pruned student <- teacher |
| (teacher) | `rr_full_llama` | llama | full fine-tuning - the teacher of R4, also your strongest single model |

Extras: sentence-only vs. document-context ablation, LoRA / LoRMAπ, ensemble. Everything is resumable: a finished run is skipped when you re-run its cell.

In [ ]:
import base64
open("cs781_phase1.py", "w", encoding="utf-8").write(base64.b64decode("...").decode("utf-8"))
open("cs781_phase2_ner.py", "w", encoding="utf-8").write(base64.b64decode("...").decode("utf-8"))
print("wrote cs781_phase1.py and cs781_phase2_ner.py")

In [ ]:
%%writefile cs781_phase3_rr.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
CS781 Phase 3 - Rhetorical Role prediction (CL + IT) on top of YOUR Phase-1 language models.

Needs cs781_phase1.py and cs781_phase2_ner.py in the same folder (model classes, LoRA / LoRMA adapters, distillation helpers).

Model
  document -> sentences -> Phase-1 BPE tokens   [sent_1 <eos> sent_2 <eos> ...]   packed into windows of <= max_len tokens
  -> Phase-1 backbone (bidirectional attention, or causal for retnet / adapter runs)
  -> one vector per sentence = [mean of its token states ; state of its <eos> marker]
  -> + relative / absolute position-in-document + domain (CL / IT) embeddings
  -> BiGRU over the sentences of the window (document context)  -> linear -> 13 rhetorical roles
  ctx_mode "docseq" (recommended): the BiGRU runs over the sentences of the WHOLE document (training on crops of consecutive windows,
  inference on the full document), so every sentence sees the structure of the entire judgment. "doc" = overlapping windows,
  "sentence" = every sentence alone (ablation).
  Decoding is calibrated on the dev set: per-class thresholds (macro-F1) and a label-transition prior (Viterbi).
  Training uses weight averaging (EMA). Exactly one label per sentence, in the original order.

Adaptation methods (cfg "method")
  full    all weights                       lora    additive low-rank (W0 + s*BA)            rslora  LoRA with scale alpha/sqrt(r)
  lorma   multiplicative low-rank (LoRMA+ / LoRMAπ)   ia3   learned scaling vectors on keys, values and FFN activations
  kd      knowledge distillation: pruned student <- fine-tuned teacher (logits)               partial freeze embeddings + first blocks

Stages:  selftest | check | train | predict | ensemble
  python cs781_phase3_rr.py check --run rr_check --cfg '{"p1_dir": ".../explore_llama"}'
  python cs781_phase3_rr.py train --run rr_full_llama --cfg cfg_rr.json
"""
from __future__ import annotations

import argparse
import contextlib
import csv
import gc
import json
import math
import os
import random
import re
import shutil
import sys
import time
import zipfile
from dataclasses import dataclass, field, fields, asdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import cs781_phase1 as P1
import cs781_phase2_ner as N

log = P1.log
OFFICIAL_LABELS = ["Fact", "Issue", "ArgumentPetitioner", "ArgumentRespondent", "Statute", "Dissent", "PrecedentReliedUpon",
                   "PrecedentNotReliedUpon", "PrecedentOverruled", "RulingByLowerCourt", "RatioOfTheDecision",
                   "RulingByPresentCourt", "None"]
DOMAINS = ["CL", "IT"]


# ============================================================================================
# 0. Config
# ============================================================================================
@dataclass
class RCfg:
    run: str = "rr_full_v1"
    work_dir: str = P1.DEFAULT_WORK
    p1_dir: str = ""                  # Phase-1 run folder (ckpt_best.pt + tokenizer.json)
    p1_ckpt: str = "ckpt_best.pt"
    hf_repo: str = "Exploration-Lab/CS-781-Capstone"
    local_data_dir: str = ""          # folder with CL_train.json, CL_dev.json, IT_train.json, IT_dev.json, label_vocab.json, CL_test.json, IT_test.json
    train_prefix: str = "Phase_3_RR/Train"
    test_prefix: str = "Phase_3_RR/Test"
    # ---- context
    ctx_mode: str = "docseq"          # docseq = BiGRU over the whole document | doc = overlapping windows | sentence = every sentence alone
    crop_windows: int = 8             # docseq training: a crop = this many consecutive windows (~ 8 x max_len tokens)
    batch_docs: int = 2               # docseq training: crops per step
    max_len: int = 512                # tokens per window (<= Phase-1 context)
    max_sent_tokens: int = 96         # longer sentences are truncated
    win_overlap: float = 0.5          # fraction of sentences shared by consecutive windows
    # ---- method
    method: str = "full"              # full | partial | lora | rslora | lorma | ia3 | kd
    freeze_layers: int = 0
    peft_r: int = 16
    peft_alpha: float = 32.0
    peft_dropout: float = 0.05
    peft_lr: float = 5e-4
    peft_targets: str = "all"         # attn | all
    peft_experts: bool = False
    lorma_variant: str = "plus"       # plus | pi
    teacher_run: str = ""
    student_layers: int = 0           # 0 = half of the Phase-1 blocks
    kd_alpha: float = 0.5
    kd_temp: float = 2.0
    # ---- architecture of the sentence head
    bidir: str = "auto"               # auto = bidirectional attention for gpt2/llama/moe, causal for retnet
    head: str = "bigru"               # bigru (sequence model over sentences) | linear
    sent_dim: int = 384
    head_hidden: int = 256
    head_layers: int = 1
    head_dropout: float = 0.1
    # ---- optimisation
    epochs: int = 12
    patience: int = 6
    batch_size: int = 8
    eval_batch: int = 16
    lr: float = 3e-4
    llrd: float = 0.85
    head_lr: float = 1e-3
    weight_decay: float = 0.01
    warmup_frac: float = 0.06
    grad_clip: float = 1.0
    class_weight: str = "sqrt"        # none | sqrt
    label_smoothing: float = 0.0
    class_bias: bool = True           # per-class thresholds tuned on dev for macro-F1
    trans_decode: bool = True         # Viterbi with a label-transition prior estimated on the training documents (weight tuned on dev)
    ema_decay: float = 0.998          # exponential moving average of the trainable weights (0 = off); the EMA model is evaluated and saved
    seed: int = 1337
    predict_after: bool = True
    # ---- ensemble stage
    ens_runs: list = field(default_factory=list)
    ens_auto: bool = False
    ens_prefix: str = ""
    ens_max: int = 6


def apply_overrides(cfg: RCfg, ov: dict) -> RCfg:
    names = {f.name for f in fields(RCfg)}
    for k, v in ov.items():
        if k not in names:
            raise SystemExit(f"Unknown config key {k!r}. Valid: {sorted(names)}")
        setattr(cfg, k, v)
    return cfg


def run_dir3(cfg: RCfg) -> Path:
    return Path(cfg.work_dir) / "rr_runs" / cfg.run


RR_FIELDS = ["time", "run", "p1_arch", "method", "variant", "ctx_mode", "head", "bidir", "params_total_M",
             "params_trainable_M", "dev_macro_f1", "dev_macro_p", "dev_macro_r", "dev_cl_f1", "dev_it_f1", "train_s",
             "peak_gpu_mb", "infer_sent_s", "teacher_run"]


def log_csv(cfg: RCfg, row: dict):
    p = Path(cfg.work_dir) / "rr_experiments.csv"
    if p.exists() and open(p).readline().strip().split(",") != RR_FIELDS:
        p.rename(p.with_suffix(".old.csv"))
    new = not p.exists()
    with open(p, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=RR_FIELDS, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerow({"time": time.strftime("%Y-%m-%d %H:%M:%S"), **row})


# ============================================================================================
# 1. Data: tolerant JSON parsing, tokenisation, windows
# ============================================================================================
def parse_vocab(raw):
    """Same accepted formats as the official evaluator: list, label->id, or id->label. Returns (labels, id->label)."""
    if isinstance(raw, list):
        labels = [str(x) for x in raw]
        return labels, {i: l for i, l in enumerate(labels)}
    if isinstance(raw, dict) and raw:
        vals = list(raw.values())
        if all(isinstance(v, int) and not isinstance(v, bool) for v in vals):
            return list(raw.keys()), {v: k for k, v in raw.items()}
        labels = [str(v) for v in vals]
        return labels, {(int(k) if str(k).isdigit() else k): str(v) for k, v in raw.items()}
    raise SystemExit("label_vocab.json must be a non-empty list or object")


def _first(d, keys):
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return None


def load_raw_docs(path, labels, id2):
    """-> list of {"id", "sents": [str], "labels": [str] | None}"""
    raw = P1.jload(path)
    if isinstance(raw, dict):
        items = [dict(v, id=k) if isinstance(v, dict) else {"id": k, "text": v} for k, v in raw.items()]
    else:
        items = raw
    lab_set, docs = set(labels), []
    for i, r in enumerate(items):
        did = _first(r, ("id", "doc_id", "document_id", "name"))
        did = str(did) if did is not None else str(i)
        text = _first(r, ("text", "sentences", "sents")) or []
        if isinstance(text, str):
            text = [text]
        sents = [s if isinstance(s, str) else str(_first(s, ("text", "sentence")) or "") for s in text]
        ls = _first(r, ("labels", "label", "roles"))
        if ls is not None:
            conv = []
            for x in ls:
                x = "None" if x is None else (id2.get(x, x) if not isinstance(x, str) else x)
                if x not in lab_set:
                    raise SystemExit(f"{path}: document {did} has label {x!r} which is not in label_vocab.json")
                conv.append(x)
            if len(conv) != len(sents):
                raise SystemExit(f"{path}: document {did} has {len(sents)} sentences but {len(conv)} labels")
            ls = conv
        docs.append({"id": did, "sents": sents, "labels": ls})
    return docs


class RDoc:
    pass


def build_docs(tok, raw_docs, label2i, domain, eos, max_sent_tokens):
    flat = [" ".join(s.split()) for d in raw_docs for s in d["sents"]]
    encs = []
    for i in range(0, len(flat), 20000):
        encs += tok.encode_batch(flat[i:i + 20000])
    k, out = 0, []
    for d in raw_docs:
        r = RDoc()
        r.id, r.domain, r.n = d["id"], domain, len(d["sents"])
        r.sents = []
        for _ in range(r.n):
            ids = encs[k].ids[:max_sent_tokens]
            k += 1
            r.sents.append(np.asarray(ids if ids else [eos], dtype=np.int64))
        r.lens = np.array([len(s) for s in r.sents], dtype=np.int64)
        r.lab = np.array([label2i[x] for x in d["labels"]], dtype=np.int64) if d["labels"] is not None else None
        out.append(r)
    return out


def make_windows3(d, cfg):
    """Pack consecutive sentences into windows of <= max_len tokens (each sentence costs len + 1 marker)."""
    if d.n == 0:
        return []
    if cfg.ctx_mode == "sentence":
        return [(i, i + 1) for i in range(d.n)]
    cost = d.lens + 1
    wins, a = [], 0
    while a < d.n:
        b, tot = a, 0
        while b < d.n and (tot + cost[b] <= cfg.max_len or b == a):
            tot += cost[b]
            b += 1
        wins.append((a, b))
        if b >= d.n:
            break
        a += max(1, int((b - a) * (1.0 - cfg.win_overlap)))
    return wins


def abs_bucket(i):
    return i if i < 3 else 3 if i < 5 else 4 if i < 10 else 5 if i < 20 else 6


def collate3(docs, items, dev, eos):
    B = len(items)
    L = max(sum(len(docs[di].sents[i]) + 1 for i in range(a, b)) for di, a, b in items)
    M = max(b - a for _, a, b in items)
    ids = np.zeros((B, L), dtype=np.int64)
    km = np.zeros((B, L), dtype=bool)
    pool = np.zeros((B, M, L), dtype=np.float32)
    mark = np.zeros((B, M), dtype=np.int64)
    sm = np.zeros((B, M), dtype=bool)
    lab = np.full((B, M), -100, dtype=np.int64)
    rel = np.zeros((B, M), dtype=np.int64)
    ab = np.zeros((B, M), dtype=np.int64)
    dom = np.zeros(B, dtype=np.int64)
    for j, (di, a, b) in enumerate(items):
        d, p = docs[di], 0
        dom[j] = d.domain
        for m, i in enumerate(range(a, b)):
            s = d.sents[i]
            n = len(s)
            ids[j, p:p + n] = s
            ids[j, p + n] = eos
            pool[j, m, p:p + n] = 1.0 / n
            mark[j, m] = p + n
            sm[j, m] = True
            rel[j, m] = min(9, int(10 * i / max(1, d.n)))
            ab[j, m] = abs_bucket(i)
            if d.lab is not None:
                lab[j, m] = d.lab[i]
            p += n + 1
        km[j, :p] = True
    t = lambda x: torch.from_numpy(x).to(dev)
    return t(ids), t(km), t(pool), t(mark), t(sm), t(rel), t(ab), t(dom), t(lab)


def load_all(cfg: RCfg, tok, with_test=False):
    res = lambda rel: P1.resolve_file(cfg, rel)
    labels, id2 = parse_vocab(P1.jload(res(f"{cfg.train_prefix}/label_vocab.json")))
    label2i = {l: i for i, l in enumerate(labels)}
    eos = tok.token_to_id(P1.EOS)
    out = {"labels": labels, "eos": eos}
    for split, prefix, labelled in (("train", cfg.train_prefix, True), ("dev", cfg.train_prefix, True), ("test", cfg.test_prefix, False)):
        if split == "test" and not with_test:
            continue
        docs = []
        for di, dom in enumerate(DOMAINS):
            raw = load_raw_docs(res(f"{prefix}/{dom}_{split}.json"), labels, id2)
            docs += build_docs(tok, raw, label2i, di, eos, cfg.max_sent_tokens)
        out[split] = docs
    return out


# ============================================================================================
# 2. Metrics (identical to the official evaluator: pooled macro over the 13 labels, zero_division=0)
# ============================================================================================
def macro_scores(y_true, y_pred, C):
    cm = np.zeros((C, C), dtype=np.int64)
    np.add.at(cm, (y_true, y_pred), 1)
    tp = np.diag(cm).astype(np.float64)
    fp, fn = cm.sum(0) - tp, cm.sum(1) - tp
    P = np.divide(tp, tp + fp, out=np.zeros(C), where=(tp + fp) > 0)
    R = np.divide(tp, tp + fn, out=np.zeros(C), where=(tp + fn) > 0)
    F1 = np.divide(2 * P * R, P + R, out=np.zeros(C), where=(P + R) > 0)
    return {"macro_f1": float(F1.mean()), "macro_p": float(P.mean()), "macro_r": float(R.mean()),
            "P": P.tolist(), "R": R.tolist(), "F1": F1.tolist(), "support": cm.sum(1).tolist(), "cm": cm}


def pooled(docs, probs, C, bias=None, domain=None):
    ys, ps = [], []
    for d, p in zip(docs, probs):
        if domain is not None and d.domain != domain:
            continue
        lp = np.log(np.clip(p, 1e-9, 1.0))
        if bias is not None:
            lp = lp + bias[None, :]
        ys.append(d.lab)
        ps.append(lp.argmax(1))
    if not ys:
        return macro_scores(np.zeros(0, dtype=np.int64), np.zeros(0, dtype=np.int64), C)
    return macro_scores(np.concatenate(ys), np.concatenate(ps), C)


def tune_bias(docs, probs, C, min_gain=0.002, grid=(-1.0, -0.5, -0.25, 0.25, 0.5, 1.0), passes=3):
    """Coordinate ascent on one additive log-probability bias per class to maximise pooled macro-F1 on the dev set."""
    y = np.concatenate([d.lab for d in docs])
    lp = np.log(np.clip(np.concatenate(probs), 1e-9, 1.0))
    f1 = lambda b: macro_scores(y, (lp + b[None, :]).argmax(1), C)["macro_f1"]
    b = np.zeros(C, dtype=np.float64)
    cur = f1(b)
    for _ in range(passes):
        improved = False
        for c in range(C):
            best_b, best_s = 0.0, cur
            for g in grid:
                t = b.copy()
                t[c] += g
                s = f1(t)
                if s > best_s + min_gain:
                    best_s, best_b = s, g
            if best_b != 0.0:
                b[c] += best_b
                cur, improved = best_s, True
        if not improved:
            break
    return b.astype(np.float32), cur


# ============================================================================================
# 3. Parameter-efficient adapters beyond Phase 2: rsLoRA and IA3
# ============================================================================================
class RsLoRALinear(N.LoRALinear):
    """Rank-stabilised LoRA (Kalajdzievski 2023): same as LoRA but scale = alpha / sqrt(r) instead of alpha / r."""

    def __init__(self, base, r, alpha, dropout):
        super().__init__(base, r, alpha, dropout)
        self.scale = alpha / math.sqrt(r)


class IA3Out(nn.Module):
    """IA3 (Liu et al. 2022): rescale the OUTPUT of a projection (keys / values) with a learned vector, initialised to ones."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        return self.base(x) * self.l


class IA3In(nn.Module):
    """IA3: rescale the INPUT of the feed-forward down-projection (its hidden activations) with a learned vector."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.l)


def inject_peft3(lm, cfg: RCfg) -> int:
    for p in lm.parameters():
        p.requires_grad = False
    names = N.PEFT_ATTN if cfg.peft_targets == "attn" else N.PEFT_ALL
    m = cfg.method

    def wrap(base):
        if m == "lora":
            return N.LoRALinear(base, cfg.peft_r, cfg.peft_alpha, cfg.peft_dropout)
        if m == "rslora":
            return RsLoRALinear(base, cfg.peft_r, cfg.peft_alpha, cfg.peft_dropout)
        return N.LoRMALinear(base, cfg.peft_r, cfg.peft_alpha, cfg.lorma_variant)

    n = 0
    for blk in lm.blocks:
        for part in (blk.mix, blk.ffn):
            for name, child in list(part.named_children()):
                if isinstance(child, nn.Linear):
                    if m == "ia3":
                        if name in ("k", "v"):
                            setattr(part, name, IA3Out(child))
                            n += 1
                        elif name == "w2":
                            setattr(part, name, IA3In(child))
                            n += 1
                    elif name in names:
                        setattr(part, name, wrap(child))
                        n += 1
                elif name == "experts" and cfg.peft_experts and m != "ia3":
                    for ex in child:
                        for en, ec in list(ex.named_children()):
                            if isinstance(ec, nn.Linear) and en in names:
                                setattr(ex, en, wrap(ec))
                                n += 1
    return n


def apply_method3(net, cfg: RCfg):
    m = cfg.method
    if m in ("full", "kd"):
        return
    if m == "partial":
        for p in net.lm.wte.parameters():
            p.requires_grad = False
        if net.lm.wpe is not None:
            for p in net.lm.wpe.parameters():
                p.requires_grad = False
        for blk in net.lm.blocks[:cfg.freeze_layers]:
            for p in blk.parameters():
                p.requires_grad = False
        return
    if m in ("lora", "rslora", "lorma", "ia3"):
        n = inject_peft3(net.lm, cfg)
        log(f"[peft] {m}{'-' + cfg.lorma_variant if m == 'lorma' else ''}: adapted {n} layers (r={cfg.peft_r}, targets={cfg.peft_targets})")
        return
    raise SystemExit(f"unknown method {m!r}: use full | partial | lora | rslora | lorma | ia3 | kd")


# ============================================================================================
# 4. Model
# ============================================================================================
def backbone_hidden(lm, ids, km, bidir):
    B, T = ids.shape
    dev = ids.device
    pos = torch.arange(T, device=dev).unsqueeze(0)
    x = lm.wte(ids)
    if lm.wpe is not None:
        x = x + lm.wpe(pos)
    rope = None
    if lm.rope_cos is not None:
        rope = (lm.rope_cos[pos].unsqueeze(1), lm.rope_sin[pos].unsqueeze(1))
    if bidir:
        amask = km[:, None, None, :]
    else:
        qi, kj = torch.arange(T, device=dev).unsqueeze(1), torch.arange(T, device=dev).unsqueeze(0)
        amask = ((kj <= qi)[None, None] & km[:, None, None, :]) | (kj == qi)[None, None]
    for blk in lm.blocks:
        x = blk(x, rope, amask, km, None)
    return lm.norm_f(x)


class RRNet(nn.Module):
    """Phase-1 backbone + sentence pooling + (BiGRU over sentences) + classifier."""

    def __init__(self, lm, n_classes, bidir, head, sent_dim, head_hidden, head_layers, head_dropout):
        super().__init__()
        self.lm, self.bidir, self.head_type = lm, bidir, head
        d = lm.mc["n_embd"]
        self.proj = nn.Linear(2 * d, sent_dim)
        self.pos_rel = nn.Embedding(10, sent_dim)
        self.pos_abs = nn.Embedding(7, sent_dim)
        self.dom = nn.Embedding(len(DOMAINS), sent_dim)
        self.norm = nn.LayerNorm(sent_dim)
        out_d = sent_dim
        if head == "bigru":
            self.rnn = nn.GRU(sent_dim, head_hidden, num_layers=head_layers, batch_first=True, bidirectional=True,
                              dropout=0.1 if head_layers > 1 else 0.0)
            out_d = 2 * head_hidden
        self.drop = nn.Dropout(head_dropout)
        self.cls = nn.Linear(out_d, n_classes)
        for e in (self.pos_rel, self.pos_abs, self.dom):
            nn.init.normal_(e.weight, 0.0, 0.02)
        nn.init.normal_(self.cls.weight, 0.0, 0.02)
        nn.init.zeros_(self.cls.bias)

    def sent_vectors(self, ids, km, pool, mark, sm, rel, ab, dom):
        """(B, M, sent_dim): one vector per sentence of every window."""
        h = backbone_hidden(self.lm, ids, km, self.bidir)                        # (B, L, d)
        mean = torch.bmm(pool.to(h.dtype), h)                                    # mean of the sentence's token states
        mk = torch.gather(h, 1, mark.unsqueeze(-1).expand(-1, -1, h.size(-1)))   # state at the sentence's <eos> marker
        s = self.proj(torch.cat([mean, mk], dim=-1))
        s = s + self.pos_rel(rel) + self.pos_abs(ab) + self.dom(dom).unsqueeze(1)
        return self.drop(F.gelu(self.norm(s)))

    def seq_logits(self, s, lengths):
        """(B, S, sent_dim) sentence sequences (padded; `lengths` = real lengths) -> (B, S, C) role logits."""
        if self.head_type == "bigru":
            M = s.size(1)
            pk = nn.utils.rnn.pack_padded_sequence(s, lengths.cpu(), batch_first=True, enforce_sorted=False)
            s, _ = self.rnn(pk)
            s, _ = nn.utils.rnn.pad_packed_sequence(s, batch_first=True, total_length=M)
        return self.cls(self.drop(s))

    def forward(self, ids, km, pool, mark, sm, rel, ab, dom):
        return self.seq_logits(self.sent_vectors(ids, km, pool, mark, sm, rel, ab, dom), sm.sum(1))


def build_net3(cfg: RCfg, lm, C):
    arch = lm.mc["arch"]
    bidir = (arch != "retnet") if cfg.bidir == "auto" else (cfg.bidir in ("true", "yes", "1", True))
    if arch == "retnet":
        bidir = False
    log(f"[model] arch={arch} bidirectional_attention={bidir} head={cfg.head} ctx_mode={cfg.ctx_mode}")
    return RRNet(lm, C, bidir, cfg.head, cfg.sent_dim, cfg.head_hidden, cfg.head_layers, cfg.head_dropout)


def load_teacher3(cfg: RCfg, dev, labels, tok_path):
    tr = Path(cfg.work_dir) / "rr_runs" / cfg.teacher_run
    if not (tr / "best.pt").exists():
        raise SystemExit(f"teacher run not found: {tr}/best.pt (train it first)")
    ck = torch.load(tr / "best.pt", map_location="cpu", weights_only=False)
    if ck["labels"] != labels:
        raise SystemExit("teacher and student use different label sets")
    if (tr / "tokenizer.json").read_bytes() != Path(tok_path).read_bytes():
        raise SystemExit("teacher and student use different tokenizers (distillation needs identical token ids)")
    saved = RCfg()
    apply_overrides(saved, {k: v for k, v in ck["cfg"].items() if k in {f.name for f in fields(RCfg)}})
    net = build_net3(saved, P1.LM(ck["mc"]), len(labels))
    apply_method3(net, saved)
    net.load_state_dict(ck["net"])
    net.to(dev).eval()
    for p in net.parameters():
        p.requires_grad = False
    n = sum(p.numel() for p in net.parameters())
    log(f"[kd] teacher {cfg.teacher_run}: arch={ck['mc']['arch']} method={saved.method} params={n / 1e6:.1f}M")
    return net, n, ck["mc"]


def load_run3(cfg: RCfg, run_name: str, dev):
    from tokenizers import Tokenizer
    rd = Path(cfg.work_dir) / "rr_runs" / run_name
    ck = torch.load(rd / "best.pt", map_location="cpu", weights_only=False)
    saved = RCfg()
    apply_overrides(saved, {k: v for k, v in ck["cfg"].items() if k in {f.name for f in fields(RCfg)}})
    saved.work_dir = cfg.work_dir
    if cfg.local_data_dir:
        saved.local_data_dir = cfg.local_data_dir
    net = build_net3(saved, P1.LM(ck["mc"]), len(ck["labels"]))
    apply_method3(net, saved)
    net.load_state_dict(ck["net"])
    net.to(dev).eval()
    return {"rd": rd, "ck": ck, "saved": saved, "net": net, "labels": ck["labels"],
            "tok": Tokenizer.from_file(str(rd / "tokenizer.json"))}


# ============================================================================================
# 5. Inference + training
# ============================================================================================
@torch.no_grad()
def doc_probs3(net, docs, cfg, dev, amp, C, eos):
    net.eval()
    items = [(i, a, b) for i, d in enumerate(docs) for a, b in make_windows3(d, cfg)]
    order = sorted(range(len(items)), key=lambda k: int(docs[items[k][0]].lens[items[k][1]:items[k][2]].sum()))
    probs = [np.zeros((d.n, C), dtype=np.float32) for d in docs]
    wsum = [np.zeros(d.n, dtype=np.float32) for d in docs]
    for b0 in range(0, len(order), cfg.eval_batch):
        chunk = [items[k] for k in order[b0:b0 + cfg.eval_batch]]
        ids, km, pool, mark, sm, rel, ab, dom, _ = collate3(docs, chunk, dev, eos)
        with amp():
            logits = net(ids, km, pool, mark, sm, rel, ab, dom)
        pr = logits.float().softmax(-1).cpu().numpy()
        for j, (di, a, b) in enumerate(chunk):
            m = b - a
            w = np.minimum(np.arange(1, m + 1), np.arange(m, 0, -1)).astype(np.float32)   # centre sentences count more
            probs[di][a:b] += pr[j, :m] * w[:, None]
            wsum[di][a:b] += w
    return [p / np.maximum(w, 1e-6)[:, None] for p, w in zip(probs, wsum)]


def class_weights3(docs, C, mode):
    if mode == "none":
        return None
    cnt = np.bincount(np.concatenate([d.lab for d in docs]), minlength=C).astype(np.float64)
    return torch.tensor(np.clip(np.sqrt(cnt.max() / (cnt + 1.0)), 1.0, 6.0), dtype=torch.float32)


# ---- label-transition prior + calibrated decoding ---------------------------------------------------
def transition_logs(docs, C):
    """Log P(next role | role) and log P(first role) estimated (add-one smoothed) on the training documents."""
    T, S0 = np.ones((C, C)), np.ones(C)
    for d in docs:
        if d.n:
            S0[d.lab[0]] += 1
            np.add.at(T, (d.lab[:-1], d.lab[1:]), 1)
    return np.log(T / T.sum(1, keepdims=True)), np.log(S0 / S0.sum())


def viterbi_seq(lp, logT, logS, lam):
    """argmax over label paths of  sum_t lp[t, y_t] + lam * (logS[y_0] + sum_t logT[y_{t-1}, y_t])."""
    n, C = lp.shape
    if n == 0:
        return np.zeros(0, dtype=np.int64)
    score = lp[0] + lam * logS
    back = np.zeros((n, C), dtype=np.int16)
    ar = np.arange(C)
    for t in range(1, n):
        cand = score[:, None] + lam * logT
        idx = cand.argmax(0)
        back[t] = idx
        score = cand[idx, ar] + lp[t]
    path = np.empty(n, dtype=np.int64)
    path[-1] = int(score.argmax())
    for t in range(n - 1, 0, -1):
        path[t - 1] = back[t, path[t]]
    return path


def decode_all(docs, probs, bias, logT, logS, lam):
    out = []
    for p in probs:
        lp = np.log(np.clip(p, 1e-9, 1.0))
        if bias is not None:
            lp = lp + bias[None, :]
        out.append(viterbi_seq(lp, logT, logS, lam) if (lam > 0 and logT is not None) else lp.argmax(1))
    return out


def score_docs(docs, preds, C, domain=None):
    ys, ps = [], []
    for d, p in zip(docs, preds):
        if domain is None or d.domain == domain:
            ys.append(d.lab)
            ps.append(p)
    if not ys:
        return macro_scores(np.zeros(0, dtype=np.int64), np.zeros(0, dtype=np.int64), C)
    return macro_scores(np.concatenate(ys), np.concatenate(ps), C)


def calibrate(dv, probs, C, logT, logS, cfg):
    """Tune per-class thresholds, then the weight of the transition prior, on the dev set."""
    base = pooled(dv, probs, C)["macro_f1"]
    bias, f_b = tune_bias(dv, probs, C) if cfg.class_bias else (None, base)
    if bias is not None and f_b <= base + 0.002:
        bias = None
    cur = pooled(dv, probs, C, bias)["macro_f1"]
    lam = 0.0
    if cfg.trans_decode:
        for l_ in (0.1, 0.25, 0.5, 1.0):
            f = score_docs(dv, decode_all(dv, probs, bias, logT, logS, l_), C)["macro_f1"]
            log(f"[calibrate] transition weight {l_:.2f}: dev macro-F1 {f:.4f} (current {cur:.4f})")
            if f > cur + 0.003:
                lam, cur = l_, f
    return {"bias": bias, "lam": lam, "f1": cur, "f1_base": base}


# ---- whole-document sequence mode ("docseq") ---------------------------------------------------------
def pack_windows(d, a, b, max_len):
    """Partition sentences [a, b) into consecutive windows of <= max_len tokens (no overlap)."""
    cost, wins, i = d.lens + 1, [], a
    while i < b:
        j, tot = i, 0
        while j < b and (tot + cost[j] <= max_len or j == i):
            tot += cost[j]
            j += 1
        wins.append((i, j))
        i = j
    return wins


def docseq_items(docs, cfg, rng):
    """Training crops: runs of up to crop_windows consecutive windows of a document (random offset every epoch)."""
    k, items = max(1, cfg.crop_windows), []
    for di, d in enumerate(docs):
        if d.n == 0:
            continue
        wins = pack_windows(d, 0, d.n, cfg.max_len)
        if len(wins) <= k:
            items.append((di, wins))
            continue
        step = max(1, k // 2)
        starts = sorted(set([0] + list(range(rng.randrange(step), len(wins) - k + 1, step)) + [len(wins) - k]))
        items += [(di, wins[s0:s0 + k]) for s0 in starts]
    return items


def collate_docseq(docs, crops, dev, eos):
    """All windows of all crops go through the backbone together; fi maps (crop, sentence) -> row of the flattened sentence vectors."""
    win_items = [(di, a, b) for di, wins in crops for a, b in wins]
    wb = collate3(docs, win_items, dev, eos)[:8]
    M = wb[2].shape[1]
    rows, labs, j = [], [], 0
    for di, wins in crops:
        row, lb = [], []
        for a, b in wins:
            row += [j * M + m for m in range(b - a)]
            if docs[di].lab is not None:
                lb += docs[di].lab[a:b].tolist()
            j += 1
        rows.append(row)
        labs.append(lb)
    S = max(len(r) for r in rows)
    fi = np.zeros((len(crops), S), dtype=np.int64)
    smask = np.zeros((len(crops), S), dtype=bool)
    lab = np.full((len(crops), S), -100, dtype=np.int64)
    for c, (row, lb) in enumerate(zip(rows, labs)):
        fi[c, :len(row)] = row
        smask[c, :len(row)] = True
        if lb:
            lab[c, :len(lb)] = lb
    t = lambda x: torch.from_numpy(x).to(dev)
    return wb, t(fi), t(smask), t(lab)


@torch.no_grad()
def docseq_probs(net, docs, cfg, dev, amp, C, eos):
    """Backbone over the windows of a document (no grad), then ONE BiGRU pass over all its sentences."""
    net.eval()
    out = []
    for di, d in enumerate(docs):
        if d.n == 0:
            out.append(np.zeros((0, C), dtype=np.float32))
            continue
        wins, vecs = pack_windows(d, 0, d.n, cfg.max_len), []
        for b0 in range(0, len(wins), cfg.eval_batch):
            chunk = [(di, a, b) for a, b in wins[b0:b0 + cfg.eval_batch]]
            with amp():
                s = net.sent_vectors(*collate3(docs, chunk, dev, eos)[:8])
            vecs += [s[j, :b - a].float() for j, (_, a, b) in enumerate(chunk)]
        with amp():
            lg = net.seq_logits(torch.cat(vecs, 0).unsqueeze(0), torch.tensor([d.n]))
        out.append(lg[0].float().softmax(-1).cpu().numpy())
    return out


def doc_probs_any(net, docs, cfg, dev, amp, C, eos):
    if cfg.ctx_mode == "docseq":
        return docseq_probs(net, docs, cfg, dev, amp, C, eos)
    return doc_probs3(net, docs, cfg, dev, amp, C, eos)


def epoch_batches(docs, cfg, rng, dev, eos, items_all):
    if cfg.ctx_mode == "docseq":
        crops = docseq_items(docs, cfg, rng)
        rng.shuffle(crops)
        for i in range(0, len(crops), cfg.batch_docs):
            wb, fi, smask, lab = collate_docseq(docs, crops[i:i + cfg.batch_docs], dev, eos)
            yield ("docseq", wb, fi, smask, lab)
    else:
        items = list(items_all)
        rng.shuffle(items)
        for i in range(0, len(items), cfg.batch_size):
            *args, lab = collate3(docs, items[i:i + cfg.batch_size], dev, eos)
            yield ("win", tuple(args), lab)


def logits_of(net, b):
    if b[0] == "docseq":
        _, wb, fi, smask, lab = b
        s = net.sent_vectors(*wb)
        return net.seq_logits(s.reshape(-1, s.size(-1))[fi], smask.sum(1)), lab
    return net(*b[1]), b[2]


class EMA:
    """Exponential moving average of the trainable weights: much more stable on small, noisy dev sets."""

    def __init__(self, net, decay):
        self.net, self.decay = net, decay
        self.shadow = {n: p.detach().clone() for n, p in net.named_parameters() if p.requires_grad}

    @torch.no_grad()
    def update(self, step):
        d = min(self.decay, (1.0 + step) / (10.0 + step))
        for n, p in self.net.named_parameters():
            if n in self.shadow:
                self.shadow[n].mul_(d).add_(p.detach(), alpha=1.0 - d)

    @contextlib.contextmanager
    def swapped(self):
        backup = {}
        with torch.no_grad():
            for n, p in self.net.named_parameters():
                if n in self.shadow:
                    backup[n] = p.detach().clone()
                    p.copy_(self.shadow[n])
        try:
            yield
        finally:
            with torch.no_grad():
                for n, p in self.net.named_parameters():
                    if n in backup:
                        p.copy_(backup[n])


def report_text(labels, sc):
    for i, l in enumerate(labels):
        log(f"   {l:<24} P={sc['P'][i]:.3f} R={sc['R'][i]:.3f} F1={sc['F1'][i]:.3f}  support={sc['support'][i]}")


def run_train3(cfg: RCfg):
    rd = run_dir3(cfg)
    rd.mkdir(parents=True, exist_ok=True)
    jd = asdict(cfg)
    P1.jdump(jd, rd / "config.json")
    random.seed(cfg.seed), np.random.seed(cfg.seed), torch.manual_seed(cfg.seed)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    amp_dtype, amp = P1.amp_setup(dev)
    tok, lm, mc = N.load_p1(cfg)
    shutil.copy(Path(cfg.p1_dir) / "tokenizer.json", rd / "tokenizer.json")
    if cfg.method == "kd":
        if not cfg.teacher_run:
            raise SystemExit('method "kd" needs "teacher_run": the name of a finished RR run')
        lm, mc = N.make_student(lm, mc, cfg)
    data = load_all(cfg, tok)
    labels, eos, tr, dv = data["labels"], data["eos"], data["train"], data["dev"]
    C = len(labels)
    assert cfg.max_len <= mc["ctx"], "max_len exceeds the Phase-1 context length"
    log(f"[data] train {len(tr)} docs / {sum(d.n for d in tr)} sentences | dev {len(dv)} docs / {sum(d.n for d in dv)} sentences | {C} labels")
    teacher, teacher_params = None, 0
    if cfg.method == "kd":
        teacher, teacher_params, t_mc = load_teacher3(cfg, dev, labels, rd / "tokenizer.json")
        assert cfg.max_len <= t_mc["ctx"]

    net = build_net3(cfg, lm, C)
    apply_method3(net, cfg)
    net.to(dev)
    n_total = sum(p.numel() for p in net.parameters())
    n_train = sum(p.numel() for p in net.parameters() if p.requires_grad)
    log(f"[model] total params {n_total / 1e6:.1f}M | trainable {n_train / 1e6:.3f}M | method={cfg.method}")

    peft = cfg.method in ("lora", "rslora", "lorma", "ia3")
    body_lr, body_wd = (cfg.peft_lr, 0.0) if peft else (cfg.lr, cfg.weight_decay)
    nb = mc["n_layer"]

    def lr_scale(name):
        if peft or cfg.llrd >= 1.0 or not name.startswith("lm."):
            return 1.0
        m_ = re.match(r"lm\.blocks\.(\d+)\.", name)
        if m_:
            return cfg.llrd ** (nb - 1 - int(m_.group(1)))
        return cfg.llrd ** nb if name.startswith(("lm.wte", "lm.wpe")) else 1.0

    buckets = {}
    for n_, p_ in net.named_parameters():
        if not p_.requires_grad:
            continue
        if n_.startswith("lm."):
            base, wd = body_lr * lr_scale(n_), (body_wd if p_.dim() >= 2 else 0.0)
        else:
            base, wd = cfg.head_lr, (cfg.weight_decay if p_.dim() >= 2 else 0.0)
        buckets.setdefault((round(base, 12), wd), []).append(p_)
    opt = torch.optim.AdamW([{"params": ps, "lr": b_, "base": b_, "weight_decay": wd} for (b_, wd), ps in buckets.items()],
                            betas=(0.9, 0.98), eps=1e-8)
    scaler = P1.make_scaler(dev == "cuda" and amp_dtype == torch.float16)
    cw = class_weights3(tr, C, cfg.class_weight)
    cw = cw.to(dev) if cw is not None else None
    ema = EMA(net, cfg.ema_decay) if cfg.ema_decay > 0 else None
    logT, logS = transition_logs(tr, C)

    items_all = [(i, a, b) for i, d in enumerate(tr) for a, b in make_windows3(d, cfg)]
    if cfg.ctx_mode == "docseq":
        per_epoch = math.ceil(len(docseq_items(tr, cfg, random.Random(0))) / cfg.batch_docs)
    else:
        per_epoch = math.ceil(len(items_all) / cfg.batch_size)
    total_steps = max(1, cfg.epochs * per_epoch)
    log(f"[train] ctx_mode={cfg.ctx_mode}: {per_epoch} steps per epoch, {total_steps} steps in total, EMA {cfg.ema_decay}")
    rng = random.Random(cfg.seed)
    best, step, t_start, bad = {"f1": -1.0}, 0, time.time(), 0
    if dev == "cuda":
        torch.cuda.reset_peak_memory_stats()
    logf = open(rd / "log.jsonl", "a")
    for ep in range(cfg.epochs):
        net.train()
        run_loss, n_b, t_ep = 0.0, 0, time.time()
        for b in epoch_batches(tr, cfg, rng, dev, eos, items_all):
            for g in opt.param_groups:
                g["lr"] = N.lr_at(step, total_steps, g["base"], cfg.warmup_frac)
            with amp():
                logits, lab = logits_of(net, b)
                if teacher is not None:
                    with torch.no_grad():
                        t_logits, _ = logits_of(teacher, b)
            loss = F.cross_entropy(logits.float().view(-1, C), lab.view(-1), weight=cw, ignore_index=-100,
                                   label_smoothing=cfg.label_smoothing)
            if teacher is not None:
                loss = cfg.kd_alpha * loss + (1.0 - cfg.kd_alpha) * N.kd_loss(logits, t_logits, lab, cfg.kd_temp)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in net.parameters() if p.requires_grad], cfg.grad_clip)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            if ema is not None:
                ema.update(step)
            step += 1
            run_loss += float(loss)
            n_b += 1
            if step % 20 == 0:
                log(f"  ep {ep + 1} step {step}/{total_steps} loss {run_loss / n_b:.4f}")
        with (ema.swapped() if ema is not None else contextlib.nullcontext()):    # evaluate (and save) the averaged weights
            probs = doc_probs_any(net, dv, cfg, dev, amp, C, eos)
            sc = pooled(dv, probs, C)
            log(f"[epoch {ep + 1}/{cfg.epochs}] loss {run_loss / max(1, n_b):.4f} | dev macro-F1 {sc['macro_f1']:.4f} "
                f"(P {sc['macro_p']:.4f} R {sc['macro_r']:.4f}) | {time.time() - t_ep:.0f}s")
            logf.write(json.dumps({"epoch": ep + 1, "loss": run_loss / max(1, n_b), "dev_f1": sc["macro_f1"]}) + "\n")
            logf.flush()
            if sc["macro_f1"] > best["f1"]:
                best, bad = {"f1": sc["macro_f1"], "epoch": ep + 1}, 0
                torch.save({"net": net.state_dict(), "mc": mc, "labels": labels, "cfg": jd, "bias": None}, rd / "best.pt")
            else:
                bad += 1
        if bad >= cfg.patience:
            log(f"[train] early stopping: no dev improvement for {bad} epochs (best epoch {best['epoch']})")
            break
    logf.close()
    train_s = time.time() - t_start
    peak = torch.cuda.max_memory_allocated() / 2 ** 20 if dev == "cuda" else 0.0

    ck = torch.load(rd / "best.pt", map_location="cpu", weights_only=False)
    net.load_state_dict(ck["net"])
    if dev == "cuda":
        torch.cuda.synchronize()
    t_inf = time.time()
    probs = doc_probs_any(net, dv, cfg, dev, amp, C, eos)
    if dev == "cuda":
        torch.cuda.synchronize()
    infer_sent_s = sum(d.n for d in dv) / max(1e-9, time.time() - t_inf)
    cal = calibrate(dv, probs, C, logT, logS, cfg)
    preds = decode_all(dv, probs, cal["bias"], logT, logS, cal["lam"])
    sc = score_docs(dv, preds, C)
    sc_cl, sc_it = score_docs(dv, preds, C, 0), score_docs(dv, preds, C, 1)
    ck.update(bias=None if cal["bias"] is None else [float(x) for x in cal["bias"]], lam=cal["lam"],
              logT=logT.tolist(), logS=logS.tolist())
    torch.save(ck, rd / "best.pt")
    log(f"[dev] macro-F1 {cal['f1_base']:.4f} (argmax) -> {sc['macro_f1']:.4f} (class thresholds: "
        f"{'used' if cal['bias'] is not None else 'not used'}, transition weight {cal['lam']}) | P {sc['macro_p']:.4f} R {sc['macro_r']:.4f}")
    log(f"[dev] per-domain macro-F1: CL {sc_cl['macro_f1']:.4f} | IT {sc_it['macro_f1']:.4f}")
    report_text(labels, sc)
    cm = sc["cm"].copy()
    np.fill_diagonal(cm, 0)
    top = sorted(((int(cm[i, j]), labels[i], labels[j]) for i in range(C) for j in range(C) if cm[i, j] > 0), reverse=True)[:12]
    log("[dev] most frequent confusions (count, true -> predicted):")
    for c_, t_, p_ in top:
        log(f"   {c_:>5}  {t_} -> {p_}")
    variant = cfg.lorma_variant if cfg.method == "lorma" else ""
    P1.jdump({"best_epoch": best["epoch"], "macro_f1": sc["macro_f1"], "macro_f1_no_bias": cal["f1_base"],
              "macro_p": sc["macro_p"], "macro_r": sc["macro_r"], "dev_cl_f1": sc_cl["macro_f1"], "dev_it_f1": sc_it["macro_f1"],
              "per_label": {l: {"P": sc["P"][i], "R": sc["R"][i], "F1": sc["F1"][i], "support": sc["support"][i]}
                            for i, l in enumerate(labels)},
              "confusions": [{"count": c_, "true": t_, "pred": p_} for c_, t_, p_ in top], "bias": ck["bias"], "lam": cal["lam"],
              "params_total": n_total, "params_trainable": n_train, "train_seconds": train_s, "peak_gpu_mb": peak,
              "infer_sentences_per_s": infer_sent_s, "p1_arch": mc["arch"], "method": cfg.method,
              "teacher_run": cfg.teacher_run, "teacher_params": teacher_params}, rd / "dev_report.json")
    log_csv(cfg, {"run": cfg.run, "p1_arch": mc["arch"], "method": cfg.method, "variant": variant, "ctx_mode": cfg.ctx_mode,
                  "head": cfg.head, "bidir": net.bidir, "params_total_M": round(n_total / 1e6, 2),
                  "params_trainable_M": round(n_train / 1e6, 3), "dev_macro_f1": round(sc["macro_f1"], 4),
                  "dev_macro_p": round(sc["macro_p"], 4), "dev_macro_r": round(sc["macro_r"], 4),
                  "dev_cl_f1": round(sc_cl["macro_f1"], 4), "dev_it_f1": round(sc_it["macro_f1"], 4),
                  "train_s": round(train_s), "peak_gpu_mb": round(peak), "infer_sent_s": round(infer_sent_s),
                  "teacher_run": cfg.teacher_run})
    if cfg.predict_after:
        run_predict3(cfg)


def write_submission3(cfg: RCfg, rd: Path, preds: dict, test_docs):
    """preds: doc id -> list of label strings. Exactly one label per sentence, original order."""
    for d in test_docs:
        assert len(preds[d.id]) == d.n, f"{d.id}: {len(preds[d.id])} predictions for {d.n} sentences"
    ids = [d.id for d in test_docs]
    if len(set(ids)) != len(ids):
        log("[predict] WARNING: duplicate document ids across CL_test / IT_test - the output format cannot represent them separately")
    rd.mkdir(parents=True, exist_ok=True)
    pj = rd / "prediction.json"
    P1.jdump({d.id: preds[d.id] for d in test_docs}, pj)
    zp = rd / "submission.zip"
    with zipfile.ZipFile(zp, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    hist = Path(cfg.work_dir) / "rr_submissions"
    hist.mkdir(parents=True, exist_ok=True)
    shutil.copy(zp, hist / f"{cfg.run}_{time.strftime('%m%d_%H%M%S')}.zip")
    log(f"[predict] wrote {pj} ({len(test_docs)} documents, {sum(d.n for d in test_docs)} sentences) and {zp}")
    return str(zp)


def run_predict3(cfg: RCfg):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    _, amp = P1.amp_setup(dev)
    r = load_run3(cfg, cfg.run, dev)
    saved, labels, ck = r["saved"], r["labels"], r["ck"]
    data = load_all(saved, r["tok"], with_test=True)
    C, eos = len(labels), data["eos"]
    bias = np.array(ck["bias"], dtype=np.float32) if ck.get("bias") else None
    lam = float(ck.get("lam", 0.0))
    logT = np.array(ck["logT"]) if ck.get("logT") else None
    logS = np.array(ck["logS"]) if ck.get("logS") else None
    log(f"[predict] ctx_mode={saved.ctx_mode}, class thresholds {'on' if bias is not None else 'off'}, transition weight {lam}")
    probs = doc_probs_any(r["net"], data["test"], saved, dev, amp, C, eos)
    idx = decode_all(data["test"], probs, bias, logT, logS, lam)
    preds = {d.id: [labels[i] for i in p] for d, p in zip(data["test"], idx)}
    return write_submission3(cfg, r["rd"], preds, data["test"])


# ============================================================================================
# 6. Ensemble (average sentence probabilities of finished runs)
# ============================================================================================
def _ens_candidates(cfg: RCfg):
    out = []
    for f in sorted((Path(cfg.work_dir) / "rr_runs").glob("*/best.pt")):
        n = f.parent.name
        if cfg.ens_prefix:
            if not n.startswith(cfg.ens_prefix):
                continue
        elif n.startswith(("selftest", "check")):
            continue
        out.append(n)
    return out


def run_ensemble3(cfg: RCfg):
    names = list(cfg.ens_runs) or (_ens_candidates(cfg) if cfg.ens_auto else [])
    if len(names) < 2:
        raise SystemExit('need >= 2 finished runs: set "ens_runs": [...] or "ens_auto": true')
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    _, amp = P1.amp_setup(dev)
    vp, ref, dv, logT, logS = {}, {}, None, None, None
    for n in names:
        r = load_run3(cfg, n, dev)
        if not ref:
            data = load_all(r["saved"], r["tok"])
            ref = {"labels": r["labels"], "tok": r["tok"].to_str(), "eos": data["eos"]}
            dv = data["dev"]
            logT, logS = transition_logs(data["train"], len(r["labels"]))
        if r["labels"] != ref["labels"] or r["tok"].to_str() != ref["tok"]:
            log(f"[ensemble] skipping {n}: different labels / tokenizer")
        else:
            vp[n] = doc_probs_any(r["net"], dv, r["saved"], dev, amp, len(ref["labels"]), ref["eos"])
        del r
        gc.collect()
        if dev == "cuda":
            torch.cuda.empty_cache()
    labels, eos = ref["labels"], ref["eos"]
    C = len(labels)
    mean = lambda keys: [sum(vp[k][i] for k in keys) / len(keys) for i in range(len(dv))]
    single = {k: pooled(dv, vp[k], C)["macro_f1"] for k in vp}
    for k, v in sorted(single.items(), key=lambda kv: -kv[1]):
        log(f"[ensemble] single model {k:<28} dev macro-F1 {v:.4f}")
    if cfg.ens_runs or not cfg.ens_auto:
        chosen = list(vp)
    else:
        order = sorted(single, key=lambda k: -single[k])
        chosen, cur = [order[0]], single[order[0]]
        for k in order[1:]:
            if len(chosen) >= cfg.ens_max:
                break
            f = pooled(dv, mean(chosen + [k]), C)["macro_f1"]
            log(f"[ensemble] try adding {k:<28} -> {f:.4f} (current {cur:.4f})")
            if f > cur + 0.002:
                chosen.append(k)
                cur = f
    log(f"[ensemble] members: {chosen}")
    avg = mean(chosen)
    cal = calibrate(dv, avg, C, logT, logS, cfg)
    preds_dev = decode_all(dv, avg, cal["bias"], logT, logS, cal["lam"])
    sc = score_docs(dv, preds_dev, C)
    sc_cl, sc_it = score_docs(dv, preds_dev, C, 0), score_docs(dv, preds_dev, C, 1)
    log(f"[ensemble] dev macro-F1 {cal['f1_base']:.4f} (argmax) -> {sc['macro_f1']:.4f} (class thresholds "
        f"{'used' if cal['bias'] is not None else 'not used'}, transition weight {cal['lam']})")
    runs = [load_run3(cfg, n, dev) for n in chosen]
    data = load_all(runs[0]["saved"], runs[0]["tok"], with_test=True)
    plist = [doc_probs_any(r["net"], data["test"], r["saved"], dev, amp, C, eos) for r in runs]
    avg_t = [sum(p[i] for p in plist) / len(plist) for i in range(len(data["test"]))]
    idx = decode_all(data["test"], avg_t, cal["bias"], logT, logS, cal["lam"])
    preds = {d.id: [labels[i] for i in p] for d, p in zip(data["test"], idx)}
    rd = run_dir3(cfg)
    rd.mkdir(parents=True, exist_ok=True)
    P1.jdump({"members": chosen, "single_model_f1": single, "macro_f1": sc["macro_f1"], "dev_cl_f1": sc_cl["macro_f1"],
              "dev_it_f1": sc_it["macro_f1"], "lam": cal["lam"],
              "bias": None if cal["bias"] is None else [float(x) for x in cal["bias"]]}, rd / "dev_report.json")
    log_csv(cfg, {"run": cfg.run, "p1_arch": "ensemble", "method": "ensemble", "dev_macro_f1": round(sc["macro_f1"], 4),
                  "dev_macro_p": round(sc["macro_p"], 4), "dev_macro_r": round(sc["macro_r"], 4),
                  "dev_cl_f1": round(sc_cl["macro_f1"], 4), "dev_it_f1": round(sc_it["macro_f1"], 4), "teacher_run": "+".join(chosen)})
    return write_submission3(cfg, rd, preds, data["test"])


# ============================================================================================
# 7. Data diagnostics
# ============================================================================================
def diagnose3(cfg: RCfg):
    tok, _, mc = N.load_p1(cfg)
    data = load_all(cfg, tok, with_test=True)
    labels = data["labels"]
    log(f"[check] {len(labels)} labels: {labels}")
    if sorted(labels) != sorted(OFFICIAL_LABELS):
        log("[check] WARNING: label_vocab.json differs from the official 13 labels listed in the task description")
    for split in ("train", "dev", "test"):
        docs = data[split]
        for di, dom in enumerate(DOMAINS):
            dd = [d for d in docs if d.domain == di]
            if not dd:
                continue
            ns = np.array([d.n for d in dd])
            tl = np.concatenate([d.lens for d in dd]) if len(dd) else np.zeros(1)
            nw = sum(len(make_windows3(d, cfg)) for d in dd)
            log(f"[check] {split:<5} {dom}: {len(dd):>4} docs | sentences/doc mean {ns.mean():.0f} max {ns.max()} | "
                f"tokens/sentence mean {tl.mean():.1f} p95 {np.percentile(tl, 95):.0f} max {tl.max()} | windows {nw}")
    for split in ("train", "dev"):
        y = np.concatenate([d.lab for d in data[split]])
        cnt = np.bincount(y, minlength=len(labels))
        log(f"[check] {split} label distribution: " + ", ".join(f"{l}={c} ({100 * c / cnt.sum():.1f}%)" for l, c in zip(labels, cnt)))
    ids = [d.id for d in data["test"]]
    cl_ids, it_ids = {d.id for d in data["test"] if d.domain == 0}, {d.id for d in data["test"] if d.domain == 1}
    log(f"[check] test ids: {len(ids)} total, overlap between CL and IT: {len(cl_ids & it_ids)}")
    longs = sum(1 for d in data["train"] for s in d.sents if len(s) >= cfg.max_sent_tokens)
    log(f"[check] sentences truncated at {cfg.max_sent_tokens} tokens in train: {longs}")


# ============================================================================================
# 8. Self-test
# ============================================================================================
def _synthetic(tmp, rng):
    role_words = {l: [f"{l.lower()}{i}" for i in range(6)] for l in OFFICIAL_LABELS}
    common = "the court of appeal held that under section as per judgment order petitioner respondent".split()

    def sent(l):
        return " ".join(rng.choice(role_words[l] + common) for _ in range(rng.randint(6, 16))) + " ."

    def doc():
        order = ["Fact"] * rng.randint(3, 6) + ["Issue", "ArgumentPetitioner", "ArgumentRespondent", "Statute", "PrecedentReliedUpon",
                                                "RatioOfTheDecision", "RulingByPresentCourt"]
        order += [rng.choice(["None", "Dissent", "PrecedentNotReliedUpon", "PrecedentOverruled", "RulingByLowerCourt"])]
        return [sent(l) for l in order], order

    os.makedirs(tmp, exist_ok=True)
    allsent = []
    for dom in DOMAINS:
        for split, n in (("train", 10), ("dev", 4), ("test", 4)):
            docs = []
            for i in range(n):
                s, l = doc()
                allsent += s
                rec = {"id": f"{dom}_{split}_{i}", "text": s}
                if split != "test":
                    rec["labels"] = l
                docs.append(rec)
            json.dump(docs, open(os.path.join(tmp, f"{dom}_{split}.json"), "w"))
    json.dump(OFFICIAL_LABELS, open(os.path.join(tmp, "label_vocab.json"), "w"))
    return allsent


def selftest() -> bool:
    import tempfile
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
    ok = True
    rng = random.Random(0)
    tmp = tempfile.mkdtemp(prefix="cs781_rr_selftest_")
    dd = os.path.join(tmp, "data")
    allsent = _synthetic(dd, rng)
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    tok.train_from_iterator([" ".join(s.split()) for s in allsent], trainers.BpeTrainer(
        vocab_size=500, special_tokens=[P1.EOS], initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False))
    V = P1.pad64(tok.get_vocab_size())

    log("=== 1. data, windows, metrics ===")
    cfg0 = apply_overrides(RCfg(), {"local_data_dir": dd, "max_len": 96})
    data = load_all(cfg0, tok, with_test=True)
    labels, eos, tr, dv = data["labels"], data["eos"], data["train"], data["dev"]
    good = len(tr) == 20 and len(dv) == 8 and len(data["test"]) == 8 and labels == OFFICIAL_LABELS
    cover = all(sorted({i for a, b in make_windows3(d, cfg0) for i in range(a, b)}) == list(range(d.n)) for d in tr)
    toklim = all(sum(int(d.lens[i]) + 1 for i in range(a, b)) <= 96 or b - a == 1 for d in tr for a, b in make_windows3(d, cfg0))
    ok &= good and cover and toklim
    log(f"  parsing {'PASS' if good else 'FAIL'} | every sentence covered by a window {'PASS' if cover else 'FAIL'} | window token budget {'PASS' if toklim else 'FAIL'}")
    yt = np.array([0, 1, 1, 2, 2, 2, 0])
    yp = np.array([0, 1, 2, 2, 2, 1, 0])
    sc = macro_scores(yt, yp, 4)
    tp = [2, 1, 2, 0]; fp = [0, 1, 1, 0]; fn = [0, 1, 1, 0]
    f_manual = np.mean([2 * t / (2 * t + a + b) if t else 0.0 for t, a, b in zip(tp, fp, fn)])
    good = abs(sc["macro_f1"] - f_manual) < 1e-9
    ok &= good
    log(f"  macro-F1 (absent class counts as 0, as in the official scorer) {'PASS' if good else 'FAIL'}")
    items = [(0, 0, 3), (1, 1, 6)]
    batch = collate3(tr, items, "cpu", eos)
    shapes_ok = batch[2].shape[:2] == (2, 5) and batch[0].shape[0] == 2 and int(batch[4].sum()) == 8
    ok &= shapes_ok
    log(f"  collate shapes {'PASS' if shapes_ok else 'FAIL'}")

    log("=== 1b. transition prior / Viterbi / document-sequence crops / EMA ===")
    logT, logS = transition_logs(tr, len(labels))
    g1 = np.allclose(np.exp(logT).sum(1), 1.0) and abs(float(np.exp(logS).sum()) - 1.0) < 1e-6
    lp = np.log(np.array([[.6, .4], [.45, .55], [.6, .4]]))
    sticky, st0 = np.log(np.array([[.9, .1], [.1, .9]])), np.log(np.array([.5, .5]))
    g2 = list(viterbi_seq(lp, sticky, st0, 0.0)) == [0, 1, 0] and list(viterbi_seq(lp, sticky, st0, 1.0)) == [0, 0, 0]
    g3 = all([i for a, b in pack_windows(d, 0, d.n, 96) for i in range(a, b)] == list(range(d.n)) for d in tr)
    crops = docseq_items(tr, apply_overrides(RCfg(), {"max_len": 96, "crop_windows": 2}), random.Random(0))
    g4 = all({i for c_di, wins in crops if c_di == di for a, b in wins for i in range(a, b)} == set(range(d.n))
             for di, d in enumerate(tr))
    _, fi, smask, lab2 = collate_docseq(tr, crops[:2], "cpu", eos)
    g5 = fi.shape == smask.shape == lab2.shape and int(smask.sum()) == sum(b - a for _, wins in crops[:2] for a, b in wins)
    lin = nn.Linear(2, 2)
    ema_ = EMA(lin, 0.5)
    w0 = lin.weight.detach().clone()
    with torch.no_grad():
        lin.weight.add_(1.0)
    ema_.update(10 ** 6)
    with ema_.swapped():
        inside = lin.weight.detach().clone()
    g6 = torch.allclose(inside, 0.5 * w0 + 0.5 * (w0 + 1)) and torch.allclose(lin.weight.detach(), w0 + 1)
    ok &= all([g1, g2, g3, g4, g5, g6])
    log(f"  transition matrices {'PASS' if g1 else 'FAIL'} | viterbi smoothing {'PASS' if g2 else 'FAIL'} | windows partition {'PASS' if g3 else 'FAIL'} | "
        f"crops cover every sentence {'PASS' if g4 else 'FAIL'} | docseq batch {'PASS' if g5 else 'FAIL'} | EMA {'PASS' if g6 else 'FAIL'}")

    log("=== 2. adapters are the IDENTITY at initialisation; only adapters + head are trainable ===")
    for arch in P1.ARCH_PRESETS:
        for meth, var in (("lora", "plus"), ("rslora", "plus"), ("ia3", "plus"), ("lorma", "plus"), ("lorma", "pi")):
            tag = f"{arch}/{meth}" + (f"-{var}" if meth == "lorma" else "")
            try:
                torch.manual_seed(0)
                c = apply_overrides(RCfg(), {"method": meth, "lorma_variant": var, "peft_r": 4, "peft_alpha": 8.0, "peft_dropout": 0.0})
                net = build_net3(c, P1.LM(N._tiny_mc(arch, V)), len(labels)).eval()
                b = collate3(tr, [(0, 0, 4), (1, 0, 3)], "cpu", eos)
                inp = b[:8]
                with torch.no_grad():
                    ref = net(*inp)
                apply_method3(net, c)
                net.eval()
                with torch.no_grad():
                    out = net(*inp)
                diff = (ref - out).abs().max().item()
                n_tr = sum(p.numel() for p in net.parameters() if p.requires_grad)
                n_all = sum(p.numel() for p in net.parameters())
                net.train()
                net(*inp).float().pow(2).mean().backward()
                grads = any(p.grad is not None and float(p.grad.abs().sum()) > 0 for p in net.parameters() if p.requires_grad)
                frozen = all(p.grad is None for p in net.parameters() if not p.requires_grad)
                good = diff < 1e-4 and n_tr < n_all and grads and frozen
                ok &= good
                log(f"  {tag:<18} |adapted - base| = {diff:.1e}  trainable {n_tr:,}/{n_all:,}  {'PASS' if good else 'FAIL'}")
            except Exception as e:
                import traceback
                traceback.print_exc()
                log(f"  {tag:<18} FAIL: {type(e).__name__}: {e}")
                ok = False

    log("=== 3. end-to-end: every architecture / method, then ensembles ===")
    p1dirs = {}
    for arch in P1.ARCH_PRESETS:
        p1 = os.path.join(tmp, f"p1_{arch}")
        os.makedirs(p1, exist_ok=True)
        torch.save({"model": P1.LM(N._tiny_mc(arch, V)).state_dict(), "mc": N._tiny_mc(arch, V), "state": {"step": 0}},
                   os.path.join(p1, "ckpt_best.pt"))
        tok.save(os.path.join(p1, "tokenizer.json"))
        p1dirs[arch] = p1
    base = {"work_dir": os.path.join(tmp, "work"), "local_data_dir": dd, "max_len": 96, "epochs": 2, "batch_size": 8,
            "eval_batch": 8, "lr": 1e-3, "peft_lr": 3e-3, "head_lr": 3e-3, "peft_r": 4, "peft_alpha": 8.0, "sent_dim": 32,
            "head_hidden": 16, "patience": 5, "ema_decay": 0.9, "crop_windows": 2, "batch_docs": 2}
    cases = [(f"full_{a}", a, {"method": "full"}) for a in P1.ARCH_PRESETS] + [
        ("lora", "llama", {"method": "lora"}), ("rslora", "gpt2", {"method": "rslora"}), ("ia3", "moe", {"method": "ia3"}),
        ("lorma_plus", "gpt2", {"method": "lorma", "lorma_variant": "plus"}),
        ("lorma_pi", "retnet", {"method": "lorma", "lorma_variant": "pi"}),
        ("sentence_only", "llama", {"method": "full", "ctx_mode": "sentence"}),
        ("window_mode", "llama", {"method": "full", "ctx_mode": "doc"}),
        ("kd", "gpt2", {"method": "kd", "teacher_run": "selftest_full_llama", "student_layers": 1})]
    for tag, arch, extra in cases:
        try:
            cfg = apply_overrides(RCfg(), {**base, **extra, "run": f"selftest_{tag}", "p1_dir": p1dirs[arch]})
            run_train3(cfg)
            with zipfile.ZipFile(run_dir3(cfg) / "submission.zip") as z:
                assert z.namelist() == ["prediction.json"]
                pr = json.loads(z.read("prediction.json"))
            tests = {d.id: d.n for d in data["test"]}
            assert set(pr) == set(tests), "document ids"
            assert all(len(pr[k]) == tests[k] for k in tests), "exactly one label per sentence"
            assert all(x in OFFICIAL_LABELS for v in pr.values() for x in v), "labels from the vocabulary"
            rep = P1.jload(run_dir3(cfg) / "dev_report.json")
            if extra["method"] in ("lora", "rslora", "ia3", "lorma"):
                assert rep["params_trainable"] < rep["params_total"]
            if extra["method"] == "kd":
                assert rep["params_total"] < rep["teacher_params"]
            log(f"  e2e {tag:<14} PASS  (dev macro-F1 {rep['macro_f1']:.3f}, trainable {rep['params_trainable']:,} / {rep['params_total']:,})")
        except Exception as e:
            import traceback
            traceback.print_exc()
            log(f"  e2e {tag:<14} FAIL: {type(e).__name__}: {e}")
            ok = False
    try:
        for tag, ov in (("ens", {"ens_runs": ["selftest_full_llama", "selftest_full_gpt2"]}),
                        ("ens_auto", {"ens_auto": True, "ens_prefix": "selftest_"})):
            cfg = apply_overrides(RCfg(), {"work_dir": os.path.join(tmp, "work"), "local_data_dir": dd, "run": f"selftest_{tag}", **ov})
            zp = run_ensemble3(cfg)
            with zipfile.ZipFile(zp) as z:
                pr = json.loads(z.read("prediction.json"))
            assert all(len(pr[d.id]) == d.n for d in data["test"])
        log("  ensembles (explicit + automatic) PASS")
    except Exception as e:
        import traceback
        traceback.print_exc()
        log(f"  ensembles FAIL: {type(e).__name__}: {e}")
        ok = False
    shutil.rmtree(tmp, ignore_errors=True)
    log("SELFTEST " + ("PASSED" if ok else "FAILED"))
    return ok


# ============================================================================================
# 9. CLI
# ============================================================================================
def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("stage", choices=["selftest", "check", "train", "predict", "ensemble"])
    ap.add_argument("--run", default=None)
    ap.add_argument("--cfg", default="", help="JSON string or JSON file with config overrides")
    args = ap.parse_args(argv)
    if args.stage == "selftest":
        return 0 if selftest() else 1
    cfg = RCfg()
    ov = {}
    if args.cfg:
        ov = json.loads(args.cfg) if args.cfg.strip().startswith("{") else P1.jload(args.cfg)
    if args.run:
        ov["run"] = args.run
    apply_overrides(cfg, ov)
    Path(cfg.work_dir).mkdir(parents=True, exist_ok=True)
    {"check": diagnose3, "train": run_train3, "predict": run_predict3, "ensemble": run_ensemble3}[args.stage](cfg)
    return 0


if __name__ == "__main__":
    sys.exit(main())

## Self-test (once, ~5-8 min). Must end with `SELFTEST PASSED`. Checks adapters start as the identity and runs every method end to end on synthetic data.

In [ ]:
!python cs781_phase3_rr.py selftest

## Setup: locate the Phase-1 backbones

In [ ]:
import os, json, glob, subprocess
WORK = "/kaggle/working/cs781" if os.path.exists("/kaggle/working") else "./cs781_work"
P1_DIRS = {a: f"{WORK}/runs/explore_{a}" for a in ["llama", "gpt2", "retnet", "moe"]}
P1_CKPT, missing = {}, []
for a, d in P1_DIRS.items():
    if not os.path.exists(f"{d}/tokenizer.json"): missing.append(a)
    elif os.path.exists(f"{d}/ckpt_best.pt"): P1_CKPT[a] = "ckpt_best.pt"
    elif os.path.exists(f"{d}/ckpt_last.pt"): P1_CKPT[a] = "ckpt_last.pt"
    else: missing.append(a)
for a, d in P1_DIRS.items(): print(f"{a:<7}", f"OK ({P1_CKPT[a]})" if a in P1_CKPT else "MISSING", d)

def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print(line, end="")
    p.wait(); return p.returncode

def run_rr(run, arch, extra):
    if os.path.exists(f"{WORK}/rr_runs/{run}/submission.zip"):
        print("already done:", run, "(delete that folder to re-run)"); return
    if arch not in P1_CKPT:
        print(f"Phase-1 backbone '{arch}' is missing -> train it first, then re-run the setup cell."); return
    cfg = {"p1_dir": P1_DIRS[arch], "p1_ckpt": P1_CKPT[arch], **extra}
    fn = f"cfg_rr_{run}.json"; json.dump(cfg, open(fn, "w"), indent=1)
    print("=" * 100, "\n", run, "|", arch, "|", extra, "\n", "=" * 100)
    print("exit code", sh(f"python cs781_phase3_rr.py train --run {run} --cfg {fn}"))
    report(run)

def report(run, top=8):
    p = f"{WORK}/rr_runs/{run}/dev_report.json"
    if not os.path.exists(p): print(f"\n{run}: no dev_report.json (not finished)"); return
    r = json.load(open(p))
    print(f"\n=== {run} | {r.get('p1_arch', 'ensemble')} / {r.get('method', 'ensemble')} | best epoch {r.get('best_epoch')} | transition weight {r.get('lam')}")
    print(f"dev macro-F1 {r['macro_f1']:.3f}  (argmax only: {r.get('macro_f1_no_bias', float('nan')):.3f})   P {r.get('macro_p', float('nan')):.3f}   R {r.get('macro_r', float('nan')):.3f}   CL {r.get('dev_cl_f1', float('nan')):.3f}   IT {r.get('dev_it_f1', float('nan')):.3f}")
    for l, v in r.get("per_label", {}).items(): print(f"   {l:<24} P={v['P']:.2f} R={v['R']:.2f} F1={v['F1']:.2f} support={v['support']}")
    for c in r.get("confusions", [])[:top]: print(f"   confusion: {c['count']:>4}  {c['true']} -> {c['pred']}")

## (Only if a backbone is MISSING) train it now - about 0.75 h each, resumes if partially done

In [ ]:
for a in missing:
    fn = f"cfg_p1_{a}.json"; json.dump({"arch": a, "time_budget_hours": 0.75}, open(fn, "w"))
    print("training Phase-1 backbone:", a); sh(f"python cs781_phase1.py train --run explore_{a} --preset turbo --cfg {fn}")
if not missing: print("nothing to train - all four backbones exist")

In [ ]:
RUN_CONTINUE, CONT_ARCH, CONT_HOURS = False, "llama", 3.0
if RUN_CONTINUE and CONT_ARCH in P1_CKPT:
    fn = f"cfg_p1_{CONT_ARCH}_cont.json"
    json.dump({"arch": CONT_ARCH, "time_budget_hours": CONT_HOURS, "init_from": f"{P1_DIRS[CONT_ARCH]}/{P1_CKPT[CONT_ARCH]}",
               "lr": 4e-4, "warmup_steps": 100, "seed": 2024}, open(fn, "w"))
    print("exit code", sh(f"python cs781_phase1.py train --run explore_{CONT_ARCH}_cont --preset turbo --cfg {fn}"))
    # afterwards use the stronger backbone:
    P1_DIRS[CONT_ARCH] = f"{WORK}/runs/explore_{CONT_ARCH}_cont"; P1_CKPT[CONT_ARCH] = "ckpt_best.pt"
    print(CONT_ARCH, "-> now uses", P1_DIRS[CONT_ARCH])

## Check the real data first (1-2 min)
Prints, per split and domain: documents, sentences per document, tokens per sentence, number of windows, label distribution, and whether CL and IT test ids overlap (the output format maps *ids* to labels, so overlapping ids would be a problem). If anything looks off, paste the output to me.

In [ ]:
sh("python cs781_phase3_rr.py check --run rr_check --cfg " + json.dumps(json.dumps({"p1_dir": P1_DIRS["llama"], "p1_ckpt": P1_CKPT.get("llama", "ckpt_best.pt")})))

## Teacher / strong baseline: full fine-tuning of llama (also the teacher of R4)

In [ ]:
run_rr("rr2_full_llama", "llama", {"method": "full", "epochs": 10, "lr": 3e-4, "head_layers": 2})

## R1 - LoRMA+ (gpt2) - compulsory
`h = (alpha/r BA + I) W0 x`: multiplicative low-rank adaptation. Backbone frozen and causal; document context comes from the BiGRU over sentence vectors.

In [ ]:
ADAPT = {"bidir": "false", "peft_r": 32, "peft_alpha": 64, "epochs": 14, "peft_lr": 1e-3, "head_layers": 2}
run_rr("rr2_lorma_gpt2", "gpt2", {"method": "lorma", "lorma_variant": "plus", **ADAPT})

## R2 - rsLoRA (llama): LoRA with the rank-stabilised scale `alpha / sqrt(r)`

In [ ]:
run_rr("rr2_rslora_llama", "llama", {"method": "rslora", **{**ADAPT, "peft_alpha": 16}})

## R3 - IA3 (moe): learned scaling vectors on keys, values and FFN activations (a few thousand trainable parameters)

In [ ]:
run_rr("rr2_ia3_moe", "moe", {"method": "ia3", "bidir": "false", "epochs": 20, "peft_lr": 3e-3, "head_layers": 2})

## R4 - knowledge distillation (retnet student pruned to 6 of 12 blocks <- `rr_full_llama` teacher): `0.5 CE + 0.5 T^2 KL(teacher || student)`, T = 2

In [ ]:
run_rr("rr2_kd_retnet", "retnet", {"method": "kd", "teacher_run": "rr2_full_llama", "student_layers": 6, "epochs": 10, "lr": 3e-4, "head_layers": 2})

## Analysis runs and extras (optional)
* **Context ablation** (required in the analysis): the same full fine-tuning but every sentence is classified alone.
* More full fine-tunings and adapters feed the ensemble.

In [ ]:
run_rr("rr2_full_llama_windows",  "llama", {"method": "full", "ctx_mode": "doc", "epochs": 10, "lr": 3e-4, "head_layers": 2})
run_rr("rr2_full_llama_sentence", "llama", {"method": "full", "ctx_mode": "sentence", "epochs": 10, "lr": 3e-4})
run_rr("rr2_full_gpt2", "gpt2", {"method": "full", "epochs": 10, "lr": 3e-4, "head_layers": 2})
run_rr("rr2_full_moe",  "moe",  {"method": "full", "epochs": 10, "lr": 3e-4, "head_layers": 2})
# run_rr("rr2_full_llama_s2", "llama",  {"method": "full", "epochs": 10, "lr": 3e-4, "head_layers": 2, "seed": 7})
# run_rr("rr2_full_retnet",   "retnet", {"method": "full", "epochs": 10, "lr": 3e-4, "head_layers": 2})
# run_rr("rr2_lora_llama",    "llama",  {"method": "lora", **ADAPT})
# run_rr("rr2_lormapi_gpt2",  "gpt2",   {"method": "lorma", "lorma_variant": "pi", **ADAPT})

In [ ]:
json.dump({"ens_auto": True, "ens_prefix": "rr2_", "ens_max": 6}, open("cfg_rr_ens.json", "w"))
print("exit code", sh("python cs781_phase3_rr.py ensemble --run rr2_ensemble --cfg cfg_rr_ens.json"))
report("rr2_ensemble", top=0)

## Results, efficiency comparison and the files to submit

In [ ]:
import pandas as pd, shutil
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
df = pd.read_csv(f"{WORK}/rr_experiments.csv"); df = df[~df.run.str.startswith(("selftest", "rr_check"))]
cols = ["run", "p1_arch", "method", "variant", "ctx_mode", "params_total_M", "params_trainable_M", "dev_macro_f1", "dev_macro_p", "dev_macro_r",
        "dev_cl_f1", "dev_it_f1", "train_s", "peak_gpu_mb", "infer_sent_s"]
display(df[cols].sort_values("dev_macro_f1", ascending=False))
os.makedirs("phase3_submissions", exist_ok=True)
REQUIRED = ["rr2_lorma_gpt2", "rr2_rslora_llama", "rr2_ia3_moe", "rr2_kd_retnet"]
EXTRA = ["rr2_full_llama", "rr2_ensemble"]
for r in REQUIRED + EXTRA:
    src = f"{WORK}/rr_runs/{r}/submission.zip"
    if os.path.exists(src):
        shutil.copy(src, f"phase3_submissions/{r}.zip")
        row = df[df.run == r].tail(1)
        print(("REQUIRED " if r in REQUIRED else "extra    "), r, "| dev macro-F1", float(row.dev_macro_f1.iloc[0]) if len(row) else "?")
    else: print("missing:", r)

# CS781 Phase 4 - Court Judgment Prediction (binary outcome from a long judgment)

**Append at the end of your existing notebook** (same session, so `/kaggle/working/cs781/runs/explore_*` exist) or run standalone with your Phase-1 output attached.

### How the model reads a judgment
Cases are far longer than the 512-token window, so each case is split into chunks **built from the end of the document backwards** (the last chunk is always complete), a configurable subset of chunks goes through the Phase-1 backbone (`select`: tail / head / head_tail / uniform, at most `max_chunks`), every chunk becomes a vector, and an **attention pool over chunks** (plus the last-chunk vector, plus chunk-position features) feeds an MLP. The decision threshold is tuned on dev for macro-F1.
The default `select: tail` reflects that the closing reasoning of a judgment carries most of the outcome signal; the ablation cells below test this on *your* data (tail vs. head vs. head+tail vs. uniform, pooling variants) - that is the evidence the analysis asks for.

| Run | Phase-1 backbone | Technique |
|---|---|---|
| `cj_lorma_gpt2` | gpt2 | **LoRMA+** (compulsory) |
| `cj_full_llama` | llama | **full fine-tuning** (also the teacher of the KD run) |
| `cj_adapter_moe` | moe | **Houlsby adapters** |
| `cj_kd_retnet` | retnet | **knowledge distillation** (compulsory): pruned student <- teacher |

Extras: BitFit, layer freezing, LoRA, pooling / section ablations, ensemble. Everything is resumable - a finished run is skipped when you re-run its cell.
Each main run is limited by `BUDGET` minutes of training (the LR schedule follows whichever of epochs / time ends first), so total time is predictable: 4 runs ~ 4 x (BUDGET + a few minutes for dev/test inference).

In [ ]:
import base64
open("cs781_phase1.py", "w", encoding="utf-8").write(base64.b64decode("...").decode("utf-8"))
open("cs781_phase2_ner.py", "w", encoding="utf-8").write(base64.b64decode("...").decode("utf-8"))
print("wrote cs781_phase1.py and cs781_phase2_ner.py")

In [ ]:
%%writefile cs781_phase4_cjpe.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
CS781 Phase 4 - Court Judgment Prediction (binary outcome, long legal documents) on top of YOUR Phase-1 language models.

Needs cs781_phase1.py and cs781_phase2_ner.py in the same folder (model classes, LoRA / LoRMA adapters, distillation helper).

Model (hierarchical, long-document aware)
  case text -> Phase-1 BPE tokens -> chunks of max_len tokens (built from the END of the document backwards, so the last chunk is complete)
  -> chunk selection ("select": tail | head | head_tail | uniform) keeps at most max_chunks chunks
  -> Phase-1 backbone (bidirectional attention, or causal for retnet) -> chunk vector = [mean of token states ; state of the closing <eos>]
  -> + chunk-position embeddings (relative position in the case, "is last chunk")
  -> pooling over chunks ("pool": attn | gru | mean | last) -> [pooled ; last-chunk vector] -> MLP -> 2 logits
  Decision threshold tuned on dev for macro-F1.   Output: [{"id": ..., "pred": 0|1}, ...]

Adaptation methods (cfg "method")
  full      all weights                         partial   freeze embeddings + the first `freeze_layers` blocks
  bitfit    only biases + normalisation scales  adapter   Houlsby bottleneck adapters after every attention/FFN sub-layer
  lora      additive low-rank                   lorma     multiplicative low-rank (LoRMA+ / LoRMAπ)
  kd        knowledge distillation: pruned student <- fine-tuned teacher (logits)

Stages:  selftest | check | probe | train | predict | ensemble
  check  : data statistics + duplicate / train-test overlap report      probe : which part of a case carries the label? (diagnostic only)
  python cs781_phase4_cjpe.py check --run cj_check --cfg '{"p1_dir": ".../explore_llama"}'
  python cs781_phase4_cjpe.py train --run cj_full_llama --cfg cfg_cj.json
"""
from __future__ import annotations

import argparse
import contextlib
import csv
import gc
import hashlib
import json
import math
import os
import random
import re
import shutil
import sys
import time
import zipfile
from dataclasses import dataclass, field, fields, asdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import cs781_phase1 as P1
import cs781_phase2_ner as N

log = P1.log


# ============================================================================================
# 0. Config
# ============================================================================================
@dataclass
class JCfg:
    run: str = "cj_full_v1"
    work_dir: str = P1.DEFAULT_WORK
    p1_dir: str = ""
    p1_ckpt: str = "ckpt_best.pt"
    hf_repo: str = "Exploration-Lab/CS-781-Capstone"
    local_data_dir: str = ""          # folder with train.json, dev.json, test.json
    train_prefix: str = "Phase_4_CJPE/Train"
    test_file: str = "Phase_4_CJPE/Test/test.json"
    max_train_docs: int = 0           # debug / speed: use at most this many training cases (0 = all)
    dedupe_train: bool = True         # drop exact duplicate cases (the single- and multi-instance sets overlap) from train AND dev
    probe_docs: int = 12000           # probe stage: training cases used
    # ---- long-document representation
    max_len: int = 512                # tokens per chunk (<= Phase-1 context)
    max_chunks: int = 8               # chunks per case that go through the backbone
    select: str = "tail"              # tail | head | head_tail | uniform   (which chunks are kept)
    overlap: int = 0                  # tokens shared by consecutive chunks
    head_chunks: int = 0              # select=head_tail: chunks taken from the START of the case (0 = max_chunks // 4)
    pool: str = "attn"                # attn | gru | mean | last   (how chunk vectors are combined)
    chunk_dim: int = 384
    head_dropout: float = 0.1
    bidir: str = "auto"               # auto = bidirectional attention for gpt2/llama/moe, causal for retnet
    # ---- method
    method: str = "full"              # full | partial | bitfit | adapter | lora | lorma | kd
    freeze_layers: int = 0
    adapter_dim: int = 64
    peft_r: int = 16
    peft_alpha: float = 32.0
    peft_dropout: float = 0.05
    peft_lr: float = 5e-4
    peft_targets: str = "all"
    peft_experts: bool = False
    lorma_variant: str = "plus"
    teacher_run: str = ""
    student_layers: int = 0
    kd_alpha: float = 0.5
    kd_temp: float = 2.0
    # ---- optimisation
    epochs: int = 6
    docs_per_epoch: int = 0           # 0 = all training cases per epoch; else a random subset (faster epochs for huge training sets)
    batch_docs: int = 2
    accum: int = 2
    eval_docs: int = 4
    lr: float = 2e-4
    llrd: float = 0.85
    head_lr: float = 1e-3
    weight_decay: float = 0.01
    warmup_frac: float = 0.06
    grad_clip: float = 1.0
    class_weight: str = "balanced"    # none | balanced
    label_smoothing: float = 0.0
    ema_decay: float = 0.995
    time_budget_min: float = 45.0     # hard wall-clock limit of the training loop (the LR schedule follows whichever of epochs / time ends first)
    dev_max_docs: int = 600           # model selection on a fixed random subset of dev (final scores use all of dev)
    eval_every_steps: int = 3000      # also evaluate every N optimiser steps (0 = only at epoch ends) - time-limited runs never finish an epoch
    patience: int = 4                 # evaluations without improvement before stopping
    seed: int = 1337
    predict_after: bool = True
    # ---- ensemble stage
    ens_runs: list = field(default_factory=list)
    ens_auto: bool = False
    ens_prefix: str = ""
    ens_max: int = 6


def apply_overrides(cfg: JCfg, ov: dict) -> JCfg:
    names = {f.name for f in fields(JCfg)}
    for k, v in ov.items():
        if k not in names:
            raise SystemExit(f"Unknown config key {k!r}. Valid: {sorted(names)}")
        setattr(cfg, k, v)
    return cfg


def run_dir4(cfg: JCfg) -> Path:
    return Path(cfg.work_dir) / "cj_runs" / cfg.run


CJ_FIELDS = ["time", "run", "p1_arch", "method", "pool", "select", "max_chunks", "params_total_M", "params_trainable_M",
             "dev_macro_f1", "dev_precision", "dev_recall", "dev_acc", "threshold", "train_s", "peak_gpu_mb", "infer_docs_s",
             "teacher_run"]


def log_csv(cfg: JCfg, row: dict):
    p = Path(cfg.work_dir) / "cj_experiments.csv"
    if p.exists() and open(p).readline().strip().split(",") != CJ_FIELDS:
        p.rename(p.with_suffix(".old.csv"))
    new = not p.exists()
    with open(p, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=CJ_FIELDS, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerow({"time": time.strftime("%Y-%m-%d %H:%M:%S"), **row})


# ============================================================================================
# 1. Data
# ============================================================================================
class JDoc:
    pass


def _first(d, keys):
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return None


def load_split(cfg: JCfg, rel: str, tok, labelled: bool, max_docs: int = 0):
    raw = P1.jload(P1.resolve_file(cfg, rel))
    items = raw if isinstance(raw, list) else [dict(v, id=k) if isinstance(v, dict) else {"id": k, "text": v} for k, v in raw.items()]
    if max_docs:
        items = items[:max_docs]
    assert tok.get_vocab_size() < 65536, "token ids are stored as uint16"
    ids, texts, labels = [], [], []
    for i, r in enumerate(items):
        did = _first(r, ("id", "case_id", "doc_id", "document_id"))
        ids.append(str(did) if did is not None else str(i))
        t = _first(r, ("text", "document", "case_text", "content")) or ""
        texts.append(" ".join(" ".join(t).split() if isinstance(t, list) else t.split()))
        if labelled:
            y = _first(r, ("label", "gold", "labels", "y", "outcome", "decision"))
            if y is None or isinstance(y, str) and not y.strip().lstrip("-").isdigit() and y not in ("0", "1"):
                raise SystemExit(f"{rel}: case {ids[-1]} has no usable binary label (got {y!r})")
            y = int(float(y))
            if y not in (0, 1):
                raise SystemExit(f"{rel}: case {ids[-1]} has label {y}, expected 0 or 1")
            labels.append(y)
    docs = []
    for b0 in range(0, len(texts), 128):
        for k, e in enumerate(tok.encode_batch(texts[b0:b0 + 128])):
            d = JDoc()
            d.id, d.tok = ids[b0 + k], np.asarray(e.ids, dtype=np.uint16)
            d.n, d.label = len(d.tok), (labels[b0 + k] if labelled else -1)
            docs.append(d)
    if len({d.id for d in docs}) != len(docs):
        log(f"[data] WARNING: duplicate case ids in {rel}")
    return docs


def chunk_bounds(n: int, L: int, overlap: int):
    """Chunks of <= L tokens built from the END of the document backwards (the last chunk is always complete)."""
    if n <= 0:
        return [(0, 0)]
    step = max(1, L - overlap)
    out, e = [], n
    while e > 0:
        s = max(0, e - L)
        out.append((s, e))
        if s == 0:
            break
        e -= step
    return out[::-1]


def select_chunks(m: int, k: int, strategy: str, head: int = 0):
    """Indices (into the m chunks of a case) of the chunks that are kept, in document order."""
    if m <= k:
        return list(range(m))
    if strategy == "head":
        return list(range(k))
    if strategy == "head_tail":
        h = min(k, head if head > 0 else max(1, k // 4))
        return list(range(h)) + list(range(m - (k - h), m))
    if strategy == "uniform":
        return sorted({int(x) for x in np.linspace(0, m - 1, k).round()})
    return list(range(m - k, m))                       # tail (default)


def collate_docs(docs, idxs, cfg: JCfg, dev, eos, n_chunks):
    chunks, owner, slot, rel, last, counts = [], [], [], [], [], []
    for b, di in enumerate(idxs):
        d = docs[di]
        bounds = chunk_bounds(d.n, cfg.max_len - 1, cfg.overlap)
        sel = select_chunks(len(bounds), n_chunks, cfg.select, cfg.head_chunks)
        for k_, j in enumerate(sel):
            s, e = bounds[j]
            chunks.append(d.tok[s:e])
            owner.append(b)
            slot.append(k_)
            rel.append(min(7, int(8 * j / len(bounds))))
            last.append(int(j == len(bounds) - 1))
        counts.append(len(sel))
    L = max(len(c) for c in chunks) + 1
    ids = np.zeros((len(chunks), L), dtype=np.int64)
    km = np.zeros((len(chunks), L), dtype=bool)
    for i, c in enumerate(chunks):
        n = len(c)
        ids[i, :n] = c
        ids[i, n] = eos
        km[i, :n + 1] = True
    y = np.array([docs[di].label for di in idxs], dtype=np.int64)
    t = lambda x: torch.from_numpy(x).to(dev)
    return t(ids), t(km), t(np.array(owner)), t(np.array(slot)), t(np.array(rel)), t(np.array(last)), t(np.array(counts)), t(y)


def dedupe_docs(docs):
    """Drop exact duplicate cases (same token sequence). Returns (kept, n_removed, n_removed_with_conflicting_label)."""
    seen, keep, conflict = {}, [], 0
    for d in docs:
        h = hashlib.md5(d.tok.tobytes()).hexdigest()
        if h in seen:
            conflict += int(seen[h] != d.label)
            continue
        seen[h] = d.label
        keep.append(d)
    return keep, len(docs) - len(keep), conflict


def overlap_report(tr, dv, te):
    """How many dev / test cases also occur in train: identical text, identical first 256 tokens, identical last 256 tokens.
    (A large overlap, especially with agreeing labels, means the benchmark can be 'solved' by memorising train cases.)"""
    def key(d, mode):
        t = d.tok[:256] if mode == "first 256 tokens" else d.tok[-256:] if mode == "last 256 tokens" else d.tok
        return hashlib.md5(t.tobytes()).hexdigest()
    rows = []
    for mode in ("full text", "first 256 tokens", "last 256 tokens"):
        tmap = {}
        for d in tr:
            tmap.setdefault(key(d, mode), []).append(d.label)
        for name, docs in (("dev", dv), ("test", te)):
            hit = [(d, tmap[key(d, mode)]) for d in docs if key(d, mode) in tmap]
            agree = float(np.mean([(np.mean(ls) >= 0.5) == bool(d.label) for d, ls in hit])) if (name == "dev" and hit) else None
            rows.append((name, mode, len(hit), len(docs), agree))
            log(f"[overlap] {name:<4} cases whose {mode:<16} also occurs in train: {len(hit):>5}/{len(docs)}"
                + (f" | train label agrees for {100 * agree:.1f}% of them" if agree is not None else ""))
    return rows


# ============================================================================================
# 2. Metrics (identical definitions to the official evaluator)
# ============================================================================================
def binary_scores(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat)
    tp = np.array([((y == c) & (yhat == c)).sum() for c in (0, 1)], dtype=np.float64)
    fp = np.array([((y != c) & (yhat == c)).sum() for c in (0, 1)], dtype=np.float64)
    fn = np.array([((y == c) & (yhat != c)).sum() for c in (0, 1)], dtype=np.float64)
    f1 = np.divide(2 * tp, 2 * tp + fp + fn, out=np.zeros(2), where=(2 * tp + fp + fn) > 0)
    prec = tp[1] / (tp[1] + fp[1]) if tp[1] + fp[1] else 0.0
    rec = tp[1] / (tp[1] + fn[1]) if tp[1] + fn[1] else 0.0
    return {"macro_f1": float(f1.mean()), "f1_per_class": f1.tolist(), "precision": float(prec), "recall": float(rec),
            "acc": float((y == yhat).mean()) if len(y) else 0.0, "tp": tp.tolist(), "fp": fp.tolist(), "fn": fn.tolist()}


def best_threshold(y, p1, min_gain=0.003):
    base = binary_scores(y, (p1 >= 0.5).astype(int))["macro_f1"]
    best_t, best_f = 0.5, base
    for t in np.arange(0.2, 0.801, 0.02):
        f = binary_scores(y, (p1 >= t).astype(int))["macro_f1"]
        if f > best_f + min_gain:
            best_t, best_f = float(round(t, 2)), f
    return best_t, best_f, base


# ============================================================================================
# 3. Adaptation methods beyond Phase 2: Houlsby adapters, BitFit, layer freezing
# ============================================================================================
class AdapterWrap(nn.Module):
    """Houlsby-style bottleneck adapter applied to the OUTPUT of a sub-layer (attention / retention / FFN) before the residual add.
    The up-projection is zero-initialised, so the network starts exactly as the Phase-1 model."""

    def __init__(self, sub, d, r):
        super().__init__()
        self.sub = sub
        self.down, self.up = nn.Linear(d, r), nn.Linear(r, d)
        nn.init.normal_(self.down.weight, 0.0, 0.02)
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, *a, **k):
        y = self.sub(*a, **k)
        return y + self.up(F.gelu(self.down(y)))


def apply_method4(net, cfg: JCfg):
    m, lm = cfg.method, net.lm
    if m in ("full", "kd"):
        return
    if m == "partial":
        for p in lm.wte.parameters():
            p.requires_grad = False
        if lm.wpe is not None:
            for p in lm.wpe.parameters():
                p.requires_grad = False
        for blk in lm.blocks[:cfg.freeze_layers]:
            for p in blk.parameters():
                p.requires_grad = False
        return
    if m == "bitfit":                                    # bias terms + normalisation scales (the models without biases train their norm scales)
        keys = (".n1.", ".n2.", "norm_f", ".qn.", ".kn.", ".on.")
        for n, p in lm.named_parameters():
            p.requires_grad = n.endswith("bias") or any(k in n for k in keys)
        log(f"[bitfit] trainable backbone parameters: {sum(p.numel() for p in lm.parameters() if p.requires_grad):,}")
        return
    if m == "adapter":
        for p in lm.parameters():
            p.requires_grad = False
        d = lm.mc["n_embd"]
        for blk in lm.blocks:
            blk.mix = AdapterWrap(blk.mix, d, cfg.adapter_dim)
            blk.ffn = AdapterWrap(blk.ffn, d, cfg.adapter_dim)
        log(f"[adapter] Houlsby adapters (bottleneck {cfg.adapter_dim}) after every attention and FFN sub-layer")
        return
    if m in ("lora", "lorma"):
        n = N.inject_peft(lm, cfg)
        log(f"[peft] {m}{'-' + cfg.lorma_variant if m == 'lorma' else ''}: wrapped {n} linear layers (r={cfg.peft_r})")
        return
    raise SystemExit(f"unknown method {m!r}: use full | partial | bitfit | adapter | lora | lorma | kd")


# ============================================================================================
# 4. Model
# ============================================================================================
def backbone_hidden(lm, ids, km, bidir):
    B, T = ids.shape
    dev = ids.device
    pos = torch.arange(T, device=dev).unsqueeze(0)
    x = lm.wte(ids)
    if lm.wpe is not None:
        x = x + lm.wpe(pos)
    rope = None
    if lm.rope_cos is not None:
        rope = (lm.rope_cos[pos].unsqueeze(1), lm.rope_sin[pos].unsqueeze(1))
    if bidir:
        amask = km[:, None, None, :]
    else:
        qi, kj = torch.arange(T, device=dev).unsqueeze(1), torch.arange(T, device=dev).unsqueeze(0)
        amask = ((kj <= qi)[None, None] & km[:, None, None, :]) | (kj == qi)[None, None]
    for blk in lm.blocks:
        x = blk(x, rope, amask, km, None)
    return lm.norm_f(x)


class CJNet(nn.Module):
    """Phase-1 backbone + chunk encoder + (BiGRU) + attention pooling over chunks + MLP classifier."""

    def __init__(self, lm, bidir, pool, chunk_dim, head_dropout):
        super().__init__()
        self.lm, self.bidir, self.pool = lm, bidir, pool
        d = lm.mc["n_embd"]
        self.proj = nn.Linear(2 * d, chunk_dim)
        self.norm = nn.LayerNorm(chunk_dim)
        self.pos = nn.Embedding(8, chunk_dim)
        self.lastemb = nn.Embedding(2, chunk_dim)
        for e in (self.pos, self.lastemb):
            nn.init.normal_(e.weight, 0.0, 0.02)
        if pool == "gru":
            self.rnn = nn.GRU(chunk_dim, chunk_dim // 2, batch_first=True, bidirectional=True)
        self.att = nn.Linear(chunk_dim, 1)
        self.drop = nn.Dropout(head_dropout)
        self.cls = nn.Sequential(nn.Linear(2 * chunk_dim, chunk_dim), nn.GELU(), nn.Dropout(head_dropout), nn.Linear(chunk_dim, 2))

    def forward(self, ids, km, owner, slot, rel, last, counts):
        h = backbone_hidden(self.lm, ids, km, self.bidir)                              # (N, L, d)
        kf = km.unsqueeze(-1).to(h.dtype)
        mean = (h * kf).sum(1) / kf.sum(1).clamp(min=1.0)                              # mean of the chunk's token states
        lastv = h[torch.arange(h.size(0), device=h.device), km.sum(1) - 1]             # state of the closing <eos>
        c = self.proj(torch.cat([mean, lastv], dim=-1))
        c = self.drop(F.gelu(self.norm(c + self.pos(rel) + self.lastemb(last))))      # (N, D)
        B, Cm = counts.numel(), int(counts.max())
        pad = c.new_zeros(B, Cm, c.size(-1)).index_put((owner, slot), c)               # (B, Cm, D) chunk sequences
        mask = torch.arange(Cm, device=c.device).unsqueeze(0) < counts.unsqueeze(1)
        if self.pool == "gru":
            pk = nn.utils.rnn.pack_padded_sequence(pad, counts.cpu(), batch_first=True, enforce_sorted=False)
            o, _ = self.rnn(pk)
            pad, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True, total_length=Cm)
        last_c = pad[torch.arange(B, device=c.device), counts - 1]
        if self.pool == "mean":
            pooled = (pad * mask.unsqueeze(-1).to(pad.dtype)).sum(1) / mask.sum(1, keepdim=True).to(pad.dtype)
        elif self.pool == "last":
            pooled = last_c
        else:                                                                          # attention pooling over chunks (attn | gru)
            sc = self.att(pad).squeeze(-1).float().masked_fill(~mask, -1e9)
            pooled = (sc.softmax(-1).unsqueeze(-1).to(pad.dtype) * pad).sum(1)
        return self.cls(self.drop(torch.cat([pooled, last_c], dim=-1)))                # (B, 2)


def build_net4(cfg: JCfg, lm):
    arch = lm.mc["arch"]
    bidir = (arch != "retnet") if cfg.bidir == "auto" else (cfg.bidir in ("true", "yes", "1", True))
    if arch == "retnet":
        bidir = False
    log(f"[model] arch={arch} bidirectional_attention={bidir} pool={cfg.pool} select={cfg.select} max_chunks={cfg.max_chunks}")
    return CJNet(lm, bidir, cfg.pool, cfg.chunk_dim, cfg.head_dropout)


def load_run4(cfg: JCfg, run_name: str, dev):
    from tokenizers import Tokenizer
    rd = Path(cfg.work_dir) / "cj_runs" / run_name
    ck = torch.load(rd / "best.pt", map_location="cpu", weights_only=False)
    saved = JCfg()
    apply_overrides(saved, {k: v for k, v in ck["cfg"].items() if k in {f.name for f in fields(JCfg)}})
    saved.work_dir = cfg.work_dir
    if cfg.local_data_dir:
        saved.local_data_dir = cfg.local_data_dir
    net = build_net4(saved, P1.LM(ck["mc"]))
    apply_method4(net, saved)
    net.load_state_dict(ck["net"])
    net.to(dev).eval()
    return {"rd": rd, "ck": ck, "saved": saved, "net": net, "tok": Tokenizer.from_file(str(rd / "tokenizer.json"))}


def load_teacher4(cfg: JCfg, dev, tok_path):
    tr = Path(cfg.work_dir) / "cj_runs" / cfg.teacher_run
    if not (tr / "best.pt").exists():
        raise SystemExit(f"teacher run not found: {tr}/best.pt (train it first)")
    if (tr / "tokenizer.json").read_bytes() != Path(tok_path).read_bytes():
        raise SystemExit("teacher and student use different tokenizers (distillation needs identical token ids)")
    r = load_run4(cfg, cfg.teacher_run, dev)
    for p in r["net"].parameters():
        p.requires_grad = False
    n = sum(p.numel() for p in r["net"].parameters())
    log(f"[kd] teacher {cfg.teacher_run}: arch={r['ck']['mc']['arch']} method={r['saved'].method} params={n / 1e6:.1f}M")
    return r["net"], n, r["ck"]["mc"], r["saved"]


def kd_loss_cls(s_logits, t_logits, T):
    """T^2 * KL(softmax(teacher/T) || softmax(student/T)), averaged over the batch."""
    sl = F.log_softmax(s_logits.float() / T, dim=-1)
    tp = F.softmax(t_logits.float() / T, dim=-1)
    return (tp * (torch.log(tp.clamp_min(1e-9)) - sl)).sum(-1).mean() * (T * T)


# ============================================================================================
# 5. Inference, EMA, training
# ============================================================================================
@torch.no_grad()
def doc_probs4(net, docs, cfg, dev, amp, eos, idxs=None, n_chunks=None):
    net.eval()
    idxs = list(range(len(docs))) if idxs is None else list(idxs)
    order = sorted(range(len(idxs)), key=lambda i: docs[idxs[i]].n)            # similar lengths together -> less padding
    out = np.zeros((len(idxs), 2), dtype=np.float32)
    for b0 in range(0, len(order), cfg.eval_docs):
        sel = order[b0:b0 + cfg.eval_docs]
        batch = collate_docs(docs, [idxs[i] for i in sel], cfg, dev, eos, n_chunks or cfg.max_chunks)
        with amp():
            lg = net(*batch[:7])
        out[sel] = lg.float().softmax(-1).cpu().numpy()
    return out


class EMA:
    def __init__(self, net, decay):
        self.net, self.decay = net, decay
        self.shadow = {n: p.detach().clone() for n, p in net.named_parameters() if p.requires_grad}

    @torch.no_grad()
    def update(self, step):
        d = min(self.decay, (1.0 + step) / (10.0 + step))
        for n, p in self.net.named_parameters():
            if n in self.shadow:
                self.shadow[n].mul_(d).add_(p.detach(), alpha=1.0 - d)

    @contextlib.contextmanager
    def swapped(self):
        backup = {}
        with torch.no_grad():
            for n, p in self.net.named_parameters():
                if n in self.shadow:
                    backup[n] = p.detach().clone()
                    p.copy_(self.shadow[n])
        try:
            yield
        finally:
            with torch.no_grad():
                for n, p in self.net.named_parameters():
                    if n in backup:
                        p.copy_(backup[n])


def lr_prog(base, prog, warm):
    if prog < warm:
        return base * max(prog, 1e-3) / warm
    return base * max(0.0, (1.0 - prog) / (1.0 - warm))


def run_train4(cfg: JCfg):
    rd = run_dir4(cfg)
    rd.mkdir(parents=True, exist_ok=True)
    jd = asdict(cfg)
    P1.jdump(jd, rd / "config.json")
    random.seed(cfg.seed), np.random.seed(cfg.seed), torch.manual_seed(cfg.seed)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    amp_dtype, amp = P1.amp_setup(dev)
    tok, lm, mc = N.load_p1(cfg)
    shutil.copy(Path(cfg.p1_dir) / "tokenizer.json", rd / "tokenizer.json")
    if cfg.method == "kd":
        if not cfg.teacher_run:
            raise SystemExit('method "kd" needs "teacher_run": the name of a finished CJPE run')
        lm, mc = N.make_student(lm, mc, cfg)
    eos = tok.token_to_id(P1.EOS)
    assert cfg.max_len <= mc["ctx"], "max_len exceeds the Phase-1 context length"
    t0 = time.time()
    tr = load_split(cfg, f"{cfg.train_prefix}/train.json", tok, True, cfg.max_train_docs)
    dv = load_split(cfg, f"{cfg.train_prefix}/dev.json", tok, True)
    if cfg.dedupe_train:
        tr, n1, c1 = dedupe_docs(tr)
        dv, n2, c2 = dedupe_docs(dv)
        log(f"[data] removed exact duplicates: {n1} from train ({c1} with a conflicting label), {n2} from dev ({c2} conflicting)")
    ytr, ydv = np.array([d.label for d in tr]), np.array([d.label for d in dv])
    log(f"[data] train {len(tr)} cases (label-1 share {ytr.mean():.2f}) | dev {len(dv)} cases (label-1 share {ydv.mean():.2f}) | "
        f"tokens/case median {int(np.median([d.n for d in tr]))} | tokenised in {time.time() - t0:.0f}s")
    teacher, teacher_params = None, 0
    if cfg.method == "kd":
        teacher, teacher_params, t_mc, _ = load_teacher4(cfg, dev, rd / "tokenizer.json")
        assert cfg.max_len <= t_mc["ctx"]

    net = build_net4(cfg, lm)
    apply_method4(net, cfg)
    net.to(dev)
    n_total = sum(p.numel() for p in net.parameters())
    n_train = sum(p.numel() for p in net.parameters() if p.requires_grad)
    log(f"[model] total params {n_total / 1e6:.1f}M | trainable {n_train / 1e6:.3f}M | method={cfg.method}")

    peft = cfg.method in ("lora", "lorma", "adapter", "bitfit")
    body_lr, body_wd = (cfg.peft_lr, 0.0) if peft else (cfg.lr, cfg.weight_decay)
    nb = mc["n_layer"]

    def lr_scale(name):
        if peft or cfg.llrd >= 1.0 or not name.startswith("lm."):
            return 1.0
        m_ = re.match(r"lm\.blocks\.(\d+)\.", name)
        if m_:
            return cfg.llrd ** (nb - 1 - int(m_.group(1)))
        return cfg.llrd ** nb if name.startswith(("lm.wte", "lm.wpe")) else 1.0

    buckets = {}
    for n_, p_ in net.named_parameters():
        if not p_.requires_grad:
            continue
        if n_.startswith("lm."):
            base, wd = body_lr * lr_scale(n_), (body_wd if p_.dim() >= 2 else 0.0)
        else:
            base, wd = cfg.head_lr, (cfg.weight_decay if p_.dim() >= 2 else 0.0)
        buckets.setdefault((round(base, 12), wd), []).append(p_)
    opt = torch.optim.AdamW([{"params": ps, "lr": b_, "base": b_, "weight_decay": wd} for (b_, wd), ps in buckets.items()],
                            betas=(0.9, 0.98), eps=1e-8)
    scaler = P1.make_scaler(dev == "cuda" and amp_dtype == torch.float16)
    cw = None
    if cfg.class_weight == "balanced":
        cnt = np.bincount(ytr, minlength=2).astype(np.float64)
        cw = torch.tensor(np.clip(len(ytr) / (2.0 * np.maximum(cnt, 1)), 0.25, 4.0), dtype=torch.float32, device=dev)
    ema = EMA(net, cfg.ema_decay) if cfg.ema_decay > 0 else None

    rng = np.random.default_rng(cfg.seed)
    dev_sub = np.sort(rng.permutation(len(dv))[:cfg.dev_max_docs]) if cfg.dev_max_docs and len(dv) > cfg.dev_max_docs else np.arange(len(dv))
    n_epoch = len(tr) if not cfg.docs_per_epoch else min(cfg.docs_per_epoch, len(tr))
    steps_per_epoch = max(1, math.ceil(n_epoch / (cfg.batch_docs * cfg.accum)))
    total_steps = cfg.epochs * steps_per_epoch
    budget = cfg.time_budget_min * 60.0
    log(f"[train] {n_epoch} cases/epoch, {steps_per_epoch} steps/epoch x {cfg.epochs} epochs; time budget {cfg.time_budget_min} min; EMA {cfg.ema_decay}")
    best, step, bad, t_start, stop = {"f1": -1.0}, 0, 0, time.time(), False
    if dev == "cuda":
        torch.cuda.reset_peak_memory_stats()
    logf = open(rd / "log.jsonl", "a")

    def do_eval(tag, loss_val):
        """Evaluate the (EMA-averaged) weights on the dev subset; keep the best by threshold-tuned macro-F1."""
        nonlocal best, bad
        with (ema.swapped() if ema is not None else contextlib.nullcontext()):
            pdv_ = doc_probs4(net, dv, cfg, dev, amp, eos, idxs=dev_sub)
            sc_ = binary_scores(ydv[dev_sub], pdv_.argmax(1))
            thr_, f_tuned, _ = best_threshold(ydv[dev_sub], pdv_[:, 1])
            log(f"[{tag}] loss {loss_val:.4f} | dev(subset {len(dev_sub)}) macro-F1 {sc_['macro_f1']:.4f} at 0.5, "
                f"{f_tuned:.4f} at tuned threshold {thr_} | acc {sc_['acc']:.3f} | {time.time() - t_start:.0f}s")
            logf.write(json.dumps({"tag": tag, "step": step, "loss": loss_val, "dev_f1": sc_["macro_f1"], "dev_f1_tuned": f_tuned}) + "\n")
            logf.flush()
            if f_tuned > best["f1"]:
                best, bad = {"f1": f_tuned, "epoch": tag}, 0
                torch.save({"net": net.state_dict(), "mc": mc, "cfg": jd, "threshold": 0.5}, rd / "best.pt")
            else:
                bad += 1
        net.train()

    for ep in range(cfg.epochs):
        net.train()
        order = rng.permutation(len(tr))[:n_epoch]
        run_loss, n_b, early = 0.0, 0, False
        micro = cfg.batch_docs
        for g0 in range(0, len(order), micro * cfg.accum):
            prog = max(step / total_steps, (time.time() - t_start) / budget if budget > 0 else 0.0)
            if prog >= 1.0:
                stop = True
                break
            for g in opt.param_groups:
                g["lr"] = lr_prog(g["base"], prog, cfg.warmup_frac)
            group = order[g0:g0 + micro * cfg.accum]
            for m0 in range(0, len(group), micro):
                batch = collate_docs(tr, group[m0:m0 + micro], cfg, dev, eos, cfg.max_chunks)
                y = batch[7]
                with amp():
                    logits = net(*batch[:7])
                    if teacher is not None:
                        with torch.no_grad():
                            t_logits = teacher(*batch[:7])
                loss = F.cross_entropy(logits.float(), y, weight=cw, label_smoothing=cfg.label_smoothing)
                if teacher is not None:
                    loss = cfg.kd_alpha * loss + (1.0 - cfg.kd_alpha) * kd_loss_cls(logits, t_logits, cfg.kd_temp)
                scaler.scale(loss * (len(y) / len(group))).backward()
                run_loss += float(loss.detach())
                n_b += 1
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in net.parameters() if p.requires_grad], cfg.grad_clip)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            if ema is not None:
                ema.update(step)
            step += 1
            if step % 20 == 0:
                log(f"  ep {ep + 1} step {step}/{total_steps} loss {run_loss / max(1, n_b):.4f} ({time.time() - t_start:.0f}s)")
            if cfg.eval_every_steps and step % cfg.eval_every_steps == 0:
                do_eval(f"step {step}", run_loss / max(1, n_b))
                if bad >= cfg.patience:
                    early = stop = True
                    break
        if not early:
            do_eval(f"epoch {ep + 1}/{cfg.epochs}", run_loss / max(1, n_b))
        if early or bad >= cfg.patience:
            log(f"[train] early stopping: no dev improvement in {bad} evaluations (best: {best['epoch']})")
            break
        if stop:
            log("[train] time budget reached")
            break
    logf.close()
    train_s = time.time() - t_start
    peak = torch.cuda.max_memory_allocated() / 2 ** 20 if dev == "cuda" else 0.0

    ck = torch.load(rd / "best.pt", map_location="cpu", weights_only=False)
    net.load_state_dict(ck["net"])
    if dev == "cuda":
        torch.cuda.synchronize()
    t_inf = time.time()
    pdv = doc_probs4(net, dv, cfg, dev, amp, eos)
    if dev == "cuda":
        torch.cuda.synchronize()
    infer_docs_s = len(dv) / max(1e-9, time.time() - t_inf)
    thr, f_thr, f_base = best_threshold(ydv, pdv[:, 1])
    sc = binary_scores(ydv, (pdv[:, 1] >= thr).astype(int))
    ck["threshold"] = thr
    torch.save(ck, rd / "best.pt")
    np.save(rd / "dev_probs.npy", pdv)
    P1.jdump([d.id for d in dv], rd / "dev_ids.json")
    log(f"[dev] macro-F1 {f_base:.4f} (threshold 0.5) -> {sc['macro_f1']:.4f} (threshold {thr}) | precision {sc['precision']:.4f} "
        f"recall {sc['recall']:.4f} acc {sc['acc']:.4f}")
    log(f"[dev] per-class F1 {[round(x, 4) for x in sc['f1_per_class']]} | confusion tp {sc['tp']} fp {sc['fp']} fn {sc['fn']}")
    # error analysis: confident mistakes + accuracy by case length
    wrong = [(abs(pdv[i, 1] - 0.5), dv[i].id, int(ydv[i]), float(pdv[i, 1]), dv[i].n) for i in range(len(dv)) if (pdv[i, 1] >= thr) != ydv[i]]
    wrong.sort(reverse=True)
    errs = [{"id": i, "gold": g, "p1": round(p, 3), "tokens": n, "type": "false_positive" if g == 0 else "false_negative"} for _, i, g, p, n in wrong[:30]]
    P1.jdump(errs, rd / "errors.json")
    lens = np.array([d.n for d in dv])
    qs = np.quantile(lens, [0.25, 0.5, 0.75])
    pred_dv = (pdv[:, 1] >= thr).astype(int)
    for name, mk in (("shortest 25%", lens <= qs[0]), ("25-50%", (lens > qs[0]) & (lens <= qs[1])),
                     ("50-75%", (lens > qs[1]) & (lens <= qs[2])), ("longest 25%", lens > qs[2])):
        if mk.any():
            log(f"[dev] accuracy for {name:<13} cases ({int(mk.sum()):>4}): {float((pred_dv[mk] == ydv[mk]).mean()):.3f}")
    variant = cfg.lorma_variant if cfg.method == "lorma" else ""
    P1.jdump({"best_epoch": best["epoch"], "scores": sc, "threshold": thr, "macro_f1_default_threshold": f_base,
              "params_total": n_total, "params_trainable": n_train, "train_seconds": train_s, "peak_gpu_mb": peak,
              "infer_docs_per_s": infer_docs_s, "p1_arch": mc["arch"], "method": cfg.method, "teacher_run": cfg.teacher_run,
              "teacher_params": teacher_params}, rd / "dev_report.json")
    log_csv(cfg, {"run": cfg.run, "p1_arch": mc["arch"], "method": cfg.method + (f"-{variant}" if variant else ""), "pool": cfg.pool,
                  "select": cfg.select, "max_chunks": cfg.max_chunks, "params_total_M": round(n_total / 1e6, 2),
                  "params_trainable_M": round(n_train / 1e6, 3), "dev_macro_f1": round(sc["macro_f1"], 4),
                  "dev_precision": round(sc["precision"], 4), "dev_recall": round(sc["recall"], 4), "dev_acc": round(sc["acc"], 4),
                  "threshold": thr, "train_s": round(train_s), "peak_gpu_mb": round(peak), "infer_docs_s": round(infer_docs_s, 2),
                  "teacher_run": cfg.teacher_run})
    if cfg.predict_after:
        run_predict4(cfg)


def write_submission4(cfg: JCfg, rd: Path, ids, preds):
    out = [{"id": i, "pred": int(p)} for i, p in zip(ids, preds)]
    assert len(out) == len(set(ids)) and all(o["pred"] in (0, 1) for o in out)
    rd.mkdir(parents=True, exist_ok=True)
    pj = rd / "prediction.json"
    P1.jdump(out, pj)
    zp = rd / "submission.zip"
    with zipfile.ZipFile(zp, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    hist = Path(cfg.work_dir) / "cj_submissions"
    hist.mkdir(parents=True, exist_ok=True)
    shutil.copy(zp, hist / f"{cfg.run}_{time.strftime('%m%d_%H%M%S')}.zip")
    log(f"[predict] wrote {pj} ({len(out)} cases, share of 1 = {np.mean([o['pred'] for o in out]):.2f}) and {zp}")
    return str(zp)


def run_predict4(cfg: JCfg):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    _, amp = P1.amp_setup(dev)
    r = load_run4(cfg, cfg.run, dev)
    saved = r["saved"]
    eos = r["tok"].token_to_id(P1.EOS)
    te = load_split(saved, saved.test_file, r["tok"], False)
    p = doc_probs4(r["net"], te, saved, dev, amp, eos)
    np.save(r["rd"] / "test_probs.npy", p)
    P1.jdump([d.id for d in te], r["rd"] / "test_ids.json")
    thr = float(r["ck"].get("threshold", 0.5))
    log(f"[predict] {len(te)} test cases, threshold {thr}")
    return write_submission4(cfg, r["rd"], [d.id for d in te], (p[:, 1] >= thr).astype(int))


# ============================================================================================
# 6. Ensemble: average the saved probabilities of finished runs (no network needed)
# ============================================================================================
def run_ensemble4(cfg: JCfg):
    root = Path(cfg.work_dir) / "cj_runs"
    if cfg.ens_runs:
        names = list(cfg.ens_runs)
    elif cfg.ens_auto:
        names = [f.parent.name for f in sorted(root.glob("*/test_probs.npy"))
                 if (f.parent / "dev_probs.npy").exists() and ((f.parent.name.startswith(cfg.ens_prefix)) if cfg.ens_prefix
                                                              else not f.parent.name.startswith(("selftest", "check")))]
    else:
        names = []
    if len(names) < 2:
        raise SystemExit('need >= 2 finished runs: set "ens_runs": [...] or "ens_auto": true')
    dev_ids = P1.jload(root / names[0] / "dev_ids.json")
    test_ids = P1.jload(root / names[0] / "test_ids.json")
    dv_p, te_p = {}, {}
    for n in names:
        if P1.jload(root / n / "dev_ids.json") != dev_ids or P1.jload(root / n / "test_ids.json") != test_ids:
            log(f"[ensemble] skipping {n}: different case ids / order")
            continue
        dv_p[n], te_p[n] = np.load(root / n / "dev_probs.npy"), np.load(root / n / "test_probs.npy")
    # gold labels of dev, read once from the dev file through the first run's config
    saved = load_saved_cfg(cfg, names[0])
    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(str(root / names[0] / "tokenizer.json"))
    dvd = load_split(saved, f"{saved.train_prefix}/dev.json", tok, True)
    if saved.dedupe_train:
        dvd, _, _ = dedupe_docs(dvd)
    ydv = np.array([d.label for d in dvd])
    assert len(ydv) == len(next(iter(dv_p.values()))), "dev size differs from the saved probabilities"
    single = {k: binary_scores(ydv, v.argmax(1))["macro_f1"] for k, v in dv_p.items()}
    for k, v in sorted(single.items(), key=lambda kv: -kv[1]):
        log(f"[ensemble] single model {k:<28} dev macro-F1 {v:.4f}")
    mean = lambda keys, d: sum(d[k] for k in keys) / len(keys)
    if cfg.ens_runs or not cfg.ens_auto:
        chosen = list(dv_p)
    else:
        order = sorted(single, key=lambda k: -single[k])
        chosen, cur = [order[0]], single[order[0]]
        for k in order[1:]:
            if len(chosen) >= cfg.ens_max:
                break
            f = binary_scores(ydv, mean(chosen + [k], dv_p).argmax(1))["macro_f1"]
            log(f"[ensemble] try adding {k:<28} -> {f:.4f} (current {cur:.4f})")
            if f > cur + 0.002:
                chosen.append(k)
                cur = f
    log(f"[ensemble] members: {chosen}")
    pd_, pt_ = mean(chosen, dv_p), mean(chosen, te_p)
    thr, f_thr, f_base = best_threshold(ydv, pd_[:, 1])
    sc = binary_scores(ydv, (pd_[:, 1] >= thr).astype(int))
    log(f"[ensemble] dev macro-F1 {f_base:.4f} -> {sc['macro_f1']:.4f} (threshold {thr}) | precision {sc['precision']:.4f} recall {sc['recall']:.4f}")
    rd = run_dir4(cfg)
    rd.mkdir(parents=True, exist_ok=True)
    P1.jdump({"members": chosen, "single_model_f1": single, "scores": sc, "threshold": thr}, rd / "dev_report.json")
    log_csv(cfg, {"run": cfg.run, "p1_arch": "ensemble", "method": "ensemble", "dev_macro_f1": round(sc["macro_f1"], 4),
                  "dev_precision": round(sc["precision"], 4), "dev_recall": round(sc["recall"], 4), "dev_acc": round(sc["acc"], 4),
                  "threshold": thr, "teacher_run": "+".join(chosen)})
    return write_submission4(cfg, rd, test_ids, (pt_[:, 1] >= thr).astype(int))


def load_saved_cfg(cfg: JCfg, run_name: str) -> JCfg:
    saved = JCfg()
    d = P1.jload(Path(cfg.work_dir) / "cj_runs" / run_name / "config.json")
    apply_overrides(saved, {k: v for k, v in d.items() if k in {f.name for f in fields(JCfg)}})
    saved.work_dir = cfg.work_dir
    if cfg.local_data_dir:
        saved.local_data_dir = cfg.local_data_dir
    return saved


# ============================================================================================
# 7. Data diagnostics
# ============================================================================================
def diagnose4(cfg: JCfg):
    tok, _, mc = N.load_p1(cfg)
    splits = {}
    for name, rel, lab in (("train", f"{cfg.train_prefix}/train.json", True), ("dev", f"{cfg.train_prefix}/dev.json", True),
                           ("test", cfg.test_file, False)):
        docs = load_split(cfg, rel, tok, lab, cfg.max_train_docs if name == "train" else 0)
        splits[name] = docs
        n = np.array([d.n for d in docs])
        nch = np.array([len(chunk_bounds(d.n, cfg.max_len - 1, cfg.overlap)) for d in docs])
        line = (f"[check] {name:<5}: {len(docs):>6} cases | tokens/case mean {n.mean():.0f} median {np.median(n):.0f} p90 {np.percentile(n, 90):.0f} "
                f"max {n.max()} | chunks/case median {np.median(nch):.0f} p90 {np.percentile(nch, 90):.0f} | "
                f"covered by max_chunks={cfg.max_chunks}: {100 * np.mean(nch <= cfg.max_chunks):.0f}% of cases fully")
        if lab:
            y = np.array([d.label for d in docs])
            line += f" | label 1: {100 * y.mean():.1f}%"
        log(line)
        _, n_dup, n_conf = dedupe_docs(docs)
        log(f"[check] {name}: {n_dup} exact duplicate cases inside this split ({n_conf} of them with a conflicting label)")
    if mc["ctx"] < cfg.max_len:
        log(f"[check] WARNING: max_len {cfg.max_len} > Phase-1 context {mc['ctx']}")
    overlap_report(splits["train"], splits["dev"], splits["test"])


def run_probe(cfg: JCfg):
    """DIAGNOSTIC ONLY (not a submission): which part of a case carries the label? A bag-of-n-grams logistic regression is fitted on
    one region of the Phase-1 token sequence at a time (first / last / middle tokens) and scored on dev. The answer tells you which
    chunks the Phase-1 model must see (select / max_chunks), and is the evidence for 'which sections matter' in the report."""
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.linear_model import LogisticRegression
    except ImportError:
        raise SystemExit("probe needs scikit-learn:  pip install scikit-learn")
    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(str(Path(cfg.p1_dir) / "tokenizer.json"))
    tr = load_split(cfg, f"{cfg.train_prefix}/train.json", tok, True, cfg.max_train_docs)
    dv = load_split(cfg, f"{cfg.train_prefix}/dev.json", tok, True)
    tr, n1, _ = dedupe_docs(tr)
    dv, n2, _ = dedupe_docs(dv)
    rng = np.random.default_rng(0)
    trs = [tr[i] for i in rng.permutation(len(tr))[:cfg.probe_docs]]
    ytr, ydv = np.array([d.label for d in trs]), np.array([d.label for d in dv])
    log(f"[probe] {len(trs)} unique training cases, {len(dv)} unique dev cases (removed {n1}/{n2} duplicates)")
    mid = lambda t: t[max(0, len(t) // 2 - 512): len(t) // 2 + 512]
    regions = {"first 128 tokens": lambda t: t[:128], "first 512 tokens": lambda t: t[:512], "first 2048 tokens": lambda t: t[:2048],
               "last 128 tokens": lambda t: t[-128:], "last 512 tokens": lambda t: t[-512:], "last 2048 tokens": lambda t: t[-2048:],
               "middle 1024 tokens": mid, "first 1024 + last 1024": lambda t: np.concatenate([t[:1024], t[-1024:]]),
               "first 4096 + last 4096": lambda t: np.concatenate([t[:4096], t[-4096:]]) if len(t) > 8192 else t}
    text = lambda t: " ".join(f"t{int(x)}" for x in t)
    sizes = {"first 128 tokens": 128, "first 512 tokens": 512, "first 2048 tokens": 2048, "last 128 tokens": 128, "last 512 tokens": 512,
             "last 2048 tokens": 2048, "middle 1024 tokens": 1024, "first 1024 + last 1024": 2048, "first 4096 + last 4096": 8192}
    results, best = {}, (-1.0, None, None, None)
    for name, fn in regions.items():
        vec = TfidfVectorizer(token_pattern=r"\S+", ngram_range=(1, 2), min_df=3, max_features=300000, sublinear_tf=True, lowercase=False)
        Xtr = vec.fit_transform([text(fn(d.tok)) for d in trs])
        Xdv = vec.transform([text(fn(d.tok)) for d in dv])
        clf = LogisticRegression(max_iter=300, C=4.0).fit(Xtr, ytr)
        p = clf.predict_proba(Xdv)[:, 1]
        sc = binary_scores(ydv, (p >= 0.5).astype(int))
        thr, f_t, _ = best_threshold(ydv, p)
        results[name] = {"macro_f1": sc["macro_f1"], "acc": sc["acc"], "macro_f1_tuned": f_t}
        log(f"[probe] {name:<26} dev macro-F1 {sc['macro_f1']:.4f} | acc {sc['acc']:.3f} | tuned-threshold {f_t:.4f}")
        if f_t > best[0] + 0.003 or (abs(f_t - best[0]) <= 0.003 and best[1] is not None and sizes[name] < sizes[best[1]]):
            best = (max(f_t, best[0]), name, vec, clf)         # ties (within 0.003): prefer the shorter region
    f_t, name, vec, clf = best
    names, coef = vec.get_feature_names_out(), clf.coef_[0]
    order = np.argsort(coef)
    show = lambda f: tok.decode([int(x[1:]) for x in f.split()]).replace("\n", " ").strip()
    log(f"[probe] BEST region: {name} (macro-F1 {f_t:.4f}). Strongest n-grams there:")
    log("[probe]   -> label 1: " + " | ".join(show(names[i]) for i in order[::-1][:15]))
    log("[probe]   -> label 0: " + " | ".join(show(names[i]) for i in order[:15]))
    P1.jdump(results, Path(cfg.work_dir) / "cj_probe.json")
    return results


# ============================================================================================
# 8. Self-test
# ============================================================================================
def _synthetic(tmp, rng):
    common = "the court appeal petitioner respondent section order judgment high learned counsel submitted held impugned".split()

    def doc(y):
        body = [rng.choice(common) for _ in range(rng.randint(500, 1800))]
        tail = [rng.choice(common) for _ in range(40)] + (["allowed", "set", "aside"] if y else ["dismissed", "no", "merit"]) * 3
        return " ".join(body + tail + [rng.choice(common) for _ in range(10)])

    os.makedirs(tmp, exist_ok=True)
    texts = []
    for split, n in (("train", 40), ("dev", 12), ("test", 10)):
        recs = []
        for i in range(n):
            y = i % 2
            t = doc(y)
            texts.append(t)
            r = {"id": f"{split}_{i}", "text": t}
            if split != "test":
                r["label"] = y
            recs.append(r)
        json.dump(recs, open(os.path.join(tmp, f"{split}.json"), "w"))
    return texts


def selftest() -> bool:
    import tempfile
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
    ok = True
    rng = random.Random(0)
    tmp = tempfile.mkdtemp(prefix="cs781_cj_selftest_")
    dd = os.path.join(tmp, "data")
    texts = _synthetic(dd, rng)
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    tok.train_from_iterator([" ".join(t.split()) for t in texts], trainers.BpeTrainer(
        vocab_size=500, special_tokens=[P1.EOS], initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False))
    V = P1.pad64(tok.get_vocab_size())

    log("=== 1. chunking, selection, metrics ===")
    b = chunk_bounds(1000, 63, 0)
    g1 = b[-1][1] == 1000 and b[0][0] == 0 and all(e - s <= 63 for s, e in b) and all(b[i][1] == b[i + 1][0] for i in range(len(b) - 1))
    g2 = chunk_bounds(0, 63, 0) == [(0, 0)] and chunk_bounds(10, 63, 0) == [(0, 10)]
    g3 = select_chunks(10, 3, "tail") == [7, 8, 9] and select_chunks(10, 3, "head") == [0, 1, 2] and \
        select_chunks(10, 4, "head_tail") == [0, 7, 8, 9] and select_chunks(10, 4, "uniform")[0] == 0 and select_chunks(10, 4, "uniform")[-1] == 9
    g3b = select_chunks(10, 4, "head_tail", 2) == [0, 1, 8, 9] and select_chunks(10, 2, "head_tail", 1) == [0, 9]
    ok &= g3b
    log(f"  explicit head size for head_tail {'PASS' if g3b else 'FAIL'}")
    y = np.array([0, 0, 1, 1, 1, 0])
    yh = np.array([0, 1, 1, 1, 0, 0])
    sc = binary_scores(y, yh)
    f_man = np.mean([2 * 2 / (2 * 2 + 1 + 1), 2 * 2 / (2 * 2 + 1 + 1)])
    g4 = abs(sc["macro_f1"] - f_man) < 1e-9 and abs(sc["precision"] - 2 / 3) < 1e-9 and abs(sc["recall"] - 2 / 3) < 1e-9
    t_, f_, f0 = best_threshold(np.array([0, 0, 1, 1]), np.array([0.2, 0.45, 0.4, 0.9]))
    g5 = f_ >= f0
    ok &= all([g1, g2, g3, g4, g5])
    log(f"  chunks from the end {'PASS' if g1 else 'FAIL'} | edge cases {'PASS' if g2 else 'FAIL'} | selection strategies {'PASS' if g3 else 'FAIL'} | "
        f"metrics = evaluator definitions {'PASS' if g4 else 'FAIL'} | threshold tuning {'PASS' if g5 else 'FAIL'}")

    cfg0 = apply_overrides(JCfg(), {"local_data_dir": dd, "max_len": 64, "max_chunks": 3})
    tr = load_split(cfg0, "Phase_4_CJPE/Train/train.json", tok, True)
    eos = tok.token_to_id(P1.EOS)
    batch = collate_docs(tr, [0, 1], cfg0, "cpu", eos, 3)
    g6 = len(tr) == 40 and batch[0].shape[0] == int(batch[6].sum()) and batch[0].shape[1] <= 64 and int(batch[1].sum(1).max()) <= 64
    ok &= g6
    log(f"  collate (chunks of <= max_len incl. <eos>) {'PASS' if g6 else 'FAIL'}")

    d1, d2 = JDoc(), JDoc()
    d1.tok, d1.label = np.arange(10, dtype=np.uint16), 1
    d2.tok, d2.label = np.arange(10, dtype=np.uint16), 0
    kept, nrem, nconf = dedupe_docs([d1, d2, tr[0]])
    rows = overlap_report(tr[:5], tr[:3] + tr[10:12], tr[3:6])
    g7 = len(kept) == 2 and nrem == 1 and nconf == 1 and rows[0][2] == 3 and rows[1][2] == 2
    ok &= g7
    log(f"  exact-duplicate removal + train/dev/test overlap report {'PASS' if g7 else 'FAIL'}")

    log("=== 2. adapters / BitFit / freezing: identity at init, only the intended parameters train ===")
    for arch in P1.ARCH_PRESETS:
        for meth, var in (("adapter", "plus"), ("lora", "plus"), ("lorma", "plus"), ("lorma", "pi"), ("bitfit", "plus"), ("partial", "plus")):
            tag = f"{arch}/{meth}" + (f"-{var}" if meth == "lorma" else "")
            try:
                torch.manual_seed(0)
                c = apply_overrides(JCfg(), {"method": meth, "lorma_variant": var, "peft_r": 4, "peft_alpha": 8.0, "peft_dropout": 0.0,
                                             "adapter_dim": 8, "freeze_layers": 1, "chunk_dim": 32, "max_len": 64, "max_chunks": 3})
                net = build_net4(c, P1.LM(N._tiny_mc(arch, V))).eval()
                inp = collate_docs(tr, [0, 1], c, "cpu", eos, 3)[:7]
                with torch.no_grad():
                    ref = net(*inp)
                apply_method4(net, c)
                net.eval()
                with torch.no_grad():
                    out = net(*inp)
                diff = (ref - out).abs().max().item()
                n_tr = sum(p.numel() for p in net.parameters() if p.requires_grad)
                n_all = sum(p.numel() for p in net.parameters())
                net.train()
                net(*inp).float().pow(2).mean().backward()
                grads = any(p.grad is not None and float(p.grad.abs().sum()) > 0 for p in net.parameters() if p.requires_grad)
                good = diff < 1e-4 and n_tr < n_all and grads
                ok &= good
                log(f"  {tag:<18} |adapted - base| = {diff:.1e}  trainable {n_tr:,}/{n_all:,}  {'PASS' if good else 'FAIL'}")
            except Exception as e:
                import traceback
                traceback.print_exc()
                log(f"  {tag:<18} FAIL: {type(e).__name__}: {e}")
                ok = False

    log("=== 3. end-to-end: every architecture / method / pooling, then ensembles ===")
    p1dirs = {}
    for arch in P1.ARCH_PRESETS:
        p1 = os.path.join(tmp, f"p1_{arch}")
        os.makedirs(p1, exist_ok=True)
        torch.save({"model": P1.LM(N._tiny_mc(arch, V)).state_dict(), "mc": N._tiny_mc(arch, V), "state": {"step": 0}},
                   os.path.join(p1, "ckpt_best.pt"))
        tok.save(os.path.join(p1, "tokenizer.json"))
        p1dirs[arch] = p1
    base = {"work_dir": os.path.join(tmp, "work"), "local_data_dir": dd, "max_len": 64, "max_chunks": 3, "epochs": 2, "batch_docs": 2,
            "accum": 1, "eval_docs": 2, "lr": 1e-3, "peft_lr": 3e-3, "head_lr": 3e-3, "peft_r": 4, "peft_alpha": 8.0, "chunk_dim": 32,
            "adapter_dim": 8, "ema_decay": 0.9, "time_budget_min": 5, "patience": 5, "dev_max_docs": 8}
    cases = [(f"full_{a}", a, {"method": "full"}) for a in P1.ARCH_PRESETS] + [
        ("lorma_plus", "gpt2", {"method": "lorma", "lorma_variant": "plus"}),
        ("lorma_pi", "retnet", {"method": "lorma", "lorma_variant": "pi"}),
        ("lora", "llama", {"method": "lora"}), ("adapter", "moe", {"method": "adapter"}),
        ("bitfit", "gpt2", {"method": "bitfit"}), ("partial", "llama", {"method": "partial", "freeze_layers": 1}),
        ("pool_mean", "llama", {"method": "full", "pool": "mean"}), ("pool_gru", "llama", {"method": "full", "pool": "gru"}),
        ("pool_last", "llama", {"method": "full", "pool": "last"}), ("head_tail", "llama", {"method": "full", "select": "head_tail"}),
        ("kd", "gpt2", {"method": "kd", "teacher_run": "selftest_full_llama", "student_layers": 1})]
    for tag, arch, extra in cases:
        try:
            cfg = apply_overrides(JCfg(), {**base, **extra, "run": f"selftest_{tag}", "p1_dir": p1dirs[arch]})
            run_train4(cfg)
            with zipfile.ZipFile(run_dir4(cfg) / "submission.zip") as z:
                assert z.namelist() == ["prediction.json"]
                pr = json.loads(z.read("prediction.json"))
            assert isinstance(pr, list) and len(pr) == 10 and [x["id"] for x in pr] == [f"test_{i}" for i in range(10)]
            assert all(set(x) == {"id", "pred"} and type(x["pred"]) is int and x["pred"] in (0, 1) for x in pr)
            rep = P1.jload(run_dir4(cfg) / "dev_report.json")
            if extra["method"] in ("lora", "lorma", "adapter", "bitfit", "partial"):
                assert rep["params_trainable"] < rep["params_total"]
            if extra["method"] == "kd":
                assert rep["params_total"] < rep["teacher_params"]
            log(f"  e2e {tag:<12} PASS  (dev macro-F1 {rep['scores']['macro_f1']:.3f}, trainable {rep['params_trainable']:,} / {rep['params_total']:,})")
        except Exception as e:
            import traceback
            traceback.print_exc()
            log(f"  e2e {tag:<12} FAIL: {type(e).__name__}: {e}")
            ok = False
    try:
        pcfg = apply_overrides(JCfg(), {"work_dir": os.path.join(tmp, "work"), "local_data_dir": dd, "p1_dir": p1dirs["llama"], "probe_docs": 40})
        res = run_probe(pcfg)
        assert res["last 512 tokens"]["macro_f1_tuned"] > res["first 512 tokens"]["macro_f1_tuned"] + 0.1, res      # synthetic labels sit at the END of every case
        best_region = max(res, key=lambda k: res[k]["macro_f1_tuned"])
        log(f"  probe stage PASS (best region on synthetic data: {best_region})")
    except ImportError:
        log("  probe stage skipped (scikit-learn not installed)")
    except Exception as e:
        import traceback
        traceback.print_exc()
        log(f"  probe stage FAIL: {type(e).__name__}: {e}")
        ok = False
    try:
        for tag, ov in (("ens", {"ens_runs": ["selftest_full_llama", "selftest_full_gpt2"]}),
                        ("ens_auto", {"ens_auto": True, "ens_prefix": "selftest_"})):
            cfg = apply_overrides(JCfg(), {"work_dir": os.path.join(tmp, "work"), "local_data_dir": dd, "run": f"selftest_{tag}", **ov})
            zp = run_ensemble4(cfg)
            with zipfile.ZipFile(zp) as z:
                pr = json.loads(z.read("prediction.json"))
            assert len(pr) == 10 and all(x["pred"] in (0, 1) for x in pr)
        log("  ensembles (explicit + automatic) PASS")
    except Exception as e:
        import traceback
        traceback.print_exc()
        log(f"  ensembles FAIL: {type(e).__name__}: {e}")
        ok = False
    shutil.rmtree(tmp, ignore_errors=True)
    log("SELFTEST " + ("PASSED" if ok else "FAILED"))
    return ok


# ============================================================================================
# 9. CLI
# ============================================================================================
def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("stage", choices=["selftest", "check", "probe", "train", "predict", "ensemble"])
    ap.add_argument("--run", default=None)
    ap.add_argument("--cfg", default="", help="JSON string or JSON file with config overrides")
    args = ap.parse_args(argv)
    if args.stage == "selftest":
        return 0 if selftest() else 1
    cfg = JCfg()
    ov = {}
    if args.cfg:
        ov = json.loads(args.cfg) if args.cfg.strip().startswith("{") else P1.jload(args.cfg)
    if args.run:
        ov["run"] = args.run
    apply_overrides(cfg, ov)
    Path(cfg.work_dir).mkdir(parents=True, exist_ok=True)
    {"check": diagnose4, "probe": run_probe, "train": run_train4, "predict": run_predict4, "ensemble": run_ensemble4}[args.stage](cfg)
    return 0


if __name__ == "__main__":
    sys.exit(main())


## Self-test (once, ~8-12 min). Must end with `SELFTEST PASSED`. Checks chunking, metrics, adapters-at-identity and runs every method / pooling end to end on synthetic cases.

In [ ]:
!python cs781_phase4_cjpe.py selftest

## Setup

In [ ]:
import os, json, glob, subprocess
WORK = "/kaggle/working/cs781" if os.path.exists("/kaggle/working") else "./cs781_work"
BUDGET = 60          # minutes of training per main run (lower it if you are short of time, raise it for a better score)
SEL = {}             # which chunks of a case the model reads - set it AFTER the probe cell below (default: tail, 8 chunks)
P1_DIRS = {a: f"{WORK}/runs/explore_{a}" for a in ["llama", "gpt2", "retnet", "moe"]}
P1_CKPT, missing = {}, []
for a, d in P1_DIRS.items():
    if not os.path.exists(f"{d}/tokenizer.json"): missing.append(a)
    elif os.path.exists(f"{d}/ckpt_best.pt"): P1_CKPT[a] = "ckpt_best.pt"
    elif os.path.exists(f"{d}/ckpt_last.pt"): P1_CKPT[a] = "ckpt_last.pt"
    else: missing.append(a)
for a, d in P1_DIRS.items(): print(f"{a:<7}", f"OK ({P1_CKPT[a]})" if a in P1_CKPT else "MISSING", d)

def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print(line, end="")
    p.wait(); return p.returncode

def run_cj(run, arch, extra):
    if os.path.exists(f"{WORK}/cj_runs/{run}/submission.zip"):
        print("already done:", run, "(delete that folder to re-run)"); return
    if arch not in P1_CKPT:
        print(f"Phase-1 backbone '{arch}' is missing -> train it first, then re-run the setup cell."); return
    cfg = {"p1_dir": P1_DIRS[arch], "p1_ckpt": P1_CKPT[arch], "time_budget_min": BUDGET, **SEL, **extra}
    fn = f"cfg_cj_{run}.json"; json.dump(cfg, open(fn, "w"), indent=1)
    print("=" * 100, "\n", run, "|", arch, "|", extra, "\n", "=" * 100)
    print("exit code", sh(f"python cs781_phase4_cjpe.py train --run {run} --cfg {fn}"))
    report(run)

def report(run, n_err=6):
    p = f"{WORK}/cj_runs/{run}/dev_report.json"
    if not os.path.exists(p): print(f"\n{run}: no dev_report.json (not finished)"); return
    r = json.load(open(p)); sc = r["scores"]
    print(f"\n=== {run} | {r.get('p1_arch', 'ensemble')} / {r.get('method', 'ensemble')} | best epoch {r.get('best_epoch')} | threshold {r.get('threshold')}")
    print(f"dev macro-F1 {sc['macro_f1']:.3f} (at threshold 0.5: {r.get('macro_f1_default_threshold', float('nan')):.3f})  precision {sc['precision']:.3f}  recall {sc['recall']:.3f}  acc {sc['acc']:.3f}  per-class F1 {[round(x, 3) for x in sc['f1_per_class']]}")
    ep = f"{WORK}/cj_runs/{run}/errors.json"
    if os.path.exists(ep):
        for e in json.load(open(ep))[:n_err]: print("   confident error:", e)

## (Only if a backbone is MISSING) train it now - about 0.75 h each; then re-run the setup cell

In [ ]:
for a in missing:
    fn = f"cfg_p1_{a}.json"; json.dump({"arch": a, "time_budget_hours": 0.75}, open(fn, "w"))
    print("training Phase-1 backbone:", a); sh(f"python cs781_phase1.py train --run explore_{a} --preset turbo --cfg {fn}")
if not missing: print("nothing to train - all four backbones exist")

In [ ]:
RUN_CONTINUE, CONT_ARCH, CONT_HOURS = False, "llama", 3.0
if RUN_CONTINUE and CONT_ARCH in P1_CKPT:
    fn = f"cfg_p1_{CONT_ARCH}_cont.json"
    json.dump({"arch": CONT_ARCH, "time_budget_hours": CONT_HOURS, "init_from": f"{P1_DIRS[CONT_ARCH]}/{P1_CKPT[CONT_ARCH]}",
               "lr": 4e-4, "warmup_steps": 100, "seed": 2024}, open(fn, "w"))
    print("exit code", sh(f"python cs781_phase1.py train --run explore_{CONT_ARCH}_cont --preset turbo --cfg {fn}"))
    P1_DIRS[CONT_ARCH] = f"{WORK}/runs/explore_{CONT_ARCH}_cont"; P1_CKPT[CONT_ARCH] = "ckpt_best.pt"
    print(CONT_ARCH, "-> now uses", P1_DIRS[CONT_ARCH])

## Check the real data (1-3 min): label balance, case lengths in tokens, how much of a case `max_chunks` covers, duplicate ids

In [ ]:
sh("python cs781_phase4_cjpe.py check --run cj_check --cfg " + json.dumps(json.dumps({"p1_dir": P1_DIRS["llama"], "p1_ckpt": P1_CKPT.get("llama", "ckpt_best.pt")})))

In [ ]:
sh("python cs781_phase4_cjpe.py probe --run cj_probe --cfg " + json.dumps(json.dumps({"p1_dir": P1_DIRS["llama"], "p1_ckpt": P1_CKPT.get("llama", "ckpt_best.pt"), "work_dir": WORK})))

In [ ]:
# Pick ONE and run the cell (the default {} = last 8 chunks):
SEL = {"select": "tail", "max_chunks": 2}                               # label signal at the END of the case (e.g. the last 512 / 2048 tokens)
# SEL = {"select": "head", "max_chunks": 2}                             # label signal at the START (headnote / first lines)
# SEL = {"select": "head_tail", "head_chunks": 1, "max_chunks": 2}     # both ends
# SEL = {"select": "uniform", "max_chunks": 8}                          # spread over the whole case
print("chunks per case:", SEL or "default (tail, 8)")

## Teacher / strongest single model - full fine-tuning (llama)

In [ ]:
run_cj("cj_full_llama", "llama", {"method": "full", "epochs": 6, "lr": 2e-4})

## C1 - LoRMA+ (gpt2) - compulsory: `h = (alpha/r BA + I) W0 x`, backbone frozen and causal; the BiGRU + attention pool over chunks carries the document-level reasoning

In [ ]:
ADAPT = {"bidir": "false", "pool": "gru", "epochs": 8, "peft_lr": 1e-3}
run_cj("cj_lorma_gpt2", "gpt2", {"method": "lorma", "lorma_variant": "plus", "peft_r": 32, "peft_alpha": 64, **ADAPT})

## C3 - Houlsby adapters (moe): bottleneck adapters (zero-initialised up-projection) after every attention and FFN sub-layer

In [ ]:
run_cj("cj_adapter_moe", "moe", {"method": "adapter", "adapter_dim": 64, **ADAPT})

## C4 - knowledge distillation (retnet student pruned to 6 of 12 blocks <- `cj_full_llama` teacher): `0.5 CE + 0.5 T^2 KL(teacher || student)`, T = 2

In [ ]:
run_cj("cj_kd_retnet", "retnet", {"method": "kd", "teacher_run": "cj_full_llama", "student_layers": 6, "epochs": 6, "lr": 2e-4})

In [ ]:
run_cj("cj_full_gpt2",     "gpt2",   {"method": "full", "epochs": 6, "lr": 2e-4})
run_cj("cj_full_moe",      "moe",    {"method": "f ull", "epochs": 6, "lr": 2e-4})
# run_cj("cj_full_retnet",   "retnet", {"method": "full", "epochs": 6, "lr": 2e-4})
# run_cj("cj_bitfit_llama",  "llama",  {"method": "bitfit", **ADAPT})
# run_cj("cj_partial_llama", "llama",  {"method": "partial", "freeze_layers": 8, "epochs": 6, "lr": 2e-4})
# run_cj("cj_lora_llama",    "llama",  {"method": "lora", "peft_r": 32, "peft_alpha": 64, **ADAPT})

## Analysis ablations (the evidence for the written analysis) - equal small budgets so the runs are comparable
* **Which part of the case matters?** `select`: tail / head / head_tail / uniform.
* **Hierarchical vs. flat:** `pool`: attn / gru / mean / last (`last` = only the final chunk, i.e. no hierarchy).
Set `RUN_ABLATIONS = True` (about 20 min each).

In [ ]:
RUN_ABLATIONS = False
ABL = {"method": "full", "epochs": 4, "time_budget_min": 20, "docs_per_epoch": 2000}
if RUN_ABLATIONS:
    for sel in ["tail", "head", "head_tail", "uniform"]:
        run_cj(f"cj_abl_select_{sel}", "llama", {**ABL, "select": sel})
    for pool in ["attn", "gru", "mean", "last"]:
        run_cj(f"cj_abl_pool_{pool}", "llama", {**ABL, "pool": pool})

In [ ]:
json.dump({"ens_auto": True, "ens_prefix": "cj_", "ens_max": 6}, open("cfg_cj_ens.json", "w"))
print("exit code", sh("python cs781_phase4_cjpe.py ensemble --run cj_ensemble --cfg cfg_cj_ens.json"))
report("cj_ensemble", n_err=0)

## Results, efficiency comparison and the files to submit

In [ ]:
import pandas as pd, shutil
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
df = pd.read_csv(f"{WORK}/cj_experiments.csv"); df = df[~df.run.str.startswith(("selftest", "cj_check"))]
cols = ["run", "p1_arch", "method", "pool", "select", "max_chunks", "params_total_M", "params_trainable_M", "dev_macro_f1", "dev_precision",
        "dev_recall", "dev_acc", "threshold", "train_s", "peak_gpu_mb", "infer_docs_s"]
display(df[cols].sort_values("dev_macro_f1", ascending=False))
os.makedirs("phase4_submissions", exist_ok=True)
REQUIRED = ["cj_lorma_gpt2", "cj_full_llama", "cj_adapter_moe", "cj_kd_retnet"]
EXTRA = ["cj_ensemble"]
for r in REQUIRED + EXTRA:
    src = f"{WORK}/cj_runs/{r}/submission.zip"
    if os.path.exists(src):
        shutil.copy(src, f"phase4_submissions/{r}.zip")
        row = df[df.run == r].tail(1)
        print(("REQUIRED " if r in REQUIRED else "extra    "), r, "| dev macro-F1", float(row.dev_macro_f1.iloc[0]) if len(row) else "?")
    else: print("missing:", r)

# CS781 Phase 5 - Legal Statute Identification (multi-label)

### How the model works
* Each case is split into 512-token chunks; during training a few random chunks are read per case, at inference up to `eval_chunks` chunks spread over the whole judgment.
* **Label-wise attention**: every statute id has its own query that attends over the chunks, so a statute mentioned in one part of a long case is still found. Its output bias starts at the statute's training frequency (important for rare statutes).
* Loss: weighted BCE (positive weights grow for rare statutes). Decisions: the global threshold, per-statute thresholds, and "at least one statute per case" are chosen by two-fold cross-fitting on dev, so the strategy is picked on data it wasn't tuned on.
* Output: unique integer statute ids per case, one object per test id.

| Run | Phase-1 backbone | Technique |
|---|---|---|
| `ls_full_llama` | llama | full fine-tuning (teacher for the distillation run) |
| `ls_lorma_gpt2` | gpt2 | **LoRMA+** (compulsory) |
| `ls_adapter_moe` | moe | Houlsby adapters |
| `ls_kd_retnet` | retnet | **knowledge distillation** (compulsory): pruned student <- llama teacher |

Extras: soft prompt tuning, BitFit, LoRA, pooling / loss ablations, and an ensemble.

In [ ]:
import base64
open("cs781_phase1.py", "w", encoding="utf-8").write(base64.b64decode("...").decode("utf-8"))
open("cs781_phase2_ner.py", "w", encoding="utf-8").write(base64.b64decode("...").decode("utf-8"))
open("cs781_phase4_cjpe.py", "w", encoding="utf-8").write(base64.b64decode("...").decode("utf-8"))
print("wrote cs781_phase1.py, cs781_phase2_ner.py, cs781_phase4_cjpe.py")

In [ ]:
%%writefile cs781_phase5_lsi.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
CS781 Phase 5 - Legal Statute Identification (multi-label) on top of YOUR Phase-1 language models.

Needs cs781_phase1.py, cs781_phase2_ner.py and cs781_phase4_cjpe.py in the same folder (model classes, adapters, distillation helpers).

Model (multi-label, long documents)
  case text -> Phase-1 BPE tokens -> chunks of max_len tokens -> Phase-1 backbone (bidirectional attention; causal for retnet)
  -> chunk vector = [mean of token states ; closing <eos> state] + chunk-position embedding
  -> pooling over chunks ("pool"):
       attn       LABEL-WISE attention: every statute has its own query over the chunks (statutes are mentioned in different places)
       chunk_max  chunk-level statute scores, document score = max over chunks
       max | mean pooled vector -> linear
  -> one logit per statute (bias initialised from the label priors).  Training reads `train_chunks` random chunks per case, inference up to
  `eval_chunks` chunks spread over the whole case.
  Losses: bce | bce_pw (positive-class weights for rare statutes) | focal | asl.
  Decision: global vs per-label thresholds (+ "at least one statute per case"), chosen by cross-fitting on dev.
  Output: [{"id": ..., "pred": [unique statute ids]}, ...]

Adaptation methods (cfg "method")
  full | partial | bitfit | adapter (Houlsby) | lora | lorma (LoRMA+ / LoRMAπ) | prompt (soft prompt tuning) | kd (pruned student <- teacher)

Stages:  selftest | check | probe | train | predict | ensemble
  python cs781_phase5_lsi.py check --run ls_check --cfg '{"p1_dir": ".../explore_llama"}'
  python cs781_phase5_lsi.py train --run ls_full_llama --cfg cfg_ls.json
"""
from __future__ import annotations

import argparse
import contextlib
import csv
import gc
import hashlib
import json
import math
import os
import pickle
import random
import re
import shutil
import sys
import time
import zipfile
from dataclasses import dataclass, field, fields, asdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import cs781_phase1 as P1
import cs781_phase2_ner as N
import cs781_phase4_cjpe as J

log = P1.log


# ============================================================================================
# 0. Config
# ============================================================================================
@dataclass
class LCfg:
    run: str = "ls_full_v1"
    work_dir: str = P1.DEFAULT_WORK
    p1_dir: str = ""
    p1_ckpt: str = "ckpt_best.pt"
    hf_repo: str = "Exploration-Lab/CS-781-Capstone"
    local_data_dir: str = ""          # folder with train.json, dev.json, test.json
    train_prefix: str = "Phase_5_LSI/Train"
    test_file: str = "Phase_5_LSI/Test/test.json"
    max_train_docs: int = 0
    dedupe_train: bool = True         # drop exact duplicate cases from train and dev
    cache_tokens: bool = True         # tokenised documents are cached in <work_dir>/cache (every run reuses them)
    # ---- long-document representation
    max_len: int = 512
    train_chunks: int = 4             # random chunks per case per training step
    eval_chunks: int = 16             # chunks per case at inference (spread over the whole case)
    train_select: str = "random"      # random | uniform | head | tail
    eval_select: str = "uniform"
    overlap: int = 0
    pool: str = "attn"                # attn (label-wise attention) | chunk_max | max | mean
    chunk_dim: int = 384
    head_dropout: float = 0.1
    bidir: str = "auto"
    # ---- method
    method: str = "full"              # full | partial | bitfit | adapter | lora | lorma | prompt | kd
    freeze_layers: int = 0
    adapter_dim: int = 64
    prompt_tokens: int = 16
    peft_r: int = 16
    peft_alpha: float = 32.0
    peft_dropout: float = 0.05
    peft_lr: float = 5e-4
    peft_targets: str = "all"
    peft_experts: bool = False
    lorma_variant: str = "plus"
    teacher_run: str = ""
    student_layers: int = 0
    kd_alpha: float = 0.5
    kd_temp: float = 2.0
    # ---- loss
    loss: str = "bce_pw"              # bce | bce_pw | focal | asl
    pos_weight_max: float = 20.0
    focal_gamma: float = 2.0
    # ---- optimisation
    epochs: int = 6
    docs_per_epoch: int = 0
    batch_docs: int = 2
    accum: int = 2
    eval_docs: int = 4
    lr: float = 2e-4
    llrd: float = 0.85
    head_lr: float = 2e-3
    weight_decay: float = 0.01
    warmup_frac: float = 0.06
    grad_clip: float = 1.0
    ema_decay: float = 0.995
    time_budget_min: float = 60.0
    dev_max_docs: int = 800
    eval_every_steps: int = 3000
    patience: int = 4
    seed: int = 1337
    predict_after: bool = True
    # ---- ensemble / probe
    ens_runs: list = field(default_factory=list)
    ens_auto: bool = False
    ens_prefix: str = ""
    ens_max: int = 6
    probe_docs: int = 6000
    probe_labels: int = 10


def apply_overrides(cfg: LCfg, ov: dict) -> LCfg:
    names = {f.name for f in fields(LCfg)}
    for k, v in ov.items():
        if k not in names:
            raise SystemExit(f"Unknown config key {k!r}. Valid: {sorted(names)}")
        setattr(cfg, k, v)
    return cfg


def run_dir5(cfg: LCfg) -> Path:
    return Path(cfg.work_dir) / "ls_runs" / cfg.run


LS_FIELDS = ["time", "run", "p1_arch", "method", "pool", "loss", "params_total_M", "params_trainable_M", "dev_macro_f1", "dev_micro_f1",
             "dev_subset_acc", "decision", "n_labels", "train_s", "peak_gpu_mb", "infer_docs_s", "teacher_run"]


def log_csv(cfg: LCfg, row: dict):
    p = Path(cfg.work_dir) / "ls_experiments.csv"
    if p.exists() and open(p).readline().strip().split(",") != LS_FIELDS:
        p.rename(p.with_suffix(".old.csv"))
    new = not p.exists()
    with open(p, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=LS_FIELDS, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerow({"time": time.strftime("%Y-%m-%d %H:%M:%S"), **row})


# ============================================================================================
# 1. Data (tolerant parsing, cached tokenisation)
# ============================================================================================
class LDoc:
    pass


def _first(d, keys):
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return None


def load_split5(cfg: LCfg, rel: str, tok, labelled: bool, max_docs: int = 0, tok_path: str = ""):
    """-> list of LDoc(id, tok uint16, n, labels [int])."""
    path = P1.resolve_file(cfg, rel)
    cache = None
    if cfg.cache_tokens and tok_path:
        key = hashlib.md5(open(tok_path, "rb").read() + f"{os.path.basename(path)}|{os.path.getsize(path)}|{max_docs}".encode()).hexdigest()[:12]
        cache = Path(cfg.work_dir) / "cache" / f"lsi_{key}.pkl"
        if cache.exists():
            with open(cache, "rb") as f:
                recs = pickle.load(f)
            log(f"[data] {rel}: {len(recs)} cases from the token cache")
            return [_mk(i, t, l) for i, t, l in recs]
    raw = P1.jload(path)
    items = raw if isinstance(raw, list) else [dict(v, id=k) if isinstance(v, dict) else {"id": k, "text": v} for k, v in raw.items()]
    if max_docs:
        items = items[:max_docs]
    assert tok.get_vocab_size() < 65536, "token ids are stored as uint16"
    ids, texts, labels = [], [], []
    for i, r in enumerate(items):
        did = _first(r, ("id", "case_id", "doc_id", "document_id"))
        ids.append(str(did) if did is not None else str(i))
        t = _first(r, ("text", "document", "case_text", "content")) or ""
        texts.append(" ".join(" ".join(t).split() if isinstance(t, list) else t.split()))
        if labelled:
            y = _first(r, ("labels", "label", "gold", "statutes", "statute_ids")) or []
            if isinstance(y, (int, float, str)):
                y = [y]
            labels.append(sorted({int(float(x)) for x in y}))
        else:
            labels.append([])
    recs = []
    for b0 in range(0, len(texts), 128):
        for k, e in enumerate(tok.encode_batch(texts[b0:b0 + 128])):
            recs.append((ids[b0 + k], np.asarray(e.ids, dtype=np.uint16), labels[b0 + k]))
    if cache is not None:
        cache.parent.mkdir(parents=True, exist_ok=True)
        with open(cache, "wb") as f:
            pickle.dump(recs, f, protocol=4)
    return [_mk(i, t, l) for i, t, l in recs]


def _mk(i, t, l):
    d = LDoc()
    d.id, d.tok, d.n, d.labels = i, t, len(t), list(l)
    return d


def dedupe_docs5(docs):
    seen, keep, conflict = {}, [], 0
    for d in docs:
        h = hashlib.md5(d.tok.tobytes()).hexdigest()
        if h in seen:
            conflict += int(seen[h] != d.labels)
            continue
        seen[h] = d.labels
        keep.append(d)
    return keep, len(docs) - len(keep), conflict


def label_matrix(docs, L):
    Y = np.zeros((len(docs), L), dtype=np.uint8)
    for i, d in enumerate(docs):
        for x in d.labels:
            if 0 <= x < L:
                Y[i, x] = 1
    return Y


def pick_chunks(m, k, mode, rng):
    if m <= k:
        return list(range(m))
    if mode == "random":
        return sorted(rng.sample(range(m), k))
    if mode == "head":
        return list(range(k))
    if mode == "tail":
        return list(range(m - k, m))
    return sorted({int(x) for x in np.linspace(0, m - 1, k).round()})              # uniform


def collate5(docs, idxs, cfg: LCfg, dev, eos, L, n_chunks, mode, rng=None):
    chunks, owner, slot, rel, counts = [], [], [], [], []
    for b, di in enumerate(idxs):
        d = docs[di]
        bounds = J.chunk_bounds(d.n, cfg.max_len - 1, cfg.overlap)
        sel = pick_chunks(len(bounds), n_chunks, mode, rng or random)
        for k_, j in enumerate(sel):
            s, e = bounds[j]
            chunks.append(d.tok[s:e])
            owner.append(b)
            slot.append(k_)
            rel.append(min(7, int(8 * j / len(bounds))))
        counts.append(len(sel))
    T = max(len(c) for c in chunks) + 1
    ids = np.zeros((len(chunks), T), dtype=np.int64)
    km = np.zeros((len(chunks), T), dtype=bool)
    for i, c in enumerate(chunks):
        n = len(c)
        ids[i, :n] = c
        ids[i, n] = eos
        km[i, :n + 1] = True
    Y = label_matrix([docs[di] for di in idxs], L).astype(np.float32)
    t = lambda x: torch.from_numpy(x).to(dev)
    return t(ids), t(km), t(np.array(owner)), t(np.array(slot)), t(np.array(rel)), t(np.array(counts)), t(Y)


# ============================================================================================
# 2. Metrics + decision strategies
# ============================================================================================
def per_label_f1(Y, Yhat):
    Y, Yhat = Y.astype(bool), Yhat.astype(bool)
    tp = (Y & Yhat).sum(0).astype(np.float64)
    fp = (~Y & Yhat).sum(0).astype(np.float64)
    fn = (Y & ~Yhat).sum(0).astype(np.float64)
    den = 2 * tp + fp + fn
    return np.divide(2 * tp, den, out=np.zeros_like(tp), where=den > 0), tp, fp, fn


def ml_scores(Y, Yhat):
    """Same definitions as the official evaluator (sklearn, zero_division=0): macro over ALL label columns, micro, subset accuracy."""
    f1, tp, fp, fn = per_label_f1(Y, Yhat)
    den = 2 * tp.sum() + fp.sum() + fn.sum()
    P = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    R = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    return {"macro_f1": float(f1.mean()), "micro_f1": float(2 * tp.sum() / den) if den > 0 else 0.0,
            "subset_acc": float((Y.astype(bool) == Yhat.astype(bool)).all(1).mean()) if len(Y) else 0.0,
            "macro_p": float(P.mean()), "macro_r": float(R.mean()), "f1": f1, "tp": tp, "fp": fp, "fn": fn}


GRID = np.arange(0.05, 0.9501, 0.025)


def f1_grid(P, Y):
    return np.stack([per_label_f1(Y, P >= t)[0] for t in GRID])            # (len(GRID), L)


def best_global(P, Y):
    F1g = f1_grid(P, Y)
    gi = int(F1g.mean(1).argmax())
    return float(GRID[gi]), float(F1g[gi].mean())


def tune_decision(P, Y, min_support=3):
    """Global threshold maximising macro-F1, then a per-label threshold (shrunk 50% towards the global one) for labels with enough
    positives and a clear gain."""
    F1g = f1_grid(P, Y)
    gi = int(F1g.mean(1).argmax())
    t_g = float(GRID[gi])
    per = np.full(Y.shape[1], t_g)
    for l in range(Y.shape[1]):
        if Y[:, l].sum() < min_support:
            continue
        bi = int(F1g[:, l].argmax())
        if F1g[bi, l] > F1g[gi, l] + 0.02:
            per[l] = 0.5 * GRID[bi] + 0.5 * t_g
    return t_g, per


def apply_decision(P, thr, ensure_one):
    pred = P >= np.asarray(thr)[None, :]
    if ensure_one and len(P):
        empty = ~pred.any(1)
        top = (P - np.asarray(thr)[None, :]).argmax(1)
        pred[np.where(empty)[0], top[empty]] = True
    return pred


def choose_decision(P, Y, seed=0):
    """Pick global / per-label thresholds and 'at least one label' by 2-fold cross-fitting on dev (tune on one half, score on the other),
    then tune the chosen strategy on all of dev."""
    L = Y.shape[1]
    names = ["global", "global+one", "per_label", "per_label+one"]
    cross = {n: [] for n in names}
    perm = np.random.default_rng(seed).permutation(len(P))
    halves = [perm[::2], perm[1::2]]
    if min(len(h) for h in halves) >= 2:
        for a, b in ((0, 1), (1, 0)):
            t_g, per = tune_decision(P[halves[a]], Y[halves[a]])
            for n in names:
                thr = np.full(L, t_g) if n.startswith("global") else per
                cross[n].append(ml_scores(Y[halves[b]], apply_decision(P[halves[b]], thr, n.endswith("+one")))["macro_f1"])
        cross = {n: float(np.mean(v)) for n, v in cross.items()}
    else:
        cross = {n: 0.0 for n in names}
    best = max(names, key=lambda n: cross[n]) if any(cross.values()) else "global+one"
    t_g, per = tune_decision(P, Y)
    thr = np.full(L, t_g) if best.startswith("global") else per
    sc = ml_scores(Y, apply_decision(P, thr, best.endswith("+one")))
    return {"name": best, "thr": [float(x) for x in thr], "ensure_one": best.endswith("+one"), "t_global": t_g,
            "cross_fit": cross, "scores": sc}


# ============================================================================================
# 3. Methods: reuse Phase 4 (adapters / BitFit / freezing / LoRA / LoRMA) + soft prompt tuning
# ============================================================================================
def apply_method5(net, cfg: LCfg):
    if cfg.method == "prompt":
        for p in net.lm.parameters():
            p.requires_grad = False
        log(f"[prompt] {cfg.prompt_tokens} learned soft-prompt vectors prepended to every chunk; backbone frozen")
        return
    J.apply_method4(net, cfg)


def backbone5(lm, ids, km, bidir, prompt):
    B, T = ids.shape
    dev = ids.device
    x = lm.wte(ids)
    P = 0
    if prompt is not None:
        P = prompt.size(0)
        x = torch.cat([prompt.unsqueeze(0).expand(B, -1, -1).to(x.dtype), x], dim=1)
        km = torch.cat([torch.ones(B, P, dtype=torch.bool, device=dev), km], dim=1)
    T2 = T + P
    pos = torch.arange(T2, device=dev).unsqueeze(0)
    if lm.wpe is not None:
        x = x + lm.wpe(pos)
    rope = None
    if lm.rope_cos is not None:
        rope = (lm.rope_cos[pos].unsqueeze(1), lm.rope_sin[pos].unsqueeze(1))
    if bidir:
        amask = km[:, None, None, :]
    else:
        qi, kj = torch.arange(T2, device=dev).unsqueeze(1), torch.arange(T2, device=dev).unsqueeze(0)
        amask = ((kj <= qi)[None, None] & km[:, None, None, :]) | (kj == qi)[None, None]
    for blk in lm.blocks:
        x = blk(x, rope, amask, km, None)
    return lm.norm_f(x)[:, P:]


class LSINet(nn.Module):
    def __init__(self, lm, L, bidir, pool, chunk_dim, head_dropout, prompt_tokens, prior):
        super().__init__()
        self.lm, self.bidir, self.pool, self.P = lm, bidir, pool, prompt_tokens
        d = lm.mc["n_embd"]
        if prompt_tokens:
            self.prompt = nn.Parameter(torch.randn(prompt_tokens, d) * 0.02)
        self.proj = nn.Linear(2 * d, chunk_dim)
        self.norm = nn.LayerNorm(chunk_dim)
        self.pos = nn.Embedding(8, chunk_dim)
        nn.init.normal_(self.pos.weight, 0.0, 0.02)
        self.drop = nn.Dropout(head_dropout)
        bias0 = torch.tensor(np.log(prior / (1.0 - prior)), dtype=torch.float32)       # start from the label priors (huge help for rare statutes)
        if pool == "attn":
            self.q = nn.Parameter(torch.randn(L, chunk_dim) * 0.02)
            self.w = nn.Parameter(torch.randn(L, chunk_dim) * 0.02)
            self.b = nn.Parameter(bias0.clone())
        else:
            self.out = nn.Linear(chunk_dim, L)
            nn.init.normal_(self.out.weight, 0.0, 0.02)
            with torch.no_grad():
                self.out.bias.copy_(bias0)

    def forward(self, ids, km, owner, slot, rel, counts):
        h = backbone5(self.lm, ids, km, self.bidir, self.prompt if self.P else None)          # (N, T, d)
        kf = km.unsqueeze(-1).to(h.dtype)
        mean = (h * kf).sum(1) / kf.sum(1).clamp(min=1.0)
        lastv = h[torch.arange(h.size(0), device=h.device), km.sum(1) - 1]
        c = self.drop(F.gelu(self.norm(self.proj(torch.cat([mean, lastv], dim=-1)) + self.pos(rel))))        # (N, D)
        B, Cm = counts.numel(), int(counts.max())
        pad = c.new_zeros(B, Cm, c.size(-1)).index_put((owner, slot), c)
        mask = torch.arange(Cm, device=c.device).unsqueeze(0) < counts.unsqueeze(1)                  # (B, Cm)
        if self.pool == "attn":                                                                      # label-wise attention over chunks
            sc = torch.einsum("bnd,ld->bln", pad, self.q.to(pad.dtype)).float().masked_fill(~mask.unsqueeze(1), -1e9)
            ctx = torch.einsum("bln,bnd->bld", sc.softmax(-1).to(pad.dtype), pad)
            return (ctx * self.w.to(ctx.dtype)).sum(-1) + self.b.to(ctx.dtype)
        if self.pool == "mean":
            z = (pad * mask.unsqueeze(-1).to(pad.dtype)).sum(1) / mask.sum(1, keepdim=True).to(pad.dtype)
            return self.out(z)
        if self.pool == "max":
            return self.out(pad.masked_fill(~mask.unsqueeze(-1), -1e4).max(1).values)
        lg = self.out(pad)                                                                           # chunk_max: chunk-level scores, then max
        return lg.masked_fill(~mask.unsqueeze(-1), -1e4).max(1).values


def build_net5(cfg: LCfg, lm, L, prior):
    arch = lm.mc["arch"]
    bidir = (arch != "retnet") if cfg.bidir == "auto" else (cfg.bidir in ("true", "yes", "1", True))
    if arch == "retnet":
        bidir = False
    P = cfg.prompt_tokens if cfg.method == "prompt" else 0
    log(f"[model] arch={arch} bidirectional_attention={bidir} pool={cfg.pool} labels={L} loss={cfg.loss} prompt_tokens={P}")
    return LSINet(lm, L, bidir, cfg.pool, cfg.chunk_dim, cfg.head_dropout, P, prior)


def ml_loss(logits, Y, cfg: LCfg, pos_weight):
    x = logits.float()
    if cfg.loss == "bce":
        return F.binary_cross_entropy_with_logits(x, Y)
    if cfg.loss == "bce_pw":
        return F.binary_cross_entropy_with_logits(x, Y, pos_weight=pos_weight)
    if cfg.loss == "focal":
        bce = F.binary_cross_entropy_with_logits(x, Y, reduction="none")
        p = torch.sigmoid(x)
        pt = Y * p + (1 - Y) * (1 - p)
        return ((1 - pt) ** cfg.focal_gamma * bce).mean()
    if cfg.loss == "asl":                                                   # asymmetric loss: down-weights easy negatives
        p = torch.sigmoid(x)
        pm = (p - 0.05).clamp(min=0)
        return -(Y * torch.log(p.clamp(min=1e-8)) + (1 - Y) * pm ** 4 * torch.log((1 - pm).clamp(min=1e-8))).mean()
    raise SystemExit(f"unknown loss {cfg.loss!r}: use bce | bce_pw | focal | asl")


def kd_loss_ml(s, t, T):
    """Soft-target BCE between temperature-scaled student and teacher probabilities (multi-label distillation), times T^2."""
    return F.binary_cross_entropy_with_logits(s.float() / T, torch.sigmoid(t.float() / T)) * (T * T)


def load_run5(cfg: LCfg, run_name: str, dev):
    from tokenizers import Tokenizer
    rd = Path(cfg.work_dir) / "ls_runs" / run_name
    ck = torch.load(rd / "best.pt", map_location="cpu", weights_only=False)
    saved = LCfg()
    apply_overrides(saved, {k: v for k, v in ck["cfg"].items() if k in {f.name for f in fields(LCfg)}})
    saved.work_dir = cfg.work_dir
    if cfg.local_data_dir:
        saved.local_data_dir = cfg.local_data_dir
    L = ck["n_labels"]
    net = build_net5(saved, P1.LM(ck["mc"]), L, np.full(L, 0.01))
    apply_method5(net, saved)
    net.load_state_dict(ck["net"])
    net.to(dev).eval()
    return {"rd": rd, "ck": ck, "saved": saved, "net": net, "L": L, "tok": Tokenizer.from_file(str(rd / "tokenizer.json"))}


def load_teacher5(cfg: LCfg, dev, tok_path, L):
    tr = Path(cfg.work_dir) / "ls_runs" / cfg.teacher_run
    if not (tr / "best.pt").exists():
        raise SystemExit(f"teacher run not found: {tr}/best.pt (train it first)")
    if (tr / "tokenizer.json").read_bytes() != Path(tok_path).read_bytes():
        raise SystemExit("teacher and student use different tokenizers (distillation needs identical token ids)")
    r = load_run5(cfg, cfg.teacher_run, dev)
    if r["L"] != L:
        raise SystemExit("teacher and student use different label spaces")
    for p in r["net"].parameters():
        p.requires_grad = False
    n = sum(p.numel() for p in r["net"].parameters())
    log(f"[kd] teacher {cfg.teacher_run}: arch={r['ck']['mc']['arch']} method={r['saved'].method} params={n / 1e6:.1f}M")
    return r["net"], n, r["ck"]["mc"]


# ============================================================================================
# 4. Inference + training
# ============================================================================================
@torch.no_grad()
def doc_probs5(net, docs, cfg: LCfg, dev, amp, eos, L, idxs=None):
    net.eval()
    idxs = list(range(len(docs))) if idxs is None else list(idxs)
    order = sorted(range(len(idxs)), key=lambda i: docs[idxs[i]].n)
    out = np.zeros((len(idxs), L), dtype=np.float32)
    for b0 in range(0, len(order), cfg.eval_docs):
        sel = order[b0:b0 + cfg.eval_docs]
        batch = collate5(docs, [idxs[i] for i in sel], cfg, dev, eos, L, cfg.eval_chunks, cfg.eval_select)
        with amp():
            lg = net(*batch[:6])
        out[sel] = torch.sigmoid(lg.float()).cpu().numpy()
    return out


def run_train5(cfg: LCfg):
    rd = run_dir5(cfg)
    rd.mkdir(parents=True, exist_ok=True)
    jd = asdict(cfg)
    P1.jdump(jd, rd / "config.json")
    random.seed(cfg.seed), np.random.seed(cfg.seed), torch.manual_seed(cfg.seed)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    amp_dtype, amp = P1.amp_setup(dev)
    tok, lm, mc = N.load_p1(cfg)
    tok_path = str(Path(cfg.p1_dir) / "tokenizer.json")
    shutil.copy(tok_path, rd / "tokenizer.json")
    if cfg.method == "kd":
        if not cfg.teacher_run:
            raise SystemExit('method "kd" needs "teacher_run": the name of a finished LSI run')
        lm, mc = N.make_student(lm, mc, cfg)
    eos = tok.token_to_id(P1.EOS)
    P_tok = cfg.prompt_tokens if cfg.method == "prompt" else 0
    assert cfg.max_len + P_tok <= mc["ctx"], "max_len (+ prompt tokens) exceeds the Phase-1 context length"
    t0 = time.time()
    tr = load_split5(cfg, f"{cfg.train_prefix}/train.json", tok, True, cfg.max_train_docs, tok_path)
    dv = load_split5(cfg, f"{cfg.train_prefix}/dev.json", tok, True, 0, tok_path)
    if cfg.dedupe_train:
        tr, n1, c1 = dedupe_docs5(tr)
        dv, n2, c2 = dedupe_docs5(dv)
        log(f"[data] removed exact duplicates: {n1} from train ({c1} with conflicting labels), {n2} from dev ({c2} conflicting)")
    L = 1 + max(max((max(d.labels) for d in docs if d.labels), default=0) for docs in (tr, dv))
    Ytr, Ydv = label_matrix(tr, L), label_matrix(dv, L)
    freq = Ytr.sum(0)
    log(f"[data] train {len(tr)} cases | dev {len(dv)} cases | {L} statute ids ({int((freq > 0).sum())} occur in train, {int((freq < 5).sum())} have < 5 train cases) | "
        f"labels/case mean {Ytr.sum(1).mean():.2f} | loaded in {time.time() - t0:.0f}s")
    prior = np.clip((freq + 1.0) / (len(tr) + 2.0), 1e-4, 0.5)
    teacher, teacher_params = None, 0
    if cfg.method == "kd":
        teacher, teacher_params, t_mc = load_teacher5(cfg, dev, rd / "tokenizer.json", L)
        assert cfg.max_len <= t_mc["ctx"]

    net = build_net5(cfg, lm, L, prior)
    apply_method5(net, cfg)
    net.to(dev)
    n_total = sum(p.numel() for p in net.parameters())
    n_train = sum(p.numel() for p in net.parameters() if p.requires_grad)
    log(f"[model] total params {n_total / 1e6:.1f}M | trainable {n_train / 1e6:.3f}M | method={cfg.method}")

    peft = cfg.method in ("lora", "lorma", "adapter", "bitfit", "prompt")
    body_lr, body_wd = (cfg.peft_lr, 0.0) if peft else (cfg.lr, cfg.weight_decay)
    nb = mc["n_layer"]

    def lr_scale(name):
        if peft or cfg.llrd >= 1.0 or not name.startswith("lm."):
            return 1.0
        m_ = re.match(r"lm\.blocks\.(\d+)\.", name)
        if m_:
            return cfg.llrd ** (nb - 1 - int(m_.group(1)))
        return cfg.llrd ** nb if name.startswith(("lm.wte", "lm.wpe")) else 1.0

    buckets = {}
    for n_, p_ in net.named_parameters():
        if not p_.requires_grad:
            continue
        if n_.startswith("lm."):
            base, wd = body_lr * lr_scale(n_), (body_wd if p_.dim() >= 2 else 0.0)
        else:                                            # label-wise head, prompt, projection
            base, wd = cfg.head_lr, (0.0 if n_ in ("b", "prompt") or p_.dim() < 2 else cfg.weight_decay)
        buckets.setdefault((round(base, 12), wd), []).append(p_)
    opt = torch.optim.AdamW([{"params": ps, "lr": b_, "base": b_, "weight_decay": wd} for (b_, wd), ps in buckets.items()],
                            betas=(0.9, 0.98), eps=1e-8)
    scaler = P1.make_scaler(dev == "cuda" and amp_dtype == torch.float16)
    pos_weight = torch.tensor(np.clip(np.sqrt((len(tr) - freq + 1.0) / (freq + 1.0)), 1.0, cfg.pos_weight_max), dtype=torch.float32, device=dev)
    ema = J.EMA(net, cfg.ema_decay) if cfg.ema_decay > 0 else None

    rng = np.random.default_rng(cfg.seed)
    prng = random.Random(cfg.seed)
    dev_sub = np.sort(rng.permutation(len(dv))[:cfg.dev_max_docs]) if cfg.dev_max_docs and len(dv) > cfg.dev_max_docs else np.arange(len(dv))
    n_epoch = len(tr) if not cfg.docs_per_epoch else min(cfg.docs_per_epoch, len(tr))
    steps_per_epoch = max(1, math.ceil(n_epoch / (cfg.batch_docs * cfg.accum)))
    total_steps = cfg.epochs * steps_per_epoch
    budget = cfg.time_budget_min * 60.0
    log(f"[train] {n_epoch} cases/epoch, {steps_per_epoch} steps/epoch x {cfg.epochs} epochs; time budget {cfg.time_budget_min} min; "
        f"{cfg.train_chunks} random chunks per case; EMA {cfg.ema_decay}")
    best, step, bad, t_start, stop = {"f1": -1.0}, 0, 0, time.time(), False
    if dev == "cuda":
        torch.cuda.reset_peak_memory_stats()
    logf = open(rd / "log.jsonl", "a")

    def do_eval(tag, loss_val):
        nonlocal best, bad
        with (ema.swapped() if ema is not None else contextlib.nullcontext()):
            P_ = doc_probs5(net, dv, cfg, dev, amp, eos, L, idxs=dev_sub)
            t_, f_ = best_global(P_, Ydv[dev_sub])
            sc_ = ml_scores(Ydv[dev_sub], P_ >= t_)
            log(f"[{tag}] loss {loss_val:.4f} | dev(subset {len(dev_sub)}) macro-F1 {f_:.4f} micro-F1 {sc_['micro_f1']:.4f} at global threshold {t_:.3f} | "
                f"{time.time() - t_start:.0f}s")
            logf.write(json.dumps({"tag": tag, "step": step, "loss": loss_val, "dev_macro_f1": f_, "dev_micro_f1": sc_["micro_f1"]}) + "\n")
            logf.flush()
            if f_ > best["f1"]:
                best, bad = {"f1": f_, "epoch": tag}, 0
                torch.save({"net": net.state_dict(), "mc": mc, "cfg": jd, "n_labels": L, "decision": None}, rd / "best.pt")
            else:
                bad += 1
        net.train()

    for ep in range(cfg.epochs):
        net.train()
        order = rng.permutation(len(tr))[:n_epoch]
        run_loss, n_b, early = 0.0, 0, False
        micro = cfg.batch_docs
        for g0 in range(0, len(order), micro * cfg.accum):
            prog = max(step / total_steps, (time.time() - t_start) / budget if budget > 0 else 0.0)
            if prog >= 1.0:
                stop = True
                break
            for g in opt.param_groups:
                g["lr"] = J.lr_prog(g["base"], prog, cfg.warmup_frac)
            group = order[g0:g0 + micro * cfg.accum]
            for m0 in range(0, len(group), micro):
                batch = collate5(tr, group[m0:m0 + micro], cfg, dev, eos, L, cfg.train_chunks, cfg.train_select, prng)
                Y = batch[6]
                with amp():
                    logits = net(*batch[:6])
                    if teacher is not None:
                        with torch.no_grad():
                            t_logits = teacher(*batch[:6])
                loss = ml_loss(logits, Y, cfg, pos_weight)
                if teacher is not None:
                    loss = cfg.kd_alpha * loss + (1.0 - cfg.kd_alpha) * kd_loss_ml(logits, t_logits, cfg.kd_temp)
                scaler.scale(loss * (len(Y) / len(group))).backward()
                run_loss += float(loss.detach())
                n_b += 1
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in net.parameters() if p.requires_grad], cfg.grad_clip)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            if ema is not None:
                ema.update(step)
            step += 1
            if step % 20 == 0:
                log(f"  ep {ep + 1} step {step}/{total_steps} loss {run_loss / max(1, n_b):.4f} ({time.time() - t_start:.0f}s)")
            if cfg.eval_every_steps and step % cfg.eval_every_steps == 0:
                do_eval(f"step {step}", run_loss / max(1, n_b))
                if bad >= cfg.patience:
                    early = stop = True
                    break
        if not early:
            do_eval(f"epoch {ep + 1}/{cfg.epochs}", run_loss / max(1, n_b))
        if early or bad >= cfg.patience:
            log(f"[train] early stopping: no dev improvement in {bad} evaluations (best: {best['epoch']})")
            break
        if stop:
            log("[train] time budget reached")
            break
    logf.close()
    train_s = time.time() - t_start
    peak = torch.cuda.max_memory_allocated() / 2 ** 20 if dev == "cuda" else 0.0

    ck = torch.load(rd / "best.pt", map_location="cpu", weights_only=False)
    net.load_state_dict(ck["net"])
    if dev == "cuda":
        torch.cuda.synchronize()
    t_inf = time.time()
    Pdv = doc_probs5(net, dv, cfg, dev, amp, eos, L)
    if dev == "cuda":
        torch.cuda.synchronize()
    infer_docs_s = len(dv) / max(1e-9, time.time() - t_inf)
    dec = choose_decision(Pdv, Ydv)
    t_g, f_g = best_global(Pdv, Ydv)
    sc = dec["scores"]
    ck["decision"] = {k: dec[k] for k in ("name", "thr", "ensure_one", "t_global")}
    torch.save(ck, rd / "best.pt")
    np.save(rd / "dev_probs.npy", Pdv)
    P1.jdump([d.id for d in dv], rd / "dev_ids.json")
    P1.jdump(label_matrix(dv, L).tolist(), rd / "dev_labels.json")
    log(f"[dev] decision strategies (2-fold cross-fit macro-F1 on dev): { {k: round(v, 4) for k, v in dec['cross_fit'].items()} } -> chosen '{dec['name']}'")
    log(f"[dev] macro-F1 {sc['macro_f1']:.4f} (global threshold {t_g:.3f} alone: {f_g:.4f}) | micro-F1 {sc['micro_f1']:.4f} | subset accuracy {sc['subset_acc']:.4f} | "
        f"macro-P {sc['macro_p']:.4f} macro-R {sc['macro_r']:.4f}")
    supp = Ydv.sum(0)
    for lo, hi, name in ((0, 1, "absent in dev"), (1, 5, "1-4 dev cases"), (5, 20, "5-19 dev cases"), (20, 10 ** 9, ">= 20 dev cases")):
        mk = (supp >= lo) & (supp < hi)
        if mk.any():
            log(f"[dev] labels with {name:<15} ({int(mk.sum()):>3}): mean F1 {sc['f1'][mk].mean():.4f}")
    worst = np.argsort(-supp)[:10]
    log("[dev] 10 most frequent statutes (id: dev cases, F1): " + ", ".join(f"{int(l)}: {int(supp[l])}, {sc['f1'][l]:.2f}" for l in worst))
    variant = cfg.lorma_variant if cfg.method == "lorma" else ""
    P1.jdump({"best_epoch": best["epoch"], "decision": ck["decision"], "cross_fit": dec["cross_fit"], "macro_f1": sc["macro_f1"],
              "micro_f1": sc["micro_f1"], "subset_acc": sc["subset_acc"], "macro_p": sc["macro_p"], "macro_r": sc["macro_r"],
              "macro_f1_global_only": f_g, "n_labels": L, "per_label_f1": [float(x) for x in sc["f1"]], "label_support_dev": supp.tolist(),
              "label_support_train": freq.tolist(), "params_total": n_total, "params_trainable": n_train, "train_seconds": train_s,
              "peak_gpu_mb": peak, "infer_docs_per_s": infer_docs_s, "p1_arch": mc["arch"], "method": cfg.method,
              "teacher_run": cfg.teacher_run, "teacher_params": teacher_params}, rd / "dev_report.json")
    log_csv(cfg, {"run": cfg.run, "p1_arch": mc["arch"], "method": cfg.method + (f"-{variant}" if variant else ""), "pool": cfg.pool, "loss": cfg.loss,
                  "params_total_M": round(n_total / 1e6, 2), "params_trainable_M": round(n_train / 1e6, 3),
                  "dev_macro_f1": round(sc["macro_f1"], 4), "dev_micro_f1": round(sc["micro_f1"], 4), "dev_subset_acc": round(sc["subset_acc"], 4),
                  "decision": dec["name"], "n_labels": L, "train_s": round(train_s), "peak_gpu_mb": round(peak),
                  "infer_docs_s": round(infer_docs_s, 2), "teacher_run": cfg.teacher_run})
    if cfg.predict_after:
        run_predict5(cfg)


def write_submission5(cfg: LCfg, rd: Path, ids, pred_mat, L):
    out = []
    for i, row in zip(ids, pred_mat):
        labs = sorted({int(x) for x in np.where(row)[0]})
        assert all(0 <= x < L for x in labs)
        out.append({"id": i, "pred": labs})
    assert len(out) == len(set(ids))
    rd.mkdir(parents=True, exist_ok=True)
    pj = rd / "prediction.json"
    P1.jdump(out, pj)
    zp = rd / "submission.zip"
    with zipfile.ZipFile(zp, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    hist = Path(cfg.work_dir) / "ls_submissions"
    hist.mkdir(parents=True, exist_ok=True)
    shutil.copy(zp, hist / f"{cfg.run}_{time.strftime('%m%d_%H%M%S')}.zip")
    log(f"[predict] wrote {pj} ({len(out)} cases, {np.mean([len(o['pred']) for o in out]):.2f} statutes per case on average) and {zp}")
    return str(zp)


def run_predict5(cfg: LCfg):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    _, amp = P1.amp_setup(dev)
    r = load_run5(cfg, cfg.run, dev)
    saved = r["saved"]
    eos = r["tok"].token_to_id(P1.EOS)
    te = load_split5(saved, saved.test_file, r["tok"], False, 0, str(r["rd"] / "tokenizer.json"))
    P = doc_probs5(r["net"], te, saved, dev, amp, eos, r["L"])
    np.save(r["rd"] / "test_probs.npy", P)
    P1.jdump([d.id for d in te], r["rd"] / "test_ids.json")
    dec = r["ck"]["decision"]
    pred = apply_decision(P, np.array(dec["thr"]), dec["ensure_one"])
    log(f"[predict] {len(te)} test cases, decision '{dec['name']}'")
    return write_submission5(cfg, r["rd"], [d.id for d in te], pred, r["L"])


# ============================================================================================
# 5. Ensemble (average saved probabilities of finished runs)
# ============================================================================================
def run_ensemble5(cfg: LCfg):
    root = Path(cfg.work_dir) / "ls_runs"
    if cfg.ens_runs:
        names = list(cfg.ens_runs)
    elif cfg.ens_auto:
        names = [f.parent.name for f in sorted(root.glob("*/test_probs.npy")) if (f.parent / "dev_probs.npy").exists()
                 and (f.parent.name.startswith(cfg.ens_prefix) if cfg.ens_prefix else not f.parent.name.startswith(("selftest", "check")))]
    else:
        names = []
    if len(names) < 2:
        raise SystemExit('need >= 2 finished runs: set "ens_runs": [...] or "ens_auto": true')
    dev_ids, test_ids = P1.jload(root / names[0] / "dev_ids.json"), P1.jload(root / names[0] / "test_ids.json")
    Ydv = np.array(P1.jload(root / names[0] / "dev_labels.json"), dtype=np.uint8)
    L = Ydv.shape[1]
    dv_p, te_p = {}, {}
    for n in names:
        if P1.jload(root / n / "dev_ids.json") != dev_ids or P1.jload(root / n / "test_ids.json") != test_ids or np.load(root / n / "dev_probs.npy").shape[1] != L:
            log(f"[ensemble] skipping {n}: different cases / label space")
            continue
        dv_p[n], te_p[n] = np.load(root / n / "dev_probs.npy"), np.load(root / n / "test_probs.npy")
    quick = lambda P: best_global(P, Ydv)[1]
    single = {k: quick(v) for k, v in dv_p.items()}
    for k, v in sorted(single.items(), key=lambda kv: -kv[1]):
        log(f"[ensemble] single model {k:<28} dev macro-F1 {v:.4f}")
    mean = lambda keys, d: sum(d[k] for k in keys) / len(keys)
    if cfg.ens_runs or not cfg.ens_auto:
        chosen = list(dv_p)
    else:
        order = sorted(single, key=lambda k: -single[k])
        chosen, cur = [order[0]], single[order[0]]
        for k in order[1:]:
            if len(chosen) >= cfg.ens_max:
                break
            f = quick(mean(chosen + [k], dv_p))
            log(f"[ensemble] try adding {k:<28} -> {f:.4f} (current {cur:.4f})")
            if f > cur + 0.002:
                chosen.append(k)
                cur = f
    log(f"[ensemble] members: {chosen}")
    Pd, Pt = mean(chosen, dv_p), mean(chosen, te_p)
    dec = choose_decision(Pd, Ydv)
    sc = dec["scores"]
    log(f"[ensemble] cross-fit { {k: round(v, 4) for k, v in dec['cross_fit'].items()} } -> '{dec['name']}' | dev macro-F1 {sc['macro_f1']:.4f} "
        f"micro-F1 {sc['micro_f1']:.4f} subset acc {sc['subset_acc']:.4f}")
    rd = run_dir5(cfg)
    rd.mkdir(parents=True, exist_ok=True)
    P1.jdump({"members": chosen, "single_model_f1": single, "decision": {k: dec[k] for k in ("name", "t_global", "ensure_one")},
              "macro_f1": sc["macro_f1"], "micro_f1": sc["micro_f1"], "subset_acc": sc["subset_acc"]}, rd / "dev_report.json")
    log_csv(cfg, {"run": cfg.run, "p1_arch": "ensemble", "method": "ensemble", "dev_macro_f1": round(sc["macro_f1"], 4),
                  "dev_micro_f1": round(sc["micro_f1"], 4), "dev_subset_acc": round(sc["subset_acc"], 4), "decision": dec["name"],
                  "n_labels": L, "teacher_run": "+".join(chosen)})
    return write_submission5(cfg, rd, test_ids, apply_decision(Pt, np.array(dec["thr"]), dec["ensure_one"]), L)


# ============================================================================================
# 6. Diagnostics: check + probe
# ============================================================================================
def diagnose5(cfg: LCfg):
    tok, _, mc = N.load_p1(cfg)
    tp = str(Path(cfg.p1_dir) / "tokenizer.json")
    sp = {}
    for name, rel, lab in (("train", f"{cfg.train_prefix}/train.json", True), ("dev", f"{cfg.train_prefix}/dev.json", True), ("test", cfg.test_file, False)):
        sp[name] = load_split5(cfg, rel, tok, lab, cfg.max_train_docs if name == "train" else 0, tp)
        docs, n = sp[name], np.array([d.n for d in sp[name]])
        nch = np.array([len(J.chunk_bounds(d.n, cfg.max_len - 1, cfg.overlap)) for d in docs])
        line = (f"[check] {name:<5}: {len(docs):>6} cases | tokens/case mean {n.mean():.0f} median {np.median(n):.0f} p90 {np.percentile(n, 90):.0f} max {n.max()} | "
                f"chunks/case median {np.median(nch):.0f} p90 {np.percentile(nch, 90):.0f} (eval_chunks={cfg.eval_chunks} covers {100 * np.mean(nch <= cfg.eval_chunks):.0f}% of cases fully)")
        if lab:
            nl = np.array([len(d.labels) for d in docs])
            line += f" | statutes/case mean {nl.mean():.2f} max {nl.max()} | cases without any statute: {int((nl == 0).sum())}"
        log(line)
        _, nd, nc = dedupe_docs5(docs)
        log(f"[check] {name}: {nd} exact duplicate cases ({nc} with conflicting label sets)")
    L = 1 + max(max((max(d.labels) for d in sp[s] if d.labels), default=0) for s in ("train", "dev"))
    Ytr, Ydv = label_matrix(sp["train"], L), label_matrix(sp["dev"], L)
    f = Ytr.sum(0)
    order = np.argsort(-f)
    log(f"[check] {L} statute ids in train+dev | cases per statute in train: max {int(f.max())}, median {int(np.median(f))}, min {int(f.min())} | "
        f"statutes with 0 / <5 / <20 train cases: {int((f == 0).sum())} / {int((f < 5).sum())} / {int((f < 20).sum())}")
    log("[check] most frequent statutes (id: train cases): " + ", ".join(f"{int(l)}: {int(f[l])}" for l in order[:12]))
    log(f"[check] statutes present in dev but absent from train: {int(((Ydv.sum(0) > 0) & (f == 0)).sum())} | "
        f"top-1 statute alone would give micro-F1 {ml_scores(Ydv, np.tile((np.arange(L) == order[0]), (len(Ydv), 1)))['micro_f1']:.3f} on dev")
    co = np.triu(Ytr.T.astype(np.float32) @ Ytr.astype(np.float32), k=1)
    top_pairs = np.dstack(np.unravel_index(np.argsort(-co.ravel())[:5], co.shape))[0]
    log("[check] most frequent statute pairs in the same case: " + ", ".join(f"({int(a)},{int(b)}): {int(co[a, b])}" for a, b in top_pairs if co[a, b] > 0))

    def key(d, mode):
        t = d.tok[:256] if mode == "first 256 tokens" else d.tok[-256:] if mode == "last 256 tokens" else d.tok
        return hashlib.md5(t.tobytes()).hexdigest()
    for mode in ("full text", "first 256 tokens", "last 256 tokens"):
        tmap = {}
        for d in sp["train"]:
            tmap.setdefault(key(d, mode), []).append(set(d.labels))
        for name in ("dev", "test"):
            hit = [(d, tmap[key(d, mode)]) for d in sp[name] if key(d, mode) in tmap]
            extra = ""
            if name == "dev" and hit:
                jac = [len(set(d.labels) & u[0]) / max(1, len(set(d.labels) | u[0])) for d, u in hit]
                extra = f" | label-set Jaccard with the train copy {np.mean(jac):.2f}"
            log(f"[overlap] {name:<4} cases whose {mode:<16} also occurs in train: {len(hit):>5}/{len(sp[name])}{extra}")


def run_probe(cfg: LCfg):
    """DIAGNOSTIC ONLY: which part of a case identifies the statutes? One-vs-rest logistic regression on bag-of-n-grams of one region at a time,
    for the most frequent statutes; reports the mean F1 on dev per region and the strongest n-grams of the best region for the top statute."""
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.linear_model import LogisticRegression
    except ImportError:
        raise SystemExit("probe needs scikit-learn:  pip install scikit-learn")
    from tokenizers import Tokenizer
    tp = str(Path(cfg.p1_dir) / "tokenizer.json")
    tok = Tokenizer.from_file(tp)
    tr = load_split5(cfg, f"{cfg.train_prefix}/train.json", tok, True, cfg.max_train_docs, tp)
    dv = load_split5(cfg, f"{cfg.train_prefix}/dev.json", tok, True, 0, tp)
    tr, _, _ = dedupe_docs5(tr)
    dv, _, _ = dedupe_docs5(dv)
    L = 1 + max(max((max(d.labels) for d in docs if d.labels), default=0) for docs in (tr, dv))
    rng = np.random.default_rng(0)
    trs = [tr[i] for i in rng.permutation(len(tr))[:cfg.probe_docs]]
    Ytr, Ydv = label_matrix(trs, L), label_matrix(dv, L)
    top = [int(l) for l in np.argsort(-Ytr.sum(0))[:cfg.probe_labels] if Ytr[:, l].sum() >= 10 and Ydv[:, l].sum() >= 1]
    log(f"[probe] {len(trs)} training cases, {len(dv)} dev cases, statutes probed: {top}")
    mid = lambda t: t[max(0, len(t) // 2 - 1024): len(t) // 2 + 1024]
    regions = {"first 512 tokens": lambda t: t[:512], "first 2048 tokens": lambda t: t[:2048], "last 512 tokens": lambda t: t[-512:],
               "last 2048 tokens": lambda t: t[-2048:], "middle 2048 tokens": mid,
               "first 4096 + last 4096": lambda t: np.concatenate([t[:4096], t[-4096:]]) if len(t) > 8192 else t}
    text = lambda t: " ".join(f"t{int(x)}" for x in t)
    results, best = {}, (-1.0, None, None, None)
    for name, fn in regions.items():
        vec = TfidfVectorizer(token_pattern=r"\S+", ngram_range=(1, 2), min_df=3, max_features=150000, sublinear_tf=True, lowercase=False)
        Xtr = vec.fit_transform([text(fn(d.tok)) for d in trs])
        Xdv = vec.transform([text(fn(d.tok)) for d in dv])
        f1s, clfs = [], {}
        for l in top:
            clf = LogisticRegression(max_iter=200, C=4.0, class_weight="balanced").fit(Xtr, Ytr[:, l])
            p = clf.predict_proba(Xdv)[:, 1]
            f1s.append(max(per_label_f1(Ydv[:, [l]], (p >= t)[:, None])[0][0] for t in (0.3, 0.5, 0.7)))
            clfs[l] = clf
        results[name] = float(np.mean(f1s))
        log(f"[probe] {name:<24} mean F1 over the probed statutes on dev: {results[name]:.4f}")
        if results[name] > best[0]:
            best = (results[name], name, vec, clfs)
    _, name, vec, clfs = best
    l0 = top[0]
    coef, names = clfs[l0].coef_[0], vec.get_feature_names_out()
    order = np.argsort(coef)[::-1][:15]
    show = lambda f: tok.decode([int(x[1:]) for x in f.split()]).replace("\n", " ").strip()
    log(f"[probe] BEST region: {name}. Strongest n-grams for statute {l0}: " + " | ".join(show(names[i]) for i in order))
    P1.jdump(results, Path(cfg.work_dir) / "ls_probe.json")
    return results


# ============================================================================================
# 7. Self-test
# ============================================================================================
def _synthetic(tmp, rng, L=8):
    filler = "the court held that appeal petitioner respondent order section act learned counsel submitted impugned judgment".split()

    def doc(labs):
        words = [rng.choice(filler) for _ in range(rng.randint(300, 1500))]
        for l in labs:
            for _ in range(rng.randint(2, 3)):
                words.insert(rng.randint(0, len(words)), f"statute{l}x")
        return " ".join(words)

    os.makedirs(tmp, exist_ok=True)
    texts = []
    for split, n in (("train", 60), ("dev", 16), ("test", 12)):
        recs = []
        for i in range(n):
            labs = sorted(rng.sample(range(L), rng.randint(1, 3))) if rng.random() < 0.9 else [0]
            t = doc(labs)
            texts.append(t)
            r = {"id": f"{split}_{i}", "text": t}
            if split != "test":
                r["labels"] = labs
            recs.append(r)
        json.dump(recs, open(os.path.join(tmp, f"{split}.json"), "w"))
    return texts


def selftest() -> bool:
    import tempfile
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
    ok = True
    rng = random.Random(0)
    tmp = tempfile.mkdtemp(prefix="cs781_ls_selftest_")
    dd = os.path.join(tmp, "data")
    texts = _synthetic(dd, rng)
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    tok.train_from_iterator([" ".join(t.split()) for t in texts], trainers.BpeTrainer(
        vocab_size=500, special_tokens=[P1.EOS], initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False))
    V = P1.pad64(tok.get_vocab_size())
    eos = tok.token_to_id(P1.EOS)

    log("=== 1. data, cache, metrics, thresholds, losses ===")
    p1x = os.path.join(tmp, "p1_cache")
    os.makedirs(p1x, exist_ok=True)
    tok.save(os.path.join(p1x, "tokenizer.json"))
    cfg0 = apply_overrides(LCfg(), {"local_data_dir": dd, "work_dir": os.path.join(tmp, "work0"), "max_len": 64})
    tr = load_split5(cfg0, "Phase_5_LSI/Train/train.json", tok, True, 0, os.path.join(p1x, "tokenizer.json"))
    tr2 = load_split5(cfg0, "Phase_5_LSI/Train/train.json", tok, True, 0, os.path.join(p1x, "tokenizer.json"))     # second call: from the cache
    g1 = len(tr) == 60 and all(np.array_equal(a.tok, b.tok) and a.labels == b.labels for a, b in zip(tr, tr2)) and any((Path(cfg0.work_dir) / "cache").glob("lsi_*.pkl"))
    L = 8
    Y = np.array([[1, 0, 1, 0], [0, 1, 0, 0], [1, 1, 0, 0], [0, 0, 0, 0]], dtype=np.uint8)
    Yh = np.array([[1, 0, 0, 0], [0, 1, 1, 0], [1, 1, 0, 0], [0, 0, 0, 0]], dtype=np.uint8)
    sc = ml_scores(Y, Yh)
    f_ref = [2 * 2 / (2 * 2 + 0 + 0), 2 * 2 / (2 * 2 + 0 + 0), 0.0, 0.0]            # per-label F1 by hand
    g2 = abs(sc["macro_f1"] - np.mean(f_ref)) < 1e-9 and abs(sc["subset_acc"] - 0.5) < 1e-9 and abs(sc["micro_f1"] - 2 * 4 / (2 * 4 + 1 + 1)) < 1e-9
    P = np.array([[0.9, 0.1, 0.2], [0.2, 0.3, 0.1], [0.6, 0.55, 0.1]])
    pr = apply_decision(P, np.full(3, 0.5), True)
    g3 = pr[1].sum() == 1 and pr[1, 1] and not apply_decision(P, np.full(3, 0.5), False)[1].any()
    Pm = np.random.default_rng(0).random((40, 6))
    Ym = (Pm > 0.7).astype(np.uint8)
    dec = choose_decision(Pm, Ym)
    g4 = dec["name"] in ("global", "global+one", "per_label", "per_label+one") and len(dec["thr"]) == 6 and dec["scores"]["macro_f1"] >= 0.5
    cfgl = apply_overrides(LCfg(), {})
    lg, Yt = torch.randn(4, 6), torch.randint(0, 2, (4, 6)).float()
    pw = torch.full((6,), 3.0)
    g5 = all(float(ml_loss(lg, Yt, apply_overrides(LCfg(), {"loss": k}), pw)) > 0 for k in ("bce", "bce_pw", "focal", "asl")) and \
        abs(float(kd_loss_ml(lg, lg, 2.0)) - float(F.binary_cross_entropy_with_logits(lg / 2.0, torch.sigmoid(lg / 2.0)) * 4)) < 1e-6
    g6 = pick_chunks(10, 3, "tail", random) == [7, 8, 9] and pick_chunks(10, 3, "uniform", random)[0] == 0 and pick_chunks(10, 3, "uniform", random)[-1] == 9 \
        and len(pick_chunks(10, 3, "random", random.Random(1))) == 3
    ok &= all([g1, g2, g3, g4, g5, g6])
    log(f"  token cache {'PASS' if g1 else 'FAIL'} | metrics = evaluator definitions {'PASS' if g2 else 'FAIL'} | 'at least one statute' rule {'PASS' if g3 else 'FAIL'} | "
        f"decision strategies {'PASS' if g4 else 'FAIL'} | losses {'PASS' if g5 else 'FAIL'} | chunk picking {'PASS' if g6 else 'FAIL'}")

    log("=== 2. methods: identity at init (adapters / LoRMA / BitFit / freezing), prompt tuning trains only prompt + head ===")
    prior = np.full(L, 0.2)
    for arch in P1.ARCH_PRESETS:
        for meth, var in (("adapter", "plus"), ("lora", "plus"), ("lorma", "plus"), ("lorma", "pi"), ("bitfit", "plus"), ("partial", "plus"), ("prompt", "plus")):
            tag = f"{arch}/{meth}" + (f"-{var}" if meth == "lorma" else "")
            try:
                torch.manual_seed(0)
                c = apply_overrides(LCfg(), {"method": meth, "lorma_variant": var, "peft_r": 4, "peft_alpha": 8.0, "peft_dropout": 0.0, "adapter_dim": 8,
                                             "freeze_layers": 1, "chunk_dim": 32, "max_len": 64, "prompt_tokens": 4})
                for pool in ("attn", "chunk_max", "max", "mean") if (arch == "llama" and meth == "adapter") else ("attn",):
                    c.pool = pool
                    net = build_net5(c, P1.LM(N._tiny_mc(arch, V)), L, prior).eval()
                    inp = collate5(tr, [0, 1], c, "cpu", eos, L, 3, "uniform")[:6]
                    with torch.no_grad():
                        ref = net(*inp)
                    assert ref.shape == (2, L)
                    if pool != "attn":
                        continue
                    apply_method5(net, c)
                    net.eval()
                    with torch.no_grad():
                        out = net(*inp)
                    diff = (ref - out).abs().max().item()
                    n_tr = sum(p.numel() for p in net.parameters() if p.requires_grad)
                    n_all = sum(p.numel() for p in net.parameters())
                    net.train()
                    net(*inp).float().pow(2).mean().backward()
                    grads = any(p.grad is not None and float(p.grad.abs().sum()) > 0 for p in net.parameters() if p.requires_grad)
                    good = (diff < 1e-4 or meth == "prompt") and n_tr < n_all and grads
                    ok &= good
                    log(f"  {tag:<18} |adapted - base| = {diff:.1e}  trainable {n_tr:,}/{n_all:,}  {'PASS' if good else 'FAIL'}")
            except Exception as e:
                import traceback
                traceback.print_exc()
                log(f"  {tag:<18} FAIL: {type(e).__name__}: {e}")
                ok = False

    log("=== 3. end-to-end: architectures / methods / pooling / losses, then ensembles ===")
    p1dirs = {}
    for arch in P1.ARCH_PRESETS:
        p1 = os.path.join(tmp, f"p1_{arch}")
        os.makedirs(p1, exist_ok=True)
        torch.save({"model": P1.LM(N._tiny_mc(arch, V)).state_dict(), "mc": N._tiny_mc(arch, V), "state": {"step": 0}}, os.path.join(p1, "ckpt_best.pt"))
        tok.save(os.path.join(p1, "tokenizer.json"))
        p1dirs[arch] = p1
    base = {"work_dir": os.path.join(tmp, "work"), "local_data_dir": dd, "max_len": 64, "train_chunks": 3, "eval_chunks": 6, "epochs": 2, "batch_docs": 2,
            "accum": 1, "eval_docs": 2, "lr": 1e-3, "peft_lr": 3e-3, "head_lr": 5e-3, "peft_r": 4, "peft_alpha": 8.0, "chunk_dim": 32, "adapter_dim": 8,
            "ema_decay": 0.9, "time_budget_min": 5, "patience": 5, "dev_max_docs": 12, "prompt_tokens": 4}
    cases = [(f"full_{a}", a, {"method": "full"}) for a in P1.ARCH_PRESETS] + [
        ("lorma_plus", "gpt2", {"method": "lorma", "lorma_variant": "plus"}), ("lorma_pi", "retnet", {"method": "lorma", "lorma_variant": "pi"}),
        ("lora", "llama", {"method": "lora"}), ("adapter", "moe", {"method": "adapter"}), ("bitfit", "gpt2", {"method": "bitfit"}),
        ("partial", "llama", {"method": "partial", "freeze_layers": 1}), ("prompt", "llama", {"method": "prompt"}),
        ("pool_chunkmax", "llama", {"method": "full", "pool": "chunk_max"}), ("pool_mean", "llama", {"method": "full", "pool": "mean"}),
        ("loss_asl", "llama", {"method": "full", "loss": "asl"}), ("loss_focal", "llama", {"method": "full", "loss": "focal"}),
        ("kd", "gpt2", {"method": "kd", "teacher_run": "selftest_full_llama", "student_layers": 1})]
    for tag, arch, extra in cases:
        try:
            cfg = apply_overrides(LCfg(), {**base, **extra, "run": f"selftest_{tag}", "p1_dir": p1dirs[arch]})
            run_train5(cfg)
            with zipfile.ZipFile(run_dir5(cfg) / "submission.zip") as z:
                assert z.namelist() == ["prediction.json"]
                pr = json.loads(z.read("prediction.json"))
            assert isinstance(pr, list) and [x["id"] for x in pr] == [f"test_{i}" for i in range(12)]
            assert all(set(x) == {"id", "pred"} and isinstance(x["pred"], list) and len(set(x["pred"])) == len(x["pred"])
                       and all(type(v) is int and 0 <= v < 8 for v in x["pred"]) for x in pr)
            rep = P1.jload(run_dir5(cfg) / "dev_report.json")
            if extra["method"] in ("lora", "lorma", "adapter", "bitfit", "partial", "prompt"):
                assert rep["params_trainable"] < rep["params_total"]
            if extra["method"] == "kd":
                assert rep["params_total"] < rep["teacher_params"]
            log(f"  e2e {tag:<14} PASS  (dev macro-F1 {rep['macro_f1']:.3f}, trainable {rep['params_trainable']:,} / {rep['params_total']:,})")
        except Exception as e:
            import traceback
            traceback.print_exc()
            log(f"  e2e {tag:<14} FAIL: {type(e).__name__}: {e}")
            ok = False
    try:
        for tag, ov in (("ens", {"ens_runs": ["selftest_full_llama", "selftest_full_gpt2"]}), ("ens_auto", {"ens_auto": True, "ens_prefix": "selftest_"})):
            cfg = apply_overrides(LCfg(), {"work_dir": os.path.join(tmp, "work"), "local_data_dir": dd, "run": f"selftest_{tag}", **ov})
            zp = run_ensemble5(cfg)
            with zipfile.ZipFile(zp) as z:
                pr = json.loads(z.read("prediction.json"))
            assert len(pr) == 12
        log("  ensembles (explicit + automatic) PASS")
        pcfg = apply_overrides(LCfg(), {"work_dir": os.path.join(tmp, "work"), "local_data_dir": dd, "p1_dir": p1dirs["llama"], "probe_docs": 40, "probe_labels": 4})
        res = run_probe(pcfg)
        assert res["last 512 tokens"] >= 0.0
        log("  probe stage PASS")
    except ImportError:
        log("  probe stage skipped (scikit-learn not installed)")
    except Exception as e:
        import traceback
        traceback.print_exc()
        log(f"  ensembles / probe FAIL: {type(e).__name__}: {e}")
        ok = False
    shutil.rmtree(tmp, ignore_errors=True)
    log("SELFTEST " + ("PASSED" if ok else "FAILED"))
    return ok


# ============================================================================================
# 8. CLI
# ============================================================================================
def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("stage", choices=["selftest", "check", "probe", "train", "predict", "ensemble"])
    ap.add_argument("--run", default=None)
    ap.add_argument("--cfg", default="", help="JSON string or JSON file with config overrides")
    args = ap.parse_args(argv)
    if args.stage == "selftest":
        return 0 if selftest() else 1
    cfg = LCfg()
    ov = {}
    if args.cfg:
        ov = json.loads(args.cfg) if args.cfg.strip().startswith("{") else P1.jload(args.cfg)
    if args.run:
        ov["run"] = args.run
    apply_overrides(cfg, ov)
    Path(cfg.work_dir).mkdir(parents=True, exist_ok=True)
    {"check": diagnose5, "probe": run_probe, "train": run_train5, "predict": run_predict5, "ensemble": run_ensemble5}[args.stage](cfg)
    return 0


if __name__ == "__main__":
    sys.exit(main())

## Self-test (once, ~10 min). Must end with `SELFTEST PASSED`. Checks the metrics against the official definitions, the decision strategies, losses, identity-at-init of every adapter, and runs every method end to end.

In [ ]:
!python cs781_phase5_lsi.py selftest

## Setup

In [ ]:
import os, json, glob, subprocess
WORK = "/kaggle/working/cs781" if os.path.exists("/kaggle/working") else "./cs781_work"
BUDGET = 60            # training minutes per main run (the schedule follows whichever of epochs / time ends first)
SEL = {}               # optional overrides for every run, e.g. {"train_chunks": 6, "eval_chunks": 24}
P1_DIRS = {a: f"{WORK}/runs/explore_{a}" for a in ["llama", "gpt2", "retnet", "moe"]}
P1_CKPT, missing = {}, []
for a, d in P1_DIRS.items():
    if not os.path.exists(f"{d}/tokenizer.json"): missing.append(a)
    elif os.path.exists(f"{d}/ckpt_best.pt"): P1_CKPT[a] = "ckpt_best.pt"
    elif os.path.exists(f"{d}/ckpt_last.pt"): P1_CKPT[a] = "ckpt_last.pt"
    else: missing.append(a)
for a, d in P1_DIRS.items(): print(f"{a:<7}", f"OK ({P1_CKPT[a]})" if a in P1_CKPT else "MISSING", d)

def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print(line, end="")
    p.wait(); return p.returncode

def run_ls(run, arch, extra):
    if os.path.exists(f"{WORK}/ls_runs/{run}/submission.zip"):
        print("already done:", run, "(delete that folder to re-run)"); return
    if arch not in P1_CKPT:
        print(f"Phase-1 backbone '{arch}' is missing -> train it first, then re-run the setup cell."); return
    cfg = {"p1_dir": P1_DIRS[arch], "p1_ckpt": P1_CKPT[arch], "time_budget_min": BUDGET, **SEL, **extra}
    fn = f"cfg_ls_{run}.json"; json.dump(cfg, open(fn, "w"), indent=1)
    print("=" * 100, "\n", run, "|", arch, "|", extra, "\n", "=" * 100)
    print("exit code", sh(f"python cs781_phase5_lsi.py train --run {run} --cfg {fn}"))
    report(run)

def report(run, n=10):
    p = f"{WORK}/ls_runs/{run}/dev_report.json"
    if not os.path.exists(p): print(f"\n{run}: no dev_report.json (not finished)"); return
    r = json.load(open(p))
    print(f"\n=== {run} | {r.get('p1_arch', 'ensemble')} / {r.get('method', 'ensemble')} | best {r.get('best_epoch')} | decision {r.get('decision', {}).get('name')}")
    print(f"dev macro-F1 {r['macro_f1']:.3f}  micro-F1 {r['micro_f1']:.3f}  subset acc {r['subset_acc']:.3f}  macro-P {r.get('macro_p', float('nan')):.3f}  macro-R {r.get('macro_r', float('nan')):.3f}")
    if "per_label_f1" in r:
        f1, sup = r["per_label_f1"], r["label_support_dev"]
        worst = sorted([i for i in range(len(f1)) if sup[i] > 0], key=lambda i: f1[i])[:n]
        print("   weakest statutes (id: F1, dev cases):", ", ".join(f"{i}: {f1[i]:.2f}, {sup[i]}" for i in worst))

In [ ]:
for a in missing:
    fn = f"cfg_p1_{a}.json"; json.dump({"arch": a, "time_budget_hours": 0.75}, open(fn, "w"))
    print("training Phase-1 backbone:", a); sh(f"python cs781_phase1.py train --run explore_{a} --preset turbo --cfg {fn}")
if not missing: print("nothing to train - all four backbones exist")

## Check the data first (1-3 min)
Read these lines carefully:
* **statutes per case / cases without any statute** - shows how multi-label the task is;
* **statutes with 0 / <5 / <20 train cases** - rare statutes decide Macro-F1 (every statute id counts equally);
* **statutes present in dev but absent from train** - the model cannot predict these, so they put a ceiling on the score;
* **covers X% of cases fully** - how much of each case `eval_chunks` sees;
* **overlap** lines - if dev/test cases also occur in train, the label sets of the train copies are the strongest signal.

In [ ]:
sh("python cs781_phase5_lsi.py check --run ls_check --cfg " + json.dumps(json.dumps({"p1_dir": P1_DIRS["llama"], "p1_ckpt": P1_CKPT.get("llama", "ckpt_best.pt")})))

In [ ]:
import json
pcfg = {"p1_dir": P1_DIRS["llama"], "p1_ckpt": P1_CKPT.get("llama", "ckpt_best.pt"), "work_dir": WORK}
sh("python cs781_phase5_lsi.py probe --run ls_probe --cfg " + json.dumps(json.dumps(pcfg)))

## Required run 1 - full fine-tuning (llama); also the teacher for the distillation run

In [ ]:
run_ls("ls_full_llama", "llama", {"method": "full", "epochs": 6, "lr": 2e-4})

## Required run 2 - LoRMA+ (gpt2), compulsory: `h = (alpha/r BA + I) W0 x`, backbone frozen; the label-wise attention head is trained

In [ ]:
run_ls("ls_lorma_gpt2", "gpt2", {"method": "lorma", "lorma_variant": "plus", "peft_r": 32, "peft_alpha": 64, "epochs": 10, "peft_lr": 1e-3, "head_lr": 2e-3, "bidir": "false"})

## Required run 3 - Houlsby adapters (moe): bottleneck adapters after every attention and FFN sub-layer

In [ ]:
run_ls("ls_adapter_moe", "moe", {"method": "adapter", "adapter_dim": 64, "epochs": 10, "peft_lr": 1e-3, "head_lr": 2e-3, "bidir": "false"})

## Required run 4 - knowledge distillation (retnet student pruned to 6 of 12 blocks <- `ls_full_llama` teacher): `0.5 BCE + 0.5 T^2 KL` on the statute probabilities

In [ ]:
run_ls("ls_kd_retnet", "retnet", {"method": "kd", "teacher_run": "ls_full_llama", "student_layers": 6, "epochs": 8, "lr": 2e-4, "head_lr": 2e-3})

In [ ]:
run_ls("ls_prompt_llama", "llama", {"method": "prompt", "prompt_tokens": 16, "epochs": 10, "peft_lr": 3e-3, "head_lr": 2e-3, "bidir": "false"})
run_ls("ls_bitfit_gpt2",  "gpt2",  {"method": "bitfit", "epochs": 10, "peft_lr": 2e-3, "head_lr": 2e-3, "bidir": "false"})
run_ls("ls_lora_llama",   "llama", {"method": "lora", "peft_r": 32, "peft_alpha": 64, "epochs": 10, "peft_lr": 1e-3, "head_lr": 2e-3})

In [ ]:
# optional ablations (equal budgets, run after the required runs if time allows)
run_ls("ls_abl_chunkmax", "llama", {"method": "full", "pool": "chunk_max", "epochs": 4, "time_budget_min": 25})
run_ls("ls_abl_bce",      "llama", {"method": "full", "loss": "bce", "epochs": 4, "time_budget_min": 25})
run_ls("ls_abl_asl",      "llama", {"method": "full", "loss": "asl", "epochs": 4, "time_budget_min": 25})

In [ ]:
json.dump({"ens_auto": True, "ens_prefix": "ls_", "ens_max": 6}, open("cfg_ls_ens.json", "w"))
print("exit code", sh("python cs781_phase5_lsi.py ensemble --run ls_ensemble --cfg cfg_ls_ens.json"))
report("ls_ensemble", n=0) 

## Results and the files to submit

In [ ]:
import pandas as pd, shutil
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
df = pd.read_csv(f"{WORK}/ls_experiments.csv"); df = df[~df.run.str.startswith(("selftest", "ls_check", "ls_probe"))]
cols = ["run", "p1_arch", "method", "pool", "loss", "params_total_M", "params_trainable_M", "dev_macro_f1", "dev_micro_f1", "dev_subset_acc",
        "decision", "train_s", "peak_gpu_mb", "infer_docs_s"]
display(df[cols].sort_values("dev_macro_f1", ascending=False))
os.makedirs("phase5_submissions", exist_ok=True)
REQUIRED = ["ls_full_llama", "ls_lorma_gpt2", "ls_adapter_moe", "ls_kd_retnet"]
EXTRA = ["ls_prompt_llama", "ls_bitfit_gpt2", "ls_lora_llama", "ls_ensemble"]
for r in REQUIRED + EXTRA:
    src = f"{WORK}/ls_runs/{r}/submission.zip"
    if os.path.exists(src):
        shutil.copy(src, f"phase5_submissions/{r}.zip")
        row = df[df.run == r].tail(1)
        print(("REQUIRED " if r in REQUIRED else "extra    "), r, "| dev macro-F1", float(row.dev_macro_f1.iloc[0]) if len(row) else "?")
    else: print("missing:", r)

# CS781 Phase 6 — Abstractive Legal Summarisation
Notebook version with four separate submission runs, following the workflow and layout of the Phase 3 and Phase 4 notebooks.

Run this after the Phase-1, Phase-2 and Phase-4 setup notebooks in the same Kaggle session (or make sure the required files and Phase-1 checkpoints are present).

This notebook trains four summarisation systems and saves each submission ZIP separately:
1. S1 — Full fine-tuning (Llama): teacher / strongest single-model baseline.
2. S2 — LoRMA+ (GPT-2): parameter-efficient adaptation.
3. S3 — Houlsby adapters (MoE): bottleneck adapters.
4. S4 — Knowledge distillation (RetNet): smaller student distilled from S1.
Each run writes prediction.json and submission.zip. The final cell collects the four ZIP files into phase6_submissions/.

In [ ]:
import base64
from pathlib import Path

_phase6_source = base64.b64decode("...")
Path("cs781_phase6_summ.py").write_bytes(_phase6_source)
print("Wrote cs781_phase6_summ.py:", len(_phase6_source), "bytes")

## Self-test (once)
Run the implementation checks before spending time on training. It tests sentence selection, generation/cache behaviour, adaptation methods, summary-only loss masking, end-to-end training and consensus.

In [ ]:
!python cs781_phase6_summ.py selftest

## Setup
The four runs use the Phase-1 checkpoints produced earlier. The default work directory follows the earlier notebooks: /kaggle/working/cs781.

If a backbone is missing, train it in the Phase-1 notebook first, then re-run this setup cell.

In [ ]:
import os, json, glob, subprocess, shutil
import pandas as pd

WORK = "/kaggle/working/cs781" if os.path.exists("/kaggle/working") else "./cs781_work"
BUDGET = 45  # minutes per main run; adjust to your available compute

P1_DIRS = {a: f"{WORK}/runs/explore_{a}" for a in ["llama", "gpt2", "retnet", "moe"]}
P1_CKPT, missing = {}, []

for a, d in P1_DIRS.items():
    if not os.path.exists(f"{d}/tokenizer.json"):
        missing.append(a)
    elif os.path.exists(f"{d}/ckpt_best.pt"):
        P1_CKPT[a] = "ckpt_best.pt"
    elif os.path.exists(f"{d}/ckpt_last.pt"):
        P1_CKPT[a] = "ckpt_last.pt"
    else:
        missing.append(a)

for a, d in P1_DIRS.items():
    print(f"{a:<7}", f"OK ({P1_CKPT[a]})" if a in P1_CKPT else "MISSING", d)

def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    return p.returncode

def report_sm(run):
    p = f"{WORK}/sm_runs/{run}/dev_report.json"
    if not os.path.exists(p):
        print(f"\\n{run}: no dev_report.json (run not finished)")
        return
    r = json.load(open(p))
    sc = r.get("scores", {})
    print(f"\\n=== {run} | {r.get('p1_arch')} / {r.get('method')} ===")
    print("BERTScore:", sc.get("bertscore", "N/A"),
          "| ROUGE-1:", sc.get("rouge1"),
          "| ROUGE-L:", sc.get("rougeL"),
          "| avg words:", r.get("avg_words"),
          "| trainable/total:", r.get("params_trainable"), "/", r.get("params_total"))

def run_sm(run, arch, extra):
    out_zip = f"{WORK}/sm_runs/{run}/submission.zip"
    if os.path.exists(out_zip):
        print("Already completed:", run, "(delete its run folder to retrain)")
        report_sm(run)
        return
    if arch not in P1_CKPT:
        print(f"Phase-1 backbone '{arch}' is missing. Train it first and re-run setup.")
        return

    cfg = {
        "p1_dir": P1_DIRS[arch],
        "p1_ckpt": P1_CKPT[arch],
        "time_budget_min": BUDGET,
        "predict_after": True,
        **extra
    }
    fn = f"cfg_sm_{run}.json"
    with open(fn, "w") as f:
        json.dump(cfg, f, indent=2)

    print("\\n" + "=" * 100)
    print(run, "|", arch, "|", extra)
    print("=" * 100)
    rc = sh(f"python cs781_phase6_summ.py train --run {run} --cfg {fn}")
    print("exit code:", rc)
    report_sm(run)

## Check the real data (1–3 min)
Checks train/test document lengths, reference-summary lengths, model context size, and the amount of source text retained by each selector.

In [ ]:
if "llama" in P1_CKPT:
    check_cfg = {"p1_dir": P1_DIRS["llama"], "p1_ckpt": P1_CKPT["llama"]}
    with open("cfg_sm_check.json", "w") as f:
        json.dump(check_cfg, f)
    sh("python cs781_phase6_summ.py check --run sm_check --cfg cfg_sm_check.json")
else:
    print("Llama Phase-1 checkpoint is missing; train it before running the data check.")

## S1 — Full fine-tuning (Llama)
The full fine-tuning run is the teacher for the knowledge-distillation submission (S4). Complete this run before S4.

In [ ]:
run_sm("sm_full_llama", "llama", {
    "method": "full",
    "epochs": 6,
    "lr": 2e-4
})

## S2 — LoRMA+ (GPT-2)
Parameter-efficient fine-tuning using LoRMA+ while keeping the pretrained backbone largely frozen.

In [ ]:
run_sm("sm_lorma_gpt2", "gpt2", {
    "method": "lorma",
    "lorma_variant": "plus",
    "peft_r": 32,
    "peft_alpha": 64,
    "peft_lr": 5e-4,
    "epochs": 8
})

## S3 — Houlsby adapters (MoE)
Bottleneck adapters are inserted into the model. Only the adapter-related parameters are trained.

In [ ]:
run_sm("sm_adapter_moe", "moe", {
    "method": "adapter",
    "adapter_dim": 64,
    "peft_lr": 5e-4,
    "epochs": 8
})

## S4 — Knowledge distillation (RetNet student ← Llama teacher)
Trains a smaller RetNet student using the completed S1 Llama run as teacher. This cell requires sm_full_llama/best.pt and its tokenizer.

In [ ]:
run_sm("sm_kd_retnet", "retnet", {
    "method": "kd",
    "teacher_run": "sm_full_llama",
    "student_layers": 6,
    "kd_alpha": 0.5,
    "kd_temp": 2.0,
    "epochs": 6,
    "lr": 2e-4
})

In [ ]:
# Full fine-tuning on additional Phase-1 backbones
run_sm("sm_full_gpt2", "gpt2", {"method": "full", "epochs": 6, "lr": 2e-4})
run_sm("sm_full_moe", "moe", {"method": "full", "epochs": 6, "lr": 2e-4})
run_sm("sm_full_retnet", "retnet", {"method": "full", "epochs": 6, "lr": 2e-4})

In [ ]:
# Additional PEFT / partial fine-tuning experiments on Llama
ADAPT = {"peft_r": 32, "peft_alpha": 64, "peft_lr": 5e-4, "epochs": 6}

run_sm("sm_bitfit_llama", "llama", {"method": "bitfit", **ADAPT})
run_sm("sm_partial_llama", "llama", {"method": "partial", "freeze_layers": 6, "epochs": 6, "lr": 2e-4})
run_sm("sm_lora_llama", "llama", {"method": "lora", **ADAPT})
run_sm("sm_lormapi_gpt2", "gpt2", {"method": "lorma", "lorma_variant": "pi", **ADAPT})

## Results, comparison and the four files to submit
Displays the experiment results and copies the four required submission ZIPs into phase6_submissions/.

A run is copied only if its submission.zip exists. Check the printed status before submitting.

In [ ]:
csv_path = f"{WORK}/sm_experiments.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df = df[~df.run.str.startswith(("selftest", "sm_check"))]
    cols = [c for c in [
        "run", "p1_arch", "method", "select", "params_total_M",
        "params_trainable_M", "dev_bertscore", "dev_rouge1",
        "dev_rougeL", "dev_loss", "max_new", "avg_words",
        "train_s", "peak_gpu_mb", "gen_docs_s"
    ] if c in df.columns]
    display(df[cols].sort_values("dev_bertscore", ascending=False, na_position="last"))
else:
    print("No experiment log found yet:", csv_path)

os.makedirs("phase6_submissions", exist_ok=True)
REQUIRED = [
    "sm_full_llama",
    "sm_lorma_gpt2",
    "sm_adapter_moe",
    "sm_kd_retnet"
]

for run in REQUIRED:
    src = f"{WORK}/sm_runs/{run}/submission.zip"
    if os.path.exists(src):
        dst = f"phase6_submissions/{run}.zip"
        shutil.copy2(src, dst)
        print("READY:", run, "->", dst)
    else:
        print("MISSING:", run, "- run its training cell first")

print("\\nSubmission folder contents:")
for f in sorted(glob.glob("phase6_submissions/*.zip")):
    print(" ", f, f"({os.path.getsize(f):,} bytes)") 

# CS781 Phase 7 - Prior Case Retrieval (PCR)

**Append at the end of your existing notebook** (same session, so `/kaggle/working/cs781/runs/explore_*` exist), or run standalone with your Phase-1 output attached.

### How the retriever works
* **Dual encoder:** the same Phase-1 backbone encodes the query case and every candidate case. A case is split into 512-token chunks (a few random chunks per case while training, up to `eval_chunks` spread over the case at inference); chunk vectors are combined by `pool`: `mean`, learned `attn`, or `maxsim` (late interaction - each query chunk is matched with its best candidate chunk).
* **Training:** contrastive InfoNCE with in-batch negatives **plus hard negatives mined with BM25** from the split's own candidate pool; other relevant cases are masked so they are never treated as negatives. A pairwise `triplet` loss is available for the ablation.
* **Ranking:** neural similarity, **fused with a lexical BM25 score** (z-normalised per query; the weight is tuned on dev by Micro-F1@7 and always keeps at least 20% neural weight, so the Phase-1 model always contributes). The dev log prints neural-only, BM25-only and fused scores - that comparison is your analysis of what the Phase-1 representations add.
* **Output:** exactly 7 unique candidate ids per query, only from that split's candidate pool (a candidate with the query's own id is never returned); validated with the official evaluator's rules before the zip is written.

| Run | Phase-1 backbone | Technique |
|---|---|---|
| `pc_lorma_gpt2` | gpt2 | **LoRMA+** (compulsory) |
| `pc_full_llama` | llama | full fine-tuning (also the teacher of the distillation run) |
| `pc_ia3_moe` | moe | IA3 |
| `pc_kd_retnet` | retnet | **knowledge distillation** (compulsory): pruned student matches the teacher's score distribution over the same candidates |

Extras: Houlsby adapters, LoRA, BitFit, layer freezing, pooling / loss / hard-negative ablations, and an ensemble of runs.

**Honest expectation:** a from-scratch retriever can be weaker than BM25 on very long legal cases; the `check` cell prints the BM25 baseline on dev so you know what to beat. The fused ranking is tuned to be at least as good as either signal on dev.

In [ ]:
!pip -q install scikit-learn scipy

In [ ]:
import base64
open("cs781_phase1.py", "w", encoding="utf-8").write(base64.b64decode("...").decode("utf-8"))
open("cs781_phase2_ner.py", "w", encoding="utf-8").write(base64.b64decode("...").decode("utf-8"))
open("cs781_phase4_cjpe.py", "w", encoding="utf-8").write(base64.b64decode("...").decode("utf-8"))
print("wrote cs781_phase1.py, cs781_phase2_ner.py, cs781_phase4_cjpe.py")

In [ ]:
%%writefile cs781_phase7_pcr.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
CS781 Phase 7 - Prior Case Retrieval (PCR) on top of YOUR Phase-1 language models.

Needs cs781_phase1.py, cs781_phase2_ner.py and cs781_phase4_cjpe.py in the same folder (model classes, adapters, distillation helper).

Model: a DUAL ENCODER (the same Phase-1 backbone encodes queries and candidates)
  case text -> BPE tokens -> chunks of max_len tokens (a few random chunks per case while training, up to eval_chunks spread over the case at inference)
  -> Phase-1 backbone (bidirectional attention; causal for retnet) -> chunk vector = proj([mean of token states ; closing <eos> state])
  -> document representation ("pool"):
       mean    mean of chunk vectors                     -> cosine similarity
       attn    learned attention pooling over chunks     -> cosine similarity
       maxsim  late interaction: every query chunk is matched with its best candidate chunk (and vice versa), averaged
  Training: InfoNCE with in-batch negatives + HARD negatives mined with BM25 from the split's own candidate pool (false negatives masked),
  or a pairwise triplet loss. Distillation: the student matches the teacher's score distribution over the same candidates.
  Ranking: neural score, optionally FUSED with a lexical BM25 score (z-normalised per query, weight tuned on dev). Output: 7 unique ids per query.

Adaptation methods (cfg "method")
  full | partial | bitfit | adapter | lora | lorma (LoRMA+ / LoRMAπ) | ia3 | kd (pruned student <- teacher scores)

Stages:  selftest | check | train | predict | ensemble
  python cs781_phase7_pcr.py check --run pc_check --cfg '{"p1_dir": ".../explore_llama"}'
  python cs781_phase7_pcr.py train --run pc_full_llama --cfg cfg_pc.json
"""
from __future__ import annotations

import argparse
import contextlib
import csv
import gc
import hashlib
import json
import math
import os
import pickle
import random
import re
import shutil
import sys
import time
import zipfile
from dataclasses import dataclass, field, fields, asdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import cs781_phase1 as P1
import cs781_phase2_ner as N
import cs781_phase4_cjpe as J

log = P1.log
K = 7


# ============================================================================================
# 0. Config
# ============================================================================================
@dataclass
class PCfg:
    run: str = "pc_full_v1"
    work_dir: str = P1.DEFAULT_WORK
    p1_dir: str = ""
    p1_ckpt: str = "ckpt_best.pt"
    hf_repo: str = "Exploration-Lab/CS-781-Capstone"
    local_data_dir: str = ""          # folder with train.json, train_candidates.json, dev.json, dev_candidates.json, test.json, test_candidates.json
    train_prefix: str = "Phase_7_PCR/Train"
    dev_prefix: str = "Phase_7_PCR/Dev"
    test_prefix: str = "Phase_7_PCR/Test"
    max_train_queries: int = 0
    # ---- document representation
    max_len: int = 512
    train_chunks: int = 2
    eval_chunks: int = 4
    train_select: str = "random"      # random | uniform | head | tail
    eval_select: str = "uniform"
    pool: str = "mean"                # mean | attn | maxsim
    dim: int = 256
    drop: float = 0.1
    bidir: str = "auto"
    # ---- method
    method: str = "full"              # full | partial | bitfit | adapter | lora | lorma | ia3 | kd
    freeze_layers: int = 0
    adapter_dim: int = 64
    peft_r: int = 16
    peft_alpha: float = 32.0
    peft_dropout: float = 0.05
    peft_lr: float = 5e-4
    peft_targets: str = "all"
    peft_experts: bool = False
    lorma_variant: str = "plus"
    teacher_run: str = ""
    student_layers: int = 0
    kd_alpha: float = 0.5
    kd_temp: float = 2.0
    # ---- objective
    loss: str = "infonce"             # infonce | triplet
    tau: float = 0.05
    margin: float = 0.2
    batch_q: int = 6                  # queries per step
    hard_negs: int = 2                # BM25 hard negatives per query
    hard_pool: int = 30               # hard negatives are sampled from the top-N lexical neighbours that are not relevant
    # ---- optimisation
    epochs: int = 6
    lr: float = 2e-4
    llrd: float = 0.85
    head_lr: float = 1e-3
    weight_decay: float = 0.01
    warmup_frac: float = 0.06
    grad_clip: float = 1.0
    ema_decay: float = 0.995
    time_budget_min: float = 45.0
    eval_every_steps: int = 300
    dev_eval_queries: int = 60
    dev_eval_extra: int = 600         # extra random candidates in the reduced dev pool used during training
    enc_docs: int = 8                 # documents per forward pass at inference
    patience: int = 5
    seed: int = 1337
    # ---- lexical fusion
    fuse: bool = True
    alpha_min: float = 0.2            # minimum weight of the neural (Phase-1) score in the fusion: the submission never degenerates to pure BM25
    bm25_k1: float = 1.2
    bm25_b: float = 0.75
    predict_after: bool = True
    # ---- ensemble stage
    ens_runs: list = field(default_factory=list)
    ens_auto: bool = False
    ens_prefix: str = ""
    ens_max: int = 6


def apply_overrides(cfg: PCfg, ov: dict) -> PCfg:
    names = {f.name for f in fields(PCfg)}
    for k, v in ov.items():
        if k not in names:
            raise SystemExit(f"Unknown config key {k!r}. Valid: {sorted(names)}")
        setattr(cfg, k, v)
    return cfg


def run_dir7(cfg: PCfg) -> Path:
    return Path(cfg.work_dir) / "pc_runs" / cfg.run


PC_FIELDS = ["time", "run", "p1_arch", "method", "pool", "loss", "params_total_M", "params_trainable_M", "dev_f1_neural", "dev_f1_bm25", "dev_f1",
             "dev_precision", "dev_recall", "alpha", "train_s", "peak_gpu_mb", "enc_docs_s", "teacher_run"]


def log_csv(cfg: PCfg, row: dict):
    p = Path(cfg.work_dir) / "pc_experiments.csv"
    if p.exists() and open(p).readline().strip().split(",") != PC_FIELDS:
        p.rename(p.with_suffix(".old.csv"))
    new = not p.exists()
    with open(p, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=PC_FIELDS, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerow({"time": time.strftime("%Y-%m-%d %H:%M:%S"), **row})


# ============================================================================================
# 1. Data
# ============================================================================================
def _first(d, keys):
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return None


def _text_of(v):
    t = v.get("text") if isinstance(v, dict) else v
    if t is None and isinstance(v, dict):
        t = _first(v, ("document", "content", "sentences"))
    if isinstance(t, list):
        t = " ".join(str(x) for x in t)
    return " ".join(str(t or "").split())


def parse_docs(raw):
    """-> (ordered ids, dict id -> text, dict id -> relevant list or None)"""
    items = [(str(_first(r, ("id", "case_id", "doc_id", "query_id"))), r) for r in raw] if isinstance(raw, list) else [(str(k), v) for k, v in raw.items()]
    ids, texts, rel = [], {}, {}
    for did, v in items:
        ids.append(did)
        texts[did] = _text_of(v)
        r = _first(v, ("relevant_candidates", "relevant", "gold")) if isinstance(v, dict) else None
        rel[did] = [str(x) for x in r if str(x).strip()] if isinstance(r, list) else None
    return ids, texts, rel


class PSplit:
    pass


def load_split7(cfg: PCfg, prefix: str, name: str, labelled: bool, tok, tok_path: str, max_q: int = 0):
    pq = P1.resolve_file(cfg, f"{prefix}/{name}.json")
    pc = P1.resolve_file(cfg, f"{prefix}/{name}_candidates.json")
    key = hashlib.md5(open(tok_path, "rb").read() + f"{os.path.getsize(pq)}|{os.path.getsize(pc)}|{max_q}".encode()).hexdigest()[:12]
    cache = Path(cfg.work_dir) / "cache" / f"pcr_{name}_{key}.pkl"
    if cache.exists():
        with open(cache, "rb") as f:
            s = pickle.load(f)
        log(f"[data] {name}: {len(s.qids)} queries, {len(s.pool_ids)} candidates (token cache)")
        return s
    qids, qtext, rel = parse_docs(P1.jload(pq))
    if max_q:
        qids = qids[:max_q]
    pool_ids, pool_text, _ = parse_docs(P1.jload(pc))
    assert tok.get_vocab_size() < 65536
    enc = lambda texts: [np.asarray(e.ids, dtype=np.uint16) for b in range(0, len(texts), 64) for e in tok.encode_batch(texts[b:b + 64])]
    s = PSplit()
    s.name, s.qids, s.pool_ids = name, qids, pool_ids
    s.qtext, s.pool_text = [qtext[q] for q in qids], [pool_text[c] for c in pool_ids]
    s.qtok, s.pool_tok = enc(s.qtext), enc(s.pool_text)
    s.rel = {q: (rel[q] if (labelled and rel[q] is not None) else []) for q in qids}
    cache.parent.mkdir(parents=True, exist_ok=True)
    with open(cache, "wb") as f:
        pickle.dump(s, f, protocol=4)
    log(f"[data] {name}: {len(qids)} queries, {len(pool_ids)} candidates tokenised")
    return s


# ============================================================================================
# 2. Lexical retrieval (BM25) + metrics + fusion
# ============================================================================================
class BM25:
    def __init__(self, texts, k1=1.2, b=0.75):
        from sklearn.feature_extraction.text import CountVectorizer
        import scipy.sparse as sp
        self.vec = CountVectorizer(token_pattern=r"(?u)\b[a-z0-9]{2,}\b", lowercase=True, min_df=2, max_df=0.6, max_features=300000, dtype=np.float32)
        tf = self.vec.fit_transform(texts).tocoo()
        dl = np.asarray(tf.tocsr().sum(1)).ravel()
        avg = max(1.0, float(dl.mean()))
        df = np.asarray((tf.tocsr() > 0).sum(0)).ravel()
        n = tf.shape[0]
        self.idf = np.log((n - df + 0.5) / (df + 0.5) + 1.0).astype(np.float32)
        w = tf.data * (k1 + 1) / (tf.data + k1 * (1 - b + b * dl[tf.row] / avg)) * self.idf[tf.col]
        self.W = sp.csr_matrix((w.astype(np.float32), (tf.row, tf.col)), shape=tf.shape)

    def score(self, qtexts):
        q = self.vec.transform(qtexts).tocsr().astype(np.float32)
        q.data = np.log1p(q.data)                                        # damp repeated query terms (queries are whole cases)
        return np.asarray((q @ self.W.T).todense(), dtype=np.float32)


def topk_ids(S, qids, pool_ids, k=K):
    """Top-k candidate ids per query from a score matrix; a candidate with the query's own id is never returned."""
    S = np.array(S, dtype=np.float32, copy=True)
    pos = {c: i for i, c in enumerate(pool_ids)}
    for r, q in enumerate(qids):
        if q in pos:
            S[r, pos[q]] = -np.inf
    top = np.argpartition(-S, min(k, S.shape[1] - 1), axis=1)[:, :k]
    out = {}
    for r, q in enumerate(qids):
        order = top[r][np.argsort(-S[r, top[r]])]
        out[q] = [pool_ids[i] for i in order]
    return out


def f1_at_k(gold, pred, k=K):
    """Micro-F1/precision/recall @k exactly as in the official evaluator (gold set is neither truncated nor padded)."""
    tp = fp = fn = 0
    for q, labels in gold.items():
        g, p = set(labels), set(pred[q][:k])
        tp += len(g & p)
        fp += len(p - g)
        fn += len(g - p)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
    return {"f1": f1, "precision": prec, "recall": rec, "tp": tp, "fp": fp, "fn": fn}


def recall_at(gold, S, qids, pool_ids, k):
    pred = topk_ids(S, qids, pool_ids, k)
    tp = sum(len(set(gold[q]) & set(pred[q])) for q in qids)
    tot = sum(len(set(gold[q])) for q in qids)
    return tp / tot if tot else 0.0


def zrow(S):
    S = np.asarray(S, dtype=np.float32)
    return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-6)


def tune_alpha(Sn, Sb, gold, qids, pool_ids, a_min=0.0):
    """Weight of the neural score in  alpha * z(neural) + (1 - alpha) * z(BM25), chosen on dev by Micro-F1@7 among alpha >= a_min
    (ties are broken towards a balanced mixture)."""
    zn, zb, table = zrow(Sn), zrow(Sb), []
    grid = sorted({round(float(a_min), 2)} | {round(float(a), 2) for a in np.arange(0.1, 1.0001, 0.1) if a >= a_min - 1e-9})
    for a in grid:
        table.append((a, f1_at_k(gold, topk_ids(a * zn + (1 - a) * zb, qids, pool_ids))["f1"]))
    best = max(table, key=lambda t: (round(t[1], 6), -abs(t[0] - 0.5)))
    return best[0], table


def validate_pred(pred, qids, pool_ids):
    """The official evaluator's validation rules: every query, exactly 7 unique ids from the pool."""
    errs, pool = [], set(pool_ids)
    if set(pred) != set(qids):
        errs.append("query ids differ")
    for q, v in pred.items():
        if not isinstance(v, list) or len(v) != K or len(set(v)) != K or any((not isinstance(c, str)) or c not in pool for c in v):
            errs.append(f"bad prediction for {q}")
    return errs


# ============================================================================================
# 3. Methods: reuse Phase 4 + IA3
# ============================================================================================
class IA3Out(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        return self.base(x) * self.l


class IA3In(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.l)


def apply_method7(net, cfg: PCfg):
    if cfg.method != "ia3":
        J.apply_method4(net, cfg)
        return
    for p in net.lm.parameters():
        p.requires_grad = False
    n = 0
    for blk in net.lm.blocks:
        for part in (blk.mix, blk.ffn):
            for name, child in list(part.named_children()):
                if isinstance(child, nn.Linear) and name in ("k", "v"):
                    setattr(part, name, IA3Out(child))
                    n += 1
                elif isinstance(child, nn.Linear) and name == "w2":
                    setattr(part, name, IA3In(child))
                    n += 1
    log(f"[ia3] learned scaling vectors on {n} projections")


# ============================================================================================
# 4. Dual encoder
# ============================================================================================
def pick_chunks(m, k, mode, rng):
    if m <= k:
        return list(range(m))
    if mode == "random":
        return sorted(rng.sample(range(m), k))
    if mode == "head":
        return list(range(k))
    if mode == "tail":
        return list(range(m - k, m))
    return sorted({int(x) for x in np.linspace(0, m - 1, k).round()})


def collate_docs7(toks, idxs, cfg: PCfg, dev, eos, n_chunks, mode, rng=None):
    chunks, owner, slot, rel, counts = [], [], [], [], []
    for b, di in enumerate(idxs):
        t = toks[di]
        bounds = J.chunk_bounds(len(t), cfg.max_len - 1, 0)
        sel = pick_chunks(len(bounds), n_chunks, mode, rng or random)
        for k_, j in enumerate(sel):
            s, e = bounds[j]
            chunks.append(t[s:e])
            owner.append(b)
            slot.append(k_)
            rel.append(min(7, int(8 * j / len(bounds))))
        counts.append(len(sel))
    T = max(len(c) for c in chunks) + 1
    ids = np.zeros((len(chunks), T), dtype=np.int64)
    km = np.zeros((len(chunks), T), dtype=bool)
    for i, c in enumerate(chunks):
        n = len(c)
        ids[i, :n] = c
        ids[i, n] = eos
        km[i, :n + 1] = True
    t_ = lambda x: torch.from_numpy(x).to(dev)
    return t_(ids), t_(km), t_(np.array(rel)), t_(np.array(owner)), t_(np.array(slot)), t_(np.array(counts))


class PCRNet(nn.Module):
    def __init__(self, lm, bidir, pool, dim, drop):
        super().__init__()
        self.lm, self.bidir, self.pool = lm, bidir, pool
        d = lm.mc["n_embd"]
        self.proj = nn.Linear(2 * d, dim)
        self.pos = nn.Embedding(8, dim)
        nn.init.normal_(self.pos.weight, 0.0, 0.02)
        self.att = nn.Linear(dim, 1)
        self.drop = nn.Dropout(drop)

    def embed(self, ids, km, rel, owner, slot, counts):
        h = J.backbone_hidden(self.lm, ids, km, self.bidir)
        kf = km.unsqueeze(-1).to(h.dtype)
        mean = (h * kf).sum(1) / kf.sum(1).clamp(min=1.0)
        lastv = h[torch.arange(h.size(0), device=h.device), km.sum(1) - 1]
        c = self.drop(self.proj(torch.cat([mean, lastv], dim=-1)) + self.pos(rel))                  # (N, D)
        B, Cm = counts.numel(), int(counts.max())
        pad = c.new_zeros(B, Cm, c.size(-1)).index_put((owner, slot), c).float()
        mask = torch.arange(Cm, device=c.device).unsqueeze(0) < counts.unsqueeze(1)
        if self.pool == "maxsim":
            return {"pad": F.normalize(pad, dim=-1), "mask": mask}
        if self.pool == "mean":
            v = (pad * mask.unsqueeze(-1).float()).sum(1) / mask.sum(1, keepdim=True).float()
        else:
            sc = self.att(pad).squeeze(-1).float()
            sc = sc.masked_fill(~mask, torch.finfo(torch.float32).min)
            weights = torch.softmax(sc, dim=-1)
            v = (weights.unsqueeze(-1) * pad.float()).sum(1)
        return {"vec": F.normalize(v, dim=-1)}


def sim_matrix(Q, D, pool):
    """Query representations x candidate representations -> (nq, nd) similarity (cosine / late-interaction)."""
    if pool != "maxsim":
        return Q["vec"] @ D["vec"].T
    s = torch.einsum("bid,mjd->bmij", Q["pad"], D["pad"])                                           # (B, M, Cq, Cd)
    qm, dm = Q["mask"].float(), D["mask"].float()
    s_q = s.masked_fill(~D["mask"][None, :, None, :], -2.0).max(3).values                           # best candidate chunk for every query chunk
    s_q = (s_q * qm[:, None, :]).sum(2) / qm.sum(1)[:, None]
    s_d = s.masked_fill(~Q["mask"][:, None, :, None], -2.0).max(2).values                           # best query chunk for every candidate chunk
    s_d = (s_d * dm[None, :, :]).sum(2) / dm.sum(1)[None, :]
    return 0.5 * (s_q + s_d)


def build_net7(cfg: PCfg, lm):
    arch = lm.mc["arch"]
    bidir = (arch != "retnet") if cfg.bidir == "auto" else (cfg.bidir in ("true", "yes", "1", True))
    if arch == "retnet":
        bidir = False
    log(f"[model] arch={arch} bidirectional_attention={bidir} pool={cfg.pool} loss={cfg.loss} chunks train/eval {cfg.train_chunks}/{cfg.eval_chunks}")
    return PCRNet(lm, bidir, cfg.pool, cfg.dim, cfg.drop)


def load_run7(cfg: PCfg, run_name: str, dev):
    from tokenizers import Tokenizer
    rd = Path(cfg.work_dir) / "pc_runs" / run_name
    ck = torch.load(rd / "best.pt", map_location="cpu", weights_only=False)
    saved = PCfg()
    apply_overrides(saved, {k: v for k, v in ck["cfg"].items() if k in {f.name for f in fields(PCfg)}})
    saved.work_dir = cfg.work_dir
    if cfg.local_data_dir:
        saved.local_data_dir = cfg.local_data_dir
    net = build_net7(saved, P1.LM(ck["mc"]))
    apply_method7(net, saved)
    net.load_state_dict(ck["net"])
    net.to(dev).eval()
    return {"rd": rd, "ck": ck, "saved": saved, "net": net, "tok": Tokenizer.from_file(str(rd / "tokenizer.json"))}


@torch.no_grad()
def encode_set(net, toks, cfg: PCfg, dev, amp, eos, idxs=None):
    """Representations of many documents (inference): vec (n, D) or padded chunk vectors (n, eval_chunks, D) + mask, on the CPU in fp16."""
    net.eval()
    idxs = list(range(len(toks))) if idxs is None else list(idxs)
    order = sorted(range(len(idxs)), key=lambda i: len(toks[idxs[i]]))
    key = "pad" if cfg.pool == "maxsim" else "vec"
    D = net.proj.out_features
    out = torch.zeros((len(idxs), cfg.eval_chunks, D) if key == "pad" else (len(idxs), D), dtype=torch.float16)
    mask = torch.zeros((len(idxs), cfg.eval_chunks), dtype=torch.bool) if key == "pad" else None
    for b0 in range(0, len(order), cfg.enc_docs):
        sel = order[b0:b0 + cfg.enc_docs]
        batch = collate_docs7(toks, [idxs[i] for i in sel], cfg, dev, eos, cfg.eval_chunks, cfg.eval_select)
        with amp():
            r = net.embed(*batch)
        if key == "pad":
            Cm = r["pad"].shape[1]
            out[sel, :Cm] = r["pad"].half().cpu()
            mask[sel, :Cm] = r["mask"].cpu()
        else:
            out[sel] = r["vec"].half().cpu()
    return {key: out, **({"mask": mask} if mask is not None else {})}


@torch.no_grad()
def score_all(Qr, Dr, pool, dev, bq=4):
    n = (Qr["pad"] if pool == "maxsim" else Qr["vec"]).shape[0]
    D = {k: v.to(dev).float() if v.dtype != torch.bool else v.to(dev) for k, v in Dr.items()}
    rows = []
    for i in range(0, n, bq if pool == "maxsim" else 512):
        Q = {k: (v[i:i + (bq if pool == "maxsim" else 512)].to(dev).float() if v.dtype != torch.bool else v[i:i + (bq if pool == "maxsim" else 512)].to(dev))
             for k, v in Qr.items()}
        rows.append(sim_matrix(Q, D, pool).float().cpu().numpy())
    return np.concatenate(rows, 0)


def neural_scores(net, split, cfg, dev, amp, eos, pool_idx=None, q_idx=None):
    Qr = encode_set(net, split.qtok, cfg, dev, amp, eos, q_idx)
    Dr = encode_set(net, split.pool_tok, cfg, dev, amp, eos, pool_idx)
    return score_all(Qr, Dr, cfg.pool, dev)


# ============================================================================================
# 5. Training
# ============================================================================================
def run_train7(cfg: PCfg):
    rd = run_dir7(cfg)
    rd.mkdir(parents=True, exist_ok=True)
    jd = asdict(cfg)
    P1.jdump(jd, rd / "config.json")
    random.seed(cfg.seed), np.random.seed(cfg.seed), torch.manual_seed(cfg.seed)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    amp_dtype, amp = P1.amp_setup(dev)
    tok, lm, mc = N.load_p1(cfg)
    tok_path = str(Path(cfg.p1_dir) / "tokenizer.json")
    shutil.copy(tok_path, rd / "tokenizer.json")
    if cfg.method == "kd":
        if not cfg.teacher_run:
            raise SystemExit('method "kd" needs "teacher_run": the name of a finished retrieval run')
        lm, mc = N.make_student(lm, mc, cfg)
    eos = tok.token_to_id(P1.EOS)
    assert cfg.max_len <= mc["ctx"], "max_len exceeds the Phase-1 context length"
    tr = load_split7(cfg, cfg.train_prefix, "train", True, tok, tok_path, cfg.max_train_queries)
    dv = load_split7(cfg, cfg.dev_prefix, "dev", True, tok, tok_path)

    # ---- training queries, positives inside the pool, BM25 hard negatives
    pidx = {c: i for i, c in enumerate(tr.pool_ids)}
    pos = {q: [pidx[c] for c in tr.rel[q] if c in pidx] for q in tr.qids}
    train_q = [i for i, q in enumerate(tr.qids) if pos[q]]
    log(f"[data] {len(train_q)}/{len(tr.qids)} training queries have a relevant candidate in the pool | mean relevant {np.mean([len(pos[tr.qids[i]]) for i in train_q]):.2f}")
    t0 = time.time()
    bm_tr = BM25(tr.pool_text, cfg.bm25_k1, cfg.bm25_b)
    Sb_tr = bm_tr.score([tr.qtext[i] for i in train_q])
    hard = {}
    for r, i in enumerate(train_q):
        q = tr.qids[i]
        banned = set(pos[q]) | ({pidx[q]} if q in pidx else set())
        order = np.argsort(-Sb_tr[r])[:cfg.hard_pool + len(banned)]
        hard[i] = [int(j) for j in order if int(j) not in banned][:cfg.hard_pool]
    del Sb_tr
    bm_dv = BM25(dv.pool_text, cfg.bm25_k1, cfg.bm25_b)
    log(f"[data] BM25 indexes + hard negatives ready ({time.time() - t0:.0f}s)")

    teacher = None
    teacher_params = 0
    if cfg.method == "kd":
        trd = Path(cfg.work_dir) / "pc_runs" / cfg.teacher_run
        if not (trd / "best.pt").exists():
            raise SystemExit(f"teacher run not found: {trd}/best.pt (train it first)")
        if (trd / "tokenizer.json").read_bytes() != (rd / "tokenizer.json").read_bytes():
            raise SystemExit("teacher and student use different tokenizers")
        tr_ = load_run7(cfg, cfg.teacher_run, dev)
        teacher = tr_["net"]
        for p in teacher.parameters():
            p.requires_grad = False
        teacher_params = sum(p.numel() for p in teacher.parameters())
        log(f"[kd] teacher {cfg.teacher_run}: arch={tr_['ck']['mc']['arch']} method={tr_['saved'].method} pool={tr_['saved'].pool} params={teacher_params / 1e6:.1f}M")

    net = build_net7(cfg, lm)
    apply_method7(net, cfg)
    net.to(dev)
    n_total = sum(p.numel() for p in net.parameters())
    n_train = sum(p.numel() for p in net.parameters() if p.requires_grad)
    log(f"[model] total params {n_total / 1e6:.1f}M | trainable {n_train / 1e6:.3f}M | method={cfg.method}")

    peft = cfg.method in ("lora", "lorma", "adapter", "bitfit", "ia3")
    body_lr, body_wd = (cfg.peft_lr, 0.0) if peft else (cfg.lr, cfg.weight_decay)
    nb = mc["n_layer"]

    def lr_scale(name):
        if peft or cfg.llrd >= 1.0 or not name.startswith("lm."):
            return 1.0
        m_ = re.match(r"lm\.blocks\.(\d+)\.", name)
        if m_:
            return cfg.llrd ** (nb - 1 - int(m_.group(1)))
        return cfg.llrd ** nb if name.startswith(("lm.wte", "lm.wpe")) else 1.0

    buckets = {}
    for n_, p_ in net.named_parameters():
        if not p_.requires_grad:
            continue
        base, wd = (body_lr * lr_scale(n_), body_wd if p_.dim() >= 2 else 0.0) if n_.startswith("lm.") else (cfg.head_lr, cfg.weight_decay if p_.dim() >= 2 else 0.0)
        buckets.setdefault((round(base, 12), wd), []).append(p_)
    opt = torch.optim.AdamW([{"params": ps, "lr": b_, "base": b_, "weight_decay": wd} for (b_, wd), ps in buckets.items()], betas=(0.9, 0.98), eps=1e-8)
    scaler = P1.make_scaler(dev == "cuda" and amp_dtype == torch.float16)
    ema = J.EMA(net, cfg.ema_decay) if cfg.ema_decay > 0 else None

    # ---- reduced dev pool for model selection during training
    rng = np.random.default_rng(cfg.seed)
    dq = [i for i, q in enumerate(dv.qids) if dv.rel[q]][:cfg.dev_eval_queries]
    dpidx = {c: i for i, c in enumerate(dv.pool_ids)}
    need = {dpidx[c] for i in dq for c in dv.rel[dv.qids[i]] if c in dpidx}
    extra = [int(i) for i in rng.permutation(len(dv.pool_ids))[:cfg.dev_eval_extra + len(need)] if int(i) not in need][:cfg.dev_eval_extra]
    sub_pool = sorted(need | set(extra))
    sub_ids = [dv.pool_ids[i] for i in sub_pool]
    sub_gold = {dv.qids[i]: dv.rel[dv.qids[i]] for i in dq}

    steps_per_epoch = max(1, math.ceil(len(train_q) / cfg.batch_q))
    total_steps = cfg.epochs * steps_per_epoch
    budget = cfg.time_budget_min * 60.0
    log(f"[train] {len(train_q)} queries -> {steps_per_epoch} steps/epoch x {cfg.epochs} epochs ({cfg.batch_q} queries + {cfg.hard_negs} hard negatives each per step); budget {cfg.time_budget_min} min")
    best, step, bad, t_start, stop = {"f1": -1.0}, 0, 0, time.time(), False
    if dev == "cuda":
        torch.cuda.reset_peak_memory_stats()
    logf = open(rd / "log.jsonl", "a")
    prng = random.Random(cfg.seed)

    def do_eval(tag, loss_val):
        nonlocal best, bad
        with (ema.swapped() if ema is not None else contextlib.nullcontext()):
            S = neural_scores(net, dv, cfg, dev, amp, eos, sub_pool, dq)
            m = f1_at_k(sub_gold, topk_ids(S, [dv.qids[i] for i in dq], sub_ids))
            log(f"[{tag}] loss {loss_val:.4f} | reduced dev pool ({len(dq)} queries, {len(sub_ids)} candidates): Micro-F1@7 {m['f1']:.4f} "
                f"(P {m['precision']:.4f} R {m['recall']:.4f}) | {time.time() - t_start:.0f}s")
            logf.write(json.dumps({"tag": tag, "step": step, "loss": loss_val, "dev_f1": m["f1"]}) + "\n")
            logf.flush()
            if m["f1"] > best["f1"]:
                best, bad = {"f1": m["f1"], "epoch": tag}, 0
                torch.save({"net": net.state_dict(), "mc": mc, "cfg": jd, "alpha": 1.0}, rd / "best.pt")
            else:
                bad += 1
        net.train()

    for ep in range(cfg.epochs):
        net.train()
        order = list(train_q)
        prng.shuffle(order)
        run_loss, n_b, early = 0.0, 0, False
        for g0 in range(0, len(order), cfg.batch_q):
            prog = max(step / total_steps, (time.time() - t_start) / budget if budget > 0 else 0.0)
            if prog >= 1.0:
                stop = True
                break
            for g in opt.param_groups:
                g["lr"] = J.lr_prog(g["base"], prog, cfg.warmup_frac)
            qs = order[g0:g0 + cfg.batch_q]
            docs, col, target = [], {}, []
            for i in qs:                                              # one random positive per query ...
                p = prng.choice(pos[tr.qids[i]])
                if p not in col:
                    col[p] = len(docs)
                    docs.append(p)
                target.append(col[p])
            for i in qs:                                              # ... plus BM25 hard negatives
                for h in prng.sample(hard[i], min(cfg.hard_negs, len(hard[i]))):
                    if h not in col:
                        col[h] = len(docs)
                        docs.append(h)
            relmask = torch.zeros((len(qs), len(docs)), dtype=torch.bool)
            for r, i in enumerate(qs):
                for p in pos[tr.qids[i]]:
                    if p in col and col[p] != target[r]:
                        relmask[r, col[p]] = True                     # other relevant documents are not negatives
            relmask = relmask.to(dev)
            tgt = torch.tensor(target, device=dev)
            bq = collate_docs7(tr.qtok, qs, cfg, dev, eos, cfg.train_chunks, cfg.train_select, prng)
            bd = collate_docs7(tr.pool_tok, docs, cfg, dev, eos, cfg.train_chunks, cfg.train_select, prng)       # one chunk sample for student AND teacher
            with amp():
                Q, D = net.embed(*bq), net.embed(*bd)
                if teacher is not None:
                    with torch.no_grad():
                        Qt, Dt = teacher.embed(*bq), teacher.embed(*bd)
            S = sim_matrix(Q, D, cfg.pool).float()
            if cfg.loss == "triplet":
                sp = S.gather(1, tgt[:, None])
                neg = S.masked_fill(relmask, -2.0)
                neg = neg.scatter(1, tgt[:, None], -2.0)
                loss = F.relu(cfg.margin - sp + neg).masked_fill(neg <= -2.0, 0.0).sum() / max(1, int((neg > -2.0).sum()))
            else:
                logits = (S / cfg.tau).masked_fill(relmask, -1e4)
                loss = F.cross_entropy(logits, tgt)
            if teacher is not None:
                with torch.no_grad():
                    St = sim_matrix(Qt, Dt, teacher.pool).float() / cfg.tau
                s_log = F.log_softmax((S / cfg.tau).masked_fill(relmask, -1e4) / cfg.kd_temp, dim=-1)
                t_p = F.softmax(St.masked_fill(relmask, -1e4) / cfg.kd_temp, dim=-1)
                kd = (t_p * (torch.log(t_p.clamp_min(1e-9)) - s_log)).sum(-1).mean() * cfg.kd_temp ** 2
                loss = cfg.kd_alpha * loss + (1.0 - cfg.kd_alpha) * kd
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in net.parameters() if p.requires_grad], cfg.grad_clip)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            if ema is not None:
                ema.update(step)
            step += 1
            run_loss += float(loss.detach())
            n_b += 1
            if step % 20 == 0:
                log(f"  ep {ep + 1} step {step}/{total_steps} loss {run_loss / n_b:.4f} ({time.time() - t_start:.0f}s)")
            if cfg.eval_every_steps and step % cfg.eval_every_steps == 0:
                do_eval(f"step {step}", run_loss / max(1, n_b))
                if bad >= cfg.patience:
                    early = stop = True
                    break
        if not early:
            do_eval(f"epoch {ep + 1}/{cfg.epochs}", run_loss / max(1, n_b))
        if early or bad >= cfg.patience:
            log(f"[train] early stopping (best: {best['epoch']})")
            break
        if stop:
            log("[train] time budget reached")
            break
    logf.close()
    train_s = time.time() - t_start
    peak = torch.cuda.max_memory_allocated() / 2 ** 20 if dev == "cuda" else 0.0

    # ---- final evaluation on the FULL dev pool: neural, BM25, fused
    ck = torch.load(rd / "best.pt", map_location="cpu", weights_only=False)
    net.load_state_dict(ck["net"])
    if dev == "cuda":
        torch.cuda.synchronize()
    t_enc = time.time()
    Sn = neural_scores(net, dv, cfg, dev, amp, eos)
    if dev == "cuda":
        torch.cuda.synchronize()
    enc_docs_s = (len(dv.pool_ids) + len(dv.qids)) / max(1e-9, time.time() - t_enc)
    Sb = bm_dv.score(dv.qtext)
    gold = {q: dv.rel[q] for q in dv.qids}
    m_n = f1_at_k(gold, topk_ids(Sn, dv.qids, dv.pool_ids))
    m_b = f1_at_k(gold, topk_ids(Sb, dv.qids, dv.pool_ids))
    alpha, table = tune_alpha(Sn, Sb, gold, dv.qids, dv.pool_ids, cfg.alpha_min) if cfg.fuse else (1.0, [])
    m_f = f1_at_k(gold, topk_ids(alpha * zrow(Sn) + (1 - alpha) * zrow(Sb), dv.qids, dv.pool_ids)) if cfg.fuse else m_n
    ck["alpha"] = alpha
    torch.save(ck, rd / "best.pt")
    np.save(rd / "dev_scores.npy", Sn.astype(np.float16))
    np.save(rd / "dev_bm25.npy", Sb.astype(np.float16))
    P1.jdump({"qids": dv.qids, "pool": dv.pool_ids}, rd / "dev_ids.json")
    log(f"[dev] FULL pool ({len(dv.qids)} queries x {len(dv.pool_ids)} candidates): Micro-F1@7 neural {m_n['f1']:.4f} | BM25 {m_b['f1']:.4f} | "
        f"fused (alpha={alpha}) {m_f['f1']:.4f}  [P {m_f['precision']:.4f} R {m_f['recall']:.4f}]")
    if table:
        log("[dev] fusion weight alpha -> Micro-F1@7: " + ", ".join(f"{a}: {f:.3f}" for a, f in table))
    log(f"[dev] recall@7 {recall_at(gold, Sn, dv.qids, dv.pool_ids, 7):.4f} | recall@20 {recall_at(gold, Sn, dv.qids, dv.pool_ids, 20):.4f} | "
        f"recall@50 {recall_at(gold, Sn, dv.qids, dv.pool_ids, 50):.4f} (neural only: how many relevant cases are in the top-N at all)")
    variant = cfg.lorma_variant if cfg.method == "lorma" else ""
    P1.jdump({"best": best, "dev_neural": m_n, "dev_bm25": m_b, "dev_fused": m_f, "alpha": alpha, "alpha_table": table,
              "params_total": n_total, "params_trainable": n_train, "train_seconds": train_s, "peak_gpu_mb": peak, "enc_docs_per_s": enc_docs_s,
              "p1_arch": mc["arch"], "method": cfg.method, "teacher_run": cfg.teacher_run, "teacher_params": teacher_params}, rd / "dev_report.json")
    log_csv(cfg, {"run": cfg.run, "p1_arch": mc["arch"], "method": cfg.method + (f"-{variant}" if variant else ""), "pool": cfg.pool, "loss": cfg.loss,
                  "params_total_M": round(n_total / 1e6, 2), "params_trainable_M": round(n_train / 1e6, 3), "dev_f1_neural": round(m_n["f1"], 4),
                  "dev_f1_bm25": round(m_b["f1"], 4), "dev_f1": round(m_f["f1"], 4), "dev_precision": round(m_f["precision"], 4),
                  "dev_recall": round(m_f["recall"], 4), "alpha": alpha, "train_s": round(train_s), "peak_gpu_mb": round(peak),
                  "enc_docs_s": round(enc_docs_s, 1), "teacher_run": cfg.teacher_run})
    if cfg.predict_after:
        run_predict7(cfg)


def write_submission7(cfg: PCfg, rd: Path, pred: dict, qids, pool_ids):
    errs = validate_pred(pred, qids, pool_ids)
    assert not errs, errs[:3]
    rd.mkdir(parents=True, exist_ok=True)
    pj = rd / "prediction.json"
    P1.jdump({q: pred[q] for q in qids}, pj)
    zp = rd / "submission.zip"
    with zipfile.ZipFile(zp, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    hist = Path(cfg.work_dir) / "pc_submissions"
    hist.mkdir(parents=True, exist_ok=True)
    shutil.copy(zp, hist / f"{cfg.run}_{time.strftime('%m%d_%H%M%S')}.zip")
    log(f"[predict] wrote {pj} ({len(qids)} queries x {K} candidates) and {zp}")
    return str(zp)


def run_predict7(cfg: PCfg):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    _, amp = P1.amp_setup(dev)
    r = load_run7(cfg, cfg.run, dev)
    saved, ck = r["saved"], r["ck"]
    tp = str(r["rd"] / "tokenizer.json")
    eos = r["tok"].token_to_id(P1.EOS)
    te = load_split7(saved, saved.test_prefix, "test", False, r["tok"], tp)
    Sn = neural_scores(r["net"], te, saved, dev, amp, eos)
    Sb = BM25(te.pool_text, saved.bm25_k1, saved.bm25_b).score(te.qtext)
    np.save(r["rd"] / "test_scores.npy", Sn.astype(np.float16))
    np.save(r["rd"] / "test_bm25.npy", Sb.astype(np.float16))
    P1.jdump({"qids": te.qids, "pool": te.pool_ids}, r["rd"] / "test_ids.json")
    alpha = float(ck.get("alpha", 1.0))
    S = alpha * zrow(Sn) + (1 - alpha) * zrow(Sb) if saved.fuse else Sn
    log(f"[predict] {len(te.qids)} test queries x {len(te.pool_ids)} candidates, fusion alpha {alpha if saved.fuse else 1.0}")
    return write_submission7(cfg, r["rd"], topk_ids(S, te.qids, te.pool_ids), te.qids, te.pool_ids)


# ============================================================================================
# 6. Ensemble of runs (average z-normalised neural scores, then fuse with BM25)
# ============================================================================================
def run_ensemble7(cfg: PCfg):
    root = Path(cfg.work_dir) / "pc_runs"
    if cfg.ens_runs:
        names = list(cfg.ens_runs)
    elif cfg.ens_auto:
        names = [f.parent.name for f in sorted(root.glob("*/test_scores.npy")) if (f.parent / "dev_scores.npy").exists()
                 and (f.parent.name.startswith(cfg.ens_prefix) if cfg.ens_prefix else not f.parent.name.startswith(("selftest", "check")))]
    else:
        names = []
    if len(names) < 2:
        raise SystemExit('need >= 2 finished runs: set "ens_runs": [...] or "ens_auto": true')
    dids, tids = P1.jload(root / names[0] / "dev_ids.json"), P1.jload(root / names[0] / "test_ids.json")
    dev_bm, te_bm = np.load(root / names[0] / "dev_bm25.npy").astype(np.float32), np.load(root / names[0] / "test_bm25.npy").astype(np.float32)
    saved = PCfg()
    apply_overrides(saved, {k: v for k, v in P1.jload(root / names[0] / "config.json").items() if k in {f.name for f in fields(PCfg)}})
    saved.work_dir = cfg.work_dir
    if cfg.local_data_dir:
        saved.local_data_dir = cfg.local_data_dir
    pq = P1.resolve_file(saved, f"{saved.dev_prefix}/dev.json")
    _, _, rel = parse_docs(P1.jload(pq))
    gold = {q: [c for c in (rel[q] or [])] for q in dids["qids"]}
    dv_s, te_s = {}, {}
    for n in names:
        if P1.jload(root / n / "dev_ids.json") != dids or P1.jload(root / n / "test_ids.json") != tids:
            log(f"[ensemble] skipping {n}: different queries / candidate pool order")
            continue
        dv_s[n], te_s[n] = zrow(np.load(root / n / "dev_scores.npy").astype(np.float32)), zrow(np.load(root / n / "test_scores.npy").astype(np.float32))
    f1 = lambda S: f1_at_k(gold, topk_ids(S, dids["qids"], dids["pool"]))["f1"]
    single = {k: f1(v) for k, v in dv_s.items()}
    for k, v in sorted(single.items(), key=lambda kv: -kv[1]):
        log(f"[ensemble] single run {k:<28} neural-only dev Micro-F1@7 {v:.4f}")
    mean = lambda keys, d: sum(d[k] for k in keys) / len(keys)
    if cfg.ens_runs or not cfg.ens_auto:
        chosen = list(dv_s)
    else:
        order = sorted(single, key=lambda k: -single[k])
        chosen, cur = [order[0]], single[order[0]]
        for k in order[1:]:
            if len(chosen) >= cfg.ens_max:
                break
            f = f1(mean(chosen + [k], dv_s))
            log(f"[ensemble] try adding {k:<28} -> {f:.4f} (current {cur:.4f})")
            if f > cur + 0.001:
                chosen.append(k)
                cur = f
    log(f"[ensemble] members: {chosen}")
    Dn, Tn = mean(chosen, dv_s), mean(chosen, te_s)
    alpha, table = tune_alpha(Dn, dev_bm, gold, dids["qids"], dids["pool"], cfg.alpha_min) if cfg.fuse else (1.0, [])
    S_dev = alpha * zrow(Dn) + (1 - alpha) * zrow(dev_bm)
    m = f1_at_k(gold, topk_ids(S_dev, dids["qids"], dids["pool"]))
    log(f"[ensemble] dev Micro-F1@7 {m['f1']:.4f} (P {m['precision']:.4f} R {m['recall']:.4f}) with fusion alpha {alpha}")
    rd = run_dir7(cfg)
    rd.mkdir(parents=True, exist_ok=True)
    P1.jdump({"members": chosen, "single_neural_f1": single, "alpha": alpha, "dev_fused": m}, rd / "dev_report.json")
    log_csv(cfg, {"run": cfg.run, "p1_arch": "ensemble", "method": "ensemble", "dev_f1": round(m["f1"], 4), "dev_precision": round(m["precision"], 4),
                  "dev_recall": round(m["recall"], 4), "alpha": alpha, "teacher_run": "+".join(chosen)})
    S_te = alpha * zrow(Tn) + (1 - alpha) * zrow(te_bm)
    return write_submission7(cfg, rd, topk_ids(S_te, tids["qids"], tids["pool"]), tids["qids"], tids["pool"])


# ============================================================================================
# 7. Diagnostics
# ============================================================================================
def diagnose7(cfg: PCfg):
    tok, _, mc = N.load_p1(cfg)
    tp = str(Path(cfg.p1_dir) / "tokenizer.json")
    sp = {"train": load_split7(cfg, cfg.train_prefix, "train", True, tok, tp, cfg.max_train_queries),
          "dev": load_split7(cfg, cfg.dev_prefix, "dev", True, tok, tp), "test": load_split7(cfg, cfg.test_prefix, "test", False, tok, tp)}
    for name, s in sp.items():
        qn, pn = np.array([len(t) for t in s.qtok]), np.array([len(t) for t in s.pool_tok])
        nch = np.array([len(J.chunk_bounds(len(t), cfg.max_len - 1, 0)) for t in s.pool_tok])
        line = (f"[check] {name:<5}: {len(s.qids)} queries x {len(s.pool_ids)} candidates | query tokens median {int(np.median(qn))} p90 {int(np.percentile(qn, 90))} | "
                f"candidate tokens median {int(np.median(pn))} p90 {int(np.percentile(pn, 90))} max {int(pn.max())} | candidate chunks median {int(np.median(nch))} "
                f"(eval_chunks={cfg.eval_chunks} covers {100 * np.mean(nch <= cfg.eval_chunks):.0f}% fully)")
        if name != "test":
            rl = np.array([len(s.rel[q]) for q in s.qids])
            pset = set(s.pool_ids)
            inpool = np.mean([c in pset for q in s.qids for c in s.rel[q]]) if rl.sum() else 0.0
            line += f" | relevant per query mean {rl.mean():.2f} max {rl.max()} (queries without any: {int((rl == 0).sum())}) | relevant ids found in the pool: {100 * inpool:.0f}%"
        log(line)
        log(f"[check] {name}: query ids that are also in the candidate pool: {len(set(s.qids) & set(s.pool_ids))}")
    dv = sp["dev"]
    gold = {q: dv.rel[q] for q in dv.qids}
    t0 = time.time()
    Sb = BM25(dv.pool_text, cfg.bm25_k1, cfg.bm25_b).score(dv.qtext)
    m = f1_at_k(gold, topk_ids(Sb, dv.qids, dv.pool_ids))
    log(f"[check] LEXICAL BASELINE (BM25, no learning) on dev: Micro-F1@7 {m['f1']:.4f} (P {m['precision']:.4f} R {m['recall']:.4f}) | "
        f"recall@7 {recall_at(gold, Sb, dv.qids, dv.pool_ids, 7):.4f} recall@20 {recall_at(gold, Sb, dv.qids, dv.pool_ids, 20):.4f} recall@50 {recall_at(gold, Sb, dv.qids, dv.pool_ids, 50):.4f} ({time.time() - t0:.0f}s)")
    log("[check] the Phase-1-based neural retriever has to beat or complement this number; the fused ranking (neural + BM25) is tuned on dev automatically")
    ex = np.mean([len(set(dv.rel[q]) & set(sp["train"].pool_ids)) for q in dv.qids])
    log(f"[check] relevant dev candidates that also occur in the TRAIN pool (would be leakage): {ex:.2f} per query")


# ============================================================================================
# 8. Self-test
# ============================================================================================
def _synthetic(tmp, rng):
    topics = [[f"topic{t}w{i}" for i in range(12)] for t in range(8)]
    filler = "the court held that the appeal petitioner respondent order section learned counsel submitted impugned judgment evidence".split()

    def doc(t):
        words = [rng.choice(filler) for _ in range(rng.randint(150, 500))]
        for _ in range(rng.randint(15, 30)):
            words.insert(rng.randint(0, len(words)), rng.choice(topics[t]))
        sents = []
        for i in range(0, len(words), 12):
            sents.append(" ".join(words[i:i + 12]) + ".")
        return sents

    os.makedirs(tmp, exist_ok=True)
    allt = []
    for split, nq, npool in (("train", 24, 60), ("dev", 10, 40), ("test", 8, 40)):
        pool, ptopic = {}, {}
        for i in range(npool):
            t = rng.randrange(8)
            pool[f"{split[0]}{i:05d}"] = {"text": doc(t)}
            ptopic[f"{split[0]}{i:05d}"] = t
        qs = {}
        for i in range(nq):
            t = rng.randrange(8)
            rel = [c for c, tt in ptopic.items() if tt == t]
            rng.shuffle(rel)
            q = {"text": doc(t)}
            if split != "test":
                q["relevant_candidates"] = rel[:rng.randint(1, 4)]
            qs[f"q{split[0]}{i:05d}"] = q
        json.dump(qs, open(os.path.join(tmp, f"{split}.json"), "w"))
        json.dump(pool, open(os.path.join(tmp, f"{split}_candidates.json"), "w"))
        allt += [" ".join(v["text"]) for v in list(qs.values()) + list(pool.values())]
    return allt


def selftest() -> bool:
    import tempfile
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
    ok = True
    rng = random.Random(0)
    tmp = tempfile.mkdtemp(prefix="cs781_pc_selftest_")
    dd = os.path.join(tmp, "data")
    texts = _synthetic(dd, rng)
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    tok.train_from_iterator(texts, trainers.BpeTrainer(vocab_size=600, special_tokens=[P1.EOS], initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False))
    V = P1.pad64(tok.get_vocab_size())
    eos = tok.token_to_id(P1.EOS)
    p1x = os.path.join(tmp, "p1_x")
    os.makedirs(p1x, exist_ok=True)
    tok.save(os.path.join(p1x, "tokenizer.json"))

    log("=== 1. data, BM25, metrics (official definitions), fusion ===")
    cfg0 = apply_overrides(PCfg(), {"local_data_dir": dd, "work_dir": os.path.join(tmp, "work0"), "max_len": 64, "p1_dir": p1x})
    tr = load_split7(cfg0, cfg0.train_prefix, "train", True, tok, os.path.join(p1x, "tokenizer.json"))
    tr2 = load_split7(cfg0, cfg0.train_prefix, "train", True, tok, os.path.join(p1x, "tokenizer.json"))                 # from the cache
    dv = load_split7(cfg0, cfg0.dev_prefix, "dev", True, tok, os.path.join(p1x, "tokenizer.json"))
    g1 = len(tr.qids) == 24 and len(tr.pool_ids) == 60 and len(dv.pool_ids) == 40 and all(np.array_equal(a, b) for a, b in zip(tr.qtok, tr2.qtok))
    gold = {q: dv.rel[q] for q in dv.qids}
    Sb = BM25(dv.pool_text, 1.2, 0.75).score(dv.qtext)
    m_b = f1_at_k(gold, topk_ids(Sb, dv.qids, dv.pool_ids))
    g2 = m_b["recall"] > 0.9                                                  # topic words make lexical retrieval easy (F1@7 is capped by ~2 relevant per query)
    pred = topk_ids(Sb, dv.qids, dv.pool_ids)
    g3 = not validate_pred(pred, dv.qids, dv.pool_ids) and all(len(set(v)) == K for v in pred.values())
    gg = {"a": ["x", "y", "z"], "b": ["w"]}
    pp = {"a": ["x", "q", "r", "s", "t", "u", "v"], "b": ["w", "x", "y", "z", "q", "r", "s"]}
    r_ = f1_at_k(gg, pp)
    g4 = (r_["tp"], r_["fp"], r_["fn"]) == (2, 12, 2) and abs(r_["f1"] - 2 * 2 / (2 * 2 + 12 + 2)) < 1e-12
    S = np.random.default_rng(0).random((3, 12)).astype(np.float32)
    ids12 = [f"c{i}" for i in range(12)]
    S[0, 4] = 99.0
    tk = topk_ids(S, ["c4", "q1", "q2"], ids12)                              # query 0 is itself candidate c4 -> must not be returned
    g5 = "c4" not in tk["c4"] and len(tk["c4"]) == K and tk["q1"][0] != tk["q1"][1]
    a_, table = tune_alpha(np.random.default_rng(1).random((10, 40)), Sb, gold, dv.qids, dv.pool_ids)
    a2_, table2 = tune_alpha(np.random.default_rng(1).random((10, 40)), Sb, gold, dv.qids, dv.pool_ids, 0.2)
    g6 = 0.0 <= a_ <= 1.0 and len(table) == 11 and a2_ >= 0.2 and min(t[0] for t in table2) == 0.2
    bad = {q: v[:6] for q, v in pred.items()}
    g7 = bool(validate_pred(bad, dv.qids, dv.pool_ids))
    ok &= all([g1, g2, g3, g4, g5, g6, g7])
    log(f"  data + token cache {'PASS' if g1 else 'FAIL'} | BM25 finds the synthetic topics (recall@7 {m_b['recall']:.2f}) {'PASS' if g2 else 'FAIL'} | output validity {'PASS' if g3 else 'FAIL'} | "
        f"micro-F1@7 = evaluator definition {'PASS' if g4 else 'FAIL'} | query itself never retrieved {'PASS' if g5 else 'FAIL'} | fusion grid {'PASS' if g6 else 'FAIL'} | "
        f"validator rejects 6 candidates {'PASS' if g7 else 'FAIL'}")

    log("=== 2. similarity functions + adapters at identity / trainable subsets ===")
    for arch in P1.ARCH_PRESETS:
        for meth, var in (("adapter", "plus"), ("lora", "plus"), ("lorma", "plus"), ("lorma", "pi"), ("bitfit", "plus"), ("partial", "plus"), ("ia3", "plus")):
            tag = f"{arch}/{meth}" + (f"-{var}" if meth == "lorma" else "")
            try:
                torch.manual_seed(0)
                c = apply_overrides(PCfg(), {"method": meth, "lorma_variant": var, "peft_r": 4, "peft_alpha": 8.0, "peft_dropout": 0.0, "adapter_dim": 8,
                                             "freeze_layers": 1, "dim": 32, "max_len": 64})
                for pool in ("mean", "attn", "maxsim") if (arch == "llama" and meth == "adapter") else ("mean",):
                    c.pool = pool
                    net = build_net7(c, P1.LM(N._tiny_mc(arch, V))).eval()
                    Qb = collate_docs7(tr.qtok, [0, 1], c, "cpu", eos, 3, "uniform")
                    Db = collate_docs7(tr.pool_tok, [0, 1, 2], c, "cpu", eos, 3, "uniform")
                    with torch.no_grad():
                        s0 = sim_matrix(net.embed(*Qb), net.embed(*Db), pool)
                    assert s0.shape == (2, 3) and float(s0.abs().max()) <= 1.0001
                    if pool != "mean":
                        continue
                    apply_method7(net, c)
                    net.eval()
                    with torch.no_grad():
                        s1 = sim_matrix(net.embed(*Qb), net.embed(*Db), pool)
                    diff = (s0 - s1).abs().max().item()
                    n_tr = sum(p.numel() for p in net.parameters() if p.requires_grad)
                    n_all = sum(p.numel() for p in net.parameters())
                    net.train()
                    sim_matrix(net.embed(*Qb), net.embed(*Db), pool).sum().backward()
                    grads = any(p.grad is not None and float(p.grad.abs().sum()) > 0 for p in net.parameters() if p.requires_grad)
                    good = diff < 1e-4 and n_tr < n_all and grads
                    ok &= good
                    log(f"  {tag:<18} |adapted - base| = {diff:.1e}  trainable {n_tr:,}/{n_all:,}  {'PASS' if good else 'FAIL'}")
            except Exception as e:
                import traceback
                traceback.print_exc()
                log(f"  {tag:<18} FAIL: {type(e).__name__}: {e}")
                ok = False

    log("=== 3. end-to-end: architectures / methods / pooling / losses, then ensemble ===")
    p1dirs = {}
    for arch in P1.ARCH_PRESETS:
        p1 = os.path.join(tmp, f"p1_{arch}")
        os.makedirs(p1, exist_ok=True)
        torch.save({"model": P1.LM(N._tiny_mc(arch, V)).state_dict(), "mc": N._tiny_mc(arch, V), "state": {"step": 0}}, os.path.join(p1, "ckpt_best.pt"))
        tok.save(os.path.join(p1, "tokenizer.json"))
        p1dirs[arch] = p1
    base = {"work_dir": os.path.join(tmp, "work"), "local_data_dir": dd, "max_len": 64, "train_chunks": 2, "eval_chunks": 3, "epochs": 2, "batch_q": 4,
            "hard_negs": 2, "hard_pool": 8, "lr": 1e-3, "peft_lr": 3e-3, "head_lr": 3e-3, "peft_r": 4, "peft_alpha": 8.0, "adapter_dim": 8, "dim": 32,
            "ema_decay": 0.9, "time_budget_min": 5, "eval_every_steps": 0, "dev_eval_queries": 6, "dev_eval_extra": 15, "enc_docs": 4, "patience": 5}
    cases = [(f"full_{a}", a, {"method": "full"}) for a in P1.ARCH_PRESETS] + [
        ("lorma_plus", "gpt2", {"method": "lorma", "lorma_variant": "plus"}), ("lorma_pi", "retnet", {"method": "lorma", "lorma_variant": "pi"}),
        ("lora", "llama", {"method": "lora"}), ("adapter", "moe", {"method": "adapter"}), ("bitfit", "gpt2", {"method": "bitfit"}),
        ("partial", "llama", {"method": "partial", "freeze_layers": 1}), ("ia3", "moe", {"method": "ia3"}),
        ("pool_attn", "llama", {"method": "full", "pool": "attn"}), ("pool_maxsim", "llama", {"method": "full", "pool": "maxsim"}),
        ("loss_triplet", "llama", {"method": "full", "loss": "triplet"}),
        ("kd", "gpt2", {"method": "kd", "teacher_run": "selftest_full_llama", "student_layers": 1})]
    te_ids = None
    for tag, arch, extra in cases:
        try:
            cfg = apply_overrides(PCfg(), {**base, **extra, "run": f"selftest_{tag}", "p1_dir": p1dirs[arch]})
            run_train7(cfg)
            with zipfile.ZipFile(run_dir7(cfg) / "submission.zip") as z:
                assert z.namelist() == ["prediction.json"]
                pr = json.loads(z.read("prediction.json"))
            ids_ = P1.jload(run_dir7(cfg) / "test_ids.json")
            assert not validate_pred(pr, ids_["qids"], ids_["pool"]), "official validation rules"
            rep = P1.jload(run_dir7(cfg) / "dev_report.json")
            if extra["method"] in ("lora", "lorma", "adapter", "bitfit", "partial", "ia3"):
                assert rep["params_trainable"] < rep["params_total"]
            if extra["method"] == "kd":
                assert rep["params_total"] < rep["teacher_params"]
            log(f"  e2e {tag:<13} PASS  (dev Micro-F1@7 neural {rep['dev_neural']['f1']:.3f}, BM25 {rep['dev_bm25']['f1']:.3f}, fused {rep['dev_fused']['f1']:.3f}; "
                f"trainable {rep['params_trainable']:,}/{rep['params_total']:,})")
        except Exception as e:
            import traceback
            traceback.print_exc()
            log(f"  e2e {tag:<13} FAIL: {type(e).__name__}: {e}")
            ok = False
    try:
        for tag, ov in (("ens", {"ens_runs": ["selftest_full_llama", "selftest_full_gpt2"]}), ("ens_auto", {"ens_auto": True, "ens_prefix": "selftest_full_"})):
            cfg = apply_overrides(PCfg(), {"work_dir": os.path.join(tmp, "work"), "local_data_dir": dd, "run": f"selftest_{tag}", **ov})
            zp = run_ensemble7(cfg)
            with zipfile.ZipFile(zp) as z:
                pr = json.loads(z.read("prediction.json"))
            ids_ = P1.jload(run_dir7(apply_overrides(PCfg(), {"work_dir": os.path.join(tmp, "work"), "run": "selftest_full_llama"})) / "test_ids.json")
            assert not validate_pred(pr, ids_["qids"], ids_["pool"])
        log("  ensembles (explicit + automatic) PASS")
    except Exception as e:
        import traceback
        traceback.print_exc()
        log(f"  ensembles FAIL: {type(e).__name__}: {e}")
        ok = False
    shutil.rmtree(tmp, ignore_errors=True)
    log("SELFTEST " + ("PASSED" if ok else "FAILED"))
    return ok


# ============================================================================================
# 9. CLI
# ============================================================================================
def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("stage", choices=["selftest", "check", "train", "predict", "ensemble"])
    ap.add_argument("--run", default=None)
    ap.add_argument("--cfg", default="", help="JSON string or JSON file with config overrides")
    args = ap.parse_args(argv)
    if args.stage == "selftest":
        return 0 if selftest() else 1
    cfg = PCfg()
    ov = {}
    if args.cfg:
        ov = json.loads(args.cfg) if args.cfg.strip().startswith("{") else P1.jload(args.cfg)
    if args.run:
        ov["run"] = args.run
    apply_overrides(cfg, ov)
    Path(cfg.work_dir).mkdir(parents=True, exist_ok=True)
    {"check": diagnose7, "train": run_train7, "predict": run_predict7, "ensemble": run_ensemble7}[args.stage](cfg)
    return 0


if __name__ == "__main__":
    sys.exit(main())

## Self-test (once, ~10 min). Must end with `SELFTEST PASSED`. Checks BM25, the metric against the official definition, the output rules, adapters at identity, and runs every method / pooling / loss end to end.

In [ ]:
!python cs781_phase7_pcr.py selftest

## Setup

In [ ]:
import os, json, glob, subprocess
WORK = "/kaggle/working/cs781" if os.path.exists("/kaggle/working") else "./cs781_work"
BUDGET = 45            # training minutes per main run
SEL = {}               # overrides for every run, e.g. {"eval_chunks": 8, "pool": "maxsim"}
P1_DIRS = {a: f"{WORK}/runs/explore_{a}" for a in ["llama", "gpt2", "retnet", "moe"]}
P1_CKPT, missing = {}, []
for a, d in P1_DIRS.items():
    if not os.path.exists(f"{d}/tokenizer.json"): missing.append(a)
    elif os.path.exists(f"{d}/ckpt_best.pt"): P1_CKPT[a] = "ckpt_best.pt"
    elif os.path.exists(f"{d}/ckpt_last.pt"): P1_CKPT[a] = "ckpt_last.pt"
    else: missing.append(a)
for a, d in P1_DIRS.items(): print(f"{a:<7}", f"OK ({P1_CKPT[a]})" if a in P1_CKPT else "MISSING", d)

def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print(line, end="")
    p.wait(); return p.returncode

def run_pc(run, arch, extra):
    if os.path.exists(f"{WORK}/pc_runs/{run}/submission.zip"):
        print("already done:", run, "(delete that folder to re-run)"); return
    if arch not in P1_CKPT:
        print(f"Phase-1 backbone '{arch}' is missing -> train it first, then re-run the setup cell."); return
    cfg = {"p1_dir": P1_DIRS[arch], "p1_ckpt": P1_CKPT[arch], "time_budget_min": BUDGET, **SEL, **extra}
    fn = f"cfg_pc_{run}.json"; json.dump(cfg, open(fn, "w"), indent=1)
    print("=" * 100, "\n", run, "|", arch, "|", extra, "\n", "=" * 100)
    print("exit code", sh(f"python cs781_phase7_pcr.py train --run {run} --cfg {fn}"))
    report(run)

def report(run):
    p = f"{WORK}/pc_runs/{run}/dev_report.json"
    if not os.path.exists(p): print(f"\n{run}: no dev_report.json (not finished)"); return
    r = json.load(open(p))
    print(f"\n=== {run} | {r.get('p1_arch', 'ensemble')} / {r.get('method', 'ensemble')}")
    if "dev_neural" in r:
        print(f"dev Micro-F1@7: neural only {r['dev_neural']['f1']:.4f} | BM25 only {r['dev_bm25']['f1']:.4f} | FUSED (alpha={r['alpha']}) {r['dev_fused']['f1']:.4f}  "
              f"(P {r['dev_fused']['precision']:.4f} R {r['dev_fused']['recall']:.4f})")
    else:
        print(f"dev Micro-F1@7 {r['dev_fused']['f1']:.4f} (alpha={r['alpha']}) | members {r.get('members')}")

In [ ]:
for a in missing:
    fn = f"cfg_p1_{a}.json"; json.dump({"arch": a, "time_budget_hours": 0.75}, open(fn, "w"))
    print("training Phase-1 backbone:", a); sh(f"python cs781_phase1.py train --run explore_{a} --preset turbo --cfg {fn}")
if not missing: print("nothing to train - all four backbones exist")

In [ ]:
RUN_CONTINUE, CONT_ARCH, CONT_HOURS = False, "llama", 3.0
if RUN_CONTINUE and CONT_ARCH in P1_CKPT:
    fn = f"cfg_p1_{CONT_ARCH}_cont.json"
    json.dump({"arch": CONT_ARCH, "time_budget_hours": CONT_HOURS, "init_from": f"{P1_DIRS[CONT_ARCH]}/{P1_CKPT[CONT_ARCH]}",
               "lr": 4e-4, "warmup_steps": 100, "seed": 2024}, open(fn, "w"))
    print("exit code", sh(f"python cs781_phase1.py train --run explore_{CONT_ARCH}_cont --preset turbo --cfg {fn}"))
    P1_DIRS[CONT_ARCH] = f"{WORK}/runs/explore_{CONT_ARCH}_cont"; P1_CKPT[CONT_ARCH] = "ckpt_best.pt"
    print(CONT_ARCH, "-> now uses", P1_DIRS[CONT_ARCH])

## Check the data first (3-8 min)
Read these lines:
* **queries x candidates, relevant per query** - the task's shape; the evaluator always counts 7 predictions, so queries with fewer relevant cases cap precision;
* **relevant ids found in the pool** - anything below 100% is unreachable;
* **query ids that are also in the candidate pool** - the script never returns a query as its own answer;
* **LEXICAL BASELINE (BM25)** - the Micro-F1@7 and recall@7/20/50 you have to beat or complement;
* **candidate chunks / eval_chunks coverage** - how much of each candidate the encoder sees.

In [ ]:
sh("python cs781_phase7_pcr.py check --run pc_check --cfg " + json.dumps(json.dumps({"p1_dir": P1_DIRS["llama"], "p1_ckpt": P1_CKPT.get("llama", "ckpt_best.pt")})))

## Required run 1 - full fine-tuning (llama); also the teacher of the distillation run

In [ ]:
run_pc("pc_full_llama", "llama", {"method": "full", "epochs": 6, "lr": 2e-4})

## Required run 2 - LoRMA+ (gpt2), compulsory: `h = (alpha/r BA + I) W0 x`, backbone frozen; only the adapters and the projection head are trained

In [ ]:
run_pc("pc_lorma_gpt2", "gpt2", {"method": "lorma", "lorma_variant": "plus", "peft_r": 32, "peft_alpha": 64, "epochs": 8, "peft_lr": 1e-3, "bidir": "false"})

## Required run 3 - IA3 (moe): learned scaling vectors on keys, values and FFN activations



In [ ]:
run_pc("pc_ia3_moe", "moe", {"method": "ia3", "epochs": 10, "peft_lr": 3e-3, "bidir": "false"})

## Required run 4 - knowledge distillation (retnet student pruned to 6 of 12 blocks <- `pc_full_llama` teacher): the student matches the teacher's softmax over the same candidates (`0.5 InfoNCE + 0.5 T^2 KL`)

In [ ]:
run_pc("pc_kd_retnet", "retnet", {"method": "kd", "teacher_run": "pc_full_llama", "student_layers": 6, "epochs": 6, "lr": 2e-4})

In [ ]:
run_pc("pc_adapter_gpt2", "gpt2", {"method": "adapter", "adapter_dim": 64, "epochs": 8, "peft_lr": 1e-3, "bidir": "false"})
run_pc("pc_lora_llama",   "llama", {"method": "lora", "peft_r": 32, "peft_alpha": 64, "epochs": 8, "peft_lr": 1e-3})
run_pc("pc_bitfit_moe",   "moe",   {"method": "bitfit", "epochs": 8, "peft_lr": 2e-3, "bidir": "false"})
run_pc("pc_partial_llama", "llama", {"method": "partial", "freeze_layers": 8, "epochs": 6, "lr": 2e-4})

In [ ]:
RUN_ABLATIONS = False
ABL = {"method": "full", "epochs": 3, "time_budget_min": 20}
if RUN_ABLATIONS:
    for pool in ["mean", "attn", "maxsim"]:
        run_pc(f"pc_abl_pool_{pool}", "llama", {**ABL, "pool": pool})
    run_pc("pc_abl_triplet", "llama", {**ABL, "loss": "triplet"})
    run_pc("pc_abl_nohard", "llama", {**ABL, "hard_negs": 0})

## Results and the files to submit

In [ ]:
import pandas as pd4, shutil
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
df = pd.read_csv(f"{WORK}/pc_experiments.csv"); df = df[~df.run.str.startswith(("selftest", "pc_check"))]
cols = ["run", "p1_arch", "method", "pool", "loss", "params_total_M", "params_trainable_M", "dev_f1_neural", "dev_f1_bm25", "dev_f1", "dev_precision",
        "dev_recall", "alpha", "train_s", "peak_gpu_mb", "enc_docs_s"]
display(df[cols].sort_values("dev_f1", ascending=False))
os.makedirs("phase7_submissions", exist_ok=True)
REQUIRED = ["pc_lorma_gpt2", "pc_full_llama", "pc_ia3_moe", "pc_kd_retnet"]
EXTRA = ["pc_adapter_gpt2", "pc_lora_llama", "pc_ensemble"]
for r in REQUIRED + EXTRA:
    src = f"{WORK}/pc_runs/{r}/submission.zip"
    if os.path.exists(src):
        shutil.copy(src, f"phase7_submissions/{r}.zip")
        row = df[df.run == r].tail(1)
        print(("REQUIRED " if r in REQUIRED else "extra    "), r, "| dev Micro-F1@7", float(row.dev_f1.iloc[0]) if len(row) else "?")
    else: print("missing:", r)`